<div style="direction: rtl; white-space: normal; line-height: 1;">

# Notebook 06 — Transcript Ingestion

این Notebook مرحله پردازش Transcriptهای پذیرفته‌شده در Phase 2 پروژه `RAG_finance` است.

ورودی رسمی این مرحله فقط Manifest پذیرفته‌شده Notebook 05 است:

`data/phase2/manifests/transcript_manifest_v1.json`

Notebook 05 مرحله Inventory را کامل کرده و مشخص کرده است کدام فایل‌ها Transcript قابل قبول، کدام فایل‌ها Review Material و کدام موارد Missing Source یا Recovery Candidate هستند.

وضعیت ورودی پذیرفته‌شده:

- `13` Program
- `183` Accepted Transcript Physical Files
- `30` Review Materials
- Episode 21 مجموعه `در جستجوی خوشبختی` با وضعیت `MISSING_SOURCE`
- Site 32 recovery candidate با وضعیت `RECOVERY_CANDIDATE_NOT_ACCEPTED`
- `rights_status` تمام فایل‌های Accepted برابر `EVIDENCE_REQUIRED`
- Unknown Speaker Inference ممنوع

هدف Notebook 06 این است که Transcriptهای پذیرفته‌شده را از روی Manifest بخواند و ساختار واقعی محتوای آن‌ها را برای Ingestion استخراج کند.

در Transcriptهای چندگوینده، Turnها و سؤال‌های سایر افراد باید حفظ شوند، زیرا بخشی از Context لازم برای فهم پاسخ‌های دکتر کمیل رودی هستند.

نام Speaker فقط بر اساس شواهد موجود در Source ثبت می‌شود و هیچ Speaker ناشناخته‌ای از روی متن یا Context حدس زده نمی‌شود.

`Instructional Bold Content` یک Speaker مستقل نیست و Speaker جاری را inherit می‌کند.

در این Notebook انجام نمی‌شوند:

- Persian normalization
- Deduplication pipeline
- Chunking
- Unified Schema مرحله بعد
- حدس Speaker
- ساخت Timestamp غیرواقعی
- ورود Review Material به Corpus پذیرفته‌شده
- پذیرش Recovery Candidate تأییدنشده

Persian normalization متعلق به Notebook 08، Deduplication متعلق به Notebook 09 و Chunking متعلق به Notebook 11 است.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### Start Gate — بررسی ورودی رسمی Transcript Ingestion

قبل از Parse یا Ingest کردن هر Transcript باید ورودی رسمی این Notebook مستقیماً از روی Disk بررسی شود.

Source of Truth این مرحله:

`data/phase2/manifests/transcript_manifest_v1.json`

این Cell فقط وضعیت اولیه را بررسی می‌کند و هنوز هیچ Transcriptی Parse یا تغییر داده نمی‌شود.

بررسی‌های این مرحله:

- شناسایی Root پروژه بدون Hard-code کردن مسیر Absolute مک
- بررسی وجود Manifest مورد انتظار در مسیر دقیق پروژه
- خواندن Manifest مستقیماً از Disk
- محاسبه `SHA-256`
- بررسی معتبر بودن JSON
- مشاهده ساختار سطح اول Manifest
- ثبت تعداد رکوردها یا کلیدهای اصلی بدون حدس درباره Schema داخلی

در این مرحله ساختار Manifest از قبل فرض نمی‌شود. ابتدا داده واقعی مشاهده می‌شود و مراحل بعدی بر اساس Schema واقعی Artifact پذیرفته‌شده Notebook 05 طراحی خواهند شد.

**معیار PASS:**

- Repository Root شناسایی شود.
- Manifest دقیقاً در مسیر مورد انتظار وجود داشته باشد.
- Manifest JSON معتبر باشد.
- `SHA-256` آن محاسبه شود.
- ساختار سطح اول آن قابل مشاهده باشد.

**معیار FAIL:**

- Repository Root پیدا نشود.
- Manifest در مسیر مورد انتظار وجود نداشته باشد.
- JSON خراب یا غیرقابل خواندن باشد.
- Artifact ورودی با چیزی که Notebook 05 تحویل داده سازگار نباشد.

در صورت FAIL، Transcript Ingestion شروع نمی‌شود و ابتدا همان مشکل بررسی خواهد شد.

</div>

In [1]:
# Code Cell 01

from pathlib import Path
import hashlib
import json


def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()

    for candidate in (current, *current.parents):
        if candidate.name == "RAG_finance" and (candidate / ".git").exists():
            return candidate

    raise RuntimeError("RAG_finance repository root could not be identified.")


project_root = find_project_root(Path.cwd())

transcript_manifest_path = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "transcript_manifest_v1.json"
)

assert transcript_manifest_path.is_file(), (
    "Expected transcript manifest was not found: "
    f"{transcript_manifest_path.relative_to(project_root)}"
)

manifest_bytes = transcript_manifest_path.read_bytes()
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()

with transcript_manifest_path.open("r", encoding="utf-8") as file:
    transcript_manifest_v1 = json.load(file)

print("Repository root:", project_root)
print(
    "Manifest path:",
    transcript_manifest_path.relative_to(project_root),
)
print("Manifest size (bytes):", len(manifest_bytes))
print("Manifest SHA-256:", manifest_sha256)
print("Top-level JSON type:", type(transcript_manifest_v1).__name__)

if isinstance(transcript_manifest_v1, dict):
    print("Top-level keys:", sorted(transcript_manifest_v1.keys()))

elif isinstance(transcript_manifest_v1, list):
    print("Top-level records:", len(transcript_manifest_v1))

else:
    raise AssertionError(
        "Transcript manifest top-level structure must be a dict or list."
    )

print("NOTEBOOK 06 START GATE: PASS")

Repository root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Manifest path: data/phase2/manifests/transcript_manifest_v1.json
Manifest size (bytes): 358749
Manifest SHA-256: b328f512c67971b5e0e1f82cd5e9fe2ea148a89db3fbde31aebb474e78c6a9eb
Top-level JSON type: dict
Top-level keys: ['accepted_files', 'artifact_type', 'artifact_version', 'counts', 'global_policies', 'missing_sources', 'programs', 'review_materials', 'schema_version', 'scope', 'source_name_variants']
NOTEBOOK 06 START GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی Schema واقعی `transcript_manifest_v1`

Start Gate با موفقیت PASS شد و Artifact رسمی Notebook 05 مستقیماً از Disk خوانده شد.

در این مرحله ساختار داخلی Manifest بررسی می‌شود تا Transcript Ingestion بر اساس فیلدهای واقعی Artifact طراحی شود و هیچ نام فیلد، Metadata یا رابطه‌ای از قبل حدس زده نشود.

بخش‌های اصلی Manifest که بررسی می‌شوند:

- `counts`
- `global_policies`
- `programs`
- `accepted_files`
- `review_materials`
- `missing_sources`
- `source_name_variants`

برای Collectionهای اصلی فقط ساختار و کلیدهای Record نمونه نمایش داده می‌شود و متن Transcriptها هنوز خوانده نمی‌شود.

هدف این Inspection مشخص‌کردن موارد زیر است:

- Schema واقعی رکوردهای Accepted
- نحوه اتصال فایل پذیرفته‌شده به Program و Episode
- فیلدهای مربوط به Speaker و Source
- وضعیت Rights
- Policyهای ثبت‌شده در Notebook 05
- نحوه ثبت Missing Source و Review Material

در این مرحله هیچ Artifactی ساخته یا تغییر داده نمی‌شود.

**معیار PASS:**

- تمام بخش‌های اصلی Manifest وجود داشته باشند.
- نوع داده هر بخش قابل شناسایی باشد.
- حداقل یک Record در `accepted_files` وجود داشته باشد.
- Schema نمونه Accepted File بدون خطا قابل مشاهده باشد.
- اطلاعات لازم برای طراحی مرحله بعد از خود Manifest قابل استخراج باشد.

**معیار FAIL:**

اگر یکی از بخش‌های ضروری Manifest وجود نداشته باشد، `accepted_files` خالی باشد یا ساختار داخلی Artifact قابل بررسی نباشد، Ingestion شروع نمی‌شود و ابتدا همان ناسازگاری بررسی خواهد شد.

</div>

In [2]:
# Code Cell 02

required_top_level_keys = {
    "accepted_files",
    "artifact_type",
    "artifact_version",
    "counts",
    "global_policies",
    "missing_sources",
    "programs",
    "review_materials",
    "schema_version",
    "scope",
    "source_name_variants",
}

missing_top_level_keys = sorted(
    required_top_level_keys - set(transcript_manifest_v1)
)

assert not missing_top_level_keys, (
    f"Missing required top-level keys: {missing_top_level_keys}"
)

print("Artifact type:", transcript_manifest_v1["artifact_type"])
print("Artifact version:", transcript_manifest_v1["artifact_version"])
print("Schema version:", transcript_manifest_v1["schema_version"])
print()

for section_name in (
    "counts",
    "global_policies",
    "programs",
    "accepted_files",
    "review_materials",
    "missing_sources",
    "source_name_variants",
):
    section = transcript_manifest_v1[section_name]

    print(f"{section_name}:")
    print("  type:", type(section).__name__)

    if isinstance(section, (list, dict)):
        print("  size:", len(section))

    if isinstance(section, dict):
        print("  keys:", sorted(section.keys()))

    print()

accepted_files = transcript_manifest_v1["accepted_files"]

assert isinstance(accepted_files, list), (
    "accepted_files must be a list."
)
assert accepted_files, (
    "accepted_files must not be empty."
)
assert all(isinstance(item, dict) for item in accepted_files), (
    "Every accepted_files record must be a dict."
)

print("Accepted file record keys:")
print(sorted(accepted_files[0].keys()))
print()

print("First accepted file record:")
for key, value in accepted_files[0].items():
    if isinstance(value, str) and len(value) > 300:
        display_value = value[:300] + "..."
    else:
        display_value = value

    print(f"- {key}: {display_value}")

print()
print("Counts:")
print(json.dumps(
    transcript_manifest_v1["counts"],
    ensure_ascii=False,
    indent=2,
))

print()
print("Global policies:")
print(json.dumps(
    transcript_manifest_v1["global_policies"],
    ensure_ascii=False,
    indent=2,
))

print()
print("MANIFEST SCHEMA INSPECTION: PASS")

Artifact type: phase2_transcript_inventory
Artifact version: v1
Schema version: transcript_inventory_v1

counts:
  type: dict
  size: 5
  keys: ['accepted_physical_files', 'missing_source_records', 'programs', 'recovery_candidates_not_accepted', 'review_materials']

global_policies:
  type: dict
  size: 5
  keys: ['missing_metadata_policy', 'question_response_context_preservation', 'rights_default_status', 'rights_inference_from_repository_presence', 'unknown_speaker_inference_allowed']

programs:
  type: list
  size: 13

accepted_files:
  type: list
  size: 183

review_materials:
  type: list
  size: 30

missing_sources:
  type: list
  size: 1

source_name_variants:
  type: dict
  size: 1
  keys: ['فناپ']

Accepted file record keys:
['default_body_attribution_allowed', 'episode_number', 'inventory_decision', 'metadata_layers', 'physical_file_id', 'preserve_other_turn_context', 'primary_speaker', 'program', 'program_id', 'relative_path', 'rights_basis', 'rights_status', 'season_number'

<div style="direction: rtl; white-space: normal; line-height: 1;">

### Accepted Input Integrity Gate — تطبیق Manifest با فایل‌های فیزیکی

Schema واقعی `transcript_manifest_v1` تأیید شد و مشخص است که `accepted_files` شامل `183` فایل با تصمیم صریح `ACCEPTED_FOR_INGESTION` است.

قبل از Parse کردن محتوای Transcriptها باید ثابت شود که همین `183` فایل هنوز روی Disk موجود هستند و با Artifact پذیرفته‌شده Notebook 05 تغییر نکرده‌اند.

در این مرحله برای هر Accepted File موارد زیر بررسی می‌شوند:

- `relative_path` معتبر و داخل Repository باشد.
- فایل فیزیکی واقعاً وجود داشته باشد.
- فایل یک File معمولی باشد.
- `SHA-256` فعلی فایل دقیقاً با `sha256` ثبت‌شده در Manifest برابر باشد.
- `physical_file_id` با Hash ثبت‌شده سازگار باشد.
- `inventory_decision` برابر `ACCEPTED_FOR_INGESTION` باشد.
- `rights_status` برابر Policy ثبت‌شده Manifest یعنی `EVIDENCE_REQUIRED` باشد.
- `unknown_speaker_inference_allowed` برای هیچ فایل پذیرفته‌شده فعال نباشد.
- تعداد فایل‌های پذیرفته‌شده با `counts.accepted_physical_files` برابر باشد.
- `relative_path` و `physical_file_id` در مجموعه Accepted تکراری نباشند.

این مرحله **Deduplication محتوایی** انجام نمی‌دهد. کنترل تکراری نبودن `relative_path` و `physical_file_id` فقط Validation قرارداد Manifest است و جایگزین Deduplication Notebook 09 نیست.

همچنین در این مرحله هنوز متن Transcriptها Parse، Normalize، Segment یا Chunk نمی‌شود.

**معیار PASS:**

- هر `183` فایل Accepted روی Disk موجود باشد.
- Hash تمام فایل‌ها با Manifest برابر باشد.
- هیچ Path یا `physical_file_id` تکراری وجود نداشته باشد.
- تمام تصمیم‌های Inventory و Policyهای Rights و Unknown Speaker بدون انحراف حفظ شده باشند.
- هیچ فایل تغییرکرده یا مفقودی وجود نداشته باشد.

**معیار FAIL:**

اگر حتی یک Accepted File مفقود، تغییرکرده، خارج از Repository یا ناسازگار با Manifest باشد، Transcript Ingestion شروع نمی‌شود و همان مورد ابتدا بررسی خواهد شد.

</div>

In [3]:
# Code Cell 03

accepted_count_expected = transcript_manifest_v1["counts"]["accepted_physical_files"]
rights_status_expected = transcript_manifest_v1["global_policies"]["rights_default_status"]
unknown_speaker_allowed_expected = transcript_manifest_v1["global_policies"][
    "unknown_speaker_inference_allowed"
]

assert len(accepted_files) == accepted_count_expected, (
    f"Accepted file count mismatch: manifest records={len(accepted_files)}, "
    f"declared count={accepted_count_expected}"
)

relative_paths = [record["relative_path"] for record in accepted_files]
physical_file_ids = [record["physical_file_id"] for record in accepted_files]

duplicate_relative_paths = sorted({
    value for value in relative_paths
    if relative_paths.count(value) > 1
})

duplicate_physical_file_ids = sorted({
    value for value in physical_file_ids
    if physical_file_ids.count(value) > 1
})

missing_files = []
non_file_paths = []
paths_outside_repository = []
hash_mismatches = []
physical_id_mismatches = []
invalid_inventory_decisions = []
invalid_rights_statuses = []
invalid_unknown_speaker_policies = []

for record in accepted_files:
    relative_path = Path(record["relative_path"])
    file_path = (project_root / relative_path).resolve()

    try:
        file_path.relative_to(project_root.resolve())
    except ValueError:
        paths_outside_repository.append(record["relative_path"])
        continue

    if not file_path.exists():
        missing_files.append(record["relative_path"])
        continue

    if not file_path.is_file():
        non_file_paths.append(record["relative_path"])
        continue

    current_sha256 = hashlib.sha256(file_path.read_bytes()).hexdigest()
    expected_sha256 = record["sha256"]

    if current_sha256 != expected_sha256:
        hash_mismatches.append({
            "relative_path": record["relative_path"],
            "expected": expected_sha256,
            "actual": current_sha256,
        })

    expected_physical_file_id = f"transcript-file:{expected_sha256}"

    if record["physical_file_id"] != expected_physical_file_id:
        physical_id_mismatches.append({
            "relative_path": record["relative_path"],
            "physical_file_id": record["physical_file_id"],
            "expected": expected_physical_file_id,
        })

    if record["inventory_decision"] != "ACCEPTED_FOR_INGESTION":
        invalid_inventory_decisions.append({
            "relative_path": record["relative_path"],
            "inventory_decision": record["inventory_decision"],
        })

    if record["rights_status"] != rights_status_expected:
        invalid_rights_statuses.append({
            "relative_path": record["relative_path"],
            "rights_status": record["rights_status"],
        })

    if record["unknown_speaker_inference_allowed"] != unknown_speaker_allowed_expected:
        invalid_unknown_speaker_policies.append({
            "relative_path": record["relative_path"],
            "unknown_speaker_inference_allowed": record[
                "unknown_speaker_inference_allowed"
            ],
        })

print("Accepted files declared:", accepted_count_expected)
print("Accepted files inspected:", len(accepted_files))
print("Duplicate relative paths:", len(duplicate_relative_paths))
print("Duplicate physical file IDs:", len(duplicate_physical_file_ids))
print("Paths outside repository:", len(paths_outside_repository))
print("Missing files:", len(missing_files))
print("Non-file paths:", len(non_file_paths))
print("SHA-256 mismatches:", len(hash_mismatches))
print("Physical file ID mismatches:", len(physical_id_mismatches))
print("Invalid inventory decisions:", len(invalid_inventory_decisions))
print("Invalid rights statuses:", len(invalid_rights_statuses))
print(
    "Invalid unknown-speaker policies:",
    len(invalid_unknown_speaker_policies),
)

assert not duplicate_relative_paths, (
    f"Duplicate accepted relative paths found: {duplicate_relative_paths}"
)
assert not duplicate_physical_file_ids, (
    f"Duplicate accepted physical_file_ids found: {duplicate_physical_file_ids}"
)
assert not paths_outside_repository, (
    f"Accepted paths outside repository found: {paths_outside_repository}"
)
assert not missing_files, (
    f"Accepted files missing from disk: {missing_files}"
)
assert not non_file_paths, (
    f"Accepted paths that are not files: {non_file_paths}"
)
assert not hash_mismatches, (
    f"Accepted file hash mismatches found: {hash_mismatches}"
)
assert not physical_id_mismatches, (
    f"Physical file ID mismatches found: {physical_id_mismatches}"
)
assert not invalid_inventory_decisions, (
    f"Invalid inventory decisions found: {invalid_inventory_decisions}"
)
assert not invalid_rights_statuses, (
    f"Invalid rights statuses found: {invalid_rights_statuses}"
)
assert not invalid_unknown_speaker_policies, (
    "Unknown-speaker policy mismatch found: "
    f"{invalid_unknown_speaker_policies}"
)

print()
print("ACCEPTED INPUT INTEGRITY GATE: PASS")

Accepted files declared: 183
Accepted files inspected: 183
Duplicate relative paths: 0
Duplicate physical file IDs: 0
Paths outside repository: 0
Missing files: 0
Non-file paths: 0
SHA-256 mismatches: 0
Physical file ID mismatches: 0
Invalid inventory decisions: 0
Invalid rights statuses: 0
Invalid unknown-speaker policies: 0

ACCEPTED INPUT INTEGRITY GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### Discovery ساختار واقعی فایل‌های Transcript

Integrity ورودی PASS شد و اکنون ثابت شده است که هر `183` فایل Accepted روی Disk وجود دارد و محتوای آن با Hash ثبت‌شده در Manifest یکسان است.

قبل از طراحی Parser باید قالب واقعی Markdown در مجموعه‌های مختلف بررسی شود.

این مرحله فقط **Discovery ساختاری** انجام می‌دهد. هنوز Speaker، Turn یا Utterance استخراج نمی‌شود و هیچ تصمیمی درباره معنای خطوط گرفته نمی‌شود.

از آنجا که `13` Program مختلف داریم، ممکن است قالب فایل‌های آن‌ها یکسان نباشد. بنابراین ابتدا باید مشخص شود هر مجموعه از چه الگوی Markdown استفاده می‌کند.

در این Cell موارد زیر بررسی می‌شوند:

- امکان خواندن تمام `183` فایل با `UTF-8`
- تعداد فایل‌ها به تفکیک Program
- `speaker_policy`های ثبت‌شده در Manifest
- تعداد خطوط و خطوط غیرخالی
- وجود Markerهای ساختاری مانند Heading، Bold، Blockquote و List
- یک فایل نماینده از هر Program
- چند خط ابتدایی غیرخالی از فایل نماینده، بدون تغییر متن

نمایش Markerهای Markdown در این مرحله فقط برای شناخت قالب Source است و به معنی تشخیص Speaker یا نوع محتوا نیست.

به‌طور خاص، وجود متن Bold به‌تنهایی باعث ایجاد Speaker جدید نخواهد شد. قانون `Instructional Bold Content` همچنان برقرار است و در Parser نهایی باید Speaker جاری را inherit کند.

هیچ نام Speaker از روی متن نمونه‌ها حدس زده نمی‌شود.

در این مرحله انجام نمی‌شوند:

- Persian normalization
- Deduplication
- Chunking
- Speaker inference
- Turn segmentation
- بازنویسی متن Source

**معیار PASS:**

- تمام `183` فایل بدون خطای Decode خوانده شوند.
- هیچ فایل Accepted خالی نباشد.
- فایل‌ها دقیقاً در `13` Program قابل گروه‌بندی باشند.
- برای هر Program حداقل یک فایل نماینده قابل مشاهده باشد.
- اطلاعات کافی برای طراحی Parser بر اساس قالب واقعی Source به دست آید.

**معیار FAIL:**

اگر فایل غیرقابل خواندن، فایل خالی یا ناسازگاری در تعداد Programها مشاهده شود، قبل از طراحی Parser همان مشکل بررسی خواهد شد.

</div>

In [4]:
# Code Cell 04

from collections import Counter, defaultdict
import re


files_by_program = defaultdict(list)
decode_errors = []
empty_files = []
file_profiles = []

for record in accepted_files:
    file_path = project_root / record["relative_path"]

    try:
        text = file_path.read_text(encoding="utf-8")
    except UnicodeDecodeError as exc:
        decode_errors.append({
            "relative_path": record["relative_path"],
            "error": str(exc),
        })
        continue

    lines = text.splitlines()
    nonempty_lines = [line for line in lines if line.strip()]

    if not nonempty_lines:
        empty_files.append(record["relative_path"])

    profile = {
        "program": record["program"],
        "program_id": record["program_id"],
        "relative_path": record["relative_path"],
        "speaker_policy": record["speaker_policy"],
        "total_lines": len(lines),
        "nonempty_lines": len(nonempty_lines),
        "heading_lines": sum(
            line.lstrip().startswith("#")
            for line in nonempty_lines
        ),
        "bold_only_lines": sum(
            bool(re.fullmatch(r"\s*\*\*.+\*\*\s*", line))
            for line in nonempty_lines
        ),
        "bold_prefix_lines": sum(
            bool(re.match(r"\s*\*\*[^*\n]+:\*\*", line))
            for line in nonempty_lines
        ),
        "blockquote_lines": sum(
            line.lstrip().startswith(">")
            for line in nonempty_lines
        ),
        "list_lines": sum(
            bool(re.match(r"\s*[-+*]\s+", line))
            for line in nonempty_lines
        ),
    }

    file_profiles.append(profile)
    files_by_program[record["program_id"]].append(record)


print("Accepted files profiled:", len(file_profiles))
print("UTF-8 decode errors:", len(decode_errors))
print("Empty accepted files:", len(empty_files))
print("Programs represented:", len(files_by_program))
print()

speaker_policy_counts = Counter(
    record["speaker_policy"]
    for record in accepted_files
)

print("Speaker policies:")
for policy, count in sorted(speaker_policy_counts.items()):
    print(f"- {policy}: {count}")

print()
print("Aggregate Markdown structure:")
for field in (
    "heading_lines",
    "bold_only_lines",
    "bold_prefix_lines",
    "blockquote_lines",
    "list_lines",
):
    print(
        f"- {field}:",
        sum(profile[field] for profile in file_profiles),
    )

print()
print("Representative source samples by program:")
print("=" * 80)

for program_id in sorted(files_by_program):
    records = sorted(
        files_by_program[program_id],
        key=lambda item: item["relative_path"],
    )

    representative = records[0]
    representative_path = project_root / representative["relative_path"]
    representative_text = representative_path.read_text(encoding="utf-8")

    sample_lines = [
        line
        for line in representative_text.splitlines()
        if line.strip()
    ][:8]

    policy_counts = Counter(
        record["speaker_policy"]
        for record in records
    )

    print()
    print("Program:", representative["program"])
    print("Program ID:", program_id)
    print("Accepted files:", len(records))
    print("Speaker policies:", dict(sorted(policy_counts.items())))
    print("Representative file:", representative["relative_path"])
    print("First 8 non-empty lines:")

    for index, line in enumerate(sample_lines, start=1):
        display_line = line if len(line) <= 220 else line[:220] + "..."
        print(f"  {index:02d}: {display_line}")

    print("-" * 80)

assert len(file_profiles) == len(accepted_files), (
    "Not all accepted files were successfully profiled."
)
assert not decode_errors, (
    f"UTF-8 decode errors found: {decode_errors}"
)
assert not empty_files, (
    f"Empty accepted transcript files found: {empty_files}"
)
assert len(files_by_program) == transcript_manifest_v1["counts"]["programs"], (
    "Program count derived from accepted files does not match Manifest."
)
assert all(files_by_program.values()), (
    "At least one program has no accepted files."
)

print()
print("TRANSCRIPT SOURCE FORMAT DISCOVERY: PASS")

Accepted files profiled: 183
UTF-8 decode errors: 0
Empty accepted files: 0
Programs represented: 13

Speaker policies:
- EXPLICIT_MULTI_SPEAKER: 77
- HUMAN_CONFIRMED_SOLO: 63
- KNOWN_PARTICIPANTS_UNLABELED_TURNS: 1
- MIXED_EXPLICIT_AND_CONTEXTUAL: 38
- NARRATIVE_AUDIENCE_CUES: 1
- PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED: 3

Aggregate Markdown structure:
- heading_lines: 318
- bold_only_lines: 10171
- bold_prefix_lines: 42
- blockquote_lines: 249
- list_lines: 387

Representative source samples by program:

Program: عصر شیرین
Program ID: program:asre-shirin
Accepted files: 26
Speaker policies: {'EXPLICIT_MULTI_SPEAKER': 26}
Representative file: data/transcripts/final/asre_shirin/season-1/episode-01.md
First 8 non-empty lines:
  01: # سواد مالی برای خانم‌ها؛ واقعاً ضروری است؟
  02: برنامه «عصر شیرین» — فصل اول، قسمت اول  
  03: گویندگان: دکتر کمیل رودی، خانم متولیان و شرکت‌کنندگان کارگاه  
  04: مدت: ۲۴ دقیقه و ۱۳ ثانیه  
  05: منبع: https://www.fintelligence.ir/Blog/Detail/B094D69C

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### بررسی قراردادهای Program در Manifest

Discovery ساختار Source نشان داد که Transcriptهای پذیرفته‌شده از یک قالب واحد پیروی نمی‌کنند.

در `183` فایل پذیرفته‌شده، شش Policy متفاوت برای وضعیت Speaker وجود دارد:

- `EXPLICIT_MULTI_SPEAKER`
- `HUMAN_CONFIRMED_SOLO`
- `KNOWN_PARTICIPANTS_UNLABELED_TURNS`
- `MIXED_EXPLICIT_AND_CONTEXTUAL`
- `NARRATIVE_AUDIENCE_CUES`
- `PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED`

بنابراین قبل از طراحی Parser باید بررسی شود که Notebook 05 در بخش `programs` چه اطلاعات و تصمیم‌هایی را برای هر Program ثبت کرده است.

این مرحله فقط Metadata موجود در Manifest را Inspect می‌کند و هیچ Transcriptی Parse نمی‌شود.

برای هر Program موارد زیر از خود Manifest نمایش داده می‌شوند:

- کل Record ثبت‌شده برای Program
- `program_id`
- فیلدهای واقعی موجود
- Policyها و Metadataهای ثبت‌شده
- اطلاعات Speaker فقط در صورتی که صریحاً در Manifest وجود داشته باشند

در این مرحله هیچ نام Speaker، نقش گفتگو یا مرز Turn از روی متن Source استنباط نمی‌شود.

همچنین هیچ تلاشی برای یکسان‌سازی نام‌ها انجام نمی‌شود. اگر Alias یا Variantی وجود داشته باشد، فقط همان چیزی که Manifest ثبت کرده نمایش داده می‌شود.

**هدف این مرحله:**

مشخص شود آیا طراحی Transcript Parser باید فقط بر اساس `speaker_policy` هر فایل انجام شود یا Manifest برای Programهای مختلف قراردادها و محدودیت‌های دقیق‌تری نیز ثبت کرده است.

**معیار PASS:**

- `13` Program Record وجود داشته باشد.
- تمام Recordها `dict` باشند.
- هر Program دارای `program_id` یکتا باشد.
- تمام `program_id`های Accepted Fileها در Registry برنامه‌ها وجود داشته باشند.
- ساختار واقعی هر Program بدون خطا قابل مشاهده باشد.

**معیار FAIL:**

اگر Program Registry ناقص، دارای ID تکراری یا با `accepted_files` ناسازگار باشد، طراحی Parser متوقف می‌شود تا همان ناسازگاری بررسی شود.

</div>

In [5]:
# Code Cell 05

program_records = transcript_manifest_v1["programs"]

assert isinstance(program_records, list), (
    "programs must be a list."
)

assert len(program_records) == transcript_manifest_v1["counts"]["programs"], (
    "Program record count does not match Manifest counts.programs."
)

assert all(isinstance(record, dict) for record in program_records), (
    "Every program record must be a dict."
)

program_key_union = sorted({
    key
    for record in program_records
    for key in record.keys()
})

print("Program records:", len(program_records))
print("Program record key union:")
print(program_key_union)
print()

program_ids = [
    record.get("program_id")
    for record in program_records
]

missing_program_ids = [
    index
    for index, program_id in enumerate(program_ids)
    if not program_id
]

duplicate_program_ids = sorted({
    program_id
    for program_id in program_ids
    if program_id is not None and program_ids.count(program_id) > 1
})

accepted_program_ids = {
    record["program_id"]
    for record in accepted_files
}

registered_program_ids = {
    program_id
    for program_id in program_ids
    if program_id is not None
}

unregistered_accepted_program_ids = sorted(
    accepted_program_ids - registered_program_ids
)

unused_registered_program_ids = sorted(
    registered_program_ids - accepted_program_ids
)

print("Missing program_id values:", len(missing_program_ids))
print("Duplicate program_id values:", len(duplicate_program_ids))
print(
    "Accepted program IDs missing from registry:",
    len(unregistered_accepted_program_ids),
)
print(
    "Registered program IDs without accepted files:",
    len(unused_registered_program_ids),
)
print()

print("Program registry:")
print("=" * 80)

for index, record in enumerate(program_records, start=1):
    print()
    print(f"Program record {index:02d}")
    print(json.dumps(
        record,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ))
    print("-" * 80)

assert not missing_program_ids, (
    f"Program records without program_id found at indexes: {missing_program_ids}"
)
assert not duplicate_program_ids, (
    f"Duplicate program_id values found: {duplicate_program_ids}"
)
assert not unregistered_accepted_program_ids, (
    "Accepted files reference program IDs not present in program registry: "
    f"{unregistered_accepted_program_ids}"
)
assert not unused_registered_program_ids, (
    "Program registry contains programs without accepted files: "
    f"{unused_registered_program_ids}"
)

print()
print("PROGRAM REGISTRY INSPECTION: PASS")

Program records: 13
Program record key union:
['human_confirmed_context', 'program', 'program_id', 'speaker_aliases', 'speaker_policy']

Missing program_id values: 0
Duplicate program_id values: 0
Accepted program IDs missing from registry: 0
Registered program IDs without accepted files: 0

Program registry:

Program record 01
{
  "human_confirmed_context": {
    "format": "SOLO",
    "primary_speaker": "Dr. Komeil Roudi",
    "speaker_attribution_required": false
  },
  "program": "در جستجوی خوشبختی",
  "program_id": "program:dar-jostojo-ye-khoshbakhti",
  "speaker_aliases": {
    "دکتر کمیل رودی": "Dr. Komeil Roudi"
  },
  "speaker_policy": {
    "default_body_attribution_allowed": true,
    "policy": "HUMAN_CONFIRMED_SOLO",
    "preserve_other_turn_context": false,
    "primary_speaker": "Dr. Komeil Roudi",
    "question_response_context_required": false,
    "unknown_speaker_inference_allowed": false
  }
}
---------------------------------------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### Discovery نشانه‌های مرزی و Bold Content در Source

Program Registry تأیید کرد که سیاست Speaker بین مجموعه‌ها متفاوت است و در چند Program، انتساب پیش‌فرض متن به Speaker اصلی صریحاً ممنوع است.

قبل از نوشتن منطق Parse باید بررسی شود که نشانه‌های ظاهری Markdown واقعاً در Source چگونه استفاده شده‌اند.

به‌خصوص وجود `10171` خط Bold در Discovery قبلی نشان می‌دهد که نمی‌توان این قانون ساده را به کار برد:

`Bold line = Speaker`

چنین فرضی ممکن است `Instructional Bold Content` را به اشتباه Speaker مستقل تشخیص دهد.

در این مرحله تمام Bold lineها فقط به‌عنوان **Marker مشاهده‌شده در Source** بررسی می‌شوند و هیچ‌کدام هنوز Speaker طبقه‌بندی نمی‌شوند.

برای هر Program موارد زیر اندازه‌گیری می‌شوند:

- تعداد فایل‌های دارای `---`
- تعداد فایل‌های دارای Bold-only line
- تعداد کل Bold-only lineها
- تعداد مقدارهای یکتای Bold-only
- تعداد فایل‌هایی که هیچ Bold-only line ندارند
- پرتکرارترین متن‌های Bold-only به همان شکل خام Source
- تعداد Bold-prefix lineها

این Inspection کمک می‌کند مشخص شود کدام مجموعه‌ها واقعاً از Bold برای Speaker label استفاده می‌کنند و در کدام مجموعه‌ها Bold کاربرد دیگری نیز دارد.

در این مرحله:

- هیچ Bold Content به Speaker تبدیل نمی‌شود.
- هیچ Alias جدید ساخته نمی‌شود.
- هیچ Speaker از Context حدس زده نمی‌شود.
- متن فارسی Normalize نمی‌شود.
- Turn segmentation انجام نمی‌شود.
- Deduplication یا Chunking انجام نمی‌شود.

متن Markerها دقیقاً به شکل موجود در فایل خوانده می‌شود و هیچ تغییر زبانی روی آن انجام نمی‌شود.

**معیار PASS:**

- هر `183` فایل بررسی شود.
- هر `13` Program در گزارش حاضر باشد.
- الگوی استفاده از Bold و Separator برای هر Program قابل مشاهده باشد.
- هیچ تصمیم Speaker attribution در این مرحله اعمال نشود.

**معیار FAIL:**

اگر بخشی از Accepted Corpus در این Inspection پوشش داده نشود یا Programی از گزارش حذف شود، طراحی Parser ادامه پیدا نمی‌کند.

</div>

In [6]:
# Code Cell 06

bold_only_pattern = re.compile(r"^\s*\*\*(.+?)\*\*\s*$")
bold_prefix_pattern = re.compile(r"^\s*\*\*.+?:\*\*")

marker_profiles = []
bold_values_by_program = defaultdict(Counter)

for record in accepted_files:
    file_path = project_root / record["relative_path"]
    text = file_path.read_text(encoding="utf-8")
    lines = text.splitlines()

    bold_only_values = []

    for line in lines:
        match = bold_only_pattern.fullmatch(line)

        if match:
            raw_value = match.group(1)
            bold_only_values.append(raw_value)
            bold_values_by_program[record["program_id"]][raw_value] += 1

    marker_profiles.append({
        "program": record["program"],
        "program_id": record["program_id"],
        "relative_path": record["relative_path"],
        "has_separator": any(
            line.strip() == "---"
            for line in lines
        ),
        "bold_only_count": len(bold_only_values),
        "bold_prefix_count": sum(
            bool(bold_prefix_pattern.match(line))
            for line in lines
        ),
    })


print("Files inspected:", len(marker_profiles))
print("Programs represented:", len({
    profile["program_id"]
    for profile in marker_profiles
}))
print()

print("Markdown boundary-marker profile by program:")
print("=" * 80)

for program_id in sorted(files_by_program):
    program_file_profiles = [
        profile
        for profile in marker_profiles
        if profile["program_id"] == program_id
    ]

    program_name = program_file_profiles[0]["program"]
    bold_counter = bold_values_by_program[program_id]

    files_with_separator = sum(
        profile["has_separator"]
        for profile in program_file_profiles
    )

    files_with_bold_only = sum(
        profile["bold_only_count"] > 0
        for profile in program_file_profiles
    )

    files_without_bold_only = sum(
        profile["bold_only_count"] == 0
        for profile in program_file_profiles
    )

    total_bold_only = sum(
        profile["bold_only_count"]
        for profile in program_file_profiles
    )

    total_bold_prefix = sum(
        profile["bold_prefix_count"]
        for profile in program_file_profiles
    )

    print()
    print("Program:", program_name)
    print("Program ID:", program_id)
    print("Accepted files:", len(program_file_profiles))
    print("Files with --- separator:", files_with_separator)
    print("Files with Bold-only lines:", files_with_bold_only)
    print("Files without Bold-only lines:", files_without_bold_only)
    print("Total Bold-only lines:", total_bold_only)
    print("Unique Bold-only values:", len(bold_counter))
    print("Bold-prefix lines:", total_bold_prefix)

    if bold_counter:
        ranked_values = sorted(
            bold_counter.items(),
            key=lambda item: (-item[1], item[0]),
        )[:15]

        print("Most frequent raw Bold-only values:")

        for raw_value, count in ranked_values:
            display_value = (
                raw_value
                if len(raw_value) <= 140
                else raw_value[:140] + "..."
            )
            print(f"  - {count:4d} × {display_value}")
    else:
        print("Most frequent raw Bold-only values: NONE")

    print("-" * 80)


assert len(marker_profiles) == len(accepted_files), (
    "Not all accepted files were included in marker discovery."
)

assert {
    profile["program_id"]
    for profile in marker_profiles
} == {
    record["program_id"]
    for record in program_records
}, (
    "Program coverage in marker discovery does not match Program Registry."
)

print()
print("SOURCE BOUNDARY MARKER DISCOVERY: PASS")

Files inspected: 183
Programs represented: 13

Markdown boundary-marker profile by program:

Program: عصر شیرین
Program ID: program:asre-shirin
Accepted files: 26
Files with --- separator: 0
Files with Bold-only lines: 26
Files without Bold-only lines: 0
Total Bold-only lines: 905
Unique Bold-only values: 26
Bold-prefix lines: 42
Most frequent raw Bold-only values:
  -  440 × دکتر کمیل رودی
  -  213 × خانم متولیان
  -   73 × یکی از شرکت‌کنندگان
  -   70 × شرکت‌کننده
  -   19 × یکی دیگر از شرکت‌کنندگان
  -   10 × شرکت‌کننده 5
  -    9 × شرکت‌کننده 1
  -    9 × شرکت‌کننده ۱
  -    8 × شرکت‌کننده 4
  -    8 × شرکت‌کننده ۳
  -    6 × شرکت‌کننده 2
  -    6 × شرکت‌کننده ۲
  -    5 × شرکت‌کننده 3
  -    4 × شرکت‌کننده 8
  -    4 × شرکت‌کننده ۵
--------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Accepted files: 11
Files with --- separator: 11
Files with Bold-only lines: 0
Files without Bold-only lines: 11
Total Bol

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### بررسی Candidateهای واقعی Speaker Marker بدون Attribution

بررسی Markerهای Markdown نشان داد که استفاده از Bold در کل Corpus یکسان نیست.

از `13` Program فقط چهار مجموعه دارای `Bold-only line` هستند:

- `عصر شیرین`
- `در جستجوی خوشبختی`
- `خانه تو`
- `یونی تهران`

در سایر Programها Bold-only برای تعیین مرز Speaker وجود ندارد.

همچنین در مجموعه `عصر شیرین` تعداد `42` مورد `Bold-prefix` مشاهده شده است که قبل از طراحی Parser باید ماهیت واقعی آن‌ها مشخص شود.

در این مرحله تمام مقدارهای یکتای Bold-only و Bold-prefix مستقیماً از Source استخراج و همراه با یک نمونه Context نمایش داده می‌شوند.

هدف، **مشاهده و طبقه‌بندی‌نکردن** است.

برای هر Bold-only Marker نمایش داده می‌شود:

- Program
- متن خام Marker
- تعداد تکرار
- فایل اولین مشاهده
- خط غیرخالی قبل از Marker
- خود Marker
- خط غیرخالی بعد از Marker

برای `Bold-prefix` نیز متن خام Prefixها و نمونه‌های واقعی نمایش داده می‌شوند.

این اطلاعات برای تشخیص تفاوت میان موارد زیر لازم است:

- Speaker label صریح
- Role label صریح
- Metadata
- Instructional Bold Content
- سایر کاربردهای Markdown

در این Cell هیچ Marker به Speaker تبدیل نمی‌شود و هیچ Alias یا Speaker جدیدی ساخته نمی‌شود.

به‌خصوص:

`Instructional Bold Content` حتی اگر از نظر ظاهری شبیه Bold-only باشد، Speaker مستقل محسوب نخواهد شد و در Parser نهایی باید Speaker جاری را inherit کند.

همچنان ممنوع است:

- Speaker inference
- حدس نام Speaker از Context
- Persian normalization
- Deduplication
- Chunking
- Turn segmentation

**معیار PASS:**

- تمام Bold-only valueهای یکتا از Corpus قابل مشاهده باشند.
- تمام Bold-prefixهای مشاهده‌شده قابل بازرسی باشند.
- Context واقعی حداقل اولین occurrence هر Marker ثبت شود.
- هیچ Attribution خودکار در این مرحله انجام نشود.

**معیار FAIL:**

اگر Markerهای کشف‌شده بدون Context قابل بررسی نباشند یا بخشی از Corpus از Inspection جا بماند، طراحی Ruleهای Parser ادامه پیدا نمی‌کند.

</div>

In [7]:
# Code Cell 07

bold_only_occurrences = defaultdict(list)
bold_prefix_occurrences = defaultdict(list)

for record in accepted_files:
    file_path = project_root / record["relative_path"]
    lines = file_path.read_text(encoding="utf-8").splitlines()

    nonempty_indexes = [
        index
        for index, line in enumerate(lines)
        if line.strip()
    ]

    previous_nonempty = {}
    next_nonempty = {}

    previous_index = None
    for index in range(len(lines)):
        previous_nonempty[index] = previous_index
        if lines[index].strip():
            previous_index = index

    next_index = None
    for index in range(len(lines) - 1, -1, -1):
        next_nonempty[index] = next_index
        if lines[index].strip():
            next_index = index

    for index, line in enumerate(lines):
        bold_only_match = bold_only_pattern.fullmatch(line)

        if bold_only_match:
            raw_value = bold_only_match.group(1)

            prev_index = previous_nonempty[index]
            following_index = next_nonempty[index]

            bold_only_occurrences[
                (record["program_id"], raw_value)
            ].append({
                "program": record["program"],
                "relative_path": record["relative_path"],
                "line_number": index + 1,
                "previous_line": (
                    lines[prev_index]
                    if prev_index is not None
                    else None
                ),
                "marker_line": line,
                "next_line": (
                    lines[following_index]
                    if following_index is not None
                    else None
                ),
            })

        bold_prefix_match = re.match(
            r"^\s*\*\*([^*\n]+?):\*\*(.*)$",
            line,
        )

        if bold_prefix_match:
            raw_prefix = bold_prefix_match.group(1)
            trailing_text = bold_prefix_match.group(2)

            bold_prefix_occurrences[
                (record["program_id"], raw_prefix)
            ].append({
                "program": record["program"],
                "relative_path": record["relative_path"],
                "line_number": index + 1,
                "raw_line": line,
                "trailing_text": trailing_text,
            })


print("Unique Bold-only markers:", len(bold_only_occurrences))
print(
    "Total Bold-only occurrences:",
    sum(len(items) for items in bold_only_occurrences.values()),
)
print("Unique Bold-prefix labels:", len(bold_prefix_occurrences))
print(
    "Total Bold-prefix occurrences:",
    sum(len(items) for items in bold_prefix_occurrences.values()),
)

print()
print("BOLD-ONLY MARKERS WITH FIRST CONTEXT")
print("=" * 100)

for key in sorted(
    bold_only_occurrences,
    key=lambda item: (item[0], item[1]),
):
    program_id, raw_value = key
    occurrences = bold_only_occurrences[key]
    sample = occurrences[0]

    print()
    print("Program:", sample["program"])
    print("Program ID:", program_id)
    print("Raw marker value:", raw_value)
    print("Occurrences:", len(occurrences))
    print("First file:", sample["relative_path"])
    print("Line number:", sample["line_number"])
    print("Previous non-empty line:", sample["previous_line"])
    print("Marker line:", sample["marker_line"])
    print("Next non-empty line:", sample["next_line"])
    print("-" * 100)

print()
print("BOLD-PREFIX LABELS WITH FIRST EXAMPLE")
print("=" * 100)

if bold_prefix_occurrences:
    for key in sorted(
        bold_prefix_occurrences,
        key=lambda item: (item[0], item[1]),
    ):
        program_id, raw_prefix = key
        occurrences = bold_prefix_occurrences[key]
        sample = occurrences[0]

        print()
        print("Program:", sample["program"])
        print("Program ID:", program_id)
        print("Raw prefix:", raw_prefix)
        print("Occurrences:", len(occurrences))
        print("First file:", sample["relative_path"])
        print("Line number:", sample["line_number"])
        print("Raw line:", sample["raw_line"])
        print("-" * 100)
else:
    print("NONE")

assert sum(
    len(items)
    for items in bold_only_occurrences.values()
) == sum(
    profile["bold_only_count"]
    for profile in marker_profiles
), (
    "Bold-only occurrence count does not match previous discovery."
)

assert sum(
    len(items)
    for items in bold_prefix_occurrences.values()
) == sum(
    profile["bold_prefix_count"]
    for profile in marker_profiles
), (
    "Bold-prefix occurrence count does not match previous discovery."
)

print()
print("RAW SPEAKER-MARKER CANDIDATE INSPECTION: PASS")

Unique Bold-only markers: 35
Total Bold-only occurrences: 10171
Unique Bold-prefix labels: 5
Total Bold-prefix occurrences: 42

BOLD-ONLY MARKERS WITH FIRST CONTEXT

Program: عصر شیرین
Program ID: program:asre-shirin
Raw marker value: خانم متولیان
Occurrences: 213
First file: data/transcripts/final/asre_shirin/season-1/episode-01.md
Line number: 10
Previous non-empty line: > متن نهایی با تطبیق دو نسخه موجود تنظیم شده است. متن محاوره‌ای و کامل Output حفظ شده و فقط ابهام‌هایی که نسخه دو‌فصلیِ صوت‌بازبینی‌شده پشتیبانی می‌کرد اصلاح شده‌اند.
Marker line: **خانم متولیان**
Next non-empty line: خیلی از ما توی این چند سال به نقطه‌هایی رسیدیم که گاهی خرج و برجمون با هم جور درنیومده. وقت‌هایی شده که روزهای آخر ماه کلی فشار و استرس رو تجربه کردیم و خیلی وقت‌ها نتونستیم اون چیزهایی که دوست داریم رو بخریم. حالا توی یه همچین وضعیتی چی کار می‌تونیم بکنیم که کمتر آسیب ببینیم و وضع مالیمون پایدارتر باشه؟ از قرار معلوم راه‌حل اصلی این مسئله چیزی نیست جز سواد مالی.
----------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### بررسی قرارداد Speaker در سطح فایل‌های پذیرفته‌شده

Inspection مستقیم Source نشان داد که شکل ظاهری Markdown به‌تنهایی برای تشخیص Speaker کافی نیست.

برای مثال، بعضی `Bold-only`ها Speaker Label واقعی هستند، اما بعضی دیگر محتوای آموزشی مانند فرمول یا عبارت تأکیدی‌اند. بنابراین Parser نباید صرفاً بر اساس Bold بودن یک خط، Speaker جدید ایجاد کند.

قبل از طراحی Ruleهای Parse باید مشخص شود Notebook 05 در سطح هر `Accepted File` چه تصمیم انسانی و چه Metadataای درباره ساختار Speaker ثبت کرده است.

در این مرحله فایل‌های پذیرفته‌شده بر اساس `speaker_policy` گروه‌بندی می‌شوند و موارد زیر بررسی خواهند شد:

- تعداد فایل‌های هر Policy
- مقدار `default_body_attribution_allowed`
- مقدار `preserve_other_turn_context`
- `primary_speaker`
- کلیدهای موجود در `metadata_layers`
- محتوای `human_confirmed`
- Metadataهای متفاوت یا خاص در سطح فایل

برای Policyهای کم‌تعداد یا دارای مرز حل‌نشده، مسیر فایل‌ها نیز نمایش داده می‌شود تا مشخص باشد دقیقاً کدام Sourceها نیازمند رفتار محافظه‌کارانه‌تر هستند.

این مرحله هنوز Speaker Parsing انجام نمی‌دهد.

همچنان قواعد زیر برقرار هستند:

- هیچ Speaker ناشناخته‌ای حدس زده نمی‌شود.
- Speaker Alias فقط از Manifest پذیرفته می‌شود.
- `Instructional Bold Content` Speaker مستقل نیست.
- Turnهای سایر افراد در Transcriptهای چندگوینده حفظ می‌شوند.
- فایل‌هایی که `default_body_attribution_allowed=false` دارند، نباید بدون Marker یا Evidence معتبر به Speaker اصلی نسبت داده شوند.
- Persian normalization انجام نمی‌شود.
- Deduplication انجام نمی‌شود.
- Chunking انجام نمی‌شود.

**معیار PASS:**

- هر `183` Accepted File در دقیقاً یک Speaker Policy قرار گیرد.
- Policyهای فایل با Program Registry ناسازگار نباشند.
- مقادیر `default_body_attribution_allowed`، `preserve_other_turn_context` و `unknown_speaker_inference_allowed` با قرارداد Manifest سازگار باشند.
- Metadata انسانی ثبت‌شده برای هر Policy قابل مشاهده باشد.
- فایل‌های دارای وضعیت Speaker مبهم یا مرز حل‌نشده به‌صورت صریح قابل شناسایی باشند.

**معیار FAIL:**

اگر Policy سطح فایل با Program Registry تناقض داشته باشد یا فایلی بدون قرارداد Speaker معتبر وارد مجموعه Accepted شده باشد، طراحی Parser تا بررسی همان مورد متوقف خواهد شد.

</div>

In [8]:
# Code Cell 08

program_registry_by_id = {
    record["program_id"]: record
    for record in program_records
}

files_by_speaker_policy = defaultdict(list)
policy_mismatches = []
invalid_file_contracts = []

for record in accepted_files:
    policy = record["speaker_policy"]
    files_by_speaker_policy[policy].append(record)

    program_contract = program_registry_by_id[record["program_id"]]["speaker_policy"]

    if policy != program_contract["policy"]:
        policy_mismatches.append({
            "relative_path": record["relative_path"],
            "file_policy": policy,
            "program_policy": program_contract["policy"],
        })

    expected_default_attribution = program_contract[
        "default_body_attribution_allowed"
    ]
    expected_preserve_context = program_contract[
        "preserve_other_turn_context"
    ]
    expected_unknown_inference = program_contract[
        "unknown_speaker_inference_allowed"
    ]

    if (
        record["default_body_attribution_allowed"]
        != expected_default_attribution
        or record["preserve_other_turn_context"]
        != expected_preserve_context
        or record["unknown_speaker_inference_allowed"]
        != expected_unknown_inference
    ):
        invalid_file_contracts.append({
            "relative_path": record["relative_path"],
            "program_id": record["program_id"],
            "speaker_policy": policy,
            "file_default_body_attribution_allowed": record[
                "default_body_attribution_allowed"
            ],
            "program_default_body_attribution_allowed": (
                expected_default_attribution
            ),
            "file_preserve_other_turn_context": record[
                "preserve_other_turn_context"
            ],
            "program_preserve_other_turn_context": (
                expected_preserve_context
            ),
            "file_unknown_speaker_inference_allowed": record[
                "unknown_speaker_inference_allowed"
            ],
            "program_unknown_speaker_inference_allowed": (
                expected_unknown_inference
            ),
        })


print("Accepted files:", len(accepted_files))
print("Speaker policies represented:", len(files_by_speaker_policy))
print("File/program policy mismatches:", len(policy_mismatches))
print("Invalid file-level speaker contracts:", len(invalid_file_contracts))
print()

print("FILE-LEVEL SPEAKER CONTRACTS")
print("=" * 100)

for policy in sorted(files_by_speaker_policy):
    records = sorted(
        files_by_speaker_policy[policy],
        key=lambda item: item["relative_path"],
    )

    default_attribution_values = sorted({
        record["default_body_attribution_allowed"]
        for record in records
    })

    preserve_context_values = sorted({
        record["preserve_other_turn_context"]
        for record in records
    })

    primary_speaker_values = sorted({
        str(record["primary_speaker"])
        for record in records
    })

    unknown_inference_values = sorted({
        record["unknown_speaker_inference_allowed"]
        for record in records
    })

    metadata_layer_keys = sorted({
        key
        for record in records
        for key in record["metadata_layers"].keys()
    })

    human_confirmed_variants = defaultdict(list)

    for record in records:
        human_confirmed = record["metadata_layers"].get(
            "human_confirmed",
            {}
        )

        signature = json.dumps(
            human_confirmed,
            ensure_ascii=False,
            sort_keys=True,
        )

        human_confirmed_variants[signature].append(
            record["relative_path"]
        )

    print()
    print("Speaker policy:", policy)
    print("Files:", len(records))
    print(
        "Default body attribution values:",
        default_attribution_values,
    )
    print(
        "Preserve other-turn context values:",
        preserve_context_values,
    )
    print("Primary speaker values:", primary_speaker_values)
    print(
        "Unknown speaker inference values:",
        unknown_inference_values,
    )
    print("Metadata layer keys:", metadata_layer_keys)
    print(
        "Unique human_confirmed variants:",
        len(human_confirmed_variants),
    )

    for index, (signature, paths) in enumerate(
        sorted(human_confirmed_variants.items()),
        start=1,
    ):
        print()
        print(f"  Human-confirmed variant {index}:")
        print(
            json.dumps(
                json.loads(signature),
                ensure_ascii=False,
                indent=2,
                sort_keys=True,
            )
        )
        print("  Files with this variant:", len(paths))

        preview_paths = paths[:10]

        for path in preview_paths:
            print("   -", path)

        if len(paths) > len(preview_paths):
            print(
                "   - ...",
                len(paths) - len(preview_paths),
                "more files",
            )

    if len(records) <= 5:
        print()
        print("All files in this policy:")
        for record in records:
            print(" -", record["relative_path"])

    print("-" * 100)


assert sum(
    len(records)
    for records in files_by_speaker_policy.values()
) == len(accepted_files), (
    "Not all accepted files were assigned to a speaker policy."
)

assert not policy_mismatches, (
    f"File/program speaker policy mismatches found: {policy_mismatches}"
)

assert not invalid_file_contracts, (
    "File-level speaker contract mismatches found: "
    f"{invalid_file_contracts}"
)

assert all(
    record["unknown_speaker_inference_allowed"] is False
    for record in accepted_files
), (
    "At least one accepted file allows unknown speaker inference."
)

print()
print("FILE-LEVEL SPEAKER CONTRACT INSPECTION: PASS")

Accepted files: 183
Speaker policies represented: 6
File/program policy mismatches: 0
Invalid file-level speaker contracts: 0

FILE-LEVEL SPEAKER CONTRACTS

Speaker policy: EXPLICIT_MULTI_SPEAKER
Files: 77
Default body attribution values: [False]
Preserve other-turn context values: [True]
Primary speaker values: ['Dr. Komeil Roudi']
Unknown speaker inference values: [False]
Metadata layer keys: ['human_confirmed', 'path_filename_derived', 'source_header', 'source_name_variant_status', 'source_program_name', 'source_provenance']
Unique human_confirmed variants: 1

  Human-confirmed variant 1:
{
  "program_confirmed": true,
  "speaker_policy": "EXPLICIT_MULTI_SPEAKER"
}
  Files with this variant: 77
   - data/transcripts/final/asre_shirin/season-1/episode-01.md
   - data/transcripts/final/asre_shirin/season-1/episode-02.md
   - data/transcripts/final/asre_shirin/season-1/episode-03.md
   - data/transcripts/final/asre_shirin/season-1/episode-04.md
   - data/transcripts/final/asre_shirin/s

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### Discovery مرز Metadata/Body و Markerهای غیر-Bold

تا این مرحله مشخص شد که Speaker Policy هر فایل معتبر و از قبل در Manifest ثبت شده است، اما شکل ظاهری Source میان Programها متفاوت است.

در چند مجموعه، Speaker با `Bold-only` مشخص شده است؛ در چند مجموعه دیگر چنین Markerی وجود ندارد و ساختار گفتگو ممکن است با روش دیگری ثبت شده باشد.

قبل از ساخت Parser باید دو موضوع از روی Source واقعی بررسی شوند:

1. مرز میان Metadata اولیه فایل و متن اصلی Transcript چگونه ثبت شده است؟
2. آیا Markerهای غیر-Bold مانند `label:`، Heading یا قالب‌های مشابه برای مشخص‌کردن Turn یا Speaker وجود دارند؟

این مرحله فقط Syntax موجود در Source را بررسی می‌کند و هیچ Markerی را Speaker اعلام نمی‌کند.

برای هر Program موارد زیر گزارش می‌شوند:

- تعداد فایل‌های دارای Separator دقیق `---`
- موقعیت اولین Separator در صورت وجود
- نمونه خطوط ابتدایی Body پس از Separator
- Prefixهای کوتاه قبل از `:` که در متن مشاهده می‌شوند
- Headingهای سطح دوم به بعد
- تعداد فایل‌هایی که هیچ Separator صریح ندارند

برای Patternهای `label:` فقط متن خام Prefix ثبت می‌شود. وجود چنین Prefixی به معنی Speaker بودن آن نیست؛ ممکن است Metadata، توضیح، نقش روایی یا محتوای عادی باشد.

در این مرحله هیچ فایل تغییر نمی‌کند و هیچ Parsed Artifact ساخته نمی‌شود.

همچنان ممنوع است:

- Speaker inference
- تبدیل خودکار `label:` به Speaker
- انتساب Body به Speaker در فایل‌هایی که این کار در Manifest مجاز نیست
- Persian normalization
- Deduplication
- Chunking
- حذف سؤال‌ها یا Turnهای افراد دیگر

**معیار PASS:**

- هر `183` فایل در Discovery حضور داشته باشد.
- وضعیت Separator برای تمام `13` Program مشخص شود.
- Markerهای غیر-Bold بدون Attribution قابل مشاهده باشند.
- حداقل یک نمونه Body از هر Program قابل بررسی باشد.

**معیار FAIL:**

اگر بخشی از Accepted Corpus پوشش داده نشود یا نتوان مرزهای ظاهری Source را بدون حدس بررسی کرد، طراحی Parser ادامه پیدا نمی‌کند.

</div>

In [9]:
# Code Cell 09

plain_colon_pattern = re.compile(
    r"^\s*([^#>*\-\[\]`]{1,60}?)\s*[:：]\s*(.*)$"
)
subheading_pattern = re.compile(
    r"^\s*#{2,6}\s+(.+?)\s*$"
)

body_boundary_profiles = []
colon_prefix_occurrences = defaultdict(list)
subheading_occurrences = defaultdict(list)

for record in accepted_files:
    file_path = project_root / record["relative_path"]
    lines = file_path.read_text(encoding="utf-8").splitlines()

    separator_indexes = [
        index
        for index, line in enumerate(lines)
        if line.strip() == "---"
    ]

    first_separator_index = (
        separator_indexes[0]
        if separator_indexes
        else None
    )

    if first_separator_index is not None:
        body_lines = lines[first_separator_index + 1:]
    else:
        body_lines = lines

    first_body_nonempty = [
        line
        for line in body_lines
        if line.strip()
    ][:8]

    body_boundary_profiles.append({
        "program": record["program"],
        "program_id": record["program_id"],
        "relative_path": record["relative_path"],
        "separator_count": len(separator_indexes),
        "first_separator_line": (
            first_separator_index + 1
            if first_separator_index is not None
            else None
        ),
        "first_body_nonempty": first_body_nonempty,
    })

    for index, line in enumerate(lines):
        if bold_only_pattern.fullmatch(line):
            continue

        if bold_prefix_pattern.match(line):
            continue

        colon_match = plain_colon_pattern.match(line)

        if colon_match:
            raw_prefix = colon_match.group(1)
            trailing_text = colon_match.group(2)

            colon_prefix_occurrences[
                (record["program_id"], raw_prefix)
            ].append({
                "program": record["program"],
                "relative_path": record["relative_path"],
                "line_number": index + 1,
                "raw_line": line,
                "trailing_text": trailing_text,
            })

        subheading_match = subheading_pattern.match(line)

        if subheading_match:
            raw_heading = subheading_match.group(1)

            subheading_occurrences[
                (record["program_id"], raw_heading)
            ].append({
                "program": record["program"],
                "relative_path": record["relative_path"],
                "line_number": index + 1,
                "raw_line": line,
            })


print("Files inspected:", len(body_boundary_profiles))
print(
    "Programs represented:",
    len({
        item["program_id"]
        for item in body_boundary_profiles
    }),
)
print()

print("BODY BOUNDARY PROFILE BY PROGRAM")
print("=" * 100)

for program_id in sorted(files_by_program):
    profiles = [
        item
        for item in body_boundary_profiles
        if item["program_id"] == program_id
    ]

    program_name = profiles[0]["program"]

    files_with_separator = sum(
        profile["separator_count"] > 0
        for profile in profiles
    )

    files_without_separator = sum(
        profile["separator_count"] == 0
        for profile in profiles
    )

    separator_line_numbers = sorted({
        profile["first_separator_line"]
        for profile in profiles
        if profile["first_separator_line"] is not None
    })

    representative = sorted(
        profiles,
        key=lambda item: item["relative_path"],
    )[0]

    print()
    print("Program:", program_name)
    print("Program ID:", program_id)
    print("Accepted files:", len(profiles))
    print("Files with ---:", files_with_separator)
    print("Files without ---:", files_without_separator)
    print("Observed first-separator line numbers:", separator_line_numbers)
    print("Representative file:", representative["relative_path"])
    print("First body non-empty lines:")

    for index, line in enumerate(
        representative["first_body_nonempty"],
        start=1,
    ):
        display_line = (
            line
            if len(line) <= 220
            else line[:220] + "..."
        )
        print(f"  {index:02d}: {display_line}")

    print("-" * 100)


print()
print("NON-BOLD COLON PREFIXES")
print("=" * 100)

colon_program_counts = Counter(
    program_id
    for program_id, _ in colon_prefix_occurrences
)

print("Programs with observed non-Bold colon prefixes:", len(colon_program_counts))
print("Unique program/prefix pairs:", len(colon_prefix_occurrences))
print(
    "Total occurrences:",
    sum(
        len(items)
        for items in colon_prefix_occurrences.values()
    ),
)

for key in sorted(
    colon_prefix_occurrences,
    key=lambda item: (item[0], item[1]),
):
    program_id, raw_prefix = key
    occurrences = colon_prefix_occurrences[key]
    sample = occurrences[0]

    print()
    print("Program:", sample["program"])
    print("Raw prefix:", raw_prefix)
    print("Occurrences:", len(occurrences))
    print("First file:", sample["relative_path"])
    print("Line number:", sample["line_number"])
    print("Raw line:", sample["raw_line"])
    print("-" * 100)


print()
print("SUBHEADINGS LEVEL 2+")
print("=" * 100)

print("Unique program/heading pairs:", len(subheading_occurrences))
print(
    "Total occurrences:",
    sum(
        len(items)
        for items in subheading_occurrences.values()
    ),
)

for key in sorted(
    subheading_occurrences,
    key=lambda item: (item[0], item[1]),
):
    program_id, raw_heading = key
    occurrences = subheading_occurrences[key]
    sample = occurrences[0]

    print()
    print("Program:", sample["program"])
    print("Raw heading:", raw_heading)
    print("Occurrences:", len(occurrences))
    print("First file:", sample["relative_path"])
    print("Line number:", sample["line_number"])
    print("Raw line:", sample["raw_line"])
    print("-" * 100)


assert len(body_boundary_profiles) == len(accepted_files), (
    "Not all accepted files were included in body-boundary discovery."
)

assert {
    item["program_id"]
    for item in body_boundary_profiles
} == {
    record["program_id"]
    for record in program_records
}, (
    "Program coverage in body-boundary discovery is incomplete."
)

assert all(
    profile["first_body_nonempty"]
    for profile in body_boundary_profiles
), (
    "At least one accepted file has no observable body content."
)

print()
print("BODY AND NON-BOLD MARKER DISCOVERY: PASS")

Files inspected: 183
Programs represented: 13

BODY BOUNDARY PROFILE BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Accepted files: 26
Files with ---: 0
Files without ---: 26
Observed first-separator line numbers: []
Representative file: data/transcripts/final/asre_shirin/season-1/episode-01.md
First body non-empty lines:
  01: # سواد مالی برای خانم‌ها؛ واقعاً ضروری است؟
  02: برنامه «عصر شیرین» — فصل اول، قسمت اول  
  03: گویندگان: دکتر کمیل رودی، خانم متولیان و شرکت‌کنندگان کارگاه  
  04: مدت: ۲۴ دقیقه و ۱۳ ثانیه  
  05: منبع: https://www.fintelligence.ir/Blog/Detail/B094D69C/%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%AE%D8%A7%D9%86%D9%85%D9%87%D8%A7-%D9%88%D8%A7%D9%82%D8%B9%D8%A7-%D8%B6%D8%B1%D9%88%D8...
  06: > متن نهایی با تطبیق دو نسخه موجود تنظیم شده است. متن محاوره‌ای و کامل Output حفظ شده و فقط ابهام‌هایی که نسخه دو‌فصلیِ صوت‌بازبینی‌شده پشتیبانی می‌کرد اصلاح شده‌اند.
  07: **خانم متولیان**
  08: خیلی از ما توی این چند سال به

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### طراحی Candidateهای مرز Body بر اساس ساختار واقعی Source

Discovery قبلی نشان داد که یک Rule واحد برای پیدا کردن شروع Transcript Body کافی نیست.

به‌طور مشخص، اولین `---` در همه فایل‌ها مرز Body نیست. در بخشی از Corpus بعد از Separator هنوز اطلاعات مربوط به Source و Review قرار دارد و متن Transcript چند خط بعد شروع می‌شود.

همچنین Pattern عمومی `prefix:` برای Speaker Detection قابل استفاده نیست، زیرا صدها Prefix مختلف در جمله‌های عادی Transcript وجود دارند.

بنابراین قبل از Speaker Parsing، فقط **Candidate Body Boundary** برای هر فایل تعیین می‌شود.

این مرحله هنوز Transcript را Parse نمی‌کند و هیچ Speaker Attribution انجام نمی‌دهد.

چهار خانواده ساختاری که از داده واقعی مشاهده شده‌اند:

1. فایل‌هایی که دارای Heading دقیق `# متن بازبینی نهایی` هستند  
   در این فایل‌ها Body Candidate بعد از Heading بازبینی و Blockquoteهای متصل به آن قرار می‌گیرد.

2. فایل‌هایی که Separator `---` دارند ولی بخش `# متن بازبینی نهایی` ندارند  
   در این فایل‌ها اولین محتوای غیرخالی بعد از Separator به‌عنوان Candidate بررسی می‌شود.

3. فایل‌های چندگوینده بدون Separator  
   در این فایل‌ها اولین `Bold-only` فقط به‌عنوان Candidate مرز ساختاری ثبت می‌شود؛ در این مرحله هنوز Speaker بودن آن اعلام نمی‌شود.

4. فایل‌های `HUMAN_CONFIRMED_SOLO` بدون Separator  
   با توجه به قرارداد انسانی Notebook 05، Heading اولیه فایل از Body جدا می‌شود و اولین محتوای غیرخالی بعد از Heading به‌عنوان Candidate بررسی می‌شود.

این Ruleها در این Cell فقط برای **Inspection و Coverage Check** استفاده می‌شوند و هنوز خروجی Ingestion محسوب نمی‌شوند.

در این مرحله:

- هیچ متن Source تغییر نمی‌کند.
- هیچ Bold Content به Speaker تبدیل نمی‌شود.
- هیچ `prefix:` به Speaker تبدیل نمی‌شود.
- هیچ Speaker ناشناخته‌ای حدس زده نمی‌شود.
- Persian normalization انجام نمی‌شود.
- Deduplication انجام نمی‌شود.
- Chunking انجام نمی‌شود.

**معیار PASS:**

- هر `183` فایل دقیقاً در یکی از Strategyهای قابل توضیح قرار گیرد.
- برای هیچ فایل Candidate Body مفقود نباشد.
- Candidate هر فایل مستقیماً از ساختار همان Source به دست آید.
- تمام `13` Program پوشش داده شوند.
- هیچ Attribution یا تغییر متنی در این مرحله انجام نشود.

**معیار FAIL:**

اگر فایلی در هیچ Strategy قرار نگیرد، Candidate Body نداشته باشد یا ساختار آن با قرارداد فعلی قابل توضیح نباشد، قبل از نوشتن Parser همان فایل یا خانواده ساختاری جداگانه بررسی خواهد شد.

</div>

In [10]:
# Code Cell 10

def first_nonempty_index(lines, start_index=0):
    for index in range(start_index, len(lines)):
        if lines[index].strip():
            return index
    return None


body_boundary_candidates = []
boundary_failures = []

for record in accepted_files:
    file_path = project_root / record["relative_path"]
    lines = file_path.read_text(encoding="utf-8").splitlines()

    review_heading_indexes = [
        index
        for index, line in enumerate(lines)
        if line.strip() == "# متن بازبینی نهایی"
    ]

    separator_indexes = [
        index
        for index, line in enumerate(lines)
        if line.strip() == "---"
    ]

    strategy = None
    candidate_index = None

    if len(review_heading_indexes) == 1:
        strategy = "AFTER_REVIEW_BLOCK"
        scan_index = review_heading_indexes[0] + 1

        while scan_index < len(lines):
            stripped = lines[scan_index].strip()

            if not stripped or stripped.startswith(">"):
                scan_index += 1
                continue

            candidate_index = scan_index
            break

    elif len(review_heading_indexes) > 1:
        boundary_failures.append({
            "relative_path": record["relative_path"],
            "reason": "MULTIPLE_REVIEW_HEADINGS",
            "count": len(review_heading_indexes),
        })
        continue

    elif separator_indexes:
        strategy = "AFTER_SEPARATOR"
        candidate_index = first_nonempty_index(
            lines,
            separator_indexes[0] + 1,
        )

    elif record["speaker_policy"] == "EXPLICIT_MULTI_SPEAKER":
        strategy = "FIRST_BOLD_ONLY_CANDIDATE"

        for index, line in enumerate(lines):
            if bold_only_pattern.fullmatch(line):
                candidate_index = index
                break

    elif record["speaker_policy"] == "HUMAN_CONFIRMED_SOLO":
        strategy = "SOLO_AFTER_INITIAL_HEADING"

        first_nonempty = first_nonempty_index(lines)

        if (
            first_nonempty is not None
            and lines[first_nonempty].lstrip().startswith("# ")
        ):
            candidate_index = first_nonempty_index(
                lines,
                first_nonempty + 1,
            )
        else:
            candidate_index = first_nonempty

    else:
        boundary_failures.append({
            "relative_path": record["relative_path"],
            "reason": "NO_SUPPORTED_BOUNDARY_STRATEGY",
            "speaker_policy": record["speaker_policy"],
        })
        continue

    if candidate_index is None:
        boundary_failures.append({
            "relative_path": record["relative_path"],
            "reason": "BODY_CANDIDATE_NOT_FOUND",
            "strategy": strategy,
        })
        continue

    body_boundary_candidates.append({
        "program": record["program"],
        "program_id": record["program_id"],
        "relative_path": record["relative_path"],
        "speaker_policy": record["speaker_policy"],
        "strategy": strategy,
        "candidate_line_number": candidate_index + 1,
        "candidate_raw_line": lines[candidate_index],
    })


strategy_counts = Counter(
    item["strategy"]
    for item in body_boundary_candidates
)

print("Accepted files:", len(accepted_files))
print("Boundary candidates:", len(body_boundary_candidates))
print("Boundary failures:", len(boundary_failures))
print(
    "Programs represented:",
    len({
        item["program_id"]
        for item in body_boundary_candidates
    }),
)
print()

print("Boundary strategies:")
for strategy, count in sorted(strategy_counts.items()):
    print(f"- {strategy}: {count}")

print()
print("Representative candidate by program:")
print("=" * 100)

for program_id in sorted(files_by_program):
    candidates = sorted(
        [
            item
            for item in body_boundary_candidates
            if item["program_id"] == program_id
        ],
        key=lambda item: item["relative_path"],
    )

    if not candidates:
        print()
        print("Program ID:", program_id)
        print("NO CANDIDATE")
        print("-" * 100)
        continue

    sample = candidates[0]

    display_line = (
        sample["candidate_raw_line"]
        if len(sample["candidate_raw_line"]) <= 240
        else sample["candidate_raw_line"][:240] + "..."
    )

    program_strategy_counts = Counter(
        item["strategy"]
        for item in candidates
    )

    print()
    print("Program:", sample["program"])
    print("Program ID:", program_id)
    print("Files:", len(candidates))
    print(
        "Strategies:",
        dict(sorted(program_strategy_counts.items())),
    )
    print("Representative file:", sample["relative_path"])
    print(
        "Candidate line number:",
        sample["candidate_line_number"],
    )
    print("Candidate raw line:", display_line)
    print("-" * 100)


assert len(body_boundary_candidates) == len(accepted_files), (
    "Not all accepted files received a body-boundary candidate. "
    f"Failures: {boundary_failures}"
)

assert not boundary_failures, (
    f"Body-boundary discovery failures found: {boundary_failures}"
)

assert len({
    item["program_id"]
    for item in body_boundary_candidates
}) == transcript_manifest_v1["counts"]["programs"], (
    "Body-boundary candidates do not cover all programs."
)

assert all(
    item["candidate_raw_line"].strip()
    for item in body_boundary_candidates
), (
    "At least one body-boundary candidate is empty."
)

print()
print("BODY BOUNDARY CANDIDATE COVERAGE: PASS")

Accepted files: 183
Boundary candidates: 183
Boundary failures: 0
Programs represented: 13

Boundary strategies:
- AFTER_REVIEW_BLOCK: 55
- AFTER_SEPARATOR: 13
- FIRST_BOLD_ONLY_CANDIDATE: 64
- SOLO_AFTER_INITIAL_HEADING: 51

Representative candidate by program:

Program: عصر شیرین
Program ID: program:asre-shirin
Files: 26
Strategies: {'FIRST_BOLD_ONLY_CANDIDATE': 26}
Representative file: data/transcripts/final/asre_shirin/season-1/episode-01.md
Candidate line number: 10
Candidate raw line: **خانم متولیان**
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Files: 11
Strategies: {'AFTER_REVIEW_BLOCK': 11}
Representative file: data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/بانک ملت 1.md
Candidate line number: 21
Candidate raw line: وقتی تابستان‌ها می‌رفتیم باغ پدربزرگم، او به ما یک داس می‌داد که مثلاً علف‌های هرز را بچینیم، یا یک سبد می‌داد برویم گیلاس بچینیم، یا بیل دستمان

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### بررسی Metadata منبع برای Evidence مربوط به Speaker و ساختار Transcript

Body Boundary Candidate برای تمام `183` فایل با موفقیت پیدا شد.

مرحله بعد هنوز Parse کردن Transcript نیست.

قبل از تعریف Ruleهای Speaker باید بررسی شود که آیا Notebook 05 در لایه‌های Metadata هر فایل، Evidence بیشتری از خود Source ثبت کرده است یا خیر.

دو بخش مهم در `metadata_layers` عبارت‌اند از:

- `source_header`
- `source_provenance`

ممکن است این بخش‌ها شامل اطلاعاتی مانند نام Speaker، Guest، Host، عنوان Source یا سایر نشانه‌های قابل استناد باشند.

اگر چنین اطلاعاتی وجود داشته باشد، Parser باید از همان Evidence استفاده کند و نباید اطلاعاتی را که از Manifest یا Source قابل بازیابی است از روی Context حدس بزند.

در این مرحله برای هر Program بررسی می‌شود:

- چند فایل دارای `source_header` غیرخالی هستند.
- چند فایل دارای `source_provenance` غیرخالی هستند.
- چه Keyهایی در این دو بخش واقعاً مشاهده شده‌اند.
- یک نمونه واقعی از Metadata غیرخالی هر Program چیست.
- آیا ساختار Metadata بین فایل‌های یک Program ثابت یا متغیر است.

هیچ مقدار Metadata در این مرحله Normalize یا بازنویسی نمی‌شود.

همچنان ممنوع است:

- ساخت Speaker Alias جدید از روی شباهت اسمی
- حدس Host یا Guest
- تبدیل خودکار Metadata به Speaker Attribution
- Persian normalization
- Deduplication
- Chunking
- تغییر متن Transcript

**هدف این مرحله:**

مشخص شود برای طراحی Parser چه Evidence صریحی علاوه بر `speaker_policy` و Markerهای داخل متن در اختیار داریم.

**معیار PASS:**

- تمام `183` فایل بررسی شوند.
- هر `13` Program پوشش داده شود.
- ساختار واقعی `source_header` و `source_provenance` بدون حدس قابل مشاهده باشد.
- هیچ Metadata جدیدی ساخته نشود.

**معیار FAIL:**

اگر ساختار Metadata قابل خواندن نباشد یا بخشی از Accepted Corpus از Inspection حذف شود، طراحی Parser تا رفع همان مشکل ادامه پیدا نمی‌کند.

</div>

In [11]:
# Code Cell 11

def collect_key_paths(value, prefix=""):
    key_paths = set()

    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            key_paths.add(path)
            key_paths.update(
                collect_key_paths(child, path)
            )

    elif isinstance(value, list):
        for child in value:
            key_paths.update(
                collect_key_paths(child, prefix)
            )

    return key_paths


def is_nonempty_metadata(value):
    if value is None:
        return False

    if isinstance(value, dict):
        return bool(value)

    if isinstance(value, list):
        return bool(value)

    if isinstance(value, str):
        return bool(value.strip())

    return True


metadata_inspection_records = []

for record in accepted_files:
    metadata_layers = record["metadata_layers"]

    source_header = metadata_layers.get(
        "source_header",
        {}
    )
    source_provenance = metadata_layers.get(
        "source_provenance",
        {}
    )

    metadata_inspection_records.append({
        "program": record["program"],
        "program_id": record["program_id"],
        "relative_path": record["relative_path"],
        "source_header": source_header,
        "source_provenance": source_provenance,
        "source_header_key_paths": sorted(
            collect_key_paths(source_header)
        ),
        "source_provenance_key_paths": sorted(
            collect_key_paths(source_provenance)
        ),
        "source_header_nonempty": is_nonempty_metadata(
            source_header
        ),
        "source_provenance_nonempty": is_nonempty_metadata(
            source_provenance
        ),
    })


print("Files inspected:", len(metadata_inspection_records))
print(
    "Programs represented:",
    len({
        item["program_id"]
        for item in metadata_inspection_records
    }),
)
print()

print("SOURCE METADATA PROFILE BY PROGRAM")
print("=" * 100)

for program_id in sorted(files_by_program):
    items = sorted(
        [
            item
            for item in metadata_inspection_records
            if item["program_id"] == program_id
        ],
        key=lambda item: item["relative_path"],
    )

    program_name = items[0]["program"]

    header_nonempty_items = [
        item
        for item in items
        if item["source_header_nonempty"]
    ]

    provenance_nonempty_items = [
        item
        for item in items
        if item["source_provenance_nonempty"]
    ]

    header_key_union = sorted({
        key_path
        for item in items
        for key_path in item["source_header_key_paths"]
    })

    provenance_key_union = sorted({
        key_path
        for item in items
        for key_path in item["source_provenance_key_paths"]
    })

    unique_headers = {
        json.dumps(
            item["source_header"],
            ensure_ascii=False,
            sort_keys=True,
        )
        for item in header_nonempty_items
    }

    unique_provenance = {
        json.dumps(
            item["source_provenance"],
            ensure_ascii=False,
            sort_keys=True,
        )
        for item in provenance_nonempty_items
    }

    print()
    print("Program:", program_name)
    print("Program ID:", program_id)
    print("Accepted files:", len(items))
    print(
        "Files with non-empty source_header:",
        len(header_nonempty_items),
    )
    print(
        "Files with non-empty source_provenance:",
        len(provenance_nonempty_items),
    )
    print(
        "Unique non-empty source_header objects:",
        len(unique_headers),
    )
    print(
        "Unique non-empty source_provenance objects:",
        len(unique_provenance),
    )
    print("source_header key paths:", header_key_union)
    print(
        "source_provenance key paths:",
        provenance_key_union,
    )

    if header_nonempty_items:
        sample = header_nonempty_items[0]

        print()
        print("Representative non-empty source_header:")
        print("File:", sample["relative_path"])
        print(
            json.dumps(
                sample["source_header"],
                ensure_ascii=False,
                indent=2,
                sort_keys=True,
            )
        )
    else:
        print()
        print("Representative non-empty source_header: NONE")

    if provenance_nonempty_items:
        sample = provenance_nonempty_items[0]

        print()
        print("Representative non-empty source_provenance:")
        print("File:", sample["relative_path"])
        print(
            json.dumps(
                sample["source_provenance"],
                ensure_ascii=False,
                indent=2,
                sort_keys=True,
            )
        )
    else:
        print()
        print("Representative non-empty source_provenance: NONE")

    print("-" * 100)


assert len(metadata_inspection_records) == len(accepted_files), (
    "Not all accepted files were included in source metadata inspection."
)

assert {
    item["program_id"]
    for item in metadata_inspection_records
} == {
    record["program_id"]
    for record in program_records
}, (
    "Program coverage in source metadata inspection is incomplete."
)

print()
print("SOURCE METADATA EVIDENCE INSPECTION: PASS")

Files inspected: 183
Programs represented: 13

SOURCE METADATA PROFILE BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Accepted files: 26
Files with non-empty source_header: 0
Files with non-empty source_provenance: 0
Unique non-empty source_header objects: 0
Unique non-empty source_provenance objects: 0
source_header key paths: []
source_provenance key paths: []

Representative non-empty source_header: NONE

Representative non-empty source_provenance: NONE
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Accepted files: 11
Files with non-empty source_header: 11
Files with non-empty source_provenance: 1
Unique non-empty source_header objects: 11
Unique non-empty source_provenance objects: 1
source_header key paths: ['برنامه', 'توضیحات', 'عنوان/موضوع', 'قسمت', 'مدت\u200cزمان کل', 'منبع', 'منبع آرشیوی', 'وضعیت متن']
source_provenance key paths: ['منبع', 'منبع آرشیوی']

Re

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### شناسایی فایل‌های Explicit و Contextual در مجموعه چهلستون

Program Registry برای مجموعه `چهلستون` یک قرارداد مهم ثبت کرده است:

- `38` Accepted File
- `35` فایل با Explicit Speaker Boundary
- `3` فایل با Contextual Boundary

اما این تفکیک در سطح هر Accepted File به‌صورت مستقیم ذخیره نشده است.

قبل از ساخت Parser باید مشخص شود کدام فایل‌ها دارای Markerهای صریح Speaker هستند و کدام فایل‌ها فاقد چنین Marker قابل اتکایی‌اند.

در Sourceهای چهلستون، Markerهای صریح عمدتاً به شکل زیر دیده شده‌اند:

`Speaker Label: متن`

اما Discovery قبلی ثابت کرد که هر عبارت قبل از `:` Speaker نیست. عبارت‌هایی مانند:

- `درآمد:`
- `امنیت:`
- `سؤال بعدی این است:`
- جمله‌های معمولی دارای Colon

نیز در متن وجود دارند.

بنابراین در این مرحله هیچ `prefix:` مستقیماً Speaker اعلام نمی‌شود.

برای Discovery محافظه‌کارانه فقط Prefixهایی بررسی می‌شوند که:

- در ابتدای خط قرار دارند؛
- کوتاه هستند؛
- در Corpus چهلستون بیش از یک بار تکرار شده‌اند.

این شرط صرفاً برای کاهش Noise در Inspection است و **Rule نهایی Speaker Parsing نیست**.

برای هر Candidate موارد زیر نمایش داده می‌شوند:

- متن خام Prefix
- تعداد occurrence
- تعداد فایل‌هایی که در آن دیده شده
- مسیر فایل‌ها

سپس برای هر یک از `38` فایل چهلستون تعداد Candidateهای مشاهده‌شده محاسبه می‌شود.

هدف اصلی این Cell این است که بررسی کنیم آیا الگوی Source واقعاً همان تفکیک `35 Explicit / 3 Contextual` ثبت‌شده در Manifest را نشان می‌دهد یا خیر.

در این مرحله:

- هیچ Candidate به Speaker canonical تبدیل نمی‌شود.
- هیچ نام شخصی از روی Context حدس زده نمی‌شود.
- Alias جدید ساخته نمی‌شود.
- فایل‌های Contextual به Speaker اصلی نسبت داده نمی‌شوند.
- هیچ Transcriptی هنوز Parse نمی‌شود.
- Persian normalization انجام نمی‌شود.
- Deduplication انجام نمی‌شود.
- Chunking انجام نمی‌شود.

**معیار PASS:**

- هر `38` فایل چهلستون بررسی شود.
- Candidateهای Speaker-like بدون Attribution قابل مشاهده باشند.
- فایل‌های دارای Candidate و فاقد Candidate به‌صورت مستقل مشخص شوند.
- نتیجه با قرارداد `35 Explicit / 3 Contextual` مقایسه شود.

اگر Discovery دقیقاً به `35 / 3` برسد، این یک Evidence قوی برای طراحی Parser خواهد بود.

اگر به این نسبت نرسد، هیچ Classification اجباری انجام نمی‌شود و قبل از ادامه باید Sourceهای اختلاف‌دار جداگانه بررسی شوند.

</div>

In [12]:
# Code Cell 12

chehelston_program_id = "program:chehelston"

chehelston_records = sorted(
    [
        record
        for record in accepted_files
        if record["program_id"] == chehelston_program_id
    ],
    key=lambda item: item["relative_path"],
)

assert len(chehelston_records) == 38, (
    f"Expected 38 Chehelston files, found {len(chehelston_records)}."
)

chehelston_registry = program_registry_by_id[
    chehelston_program_id
]["speaker_policy"]

expected_explicit_files = chehelston_registry[
    "explicit_boundary_files"
]
expected_contextual_files = chehelston_registry[
    "contextual_boundary_files"
]

raw_prefix_occurrence_count = Counter()
raw_prefix_file_sets = defaultdict(set)
raw_prefix_examples = {}

for record in chehelston_records:
    file_path = project_root / record["relative_path"]
    lines = file_path.read_text(encoding="utf-8").splitlines()

    for line_number, line in enumerate(lines, start=1):
        if bold_prefix_pattern.match(line):
            continue

        match = plain_colon_pattern.match(line)

        if not match:
            continue

        raw_prefix = match.group(1).strip()

        raw_prefix_occurrence_count[raw_prefix] += 1
        raw_prefix_file_sets[raw_prefix].add(
            record["relative_path"]
        )

        raw_prefix_examples.setdefault(
            raw_prefix,
            {
                "relative_path": record["relative_path"],
                "line_number": line_number,
                "raw_line": line,
            },
        )


repeated_short_prefix_candidates = sorted(
    [
        prefix
        for prefix, count in raw_prefix_occurrence_count.items()
        if count >= 2
        and len(prefix) <= 32
        and "\n" not in prefix
    ],
    key=lambda prefix: (
        -raw_prefix_occurrence_count[prefix],
        prefix,
    ),
)


print("Chehelston accepted files:", len(chehelston_records))
print("Manifest expected explicit-boundary files:", expected_explicit_files)
print("Manifest expected contextual-boundary files:", expected_contextual_files)
print()
print(
    "Repeated short colon-prefix candidates:",
    len(repeated_short_prefix_candidates),
)
print()

print("CANDIDATE PREFIX INSPECTION")
print("=" * 100)

for prefix in repeated_short_prefix_candidates:
    files = sorted(raw_prefix_file_sets[prefix])
    example = raw_prefix_examples[prefix]

    print()
    print("Raw prefix:", prefix)
    print(
        "Total occurrences:",
        raw_prefix_occurrence_count[prefix],
    )
    print("Files containing prefix:", len(files))

    for path in files:
        print(" -", path)

    print("First example:")
    print(
        f"  {example['relative_path']} "
        f"(line {example['line_number']})"
    )
    print(" ", example["raw_line"])
    print("-" * 100)


candidate_prefix_set = set(
    repeated_short_prefix_candidates
)

file_candidate_profiles = []

for record in chehelston_records:
    file_path = project_root / record["relative_path"]
    lines = file_path.read_text(encoding="utf-8").splitlines()

    observed_candidates = []

    for line_number, line in enumerate(lines, start=1):
        if bold_prefix_pattern.match(line):
            continue

        match = plain_colon_pattern.match(line)

        if not match:
            continue

        raw_prefix = match.group(1).strip()

        if raw_prefix in candidate_prefix_set:
            observed_candidates.append({
                "prefix": raw_prefix,
                "line_number": line_number,
            })

    unique_candidates = sorted({
        item["prefix"]
        for item in observed_candidates
    })

    file_candidate_profiles.append({
        "relative_path": record["relative_path"],
        "candidate_occurrences": len(observed_candidates),
        "unique_candidate_prefixes": unique_candidates,
    })


files_with_candidates = [
    profile
    for profile in file_candidate_profiles
    if profile["candidate_occurrences"] > 0
]

files_without_candidates = [
    profile
    for profile in file_candidate_profiles
    if profile["candidate_occurrences"] == 0
]


print()
print("FILE-LEVEL CANDIDATE COVERAGE")
print("=" * 100)
print("Files with candidate prefixes:", len(files_with_candidates))
print("Files without candidate prefixes:", len(files_without_candidates))
print()

print("Files without repeated-short-prefix candidates:")
for profile in files_without_candidates:
    print(" -", profile["relative_path"])

print()
print("Candidate coverage by file:")
print("-" * 100)

for profile in file_candidate_profiles:
    print()
    print("File:", profile["relative_path"])
    print(
        "Candidate occurrences:",
        profile["candidate_occurrences"],
    )
    print(
        "Unique candidate prefixes:",
        profile["unique_candidate_prefixes"],
    )


print()
print("Manifest expectation:")
print(
    f"- Explicit boundary files: {expected_explicit_files}"
)
print(
    f"- Contextual boundary files: {expected_contextual_files}"
)
print()

coverage_matches_manifest = (
    len(files_with_candidates) == expected_explicit_files
    and len(files_without_candidates) == expected_contextual_files
)

print(
    "Candidate coverage matches Manifest 35/3 split:",
    coverage_matches_manifest,
)

assert len(file_candidate_profiles) == 38, (
    "Not all Chehelston files were inspected."
)

assert (
    len(files_with_candidates)
    + len(files_without_candidates)
    == 38
), (
    "Chehelston file coverage is incomplete."
)

print()
print("CHEHELSTON BOUNDARY DISCOVERY: PASS")

Chehelston accepted files: 38
Manifest expected explicit-boundary files: 35
Manifest expected contextual-boundary files: 3

Repeated short colon-prefix candidates: 32

CANDIDATE PREFIX INSPECTION

Raw prefix: مجری
Total occurrences: 245
Files containing prefix: 35
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 10.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 11.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 12.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 13.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 14.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 15.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 16.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 17.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 18.md
 - data/transcripts/final/chehels

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### بررسی عمیق سه فایل Contextual مجموعه چهلستون

Discovery قبلی دقیقاً تقسیم `35 Explicit / 3 Contextual` ثبت‌شده در Manifest را بازتولید کرد.

سه فایل فاقد Candidate صریح عبارت‌اند از:

- `چهلستون 1.md`
- `چهلستون 2.md`
- `چهلستون 38.md`

همچنین Marker خام `مجری:` در هر `35` فایل گروه Explicit مشاهده شد و در این سه فایل وجود نداشت.

این تطابق Evidence مهمی است، اما هنوز برای سه فایل Contextual هیچ Speaker Attribution انجام نمی‌دهیم.

در این مرحله فقط ساختار واقعی همین سه Source با جزئیات بیشتری بررسی می‌شود.

برای هر فایل موارد زیر نمایش داده می‌شوند:

- Candidate شروع Body که در Cell قبلی پیدا شده است
- اولین خطوط غیرخالی Body
- تمام Prefixهای کوتاه دارای `:` در همان فایل
- خطوطی که در آن‌ها نام یا Alias ثبت‌شده دکتر کمیل رودی به‌صورت خام دیده می‌شود
- خطوطی که عبارت `مجری` در آن‌ها دیده می‌شود
- Headingها و Blockquoteهای موجود در Body

Aliasهای مورد جست‌وجو فقط از `speaker_aliases` ثبت‌شده در Program Registry گرفته می‌شوند و هیچ Alias جدیدی ساخته نمی‌شود.

وجود نام شخص در یک جمله نیز به معنی Speaker Boundary نیست. این Cell فقط محل Evidence را نشان می‌دهد.

هدف این مرحله پاسخ به این سؤال است:

آیا سه فایل Contextual دارای نشانه‌های کافی برای حفظ بخش‌های گفتگو بدون انتساب حدسی هستند، یا برای آن‌ها باید Speaker Boundary به‌صورت `UNRESOLVED` باقی بماند؟

در این مرحله همچنان ممنوع است:

- حدس Speaker از ترتیب سؤال و جواب
- انتساب متن فاقد Marker به دکتر کمیل رودی
- ساخت نام Speaker برای مخاطب یا مجری
- تبدیل هر `prefix:` به Speaker
- Persian normalization
- Deduplication
- Chunking
- حذف Context سایر افراد

**معیار PASS:**

- هر سه فایل Contextual مستقیماً از Disk بررسی شوند.
- Body Candidate هر سه فایل معتبر باشد.
- تمام Aliasهای مورد استفاده فقط از Manifest گرفته شوند.
- Evidence خام مرتبط با Speaker بدون Attribution نمایش داده شود.
- هیچ Rule حدسی در این مرحله اعمال نشود.

**معیار FAIL:**

اگر یکی از سه فایل قابل بررسی نباشد، Body Candidate آن پیدا نشود یا اطلاعات Registry با فایل‌ها ناسازگار باشد، طراحی Parser برای چهلستون متوقف می‌شود تا همان مورد بررسی شود.

</div>

In [13]:
# Code Cell 13

chehelston_contextual_paths = sorted(
    profile["relative_path"]
    for profile in files_without_candidates
)

assert len(chehelston_contextual_paths) == expected_contextual_files, (
    "Contextual Chehelston file count does not match Manifest."
)

chehelston_program_record = program_registry_by_id[
    chehelston_program_id
]

registered_aliases = sorted(
    chehelston_program_record["speaker_aliases"].keys()
)

registered_primary_speaker = chehelston_program_record[
    "speaker_policy"
]["primary_speaker"]

print("Contextual Chehelston files:", len(chehelston_contextual_paths))
print("Registered primary speaker:", registered_primary_speaker)
print("Registered source aliases:", registered_aliases)
print()

explicit_host_marker_files = []

for record in chehelston_records:
    file_path = project_root / record["relative_path"]
    lines = file_path.read_text(encoding="utf-8").splitlines()

    if any(
        re.match(r"^\s*مجری\s*[:：]", line)
        for line in lines
    ):
        explicit_host_marker_files.append(
            record["relative_path"]
        )

print(
    "Files containing exact raw 'مجری:' marker:",
    len(explicit_host_marker_files),
)
print(
    "Files without exact raw 'مجری:' marker:",
    len(chehelston_records) - len(explicit_host_marker_files),
)

host_marker_missing_paths = sorted(
    set(record["relative_path"] for record in chehelston_records)
    - set(explicit_host_marker_files)
)

print("Files without exact raw 'مجری:' marker:")
for path in host_marker_missing_paths:
    print(" -", path)

print()

assert len(explicit_host_marker_files) == expected_explicit_files, (
    "Exact 'مجری:' marker coverage does not match Manifest explicit count."
)

assert host_marker_missing_paths == chehelston_contextual_paths, (
    "Files without exact 'مجری:' marker do not match contextual files."
)


boundary_candidate_by_path = {
    item["relative_path"]: item
    for item in body_boundary_candidates
}

print("CONTEXTUAL FILE DEEP INSPECTION")
print("=" * 100)

for relative_path in chehelston_contextual_paths:
    file_path = project_root / relative_path
    lines = file_path.read_text(encoding="utf-8").splitlines()

    candidate = boundary_candidate_by_path[relative_path]
    body_start_index = candidate["candidate_line_number"] - 1
    body_lines = lines[body_start_index:]

    nonempty_body = [
        (body_start_index + index + 1, line)
        for index, line in enumerate(body_lines)
        if line.strip()
    ]

    short_colon_lines = []

    for line_number, line in nonempty_body:
        if bold_prefix_pattern.match(line):
            continue

        match = plain_colon_pattern.match(line)

        if match:
            short_colon_lines.append({
                "line_number": line_number,
                "prefix": match.group(1).strip(),
                "raw_line": line,
            })

    alias_evidence = []

    for line_number, line in nonempty_body:
        matched_aliases = [
            alias
            for alias in registered_aliases
            if alias in line
        ]

        if matched_aliases:
            alias_evidence.append({
                "line_number": line_number,
                "aliases": matched_aliases,
                "raw_line": line,
            })

    host_word_evidence = [
        {
            "line_number": line_number,
            "raw_line": line,
        }
        for line_number, line in nonempty_body
        if "مجری" in line
    ]

    structural_lines = [
        {
            "line_number": line_number,
            "raw_line": line,
        }
        for line_number, line in nonempty_body
        if (
            line.lstrip().startswith("#")
            or line.lstrip().startswith(">")
        )
    ]

    print()
    print("File:", relative_path)
    print("Body strategy:", candidate["strategy"])
    print(
        "Body candidate line:",
        candidate["candidate_line_number"],
    )
    print("Body candidate raw line:", candidate["candidate_raw_line"])
    print()

    print("First 30 non-empty body lines:")
    for line_number, line in nonempty_body[:30]:
        display_line = (
            line
            if len(line) <= 240
            else line[:240] + "..."
        )
        print(f"  {line_number:04d}: {display_line}")

    print()
    print("Short colon-prefix lines:", len(short_colon_lines))

    for item in short_colon_lines:
        display_line = (
            item["raw_line"]
            if len(item["raw_line"]) <= 240
            else item["raw_line"][:240] + "..."
        )

        print(
            f"  {item['line_number']:04d} | "
            f"prefix={item['prefix']!r} | "
            f"{display_line}"
        )

    print()
    print(
        "Lines containing registered Dr. Roudi aliases:",
        len(alias_evidence),
    )

    for item in alias_evidence:
        display_line = (
            item["raw_line"]
            if len(item["raw_line"]) <= 240
            else item["raw_line"][:240] + "..."
        )

        print(
            f"  {item['line_number']:04d} | "
            f"aliases={item['aliases']} | "
            f"{display_line}"
        )

    print()
    print(
        "Lines containing raw word 'مجری':",
        len(host_word_evidence),
    )

    for item in host_word_evidence:
        display_line = (
            item["raw_line"]
            if len(item["raw_line"]) <= 240
            else item["raw_line"][:240] + "..."
        )

        print(
            f"  {item['line_number']:04d}: "
            f"{display_line}"
        )

    print()
    print(
        "Heading/blockquote lines inside candidate body:",
        len(structural_lines),
    )

    for item in structural_lines:
        display_line = (
            item["raw_line"]
            if len(item["raw_line"]) <= 240
            else item["raw_line"][:240] + "..."
        )

        print(
            f"  {item['line_number']:04d}: "
            f"{display_line}"
        )

    print("-" * 100)


assert set(chehelston_contextual_paths).issubset(
    boundary_candidate_by_path
), (
    "At least one contextual Chehelston file lacks a body candidate."
)

print()
print("CHEHELSTON CONTEXTUAL SOURCE INSPECTION: PASS")

Contextual Chehelston files: 3
Registered primary speaker: Dr. Komeil Roudi
Registered source aliases: ['آقای دکتر رودی', 'دکتر رودی', 'دکتر کمیل رودی']

Files containing exact raw 'مجری:' marker: 35
Files without exact raw 'مجری:' marker: 3
Files without exact raw 'مجری:' marker:
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 1.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 2.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 38.md

CONTEXTUAL FILE DEEP INSPECTION

File: data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 1.md
Body strategy: AFTER_REVIEW_BLOCK
Body candidate line: 21
Body candidate raw line: شنونده برنامه «چهلستون» باشید؛ برنامه‌ای که به سواد مالی و مبانی آن می‌پردازد.

First 30 non-empty body lines:
  0021: شنونده برنامه «چهلستون» باشید؛ برنامه‌ای که به سواد مالی و مبانی آن می‌پردازد.
  0023: به نام خدا. سلام، به چهلستون خوش آمدید. اینجا رادیو گفت‌وگوست. قصد داریم درباره موض

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### بررسی عمیق سایر Policyهای دارای Speaker Boundary حل‌نشده

سه فایل Contextual چهلستون بررسی شدند و مشخص شد که برای آن‌ها انتساب Turnها بدون حدس ممکن نیست.

اکنون باید سایر فایل‌هایی که Manifest صریحاً آن‌ها را خارج از حالت `EXPLICIT_MULTI_SPEAKER` یا `HUMAN_CONFIRMED_SOLO` قرار داده بررسی کنیم.

این گروه شامل سه Policy است:

- `KNOWN_PARTICIPANTS_UNLABELED_TURNS`
- `NARRATIVE_AUDIENCE_CUES`
- `PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED`

در Corpus فعلی این سه Policy مجموعاً `5` فایل دارند:

- `کتاب باز`
- `میزان`
- دو فایل `اقیانوس آرام`
- `شراکت`

هدف این Cell تعیین Speaker نیست.

هدف فقط این است که Evidence موجود در Source و Manifest کنار هم قرار گیرد تا مشخص شود Parser برای این فایل‌ها باید چه رفتار محافظه‌کارانه‌ای داشته باشد.

برای هر فایل موارد زیر بررسی می‌شوند:

- Policy ثبت‌شده
- قرارداد Program
- Candidate شروع Body
- `known_participants` در صورت وجود
- `generic_roles_observed` در صورت وجود
- `possible_secondary_speaker` در صورت وجود
- Evidence موجود در `source_header`
- اولین خطوط واقعی Body
- خطوطی که نام Participantهای ثبت‌شده در Manifest در آن‌ها ظاهر می‌شود
- خطوطی که Cueهای روایی ثبت‌شده در Manifest در آن‌ها دیده می‌شوند

وجود نام فرد در یک جمله به معنی Speaker Attribution نیست.

برای مثال اگر Source بنویسد:

`آقای دکتر رودی، شما توضیح بدهید`

این فقط Evidence از حضور یا خطاب به شخص است و به‌تنهایی مرز آغاز Turn بعدی را تعیین نمی‌کند.

همچنین Cueهایی مانند:

`یکی از حاضران`

یا:

`یکی می‌پرسد`

صرفاً همان Label روایی موجود در Source هستند. اگر در Parser استفاده شوند باید همان Label خام حفظ شود و نباید به یک هویت واقعی تبدیل شوند.

**قاعده محافظه‌کارانه این مرحله:**

اگر Speaker Boundary از Source قابل اثبات نباشد، مقدار آن باید unresolved یا ambiguous باقی بماند؛ نه اینکه از ترتیب سؤال و جواب حدس زده شود.

در این مرحله همچنان انجام نمی‌شوند:

- Speaker inference
- ساخت Alias جدید
- انتساب خودکار سؤال و جواب
- Persian normalization
- Deduplication
- Chunking
- حذف Turnهای سایر افراد

**معیار PASS:**

- هر `5` فایل Special-policy بررسی شود.
- قرارداد Manifest هر فایل قابل بازیابی باشد.
- Participant و Cue فقط از Metadata ثبت‌شده استفاده شوند.
- Body Candidate هر فایل موجود باشد.
- هیچ Speaker جدیدی از روی Context ساخته نشود.

**معیار FAIL:**

اگر فایل Special-policy بدون قرارداد Program، بدون Body Candidate یا با Metadata ناسازگار مشاهده شود، طراحی Parser تا بررسی همان فایل متوقف خواهد شد.

</div>

In [14]:
# Code Cell 14

special_speaker_policies = {
    "KNOWN_PARTICIPANTS_UNLABELED_TURNS",
    "NARRATIVE_AUDIENCE_CUES",
    "PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED",
}

special_policy_records = sorted(
    [
        record
        for record in accepted_files
        if record["speaker_policy"] in special_speaker_policies
    ],
    key=lambda item: item["relative_path"],
)

special_policy_counts = Counter(
    record["speaker_policy"]
    for record in special_policy_records
)

print("Special-policy files:", len(special_policy_records))
print("Policy distribution:")

for policy, count in sorted(special_policy_counts.items()):
    print(f"- {policy}: {count}")

print()

assert len(special_policy_records) == 5, (
    f"Expected 5 special-policy files, found {len(special_policy_records)}."
)


def collect_literal_evidence(nonempty_body, literal_values):
    evidence = []

    for line_number, line in nonempty_body:
        matched_values = [
            value
            for value in literal_values
            if value and value in line
        ]

        if matched_values:
            evidence.append({
                "line_number": line_number,
                "matched_values": matched_values,
                "raw_line": line,
            })

    return evidence


print("SPECIAL-POLICY SOURCE INSPECTION")
print("=" * 100)

for record in special_policy_records:
    relative_path = record["relative_path"]
    program_id = record["program_id"]

    program_record = program_registry_by_id[program_id]
    policy_contract = program_record["speaker_policy"]

    candidate = boundary_candidate_by_path[relative_path]

    file_path = project_root / relative_path
    lines = file_path.read_text(encoding="utf-8").splitlines()

    body_start_index = candidate["candidate_line_number"] - 1

    nonempty_body = [
        (index + 1, line)
        for index, line in enumerate(lines)
        if index >= body_start_index and line.strip()
    ]

    known_participants = policy_contract.get(
        "known_participants",
        [],
    )

    generic_roles = policy_contract.get(
        "generic_roles_observed",
        [],
    )

    primary_speaker = policy_contract.get(
        "primary_speaker"
    )

    speaker_aliases = program_record.get(
        "speaker_aliases",
        {}
    )

    participant_literals = list(known_participants)

    if primary_speaker:
        participant_literals.append(primary_speaker)

    participant_literals.extend(
        speaker_aliases.keys()
    )

    source_header = record["metadata_layers"].get(
        "source_header",
        {}
    )

    participant_evidence = collect_literal_evidence(
        nonempty_body,
        participant_literals,
    )

    generic_role_evidence = collect_literal_evidence(
        nonempty_body,
        generic_roles,
    )

    colon_prefix_samples = []

    for line_number, line in nonempty_body:
        if bold_prefix_pattern.match(line):
            continue

        match = plain_colon_pattern.match(line)

        if match:
            colon_prefix_samples.append({
                "line_number": line_number,
                "prefix": match.group(1).strip(),
                "raw_line": line,
            })

    print()
    print("Program:", record["program"])
    print("Program ID:", program_id)
    print("File:", relative_path)
    print("Speaker policy:", record["speaker_policy"])
    print("Body strategy:", candidate["strategy"])
    print(
        "Body candidate line:",
        candidate["candidate_line_number"],
    )
    print(
        "Default body attribution allowed:",
        record["default_body_attribution_allowed"],
    )
    print(
        "Preserve other-turn context:",
        record["preserve_other_turn_context"],
    )
    print(
        "Unknown speaker inference allowed:",
        record["unknown_speaker_inference_allowed"],
    )

    print()
    print("Program policy contract:")
    print(
        json.dumps(
            policy_contract,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
    )

    print()
    print("Registered speaker aliases:")
    print(
        json.dumps(
            speaker_aliases,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
    )

    print()
    print("Source header evidence:")
    print(
        json.dumps(
            source_header,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
    )

    print()
    print("First 35 non-empty body lines:")

    for line_number, line in nonempty_body[:35]:
        display_line = (
            line
            if len(line) <= 240
            else line[:240] + "..."
        )

        print(
            f"  {line_number:04d}: "
            f"{display_line}"
        )

    print()
    print(
        "Lines containing registered participant/alias literals:",
        len(participant_evidence),
    )

    for item in participant_evidence[:25]:
        display_line = (
            item["raw_line"]
            if len(item["raw_line"]) <= 240
            else item["raw_line"][:240] + "..."
        )

        print(
            f"  {item['line_number']:04d} | "
            f"matches={item['matched_values']} | "
            f"{display_line}"
        )

    if len(participant_evidence) > 25:
        print(
            "  ...",
            len(participant_evidence) - 25,
            "more participant-evidence lines",
        )

    print()
    print(
        "Lines containing registered narrative cues:",
        len(generic_role_evidence),
    )

    for item in generic_role_evidence[:25]:
        display_line = (
            item["raw_line"]
            if len(item["raw_line"]) <= 240
            else item["raw_line"][:240] + "..."
        )

        print(
            f"  {item['line_number']:04d} | "
            f"matches={item['matched_values']} | "
            f"{display_line}"
        )

    if len(generic_role_evidence) > 25:
        print(
            "  ...",
            len(generic_role_evidence) - 25,
            "more narrative-cue lines",
        )

    print()
    print(
        "Non-Bold colon-prefix lines:",
        len(colon_prefix_samples),
    )

    for item in colon_prefix_samples[:20]:
        display_line = (
            item["raw_line"]
            if len(item["raw_line"]) <= 240
            else item["raw_line"][:240] + "..."
        )

        print(
            f"  {item['line_number']:04d} | "
            f"prefix={item['prefix']!r} | "
            f"{display_line}"
        )

    if len(colon_prefix_samples) > 20:
        print(
            "  ...",
            len(colon_prefix_samples) - 20,
            "more colon-prefix lines",
        )

    print("-" * 100)


assert all(
    record["relative_path"] in boundary_candidate_by_path
    for record in special_policy_records
), (
    "At least one special-policy file lacks a body-boundary candidate."
)

assert all(
    record["program_id"] in program_registry_by_id
    for record in special_policy_records
), (
    "At least one special-policy file lacks a Program Registry contract."
)

assert all(
    record["unknown_speaker_inference_allowed"] is False
    for record in special_policy_records
), (
    "At least one special-policy file unexpectedly allows speaker inference."
)

print()
print("SPECIAL-POLICY SOURCE INSPECTION: PASS")

Special-policy files: 5
Policy distribution:
- KNOWN_PARTICIPANTS_UNLABELED_TURNS: 1
- NARRATIVE_AUDIENCE_CUES: 1
- PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED: 3

SPECIAL-POLICY SOURCE INSPECTION

Program: اقیانوس آرام
Program ID: program:oghyanous-aram
File: data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/اقیانوس آرام 1.md
Speaker policy: PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED
Body strategy: AFTER_REVIEW_BLOCK
Body candidate line: 22
Default body attribution allowed: False
Preserve other-turn context: True
Unknown speaker inference allowed: False

Program policy contract:
{
  "default_body_attribution_allowed": false,
  "policy": "PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED",
  "preserve_other_turn_context": true,
  "primary_speaker": "Dr. Komeil Roudi",
  "question_response_context_required": true,
  "unknown_speaker_inference_allowed": false
}

Registered speaker aliases:
{}

Source header evidence:
{
  "Release": [
    "`ocean-aram-komeil-roudi`"
  ],
  "برنامه":

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### تکمیل Registry محافظه‌کارانه Speaker Markerهای چهلستون

Discovery قبلی نشان داد که `35` فایل چهلستون دارای Speaker Boundary صریح هستند و `3` فایل بدون Marker صریح باقی می‌مانند.

برای ساخت Parser نباید فقط به Markerهای پرتکرار تکیه کنیم؛ زیرا ممکن است یک مهمان فقط یک بار در کل Corpus ظاهر شده باشد.

از طرف دیگر نمی‌توان هر Prefix قبل از `:` را Speaker دانست، چون عبارت‌هایی مانند `درآمد:`، `امنیت:` و جمله‌های عادی نیز در Transcript وجود دارند.

در این مرحله Candidateهای Speaker فقط بر اساس Evidence صریح و محافظه‌کارانه جمع‌آوری می‌شوند.

Candidate قطعی در یکی از این حالت‌ها قرار می‌گیرد:

- Marker دقیق `مجری`
- Alias ثبت‌شده در Manifest برای دکتر کمیل رودی
- Prefix دارای عنوان صریح انسانی مانند `دکتر`، `آقای`، `خانم` یا `سرهنگ`
- Prefix کوتاه و تکرارشونده‌ای که در Discovery قبلی به‌عنوان Candidate مشاهده شده است

سپس Prefixهای کوتاه باقی‌مانده که هنوز در هیچ‌کدام از این گروه‌ها قرار نگرفته‌اند جداگانه نمایش داده می‌شوند.

این Prefixهای باقی‌مانده **Speaker اعلام نمی‌شوند**. هدف فقط این است که ببینیم آیا نام‌های انسانی کم‌تکراری مانند نام یک مهمان هنوز خارج از Registry مانده‌اند یا خیر.

هیچ Parser در این Cell اجرا نمی‌شود.

همچنان ممنوع است:

- تبدیل خودکار هر `prefix:` به Speaker
- حدس نام یا هویت Speaker
- ساخت Alias جدید
- نسبت‌دادن Speaker بر اساس ترتیب سؤال و جواب
- Persian normalization
- Deduplication
- Chunking

**معیار PASS:**

- هر `35` فایل Explicit چهلستون بررسی شود.
- Candidateهای قطعی بر اساس Ruleهای شفاف ساخته شوند.
- Prefixهای کوتاه باقی‌مانده به‌صورت جداگانه گزارش شوند.
- هیچ Prefix باقی‌مانده بدون Evidence به Speaker تبدیل نشود.

این مرحله آخرین Discovery مربوط به Markerهای چهلستون پیش از تعریف Parser خواهد بود.

</div>

In [15]:
# Code Cell 15

chehelston_explicit_paths = sorted(
    set(record["relative_path"] for record in chehelston_records)
    - set(chehelston_contextual_paths)
)

assert len(chehelston_explicit_paths) == expected_explicit_files, (
    "Explicit Chehelston file count does not match Manifest."
)

registered_chehelston_aliases = set(
    chehelston_program_record["speaker_aliases"].keys()
)

title_prefix_pattern = re.compile(
    r"^(?:"
    r"آقای(?:\s+دکتر)?"
    r"|خانم(?:\s+دکتر)?"
    r"|دکتر"
    r"|سرهنگ"
    r"|افسر"
    r")\s+.+$"
)

simple_name_pattern = re.compile(
    r"^[^\d\W_]+(?:[\u200c\-][^\d\W_]+)*"
    r"(?:\s+[^\d\W_]+(?:[\u200c\-][^\d\W_]+)*){0,2}$",
    re.UNICODE,
)

all_short_prefixes = Counter()
short_prefix_files = defaultdict(set)
short_prefix_examples = {}

for relative_path in chehelston_explicit_paths:
    file_path = project_root / relative_path
    lines = file_path.read_text(encoding="utf-8").splitlines()

    candidate = boundary_candidate_by_path[relative_path]
    body_start_index = candidate["candidate_line_number"] - 1

    for index in range(body_start_index, len(lines)):
        line = lines[index]

        if not line.strip():
            continue

        if bold_prefix_pattern.match(line):
            continue

        match = plain_colon_pattern.match(line)

        if not match:
            continue

        raw_prefix = match.group(1).strip()

        if len(raw_prefix) > 32:
            continue

        all_short_prefixes[raw_prefix] += 1
        short_prefix_files[raw_prefix].add(relative_path)

        short_prefix_examples.setdefault(
            raw_prefix,
            {
                "relative_path": relative_path,
                "line_number": index + 1,
                "raw_line": line,
            },
        )


verified_marker_reasons = defaultdict(set)

for prefix, count in all_short_prefixes.items():
    if prefix == "مجری":
        verified_marker_reasons[prefix].add(
            "EXACT_HOST_MARKER"
        )

    if prefix in registered_chehelston_aliases:
        verified_marker_reasons[prefix].add(
            "MANIFEST_REGISTERED_ALIAS"
        )

    if title_prefix_pattern.fullmatch(prefix):
        verified_marker_reasons[prefix].add(
            "EXPLICIT_HUMAN_TITLE_PREFIX"
        )

    if prefix in candidate_prefix_set:
        verified_marker_reasons[prefix].add(
            "REPEATED_DISCOVERY_CANDIDATE"
        )


verified_chehelston_markers = sorted(
    verified_marker_reasons
)

remaining_short_prefixes = sorted(
    [
        prefix
        for prefix in all_short_prefixes
        if prefix not in verified_marker_reasons
    ],
    key=lambda prefix: (
        -all_short_prefixes[prefix],
        prefix,
    ),
)

remaining_name_shaped_prefixes = [
    prefix
    for prefix in remaining_short_prefixes
    if simple_name_pattern.fullmatch(prefix)
]

remaining_other_prefixes = [
    prefix
    for prefix in remaining_short_prefixes
    if prefix not in remaining_name_shaped_prefixes
]


print("Explicit Chehelston files:", len(chehelston_explicit_paths))
print("Short colon prefixes observed:", len(all_short_prefixes))
print(
    "Verified marker candidates:",
    len(verified_chehelston_markers),
)
print(
    "Remaining short prefixes:",
    len(remaining_short_prefixes),
)
print(
    "Remaining name-shaped prefixes:",
    len(remaining_name_shaped_prefixes),
)
print()

print("VERIFIED CHEHELSTON MARKER CANDIDATES")
print("=" * 100)

for prefix in verified_chehelston_markers:
    example = short_prefix_examples[prefix]

    print()
    print("Raw prefix:", prefix)
    print("Occurrences:", all_short_prefixes[prefix])
    print(
        "Files:",
        len(short_prefix_files[prefix]),
    )
    print(
        "Evidence:",
        sorted(verified_marker_reasons[prefix]),
    )
    print(
        "First example:",
        f"{example['relative_path']} "
        f"(line {example['line_number']})",
    )
    print(" ", example["raw_line"])
    print("-" * 100)


print()
print("UNRESOLVED NAME-SHAPED SHORT PREFIXES")
print("=" * 100)

if remaining_name_shaped_prefixes:
    for prefix in remaining_name_shaped_prefixes:
        example = short_prefix_examples[prefix]

        print()
        print("Raw prefix:", prefix)
        print("Occurrences:", all_short_prefixes[prefix])
        print(
            "Files:",
            len(short_prefix_files[prefix]),
        )
        print(
            "First example:",
            f"{example['relative_path']} "
            f"(line {example['line_number']})",
        )
        print(" ", example["raw_line"])
        print("-" * 100)
else:
    print("NONE")


print()
print("REMAINING NON-NAME SHORT PREFIX SUMMARY")
print("=" * 100)
print(
    "Count:",
    len(remaining_other_prefixes),
)

for prefix in remaining_other_prefixes[:40]:
    print(
        f"- {all_short_prefixes[prefix]:3d} × {prefix}"
    )

if len(remaining_other_prefixes) > 40:
    print(
        "...",
        len(remaining_other_prefixes) - 40,
        "more prefixes",
    )


assert len(chehelston_explicit_paths) == 35, (
    "Chehelston explicit file coverage changed unexpectedly."
)

assert "مجری" in verified_marker_reasons, (
    "Exact host marker was not preserved."
)

assert registered_chehelston_aliases.issubset(
    set(all_short_prefixes) | {"دکتر کمیل رودی"}
), (
    "Registered Chehelston aliases are inconsistent with observed Source."
)

print()
print("CHEHELSTON SAFE MARKER REGISTRY DISCOVERY: PASS")

Explicit Chehelston files: 35
Short colon prefixes observed: 67
Verified marker candidates: 34
Remaining short prefixes: 33
Remaining name-shaped prefixes: 9

VERIFIED CHEHELSTON MARKER CANDIDATES

Raw prefix: آقای تهمتن
Occurrences: 6
Files: 1
Evidence: ['EXPLICIT_HUMAN_TITLE_PREFIX', 'REPEATED_DISCOVERY_CANDIDATE']
First example: data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 3.md (line 97)
  آقای تهمتن: اجازه بدهید مثالی بزنم. نمونه‌ای مشهور درباره صندوقی با نماد CUBA وجود دارد. در دوره‌ای که خبرهای سیاسی مربوط به بهبود روابط آمریکا و کوبا منتشر شد، قیمت این صندوق در زمانی کوتاه جهش شدیدی کرد، در حالی که نام CUBA به‌تنهایی به معنی سرمایه‌گذاری مستقیم و آزاد در کشور کوبا نبود.
----------------------------------------------------------------------------------------------------

Raw prefix: آقای داودی
Occurrences: 7
Files: 1
Evidence: ['EXPLICIT_HUMAN_TITLE_PREFIX', 'REPEATED_DISCOVERY_CANDIDATE']
First example: data/transcripts/final/chehelstion_finup_ketabbaz_ba

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### بررسی Aliasهای Source-observed باقی‌مانده در چهلستون

Discovery قبلی دو حالت خاص را نشان داد که یک Speaker در یک فایل با دو Label متفاوت ظاهر شده است:

- `حامد زحمتکش` و `زحمتکش`
- `محمدرضا منصوریان` و `منصوریان`

این دو مورد با Aliasهای Manifest متفاوت‌اند.

در این مرحله هیچ Alias بر اساس شباهت اسمی ساخته نمی‌شود.

برای پذیرش یک Alias جدید باید Evidence مستقیم از همان Source وجود داشته باشد:

1. هر دو Label به‌صورت `prefix:` در یک فایل مشاهده شده باشند.
2. Label کوتاه دقیقاً بخشی از Label بلندتر باشد.
3. هر دو Label در همان فایل در محل‌هایی ظاهر شوند که ساختار فایل Speaker Marker صریح دارد.
4. Context قبل و بعد از هر دو Label برای بررسی انسانی نمایش داده شود.

این Rule فقط برای **اثبات رابطه داخل همان Source** استفاده می‌شود و هیچ تطبیق زبانی یا Persian normalization انجام نمی‌دهد.

مواردی مانند:

- `درآمد`
- `امنیت`
- `مصرف`
- `مولوی می‌گوید`
- `پس برگردیم`

عمداً خارج از Alias Registry باقی می‌مانند.

در این Cell فقط دو Candidate زیر بررسی می‌شوند:

- `حامد زحمتکش` ↔ `زحمتکش`
- `محمدرضا منصوریان` ↔ `منصوریان`

اگر Evidence مستقیم تأیید شود، این رابطه به‌عنوان `SOURCE_OBSERVED_ALIAS` در حافظه Notebook ثبت می‌شود تا Parser بعدی بتواند هر دو Raw Label را حفظ کند و در کنار آن Canonical Speaker Label واحدی داشته باشد.

نکته مهم:

Raw Label هرگز حذف یا بازنویسی نمی‌شود. Canonicalization فقط یک Metadata ساختاری در خروجی Parser خواهد بود.

در این مرحله هنوز:

- Transcript Parse نمی‌شود.
- Artifact نوشته نمی‌شود.
- Speaker ناشناخته حدس زده نمی‌شود.
- Persian normalization انجام نمی‌شود.
- Deduplication انجام نمی‌شود.
- Chunking انجام نمی‌شود.

**معیار PASS:**

- هر دو جفت Candidate مستقیماً از Source بررسی شوند.
- Full Label و Short Label در همان فایل وجود داشته باشند.
- Short Label زیررشته Full Label باشد.
- Context هر occurrence قابل مشاهده باشد.
- Alias فقط در صورت تأیید تمام شروط به Registry موقت اضافه شود.

**معیار FAIL:**

اگر رابطه یکی از جفت‌ها با Evidence مستقیم Source قابل اثبات نباشد، آن جفت unresolved باقی می‌ماند و Parser حق Canonicalize کردن آن را نخواهد داشت.

</div>

In [16]:
# Code Cell 16

source_alias_candidates = [
    {
        "relative_path": (
            "data/transcripts/final/"
            "chehelstion_finup_ketabbaz_bankmellat/"
            "چهلستون 30.md"
        ),
        "full_label": "حامد زحمتکش",
        "short_label": "زحمتکش",
    },
    {
        "relative_path": (
            "data/transcripts/final/"
            "chehelstion_finup_ketabbaz_bankmellat/"
            "چهلستون 20.md"
        ),
        "full_label": "محمدرضا منصوریان",
        "short_label": "منصوریان",
    },
]


def collect_exact_prefix_occurrences(lines, target_prefix):
    occurrences = []

    for index, line in enumerate(lines):
        if bold_prefix_pattern.match(line):
            continue

        match = plain_colon_pattern.match(line)

        if not match:
            continue

        raw_prefix = match.group(1).strip()

        if raw_prefix != target_prefix:
            continue

        previous_index = None
        for candidate_index in range(index - 1, -1, -1):
            if lines[candidate_index].strip():
                previous_index = candidate_index
                break

        next_index = None
        for candidate_index in range(index + 1, len(lines)):
            if lines[candidate_index].strip():
                next_index = candidate_index
                break

        occurrences.append({
            "line_number": index + 1,
            "raw_line": line,
            "previous_nonempty_line": (
                lines[previous_index]
                if previous_index is not None
                else None
            ),
            "next_nonempty_line": (
                lines[next_index]
                if next_index is not None
                else None
            ),
        })

    return occurrences


source_observed_aliases = {}
alias_validation_failures = []

print("SOURCE-OBSERVED ALIAS INSPECTION")
print("=" * 100)

for candidate in source_alias_candidates:
    relative_path = candidate["relative_path"]
    full_label = candidate["full_label"]
    short_label = candidate["short_label"]

    file_path = project_root / relative_path
    lines = file_path.read_text(encoding="utf-8").splitlines()

    full_occurrences = collect_exact_prefix_occurrences(
        lines,
        full_label,
    )

    short_occurrences = collect_exact_prefix_occurrences(
        lines,
        short_label,
    )

    full_exists = bool(full_occurrences)
    short_exists = bool(short_occurrences)
    substring_relation = short_label in full_label
    explicit_file = relative_path in chehelston_explicit_paths

    validation = {
        "full_label_exists": full_exists,
        "short_label_exists": short_exists,
        "short_is_substring_of_full": substring_relation,
        "file_is_manifest_explicit_boundary": explicit_file,
    }

    accepted = all(validation.values())

    print()
    print("File:", relative_path)
    print("Full label:", full_label)
    print("Short label:", short_label)
    print("Validation:")
    for key, value in validation.items():
        print(f"  - {key}: {value}")

    print()
    print("Full-label occurrences:", len(full_occurrences))

    for occurrence in full_occurrences:
        print(
            f"  Line {occurrence['line_number']}: "
            f"{occurrence['raw_line']}"
        )
        print(
            "    Previous:",
            occurrence["previous_nonempty_line"],
        )
        print(
            "    Next:",
            occurrence["next_nonempty_line"],
        )

    print()
    print("Short-label occurrences:", len(short_occurrences))

    for occurrence in short_occurrences:
        print(
            f"  Line {occurrence['line_number']}: "
            f"{occurrence['raw_line']}"
        )
        print(
            "    Previous:",
            occurrence["previous_nonempty_line"],
        )
        print(
            "    Next:",
            occurrence["next_nonempty_line"],
        )

    if accepted:
        source_observed_aliases[short_label] = {
            "canonical_source_label": full_label,
            "evidence_type": "SOURCE_OBSERVED_ALIAS",
            "relative_path": relative_path,
            "full_label_occurrences": len(full_occurrences),
            "short_label_occurrences": len(short_occurrences),
        }

        print()
        print("Alias decision: ACCEPTED_FROM_SOURCE_EVIDENCE")
    else:
        alias_validation_failures.append({
            "relative_path": relative_path,
            "full_label": full_label,
            "short_label": short_label,
            "validation": validation,
        })

        print()
        print("Alias decision: UNRESOLVED")

    print("-" * 100)


print()
print("SOURCE-OBSERVED ALIAS REGISTRY")
print("=" * 100)

print(
    json.dumps(
        source_observed_aliases,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
)

print()
print("Validation failures:", len(alias_validation_failures))

assert not alias_validation_failures, (
    "At least one source-observed alias candidate could not be "
    f"validated: {alias_validation_failures}"
)

assert set(source_observed_aliases) == {
    "زحمتکش",
    "منصوریان",
}, (
    "Unexpected source-observed alias registry."
)

assert all(
    item["evidence_type"] == "SOURCE_OBSERVED_ALIAS"
    for item in source_observed_aliases.values()
), (
    "Source-observed aliases lack explicit evidence classification."
)

print()
print("SOURCE-OBSERVED SPEAKER ALIAS VALIDATION: PASS")

SOURCE-OBSERVED ALIAS INSPECTION

File: data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 30.md
Full label: حامد زحمتکش
Short label: زحمتکش
Validation:
  - full_label_exists: True
  - short_label_exists: True
  - short_is_substring_of_full: True
  - file_is_manifest_explicit_boundary: True

Full-label occurrences: 1
  Line 39: حامد زحمتکش: تقسیم‌بندی‌های مختلفی برای هنر وجود دارد؛ از هنرهای دیداری مثل نقاشی، خوشنویسی و مجسمه‌سازی تا موسیقی، نمایش، ادبیات، سینما و هنرهای ترکیبی.
    Previous: مجری: آقای زحمتکش، وقتی از هنر حرف می‌زنیم منظور چه حوزه‌هایی است؟
    Next: مهم‌تر از اسم دسته‌ها این است که هنر چگونه انسان را درگیر می‌کند. من خودم هنوز معلم هنر دوره راهنمایی‌ام را به یاد دارم؛ معلمی که با روش جذاب روی تخته برای ما نستعلیق می‌نوشت و همان تجربه باعث شد خط تحریری برایم جدی شود. این نشان می‌دهد یک تجربه هنری خوب ممکن است سال‌ها در شخصیت فرد بماند.

Short-label occurrences: 3
  Line 45: زحمتکش: اول «خلاقیت» است. کودک وقتی نقاشی، نمایش یا موسیقی تجربه می‌کند یاد م

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### Discovery زمان‌های واقعی موجود در Transcript Source

برای Segmentهای Transcript نباید Timestamp ساخته یا تخمین زده شود.

مدت‌زمان کلی Episode که در Header یا Metadata ثبت شده، با Timestamp یک Segment متفاوت است و نباید به‌عنوان زمان شروع یا پایان Turn استفاده شود.

در این مرحله فقط بررسی می‌شود آیا داخل **Body واقعی Transcript** نشانه زمانی صریحی وجود دارد یا خیر.

جست‌وجو به‌صورت محافظه‌کارانه انجام می‌شود و فقط قالب‌هایی بررسی می‌شوند که احتمالاً Marker زمانی ساختاری هستند، از جمله:

- `[00:12]`
- `[00:12:34]`
- `00:12`
- `00:12:34`
- Rangeهایی شبیه Subtitle مانند `00:12:34 --> 00:12:40`

جست‌وجو از Candidate شروع Body هر فایل آغاز می‌شود؛ بنابراین مدت‌زمان‌هایی که فقط در Header فایل قرار دارند وارد این Discovery نمی‌شوند.

وجود یک عدد شبیه ساعت داخل جمله عادی به‌تنهایی Timestamp محسوب نمی‌شود. Candidateها فقط زمانی ثبت می‌شوند که Pattern در ابتدای خط یا در قالب Marker مشخص قرار گرفته باشد.

در این Cell:

- Timestamp استخراج‌شده هنوز به Segment متصل نمی‌شود.
- هیچ زمان مفقودی تخمین زده نمی‌شود.
- Duration کلی Episode به Segment Timestamp تبدیل نمی‌شود.
- هیچ تبدیل زمانی بر اساس طول متن انجام نمی‌شود.

اگر Source فاقد Timestamp واقعی باشد، Parser نهایی باید:

- `start_time = None`
- `end_time = None`

را حفظ کند.

در این مرحله همچنان انجام نمی‌شوند:

- Transcript Parsing
- Speaker inference
- Persian normalization
- Deduplication
- Chunking
- Artifact write

**معیار PASS:**

- هر `183` فایل از Candidate Body به بعد بررسی شود.
- Timestamp Candidateهای واقعی در صورت وجود با مسیر و شماره خط نمایش داده شوند.
- Metadata durationهای Header وارد نتیجه نشوند.
- اگر هیچ Timestamp ساختاری وجود نداشت، این موضوع صریحاً اثبات شود.

**معیار FAIL:**

اگر فایل یا Programی از Scan حذف شود یا Pattern زمانی بدون امکان Inspection ثبت شود، طراحی Parser ادامه پیدا نمی‌کند.

</div>

In [17]:
# Code Cell 17

timestamp_patterns = {
    "SUBTITLE_RANGE": re.compile(
        r"^\s*(?:\[)?"
        r"(\d{1,2}:\d{2}(?::\d{2})?(?:[.,]\d{1,3})?)"
        r"(?:\])?\s*-->\s*(?:\[)?"
        r"(\d{1,2}:\d{2}(?::\d{2})?(?:[.,]\d{1,3})?)"
        r"(?:\])?"
    ),
    "BRACKETED_TIMESTAMP": re.compile(
        r"^\s*\["
        r"(\d{1,2}:\d{2}(?::\d{2})?)"
        r"\]\s*"
    ),
    "LEADING_TIMESTAMP": re.compile(
        r"^\s*"
        r"(\d{1,2}:\d{2}(?::\d{2})?)"
        r"\s+(?=\S)"
    ),
    "LEADING_TIMESTAMP_DASH": re.compile(
        r"^\s*"
        r"(\d{1,2}:\d{2}(?::\d{2})?)"
        r"\s*[-–—|]\s*(?=\S)"
    ),
}


def valid_timestamp_token(token):
    clean_token = token.replace(",", ".")
    main_part = clean_token.split(".", 1)[0]
    pieces = main_part.split(":")

    if len(pieces) == 2:
        minutes, seconds = pieces

        return (
            minutes.isdigit()
            and seconds.isdigit()
            and 0 <= int(seconds) <= 59
        )

    if len(pieces) == 3:
        hours, minutes, seconds = pieces

        return (
            hours.isdigit()
            and minutes.isdigit()
            and seconds.isdigit()
            and 0 <= int(minutes) <= 59
            and 0 <= int(seconds) <= 59
        )

    return False


timestamp_candidates = []
timestamp_file_profiles = []

for record in accepted_files:
    relative_path = record["relative_path"]
    candidate = boundary_candidate_by_path[relative_path]

    file_path = project_root / relative_path
    lines = file_path.read_text(encoding="utf-8").splitlines()

    body_start_index = candidate["candidate_line_number"] - 1
    file_candidates = []

    for index in range(body_start_index, len(lines)):
        line = lines[index]

        if not line.strip():
            continue

        matched_type = None
        matched_tokens = None

        range_match = timestamp_patterns[
            "SUBTITLE_RANGE"
        ].match(line)

        if range_match:
            tokens = [
                range_match.group(1),
                range_match.group(2),
            ]

            if all(valid_timestamp_token(token) for token in tokens):
                matched_type = "SUBTITLE_RANGE"
                matched_tokens = tokens

        if matched_type is None:
            for pattern_name in (
                "BRACKETED_TIMESTAMP",
                "LEADING_TIMESTAMP_DASH",
                "LEADING_TIMESTAMP",
            ):
                match = timestamp_patterns[pattern_name].match(line)

                if not match:
                    continue

                token = match.group(1)

                if valid_timestamp_token(token):
                    matched_type = pattern_name
                    matched_tokens = [token]
                    break

        if matched_type is not None:
            item = {
                "program": record["program"],
                "program_id": record["program_id"],
                "relative_path": relative_path,
                "line_number": index + 1,
                "pattern_type": matched_type,
                "tokens": matched_tokens,
                "raw_line": line,
            }

            timestamp_candidates.append(item)
            file_candidates.append(item)

    timestamp_file_profiles.append({
        "program": record["program"],
        "program_id": record["program_id"],
        "relative_path": relative_path,
        "candidate_count": len(file_candidates),
    })


print("Files scanned:", len(timestamp_file_profiles))
print(
    "Programs represented:",
    len({
        item["program_id"]
        for item in timestamp_file_profiles
    }),
)
print(
    "Files with timestamp candidates:",
    sum(
        profile["candidate_count"] > 0
        for profile in timestamp_file_profiles
    ),
)
print(
    "Files without timestamp candidates:",
    sum(
        profile["candidate_count"] == 0
        for profile in timestamp_file_profiles
    ),
)
print("Total timestamp candidates:", len(timestamp_candidates))
print()

pattern_counts = Counter(
    item["pattern_type"]
    for item in timestamp_candidates
)

print("Timestamp candidate pattern counts:")

if pattern_counts:
    for pattern_type, count in sorted(pattern_counts.items()):
        print(f"- {pattern_type}: {count}")
else:
    print("- NONE")

print()
print("TIMESTAMP COVERAGE BY PROGRAM")
print("=" * 100)

for program_id in sorted(files_by_program):
    profiles = [
        profile
        for profile in timestamp_file_profiles
        if profile["program_id"] == program_id
    ]

    program_name = profiles[0]["program"]

    files_with_candidates = sum(
        profile["candidate_count"] > 0
        for profile in profiles
    )

    total_candidates = sum(
        profile["candidate_count"]
        for profile in profiles
    )

    print()
    print("Program:", program_name)
    print("Program ID:", program_id)
    print("Files:", len(profiles))
    print(
        "Files with timestamp candidates:",
        files_with_candidates,
    )
    print(
        "Total timestamp candidates:",
        total_candidates,
    )
    print("-" * 100)


print()
print("TIMESTAMP CANDIDATE DETAILS")
print("=" * 100)

if timestamp_candidates:
    for item in timestamp_candidates:
        display_line = (
            item["raw_line"]
            if len(item["raw_line"]) <= 260
            else item["raw_line"][:260] + "..."
        )

        print()
        print("Program:", item["program"])
        print("File:", item["relative_path"])
        print("Line:", item["line_number"])
        print("Pattern:", item["pattern_type"])
        print("Tokens:", item["tokens"])
        print("Raw line:", display_line)
        print("-" * 100)
else:
    print("NO STRUCTURAL TIMESTAMP CANDIDATES FOUND IN TRANSCRIPT BODIES")


assert len(timestamp_file_profiles) == len(accepted_files), (
    "Not all accepted files were scanned for timestamps."
)

assert {
    item["program_id"]
    for item in timestamp_file_profiles
} == {
    record["program_id"]
    for record in program_records
}, (
    "Timestamp discovery does not cover all programs."
)

print()
print("STRUCTURAL TIMESTAMP DISCOVERY: PASS")

Files scanned: 183
Programs represented: 13
Files with timestamp candidates: 0
Files without timestamp candidates: 183
Total timestamp candidates: 0

Timestamp candidate pattern counts:
- NONE

TIMESTAMP COVERAGE BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Files: 26
Files with timestamp candidates: 0
Total timestamp candidates: 0
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Files: 11
Files with timestamp candidates: 0
Total timestamp candidates: 0
----------------------------------------------------------------------------------------------------

Program: چهلستون
Program ID: program:chehelston
Files: 38
Files with timestamp candidates: 0
Total timestamp candidates: 0
----------------------------------------------------------------------------------------------------

Program: در جستجوی خوشبختی
Program ID: program:dar-jostojo-ye-khoshbakhti
Files: 29
Files with 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### Freeze قرارداد Policy-aware Parser پیش از اجرای Parse

Discoveryهای انجام‌شده اکنون برای تعریف رفتار Parser کافی هستند.

این Cell هنوز Transcript را Parse نمی‌کند؛ فقط قرارداد اجرایی Parser را از Evidenceهای تأییدشده می‌سازد و اعتبارسنجی می‌کند.

شش Speaker Policy موجود در Corpus باید رفتار متفاوت داشته باشند.

#### `HUMAN_CONFIRMED_SOLO`

Manifest صریحاً اجازه داده است:

`default_body_attribution_allowed = true`

بنابراین Body این فایل‌ها می‌تواند به `primary_speaker` ثبت‌شده در Manifest نسبت داده شود.

وجود Bold تکراری در برخی فایل‌های Solo، مانند `در جستجوی خوشبختی`، Marker فرمت Source است و نباید Speaker جدید ایجاد کند.

---

#### `EXPLICIT_MULTI_SPEAKER`

Speaker فقط زمانی تغییر می‌کند که Marker صریح و از قبل تأییدشده در Source دیده شود.

سه خانواده اصلی داریم:

- `عصر شیرین` → Bold-only Speaker Marker
- `خانه تو` → Bold-only Speaker Marker
- `یونی تهران` → Bold-only Speaker Marker

Bold-onlyهایی که محتوای آموزشی هستند، مانند:

- `دارایی = سرمایه + بدهی`
- `درآمد، هزینه، سود، دارایی، بدهی و سرمایه.`

Speaker Marker نیستند و باید به Speaker جاری تعلق داشته باشند.

---

#### `MIXED_EXPLICIT_AND_CONTEXTUAL`

برای `چهلستون` دو رفتار داریم:

- `35` فایل Explicit → Markerهای `Speaker: text`
- `3` فایل Contextual → هیچ Speaker Boundary حدسی ساخته نمی‌شود.

سه فایل Contextual شناخته‌شده:

- `چهلستون 1`
- `چهلستون 2`
- `چهلستون 38`

در این سه فایل Body حفظ می‌شود ولی Speaker Boundary unresolved باقی می‌ماند.

---

#### `KNOWN_PARTICIPANTS_UNLABELED_TURNS`

در `کتاب باز` Participantها مشخص‌اند اما Turn Boundaryها Label ندارند.

بنابراین:

- متن کامل حفظ می‌شود.
- Participant list حفظ می‌شود.
- Speaker هر Segment از روی ترتیب سؤال و جواب حدس زده نمی‌شود.

---

#### `PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED`

در `اقیانوس آرام` و `شراکت` Speaker اصلی در Manifest شناخته شده است، اما:

`default_body_attribution_allowed = false`

بنابراین دانستن Primary Speaker مجوز نسبت‌دادن تمام Body به او نیست.

Speaker Boundary unresolved باقی می‌ماند.

---

#### `NARRATIVE_AUDIENCE_CUES`

در `میزان` Cueهای روایی صریح مانند:

- `یکی از حاضران`
- `یکی از والدین`
- `یکی می‌پرسد`

قابل حفظ‌اند.

این Cueها هویت واقعی Speaker نیستند و نباید به نام شخص تبدیل شوند.

---

### قرارداد Timestamp

Discovery تمام `183` فایل نشان داد:

`0 structural timestamp candidates`

بنابراین:

- `start_time = None`
- `end_time = None`

و هیچ زمان مصنوعی ساخته نمی‌شود.

---

### قرارداد Alias

دو نوع Alias مجاز است:

1. Alias ثبت‌شده در Manifest
2. Alias تأییدشده با Evidence مستقیم از همان Source

Aliasهای Source-observed تأییدشده فعلی:

- `زحمتکش` → `حامد زحمتکش`
- `منصوریان` → `محمدرضا منصوریان`

هیچ Alias دیگری از روی شباهت ظاهری ساخته نمی‌شود.

---

در این Cell یک `parser_contract_v1` فقط در حافظه Notebook ساخته می‌شود.

هنوز:

- Parsed Segment ساخته نمی‌شود.
- Artifact نوشته نمی‌شود.
- Persian normalization انجام نمی‌شود.
- Deduplication انجام نمی‌شود.
- Chunking انجام نمی‌شود.

**معیار PASS:**

- هر شش Speaker Policy قرارداد مشخص داشته باشد.
- هر `13` Program به یک Strategy معتبر متصل شود.
- سه فایل Contextual چهلستون صریحاً در قرارداد ثبت شوند.
- Timestamp policy برابر `PRESERVE_REAL_ELSE_NONE` باشد.
- Unknown-speaker inference برای تمام Strategyها ممنوع باشد.
- هیچ Program بدون Parsing Strategy باقی نماند.

</div>

In [18]:
# Code Cell 18

explicit_bold_marker_programs = {
    "program:asre-shirin",
    "program:khane-to",
    "program:uni-tehran",
}

chehelston_contextual_path_set = set(
    chehelston_contextual_paths
)

manifest_aliases_by_program = {
    record["program_id"]: dict(record.get("speaker_aliases", {}))
    for record in program_records
}

parser_contract_v1 = {
    "contract_version": "transcript_parser_contract_v1",
    "timestamp_policy": "PRESERVE_REAL_ELSE_NONE",
    "missing_timestamp_value": None,
    "unknown_speaker_inference_allowed": False,
    "preserve_raw_text": True,
    "persian_normalization_allowed": False,
    "deduplication_allowed": False,
    "chunking_allowed": False,
    "policies": {
        "HUMAN_CONFIRMED_SOLO": {
            "strategy": "SOLO_MANIFEST_ATTRIBUTION",
            "default_body_attribution_required": True,
            "speaker_boundary_source": "MANIFEST_POLICY",
            "preserve_other_turn_context": False,
            "allow_speaker_guessing": False,
        },
        "EXPLICIT_MULTI_SPEAKER": {
            "strategy": "EXPLICIT_MARKER_TURNS",
            "default_body_attribution_required": False,
            "speaker_boundary_source": "VERIFIED_SOURCE_MARKERS",
            "preserve_other_turn_context": True,
            "allow_speaker_guessing": False,
        },
        "MIXED_EXPLICIT_AND_CONTEXTUAL": {
            "strategy": "FILE_LEVEL_EXPLICIT_OR_UNRESOLVED",
            "default_body_attribution_required": False,
            "speaker_boundary_source": (
                "CHEHELSTON_EXPLICIT_MARKERS_OR_UNRESOLVED"
            ),
            "preserve_other_turn_context": True,
            "allow_speaker_guessing": False,
        },
        "KNOWN_PARTICIPANTS_UNLABELED_TURNS": {
            "strategy": "UNLABELED_TURNS_PRESERVE_CONTEXT",
            "default_body_attribution_required": False,
            "speaker_boundary_source": "UNRESOLVED",
            "preserve_other_turn_context": True,
            "allow_speaker_guessing": False,
        },
        "PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED": {
            "strategy": "PRIMARY_KNOWN_BOUNDARIES_UNRESOLVED",
            "default_body_attribution_required": False,
            "speaker_boundary_source": "UNRESOLVED",
            "preserve_other_turn_context": True,
            "allow_speaker_guessing": False,
        },
        "NARRATIVE_AUDIENCE_CUES": {
            "strategy": "NARRATIVE_CUES_WITH_UNRESOLVED_DEFAULT",
            "default_body_attribution_required": False,
            "speaker_boundary_source": "REGISTERED_NARRATIVE_CUES_ONLY",
            "preserve_other_turn_context": True,
            "allow_speaker_guessing": False,
        },
    },
    "programs": {},
    "chehelston": {
        "explicit_files": sorted(
            chehelston_explicit_paths
        ),
        "contextual_files": sorted(
            chehelston_contextual_paths
        ),
        "verified_marker_candidates": sorted(
            verified_chehelston_markers
        ),
        "source_observed_aliases": source_observed_aliases,
    },
}


for program_record in program_records:
    program_id = program_record["program_id"]
    program_policy = program_record["speaker_policy"]
    policy_name = program_policy["policy"]

    if policy_name == "HUMAN_CONFIRMED_SOLO":
        program_strategy = "SOLO_MANIFEST_ATTRIBUTION"

    elif policy_name == "EXPLICIT_MULTI_SPEAKER":
        assert program_id in explicit_bold_marker_programs, (
            "Unexpected explicit multi-speaker program without "
            f"a verified marker family: {program_id}"
        )
        program_strategy = "BOLD_ONLY_VERIFIED_MARKERS"

    elif policy_name == "MIXED_EXPLICIT_AND_CONTEXTUAL":
        assert program_id == chehelston_program_id, (
            "Unexpected MIXED_EXPLICIT_AND_CONTEXTUAL program."
        )
        program_strategy = "CHEHELSTON_FILE_LEVEL_STRATEGY"

    elif policy_name == "KNOWN_PARTICIPANTS_UNLABELED_TURNS":
        program_strategy = "UNLABELED_TURNS_UNRESOLVED"

    elif policy_name == (
        "PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED"
    ):
        program_strategy = "PRIMARY_KNOWN_UNRESOLVED"

    elif policy_name == "NARRATIVE_AUDIENCE_CUES":
        program_strategy = "REGISTERED_NARRATIVE_CUES"

    else:
        raise AssertionError(
            f"Unsupported speaker policy: {policy_name}"
        )

    parser_contract_v1["programs"][program_id] = {
        "program": program_record["program"],
        "speaker_policy": policy_name,
        "parser_strategy": program_strategy,
        "primary_speaker": program_policy.get(
            "primary_speaker"
        ),
        "default_body_attribution_allowed": (
            program_policy[
                "default_body_attribution_allowed"
            ]
        ),
        "preserve_other_turn_context": (
            program_policy[
                "preserve_other_turn_context"
            ]
        ),
        "unknown_speaker_inference_allowed": (
            program_policy[
                "unknown_speaker_inference_allowed"
            ]
        ),
        "manifest_speaker_aliases": (
            manifest_aliases_by_program[program_id]
        ),
    }


policy_names_in_corpus = {
    record["speaker_policy"]
    for record in accepted_files
}

contract_policy_names = set(
    parser_contract_v1["policies"]
)

registered_program_ids = set(
    parser_contract_v1["programs"]
)

accepted_program_ids = {
    record["program_id"]
    for record in accepted_files
}


print("Parser contract version:", parser_contract_v1["contract_version"])
print("Policies in corpus:", len(policy_names_in_corpus))
print("Policies in contract:", len(contract_policy_names))
print("Programs in contract:", len(registered_program_ids))
print(
    "Timestamp policy:",
    parser_contract_v1["timestamp_policy"],
)
print(
    "Missing timestamp value:",
    parser_contract_v1["missing_timestamp_value"],
)
print(
    "Unknown speaker inference allowed:",
    parser_contract_v1[
        "unknown_speaker_inference_allowed"
    ],
)
print()

print("PROGRAM PARSER STRATEGIES")
print("=" * 100)

for program_id in sorted(parser_contract_v1["programs"]):
    contract = parser_contract_v1["programs"][program_id]

    print()
    print("Program:", contract["program"])
    print("Program ID:", program_id)
    print("Speaker policy:", contract["speaker_policy"])
    print("Parser strategy:", contract["parser_strategy"])
    print(
        "Default body attribution allowed:",
        contract["default_body_attribution_allowed"],
    )
    print(
        "Preserve other-turn context:",
        contract["preserve_other_turn_context"],
    )
    print(
        "Unknown speaker inference allowed:",
        contract["unknown_speaker_inference_allowed"],
    )
    print(
        "Manifest aliases:",
        contract["manifest_speaker_aliases"],
    )
    print("-" * 100)


print()
print("CHEHELSTON FILE-LEVEL CONTRACT")
print("=" * 100)
print(
    "Explicit files:",
    len(parser_contract_v1["chehelston"]["explicit_files"]),
)
print(
    "Contextual files:",
    len(parser_contract_v1["chehelston"]["contextual_files"]),
)
print("Contextual paths:")

for path in parser_contract_v1["chehelston"]["contextual_files"]:
    print(" -", path)

print()
print(
    "Verified marker candidates:",
    len(
        parser_contract_v1[
            "chehelston"
        ]["verified_marker_candidates"]
    ),
)
print(
    "Source-observed aliases:",
    json.dumps(
        parser_contract_v1[
            "chehelston"
        ]["source_observed_aliases"],
        ensure_ascii=False,
        sort_keys=True,
    ),
)


assert policy_names_in_corpus == contract_policy_names, (
    "Parser contract policy coverage does not match Corpus."
)

assert registered_program_ids == accepted_program_ids, (
    "Parser contract program coverage does not match Accepted Corpus."
)

assert len(registered_program_ids) == 13, (
    "Parser contract must cover exactly 13 programs."
)

assert len(
    parser_contract_v1["chehelston"]["explicit_files"]
) == 35, (
    "Chehelston explicit-file contract must contain 35 files."
)

assert len(
    parser_contract_v1["chehelston"]["contextual_files"]
) == 3, (
    "Chehelston contextual-file contract must contain 3 files."
)

assert all(
    contract["unknown_speaker_inference_allowed"] is False
    for contract in parser_contract_v1["programs"].values()
), (
    "At least one program contract allows unknown speaker inference."
)

assert parser_contract_v1["timestamp_policy"] == (
    "PRESERVE_REAL_ELSE_NONE"
), (
    "Unexpected timestamp policy."
)

assert parser_contract_v1["missing_timestamp_value"] is None, (
    "Missing timestamps must remain None."
)

assert parser_contract_v1["persian_normalization_allowed"] is False
assert parser_contract_v1["deduplication_allowed"] is False
assert parser_contract_v1["chunking_allowed"] is False

print()
print("POLICY-AWARE PARSER CONTRACT: PASS")

Parser contract version: transcript_parser_contract_v1
Policies in corpus: 6
Policies in contract: 6
Programs in contract: 13
Timestamp policy: PRESERVE_REAL_ELSE_NONE
Missing timestamp value: None
Unknown speaker inference allowed: False

PROGRAM PARSER STRATEGIES

Program: عصر شیرین
Program ID: program:asre-shirin
Speaker policy: EXPLICIT_MULTI_SPEAKER
Parser strategy: BOLD_ONLY_VERIFIED_MARKERS
Default body attribution allowed: False
Preserve other-turn context: True
Unknown speaker inference allowed: False
Manifest aliases: {}
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Speaker policy: HUMAN_CONFIRMED_SOLO
Parser strategy: SOLO_MANIFEST_ATTRIBUTION
Default body attribution allowed: True
Preserve other-turn context: False
Unknown speaker inference allowed: False
Manifest aliases: {}
-----------------------------------------------------------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### Freeze کردن Registry دقیق Bold-only Speaker Markerها

سه Program از Strategy زیر استفاده خواهند کرد:

`BOLD_ONLY_VERIFIED_MARKERS`

این Programها عبارت‌اند از:

- `عصر شیرین`
- `خانه تو`
- `یونی تهران`

Discovery قبلی نشان داد که `Bold-only` به‌تنهایی به معنی Speaker نیست.

در `عصر شیرین` دو مقدار صریحاً به‌عنوان **Instructional Bold Content** مشاهده شدند:

- `دارایی = سرمایه + بدهی`
- `درآمد، هزینه، سود، دارایی، بدهی و سرمایه.`

این دو مقدار Speaker نیستند و باید به Turn جاری تعلق داشته باشند.

در مقابل، سایر Bold-onlyهای مشاهده‌شده در همین Program در Inspection قبلی به‌صورت Markerهای Turn دیده شدند؛ مانند:

- `دکتر کمیل رودی`
- `خانم متولیان`
- `شرکت‌کننده`
- `یکی از شرکت‌کنندگان`
- Labelهای شماره‌دار شرکت‌کنندگان

در `خانه تو` و `یونی تهران` نیز تمام Bold-onlyهای مشاهده‌شده در Source Inspection نقش Turn Label داشتند.

در این Cell، Registry فقط از **مقادیر واقعاً مشاهده‌شده در Corpus فعلی** ساخته می‌شود.

هیچ Pattern عمومی مانند:

`هر چیزی که Bold است = Speaker`

تعریف نمی‌شود.

بنابراین اگر در آینده Bold-only جدیدی وارد Source شود که در Registry فعلی وجود ندارد، Parser نباید خودکار آن را Speaker اعلام کند.

### قرارداد حفظ Label

برای هر Speaker Marker دو مفهوم از هم جدا خواهند بود:

- `speaker_raw`  
  همان Label دقیق موجود در Source.

- `speaker_canonical`  
  فقط در صورتی مقدار می‌گیرد که Evidence ثبت‌شده برای Canonicalization وجود داشته باشد.

در این مرحله فقط Registry خام Markerها Freeze می‌شود و هنوز Canonicalization کامل انجام نمی‌شود.

Aliasهای Manifest بعداً روی این Raw Labelها اعمال خواهند شد.

مخصوصاً:

`استاد → Dr. Komeil Roudi`

در `یونی تهران` از Manifest آمده است و معتبر است.

اما Labelهایی مانند:

- `مجری`
- `گوینده ۱`
- `گوینده ۲`

نباید به فرد مشخصی تبدیل شوند.

همچنان انجام نمی‌شوند:

- Parse Transcript
- Guess Speaker Identity
- Persian normalization
- Deduplication
- Chunking
- Artifact write

**معیار PASS:**

- فقط سه Program مورد انتظار Registry داشته باشند.
- تمام Bold-onlyهای مشاهده‌شده در آن‌ها یا `VERIFIED_SPEAKER_MARKER` باشند یا `INSTRUCTIONAL_BOLD_CONTENT`.
- دو Bold آموزشی عصر شیرین صریحاً از Speaker Registry خارج شوند.
- هیچ Bold-only مشاهده‌شده بدون Classification باقی نماند.
- `در جستجوی خوشبختی` وارد این Registry نشود، چون Strategy آن `HUMAN_CONFIRMED_SOLO` است.

</div>

In [19]:
# Code Cell 19

bold_marker_program_ids = {
    "program:asre-shirin",
    "program:khane-to",
    "program:uni-tehran",
}

instructional_bold_content_by_program = {
    "program:asre-shirin": {
        "دارایی = سرمایه + بدهی",
        "درآمد، هزینه، سود، دارایی، بدهی و سرمایه.",
    },
    "program:khane-to": set(),
    "program:uni-tehran": set(),
}

observed_bold_values_by_program = {
    program_id: set(
        bold_values_by_program[program_id].keys()
    )
    for program_id in bold_marker_program_ids
}

verified_bold_speaker_markers = {}

classification_failures = []

for program_id in sorted(bold_marker_program_ids):
    observed_values = observed_bold_values_by_program[
        program_id
    ]

    instructional_values = (
        instructional_bold_content_by_program[
            program_id
        ]
    )

    missing_instructional_values = (
        instructional_values - observed_values
    )

    if missing_instructional_values:
        classification_failures.append({
            "program_id": program_id,
            "reason": "EXPECTED_INSTRUCTIONAL_BOLD_NOT_OBSERVED",
            "values": sorted(missing_instructional_values),
        })

    speaker_markers = (
        observed_values - instructional_values
    )

    verified_bold_speaker_markers[
        program_id
    ] = sorted(speaker_markers)


print("Bold-marker programs:", len(bold_marker_program_ids))
print()

print("VERIFIED BOLD-ONLY REGISTRY")
print("=" * 100)

for program_id in sorted(bold_marker_program_ids):
    program_name = program_registry_by_id[
        program_id
    ]["program"]

    observed_values = sorted(
        observed_bold_values_by_program[
            program_id
        ]
    )

    instructional_values = sorted(
        instructional_bold_content_by_program[
            program_id
        ]
    )

    speaker_markers = (
        verified_bold_speaker_markers[
            program_id
        ]
    )

    classified_values = (
        set(instructional_values)
        | set(speaker_markers)
    )

    unclassified_values = (
        set(observed_values) - classified_values
    )

    print()
    print("Program:", program_name)
    print("Program ID:", program_id)
    print(
        "Observed unique Bold-only values:",
        len(observed_values),
    )
    print(
        "Verified speaker markers:",
        len(speaker_markers),
    )
    print(
        "Instructional Bold values:",
        len(instructional_values),
    )
    print(
        "Unclassified Bold-only values:",
        len(unclassified_values),
    )

    print()
    print("Verified speaker markers:")

    if speaker_markers:
        for marker in speaker_markers:
            count = bold_values_by_program[
                program_id
            ][marker]

            print(
                f"  - {count:4d} × {marker}"
            )
    else:
        print("  NONE")

    print()
    print("Instructional Bold content:")

    if instructional_values:
        for value in instructional_values:
            count = bold_values_by_program[
                program_id
            ][value]

            print(
                f"  - {count:4d} × {value}"
            )
    else:
        print("  NONE")

    print("-" * 100)


bold_only_classification = {}

for program_id in sorted(bold_marker_program_ids):
    for value in observed_bold_values_by_program[
        program_id
    ]:
        if value in instructional_bold_content_by_program[
            program_id
        ]:
            classification = "INSTRUCTIONAL_BOLD_CONTENT"
        elif value in verified_bold_speaker_markers[
            program_id
        ]:
            classification = "VERIFIED_SPEAKER_MARKER"
        else:
            classification = "UNCLASSIFIED"

        bold_only_classification[
            (program_id, value)
        ] = classification


unclassified_bold_values = [
    {
        "program_id": program_id,
        "raw_value": raw_value,
    }
    for (program_id, raw_value), classification
    in bold_only_classification.items()
    if classification == "UNCLASSIFIED"
]


print()
print("GLOBAL CLASSIFICATION SUMMARY")
print("=" * 100)

classification_counts = Counter(
    bold_only_classification.values()
)

for classification, count in sorted(
    classification_counts.items()
):
    print(
        f"- {classification}: {count}"
    )

print(
    "Classification failures:",
    len(classification_failures),
)
print(
    "Unclassified Bold-only values:",
    len(unclassified_bold_values),
)


assert not classification_failures, (
    "Bold-only registry classification failures found: "
    f"{classification_failures}"
)

assert not unclassified_bold_values, (
    "Unclassified Bold-only values remain: "
    f"{unclassified_bold_values}"
)

assert set(
    verified_bold_speaker_markers
) == bold_marker_program_ids, (
    "Verified Bold marker registry program coverage is incorrect."
)

assert instructional_bold_content_by_program[
    "program:asre-shirin"
] == {
    "دارایی = سرمایه + بدهی",
    "درآمد، هزینه، سود، دارایی، بدهی و سرمایه.",
}, (
    "Asre Shirin instructional Bold exclusion set changed unexpectedly."
)

assert "program:dar-jostojo-ye-khoshbakhti" not in (
    verified_bold_speaker_markers
), (
    "Solo Dar Jostojo program must not use the explicit Bold-marker registry."
)

assert parser_contract_v1["programs"][
    "program:uni-tehran"
]["manifest_speaker_aliases"].get(
    "استاد"
) == "Dr. Komeil Roudi", (
    "Expected Manifest alias for Uni Tehran استاد is missing."
)

print()
print("VERIFIED BOLD-ONLY SPEAKER MARKER REGISTRY: PASS")

Bold-marker programs: 3

VERIFIED BOLD-ONLY REGISTRY

Program: عصر شیرین
Program ID: program:asre-shirin
Observed unique Bold-only values: 26
Verified speaker markers: 24
Instructional Bold values: 2
Unclassified Bold-only values: 0

Verified speaker markers:
  -  213 × خانم متولیان
  -  440 × دکتر کمیل رودی
  -   70 × شرکت‌کننده
  -    9 × شرکت‌کننده 1
  -    2 × شرکت‌کننده 10
  -    6 × شرکت‌کننده 2
  -    5 × شرکت‌کننده 3
  -    8 × شرکت‌کننده 4
  -   10 × شرکت‌کننده 5
  -    1 × شرکت‌کننده 6
  -    2 × شرکت‌کننده 7
  -    4 × شرکت‌کننده 8
  -    3 × شرکت‌کننده 9
  -    9 × شرکت‌کننده ۱
  -    6 × شرکت‌کننده ۲
  -    8 × شرکت‌کننده ۳
  -    3 × شرکت‌کننده ۴
  -    4 × شرکت‌کننده ۵
  -    3 × شرکت‌کننده ۶
  -    1 × شرکت‌کننده ۷
  -    2 × شرکت‌کنندگان
  -    1 × چند شرکت‌کننده
  -   73 × یکی از شرکت‌کنندگان
  -   19 × یکی دیگر از شرکت‌کنندگان

Instructional Bold content:
  -    2 × دارایی = سرمایه + بدهی
  -    1 × درآمد، هزینه، سود، دارایی، بدهی و سرمایه.
--------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### اجرای Dry Run اولیه Policy-aware Transcript Parser

تمام Discoveryهای لازم برای اجرای نسخه اولیه Parser اکنون انجام شده‌اند.

در این مرحله Parser برای اولین بار روی تمام `183` Accepted Transcript اجرا می‌شود، اما خروجی فقط در حافظه Notebook باقی می‌ماند.

هنوز هیچ Artifact روی Disk نوشته نمی‌شود.

Parser بر اساس قرارداد Freeze‌شده رفتار می‌کند.

### رفتار فایل‌های Solo

برای Policy:

`HUMAN_CONFIRMED_SOLO`

Manifest اجازه Attribution داده است.

بنابراین هر Paragraph محتوایی به:

`primary_speaker`

ثبت‌شده در Manifest نسبت داده می‌شود.

در فایل‌هایی مانند `در جستجوی خوشبختی`، Bold تکراری Speaker به‌عنوان Marker ساختاری حذف می‌شود و متن بعدی همچنان با Attribution تأییدشده Manifest ثبت می‌شود.

---

### رفتار Multi-speakerهای Bold-based

در:

- عصر شیرین
- خانه تو
- یونی تهران

فقط Markerهای موجود در Registry تأییدشده می‌توانند Speaker Turn جدید ایجاد کنند.

Instructional Boldهای عصر شیرین Speaker را تغییر نمی‌دهند و به متن Turn جاری تعلق دارند.

---

### رفتار چهلستون Explicit

در `35` فایل Explicit، فقط Prefixهای تأییدشده می‌توانند Marker Speaker باشند.

فرمت:

`Speaker: text`

به یک Turn تبدیل می‌شود و Paragraphهای بعدی تا Marker معتبر بعدی در همان Turn باقی می‌مانند.

اگر قبل از اولین Marker متن مقدمه یا Narration وجود داشته باشد، آن متن حذف نمی‌شود و با وضعیت:

`UNRESOLVED`

حفظ می‌شود.

---

### رفتار سه فایل Contextual چهلستون

در:

- چهلستون 1
- چهلستون 2
- چهلستون 38

هیچ Speaker Boundary ساخته نمی‌شود.

Paragraphها حفظ می‌شوند ولی Speaker آنها:

`UNRESOLVED`

است.

---

### رفتار کتاب باز، اقیانوس آرام و شراکت

به دلیل نبود Boundary صریح، Paragraphها حفظ می‌شوند اما Speaker از روی ترتیب سؤال و جواب حدس زده نمی‌شود.

---

### رفتار میزان

Cueهای ثبت‌شده مانند:

- `یکی از حاضران`
- `یکی از والدین`
- `یکی می‌پرسد`

به‌عنوان `speaker_raw` روایی حفظ می‌شوند.

این Cueها هویت انسانی Canonical محسوب نمی‌شوند.

سایر Paragraphهای بدون Cue همچنان unresolved باقی می‌مانند.

---

### فیلدهای اصلی هر Parsed Utterance

در Dry Run فعلی هر واحد شامل اطلاعات زیر است:

- `utterance_id`
- `utterance_index`
- `program`
- `program_id`
- `physical_file_id`
- `transcript_unit_id`
- `relative_path`
- `speaker_raw`
- `speaker_canonical`
- `speaker_source_canonical`
- `speaker_status`
- `boundary_type`
- `text`
- `source_text`
- `source_line_start`
- `source_line_end`
- `start_time`
- `end_time`

`source_text` برای Audit، متن Source مربوط به همان واحد را نگه می‌دارد.

### Canonicalization

`speaker_canonical` فقط زمانی مقدار می‌گیرد که Manifest Evidence داشته باشیم یا خود Manifest Attribution را مجاز کرده باشد.

Aliasهای Source-observed چهلستون در فیلد جداگانه:

`speaker_source_canonical`

ثبت می‌شوند.

بنابراین مثلاً:

`زحمتکش`

می‌تواند:

`speaker_source_canonical = حامد زحمتکش`

داشته باشد، بدون اینکه هویت دیگری از روی حدس ساخته شود.

---

### Timestamp

با توجه به Discovery قبلی:

- `start_time = None`
- `end_time = None`

برای تمام Parsed Utteranceها الزامی است.

---

این مرحله هنوز Merge کردن Utteranceهای کوچک نیست.

Merge احتمالی داخل همان Speaker Boundary بعداً و پس از Validation این Parse انجام خواهد شد.

همچنان انجام نمی‌شوند:

- Persian normalization
- Deduplication
- Chunking
- Speaker guessing
- Timestamp fabrication
- Artifact write

**معیار PASS:**

- هر `183` فایل Parse شود.
- هیچ Parsed Utterance متن خالی نداشته باشد.
- هیچ Timestamp مصنوعی ایجاد نشود.
- تمام فایل‌های Solo فقط با Attribution مجاز Manifest ثبت شوند.
- سه فایل Contextual چهلستون Speaker حدسی نداشته باشند.
- فایل‌های Special-policy Speaker Canonical حدسی نداشته باشند.
- Markerهای Explicit خارج از Registry پذیرفته نشوند.
- هیچ Accepted File بدون Parsed Output باقی نماند.

این PASS فقط صحت ساختاری Dry Run را تأیید می‌کند؛ حفظ کامل محتوای Source در Cell بعدی مستقلاً Revalidate خواهد شد.

</div>

In [20]:
# Code Cell 20

final_chehelston_speaker_markers = set(
    verified_chehelston_markers
)

for raw_alias, alias_record in source_observed_aliases.items():
    final_chehelston_speaker_markers.add(raw_alias)
    final_chehelston_speaker_markers.add(
        alias_record["canonical_source_label"]
    )


def extract_bold_only_value(line):
    match = bold_only_pattern.fullmatch(line)

    if not match:
        return None

    if match.lastindex:
        return match.group(1).strip()

    stripped = line.strip()

    if stripped.startswith("**") and stripped.endswith("**"):
        return stripped[2:-2].strip()

    return None


def resolve_speaker_labels(program_id, raw_label):
    if raw_label is None:
        return {
            "speaker_canonical": None,
            "speaker_source_canonical": None,
        }

    program_contract = parser_contract_v1[
        "programs"
    ][program_id]

    manifest_aliases = program_contract[
        "manifest_speaker_aliases"
    ]

    speaker_canonical = manifest_aliases.get(
        raw_label
    )

    speaker_source_canonical = None

    if program_id == chehelston_program_id:
        source_alias_record = (
            source_observed_aliases.get(raw_label)
        )

        if source_alias_record is not None:
            speaker_source_canonical = (
                source_alias_record[
                    "canonical_source_label"
                ]
            )
        else:
            source_canonical_values = {
                item["canonical_source_label"]
                for item in source_observed_aliases.values()
            }

            if raw_label in source_canonical_values:
                speaker_source_canonical = raw_label

    return {
        "speaker_canonical": speaker_canonical,
        "speaker_source_canonical": speaker_source_canonical,
    }


def build_parsed_utterance(
    record,
    utterance_index,
    text,
    source_text,
    source_line_start,
    source_line_end,
    speaker_raw,
    speaker_canonical,
    speaker_source_canonical,
    speaker_status,
    boundary_type,
):
    assert text.strip(), (
        "Parsed utterance text must not be empty."
    )

    return {
        "utterance_id": (
            f"{record['transcript_unit_id']}"
            f":utterance:{utterance_index:05d}"
        ),
        "utterance_index": utterance_index,
        "program": record["program"],
        "program_id": record["program_id"],
        "physical_file_id": record["physical_file_id"],
        "transcript_unit_id": record["transcript_unit_id"],
        "relative_path": record["relative_path"],
        "speaker_raw": speaker_raw,
        "speaker_canonical": speaker_canonical,
        "speaker_source_canonical": (
            speaker_source_canonical
        ),
        "speaker_status": speaker_status,
        "boundary_type": boundary_type,
        "text": text,
        "source_text": source_text,
        "source_line_start": source_line_start,
        "source_line_end": source_line_end,
        "start_time": None,
        "end_time": None,
    }


def parse_source_paragraphs_unresolved(
    record,
    lines,
    body_start_index,
    boundary_type,
):
    utterances = []

    for index in range(body_start_index, len(lines)):
        line = lines[index]

        if not line.strip():
            continue

        utterance_index = len(utterances) + 1

        utterances.append(
            build_parsed_utterance(
                record=record,
                utterance_index=utterance_index,
                text=line,
                source_text=line,
                source_line_start=index + 1,
                source_line_end=index + 1,
                speaker_raw=None,
                speaker_canonical=None,
                speaker_source_canonical=None,
                speaker_status="UNRESOLVED",
                boundary_type=boundary_type,
            )
        )

    return utterances


def parse_solo_manifest_attribution(
    record,
    lines,
    body_start_index,
):
    utterances = []

    program_contract = parser_contract_v1[
        "programs"
    ][record["program_id"]]

    primary_speaker = program_contract[
        "primary_speaker"
    ]

    manifest_aliases = program_contract[
        "manifest_speaker_aliases"
    ]

    active_raw_marker = None

    for index in range(body_start_index, len(lines)):
        line = lines[index]

        if not line.strip():
            continue

        bold_value = extract_bold_only_value(line)

        if (
            bold_value is not None
            and bold_value in manifest_aliases
            and manifest_aliases[bold_value] == primary_speaker
        ):
            active_raw_marker = bold_value
            continue

        utterance_index = len(utterances) + 1

        utterances.append(
            build_parsed_utterance(
                record=record,
                utterance_index=utterance_index,
                text=line,
                source_text=line,
                source_line_start=index + 1,
                source_line_end=index + 1,
                speaker_raw=active_raw_marker,
                speaker_canonical=primary_speaker,
                speaker_source_canonical=None,
                speaker_status="MANIFEST_ATTRIBUTED",
                boundary_type="SOURCE_PARAGRAPH",
            )
        )

    return utterances


def parse_bold_explicit_turns(
    record,
    lines,
    body_start_index,
):
    program_id = record["program_id"]

    verified_markers = set(
        verified_bold_speaker_markers[
            program_id
        ]
    )

    instructional_values = (
        instructional_bold_content_by_program[
            program_id
        ]
    )

    utterances = []

    current_speaker_raw = None
    current_start_index = None
    current_end_index = None
    current_text_parts = []

    def flush_current_turn():
        nonlocal current_speaker_raw
        nonlocal current_start_index
        nonlocal current_end_index
        nonlocal current_text_parts

        if not current_text_parts:
            return

        label_resolution = resolve_speaker_labels(
            program_id,
            current_speaker_raw,
        )

        utterance_index = len(utterances) + 1

        source_text = "\n".join(
            lines[
                current_start_index:
                current_end_index + 1
            ]
        )

        utterances.append(
            build_parsed_utterance(
                record=record,
                utterance_index=utterance_index,
                text="\n\n".join(current_text_parts),
                source_text=source_text,
                source_line_start=current_start_index + 1,
                source_line_end=current_end_index + 1,
                speaker_raw=current_speaker_raw,
                speaker_canonical=label_resolution[
                    "speaker_canonical"
                ],
                speaker_source_canonical=label_resolution[
                    "speaker_source_canonical"
                ],
                speaker_status=(
                    "EXPLICIT_SOURCE_MARKER"
                    if current_speaker_raw is not None
                    else "UNRESOLVED"
                ),
                boundary_type=(
                    "BOLD_ONLY_SPEAKER_BOUNDARY"
                    if current_speaker_raw is not None
                    else "SOURCE_PARAGRAPH_UNRESOLVED"
                ),
            )
        )

        current_speaker_raw = None
        current_start_index = None
        current_end_index = None
        current_text_parts = []

    for index in range(body_start_index, len(lines)):
        line = lines[index]

        if not line.strip():
            continue

        bold_value = extract_bold_only_value(line)

        if bold_value is not None:
            if bold_value in verified_markers:
                flush_current_turn()

                current_speaker_raw = bold_value
                current_start_index = index
                current_end_index = index
                current_text_parts = []
                continue

            if bold_value in instructional_values:
                if current_start_index is None:
                    current_start_index = index
                    current_end_index = index
                    current_speaker_raw = None

                current_text_parts.append(line)
                current_end_index = index
                continue

            raise AssertionError(
                "Unclassified Bold-only value reached parser: "
                f"{program_id} | {bold_value}"
            )

        if current_start_index is None:
            current_start_index = index
            current_speaker_raw = None

        current_text_parts.append(line)
        current_end_index = index

    flush_current_turn()

    return utterances


def parse_chehelston_explicit_turns(
    record,
    lines,
    body_start_index,
):
    utterances = []

    current_speaker_raw = None
    current_start_index = None
    current_end_index = None
    current_text_parts = []

    def flush_current_turn():
        nonlocal current_speaker_raw
        nonlocal current_start_index
        nonlocal current_end_index
        nonlocal current_text_parts

        if not current_text_parts:
            return

        label_resolution = resolve_speaker_labels(
            record["program_id"],
            current_speaker_raw,
        )

        utterance_index = len(utterances) + 1

        source_text = "\n".join(
            lines[
                current_start_index:
                current_end_index + 1
            ]
        )

        utterances.append(
            build_parsed_utterance(
                record=record,
                utterance_index=utterance_index,
                text="\n\n".join(current_text_parts),
                source_text=source_text,
                source_line_start=current_start_index + 1,
                source_line_end=current_end_index + 1,
                speaker_raw=current_speaker_raw,
                speaker_canonical=label_resolution[
                    "speaker_canonical"
                ],
                speaker_source_canonical=label_resolution[
                    "speaker_source_canonical"
                ],
                speaker_status=(
                    "EXPLICIT_SOURCE_MARKER"
                    if current_speaker_raw is not None
                    else "UNRESOLVED"
                ),
                boundary_type=(
                    "COLON_SPEAKER_BOUNDARY"
                    if current_speaker_raw is not None
                    else "SOURCE_PARAGRAPH_UNRESOLVED"
                ),
            )
        )

        current_speaker_raw = None
        current_start_index = None
        current_end_index = None
        current_text_parts = []

    for index in range(body_start_index, len(lines)):
        line = lines[index]

        if not line.strip():
            continue

        match = plain_colon_pattern.match(line)

        raw_prefix = (
            match.group(1).strip()
            if match
            else None
        )

        if (
            match
            and raw_prefix
            in final_chehelston_speaker_markers
        ):
            flush_current_turn()

            current_speaker_raw = raw_prefix
            current_start_index = index
            current_end_index = index

            trailing_text = match.group(2)

            current_text_parts = (
                [trailing_text]
                if trailing_text.strip()
                else []
            )

            continue

        if current_start_index is None:
            current_start_index = index
            current_speaker_raw = None

        current_text_parts.append(line)
        current_end_index = index

    flush_current_turn()

    return utterances


def parse_narrative_cue_source(
    record,
    lines,
    body_start_index,
):
    utterances = []

    policy_contract = program_registry_by_id[
        record["program_id"]
    ]["speaker_policy"]

    generic_roles = policy_contract[
        "generic_roles_observed"
    ]

    for index in range(body_start_index, len(lines)):
        line = lines[index]

        if not line.strip():
            continue

        matched_roles = [
            role
            for role in generic_roles
            if role in line
        ]

        speaker_raw = (
            matched_roles[0]
            if matched_roles
            else None
        )

        speaker_status = (
            "NARRATIVE_CUE"
            if speaker_raw is not None
            else "UNRESOLVED"
        )

        boundary_type = (
            "REGISTERED_NARRATIVE_CUE"
            if speaker_raw is not None
            else "SOURCE_PARAGRAPH_UNRESOLVED"
        )

        utterance_index = len(utterances) + 1

        utterances.append(
            build_parsed_utterance(
                record=record,
                utterance_index=utterance_index,
                text=line,
                source_text=line,
                source_line_start=index + 1,
                source_line_end=index + 1,
                speaker_raw=speaker_raw,
                speaker_canonical=None,
                speaker_source_canonical=None,
                speaker_status=speaker_status,
                boundary_type=boundary_type,
            )
        )

    return utterances


parsed_transcript_files_v1 = []
all_parsed_utterances_v1 = []

for record in accepted_files:
    relative_path = record["relative_path"]

    candidate = boundary_candidate_by_path[
        relative_path
    ]

    body_start_index = (
        candidate["candidate_line_number"] - 1
    )

    file_path = project_root / relative_path
    lines = file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    strategy = parser_contract_v1[
        "programs"
    ][record["program_id"]][
        "parser_strategy"
    ]

    if strategy == "SOLO_MANIFEST_ATTRIBUTION":
        utterances = parse_solo_manifest_attribution(
            record,
            lines,
            body_start_index,
        )

    elif strategy == "BOLD_ONLY_VERIFIED_MARKERS":
        utterances = parse_bold_explicit_turns(
            record,
            lines,
            body_start_index,
        )

    elif strategy == "CHEHELSTON_FILE_LEVEL_STRATEGY":
        if relative_path in chehelston_contextual_path_set:
            utterances = parse_source_paragraphs_unresolved(
                record,
                lines,
                body_start_index,
                boundary_type=(
                    "CONTEXTUAL_SOURCE_PARAGRAPH_UNRESOLVED"
                ),
            )
        else:
            utterances = parse_chehelston_explicit_turns(
                record,
                lines,
                body_start_index,
            )

    elif strategy == "UNLABELED_TURNS_UNRESOLVED":
        utterances = parse_source_paragraphs_unresolved(
            record,
            lines,
            body_start_index,
            boundary_type=(
                "UNLABELED_SOURCE_PARAGRAPH_UNRESOLVED"
            ),
        )

    elif strategy == "PRIMARY_KNOWN_UNRESOLVED":
        utterances = parse_source_paragraphs_unresolved(
            record,
            lines,
            body_start_index,
            boundary_type=(
                "PRIMARY_KNOWN_BOUNDARY_UNRESOLVED"
            ),
        )

    elif strategy == "REGISTERED_NARRATIVE_CUES":
        utterances = parse_narrative_cue_source(
            record,
            lines,
            body_start_index,
        )

    else:
        raise AssertionError(
            f"Unsupported parser strategy: {strategy}"
        )

    assert utterances, (
        f"No parsed utterances produced for {relative_path}"
    )

    parsed_transcript_files_v1.append({
        "program": record["program"],
        "program_id": record["program_id"],
        "relative_path": relative_path,
        "physical_file_id": record["physical_file_id"],
        "transcript_unit_id": record["transcript_unit_id"],
        "speaker_policy": record["speaker_policy"],
        "parser_strategy": strategy,
        "body_candidate_line": (
            candidate["candidate_line_number"]
        ),
        "utterance_count": len(utterances),
        "utterances": utterances,
    })

    all_parsed_utterances_v1.extend(
        utterances
    )


print("Parsed files:", len(parsed_transcript_files_v1))
print(
    "Parsed utterances:",
    len(all_parsed_utterances_v1),
)
print(
    "Programs represented:",
    len({
        item["program_id"]
        for item in parsed_transcript_files_v1
    }),
)
print()

print("PARSED FILES BY STRATEGY")
print("=" * 100)

strategy_file_counts = Counter(
    item["parser_strategy"]
    for item in parsed_transcript_files_v1
)

for strategy, count in sorted(
    strategy_file_counts.items()
):
    utterance_count = sum(
        item["utterance_count"]
        for item in parsed_transcript_files_v1
        if item["parser_strategy"] == strategy
    )

    print(
        f"- {strategy}: "
        f"{count} files | "
        f"{utterance_count} utterances"
    )


print()
print("SPEAKER STATUS COUNTS")
print("=" * 100)

speaker_status_counts = Counter(
    item["speaker_status"]
    for item in all_parsed_utterances_v1
)

for status, count in sorted(
    speaker_status_counts.items()
):
    print(f"- {status}: {count}")


print()
print("REPRESENTATIVE PARSED UTTERANCE BY PROGRAM")
print("=" * 100)

for program_id in sorted(files_by_program):
    program_utterances = [
        item
        for item in all_parsed_utterances_v1
        if item["program_id"] == program_id
    ]

    sample = program_utterances[0]

    display_text = (
        sample["text"]
        if len(sample["text"]) <= 260
        else sample["text"][:260] + "..."
    )

    print()
    print("Program:", sample["program"])
    print("Program ID:", program_id)
    print(
        "Program utterances:",
        len(program_utterances),
    )
    print("File:", sample["relative_path"])
    print(
        "Source lines:",
        f"{sample['source_line_start']}-"
        f"{sample['source_line_end']}",
    )
    print("Speaker raw:", sample["speaker_raw"])
    print(
        "Speaker canonical:",
        sample["speaker_canonical"],
    )
    print(
        "Speaker source canonical:",
        sample["speaker_source_canonical"],
    )
    print(
        "Speaker status:",
        sample["speaker_status"],
    )
    print(
        "Boundary type:",
        sample["boundary_type"],
    )
    print("Text:", display_text)
    print("-" * 100)


assert len(parsed_transcript_files_v1) == len(
    accepted_files
), (
    "Not all accepted files were parsed."
)

assert {
    item["relative_path"]
    for item in parsed_transcript_files_v1
} == {
    record["relative_path"]
    for record in accepted_files
}, (
    "Parsed file set does not match Accepted Corpus."
)

assert all(
    item["text"].strip()
    for item in all_parsed_utterances_v1
), (
    "At least one parsed utterance has empty text."
)

assert all(
    item["start_time"] is None
    and item["end_time"] is None
    for item in all_parsed_utterances_v1
), (
    "Synthetic or unexpected timestamps were created."
)

solo_program_ids = {
    program_id
    for program_id, contract
    in parser_contract_v1["programs"].items()
    if contract["parser_strategy"]
    == "SOLO_MANIFEST_ATTRIBUTION"
}

for item in all_parsed_utterances_v1:
    if item["program_id"] in solo_program_ids:
        expected_primary = parser_contract_v1[
            "programs"
        ][item["program_id"]][
            "primary_speaker"
        ]

        assert item["speaker_canonical"] == (
            expected_primary
        ), (
            "Solo Manifest attribution mismatch: "
            f"{item['utterance_id']}"
        )


for item in all_parsed_utterances_v1:
    if item["relative_path"] in (
        chehelston_contextual_path_set
    ):
        assert item["speaker_canonical"] is None
        assert item["speaker_raw"] is None
        assert item["speaker_status"] == "UNRESOLVED"


special_unresolved_policies = {
    "KNOWN_PARTICIPANTS_UNLABELED_TURNS",
    "PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED",
}

special_unresolved_paths = {
    record["relative_path"]
    for record in accepted_files
    if record["speaker_policy"]
    in special_unresolved_policies
}

for item in all_parsed_utterances_v1:
    if item["relative_path"] in special_unresolved_paths:
        assert item["speaker_raw"] is None
        assert item["speaker_canonical"] is None
        assert item["speaker_status"] == "UNRESOLVED"


mizan_generic_roles = set(
    program_registry_by_id[
        "program:mizan"
    ]["speaker_policy"][
        "generic_roles_observed"
    ]
)

for item in all_parsed_utterances_v1:
    if item["program_id"] == "program:mizan":
        if item["speaker_raw"] is not None:
            assert item["speaker_raw"] in mizan_generic_roles
            assert item["speaker_canonical"] is None
            assert item["speaker_status"] == "NARRATIVE_CUE"


explicit_bold_program_ids = {
    "program:asre-shirin",
    "program:khane-to",
    "program:uni-tehran",
}

for item in all_parsed_utterances_v1:
    if (
        item["program_id"]
        in explicit_bold_program_ids
        and item["speaker_raw"] is not None
    ):
        assert item["speaker_raw"] in set(
            verified_bold_speaker_markers[
                item["program_id"]
            ]
        ), (
            "Unverified Bold speaker marker reached parsed output."
        )


assert len({
    item["utterance_id"]
    for item in all_parsed_utterances_v1
}) == len(all_parsed_utterances_v1), (
    "Parsed utterance IDs are not unique."
)

print()
print("POLICY-AWARE TRANSCRIPT PARSER DRY RUN: PASS")

Parsed files: 183
Parsed utterances: 15131
Programs represented: 13

PARSED FILES BY STRATEGY
- BOLD_ONLY_VERIFIED_MARKERS: 77 files | 2845 utterances
- CHEHELSTON_FILE_LEVEL_STRATEGY: 38 files | 889 utterances
- PRIMARY_KNOWN_UNRESOLVED: 3 files | 56 utterances
- REGISTERED_NARRATIVE_CUES: 1 files | 217 utterances
- SOLO_MANIFEST_ATTRIBUTION: 63 files | 10975 utterances
- UNLABELED_TURNS_UNRESOLVED: 1 files | 149 utterances

SPEAKER STATUS COUNTS
- EXPLICIT_SOURCE_MARKER: 3349
- MANIFEST_ATTRIBUTED: 10975
- NARRATIVE_CUE: 17
- UNRESOLVED: 790

REPRESENTATIVE PARSED UTTERANCE BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Program utterances: 902
File: data/transcripts/final/asre_shirin/season-1/episode-01.md
Source lines: 10-12
Speaker raw: خانم متولیان
Speaker canonical: None
Speaker source canonical: None
Speaker status: EXPLICIT_SOURCE_MARKER
Boundary type: BOLD_ONLY_SPEAKER_BOUNDARY
Text: خیلی از ما توی این چند سال به نقطه‌هایی رسیدیم که گاهی خرج و برجمون با هم جور 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### ممیزی نهایی Conservation و Lineage پس از Root-cause Analysis

Diagnostic قبلی Root Cause تمام Source Gapهای مشاهده‌شده را مشخص کرد.

نتیجه واقعی Corpus:

- تمام `183` فایل بررسی شدند.
- هیچ Lineage Failure وجود نداشت.
- هیچ Source-line Overlap وجود نداشت.
- تمام Gapهای Source ساختاری و قابل توضیح بودند.
- هیچ Instructional Content گم نشده بود.
- هیچ Source Gap توضیح‌ناپذیر وجود نداشت.

Gapهای ساختاری مشاهده‌شده دو نوع بودند:

1. Markerهای تکراری `دکتر کمیل رودی` در فایل‌های Solo مجموعه `در جستجوی خوشبختی`  
   این Markerها طبق قرارداد انسانی Manifest نقش Speaker Metadata دارند و Payload محتوایی نیستند.

2. تعداد محدودی Bold Speaker Boundary در `خانه تو` که بین آن Marker و Marker بعدی هیچ متن مستقلی وجود ندارد.  
   ساخت Utterance خالی برای این Boundaryها ممنوع است، بنابراین خود Marker می‌تواند بدون Payload باقی بماند.

این استثنا نباید به یک Rule عمومی برای حذف Speaker Marker تبدیل شود.

برای Programهای Explicit، یک Bold Speaker Marker فقط زمانی Structural Skip مجاز است که:

- Marker در Registry تأییدشده باشد؛
- بعد از آن تا Marker معتبر بعدی هیچ خط محتوایی غیرخالی وجود نداشته باشد.

بنابراین اگر Parser در آینده یک Marker دارای متن واقعی را از Lineage حذف کند، این Audit باید FAIL شود.

این Cell دوباره Source اصلی را از Disk می‌خواند و چهار Gate مستقل اجرا می‌کند:

### Exact Lineage

`source_text` هر Utterance باید دقیقاً با Range ثبت‌شده در Source اصلی یکسان باشد.

### No Overlap

هیچ Source Line نباید توسط بیش از یک Utterance مصرف شود.

### Full Coverage Accounting

هر خط غیرخالی Body باید دقیقاً یکی از این وضعیت‌ها را داشته باشد:

- Covered by parsed lineage
- Allowed Solo structural marker
- Allowed empty explicit boundary

هیچ وضعیت دیگری پذیرفته نمی‌شود.

### Exact Payload Conservation

Payload مورد انتظار مستقیماً دوباره از Source ساخته و با Parsed Payload مقایسه می‌شود.

در این مقایسه:

- Speaker Marker ساختاری حذف می‌شود.
- Instructional Bold حفظ می‌شود.
- متن بعد از `Speaker:` در چهلستون Explicit حفظ می‌شود.
- فایل‌های unresolved عیناً حفظ می‌شوند.
- Persian normalization انجام نمی‌شود.

این Cell Gate نهایی Conservation برای Dry Run Parser است.

هنوز انجام نمی‌شوند:

- Merge کردن Utteranceها
- ASR cleanup
- Persian normalization
- Deduplication
- Chunking
- Artifact write

**معیار PASS:**

- `183` فایل مستقلاً از Disk بررسی شوند.
- Lineage failure برابر صفر باشد.
- Overlap failure برابر صفر باشد.
- Unexplained source gap برابر صفر باشد.
- Payload failure برابر صفر باشد.
- Instructional-content gap برابر صفر باشد.
- مجموع Covered Lines و Structural Skipهای مجاز دقیقاً برابر تمام خطوط غیرخالی Body باشد.

**معیار FAIL:**

وجود حتی یک خط محتوایی حذف‌شده، تغییرکرده، دوباره‌مصرف‌شده یا بدون Lineage باعث توقف مرحله خواهد شد.

</div>

In [21]:
# Code Cell 21

parsed_file_by_path = {
    item["relative_path"]: item
    for item in parsed_transcript_files_v1
}

accepted_record_by_path = {
    record["relative_path"]: record
    for record in accepted_files
}


def expected_payload_fragments(
    record,
    lines,
    body_start_index,
):
    program_id = record["program_id"]

    strategy = parser_contract_v1[
        "programs"
    ][program_id]["parser_strategy"]

    fragments = []

    for index in range(body_start_index, len(lines)):
        line = lines[index]

        if not line.strip():
            continue

        if strategy == "SOLO_MANIFEST_ATTRIBUTION":
            bold_value = extract_bold_only_value(line)

            manifest_aliases = parser_contract_v1[
                "programs"
            ][program_id]["manifest_speaker_aliases"]

            primary_speaker = parser_contract_v1[
                "programs"
            ][program_id]["primary_speaker"]

            if (
                bold_value is not None
                and bold_value in manifest_aliases
                and manifest_aliases[bold_value]
                == primary_speaker
            ):
                continue

            fragments.append(line)
            continue

        if strategy == "BOLD_ONLY_VERIFIED_MARKERS":
            bold_value = extract_bold_only_value(line)

            verified_markers = set(
                verified_bold_speaker_markers[
                    program_id
                ]
            )

            if (
                bold_value is not None
                and bold_value in verified_markers
            ):
                continue

            fragments.append(line)
            continue

        if strategy == "CHEHELSTON_FILE_LEVEL_STRATEGY":
            if record["relative_path"] in (
                chehelston_contextual_path_set
            ):
                fragments.append(line)
                continue

            match = plain_colon_pattern.match(line)

            raw_prefix = (
                match.group(1).strip()
                if match
                else None
            )

            if (
                match
                and raw_prefix
                in final_chehelston_speaker_markers
            ):
                trailing_text = match.group(2)

                if trailing_text.strip():
                    fragments.append(trailing_text)

                continue

            fragments.append(line)
            continue

        if strategy in {
            "UNLABELED_TURNS_UNRESOLVED",
            "PRIMARY_KNOWN_UNRESOLVED",
            "REGISTERED_NARRATIVE_CUES",
        }:
            fragments.append(line)
            continue

        raise AssertionError(
            f"Unsupported conservation strategy: {strategy}"
        )

    return fragments


def actual_payload_fragments(parsed_file):
    fragments = []

    for utterance in parsed_file["utterances"]:
        fragments.extend(
            utterance["text"].split("\n\n")
        )

    return fragments


def is_solo_structural_marker(
    record,
    line,
):
    program_id = record["program_id"]

    strategy = parser_contract_v1[
        "programs"
    ][program_id]["parser_strategy"]

    if strategy != "SOLO_MANIFEST_ATTRIBUTION":
        return False

    bold_value = extract_bold_only_value(line)

    if bold_value is None:
        return False

    program_contract = parser_contract_v1[
        "programs"
    ][program_id]

    manifest_aliases = program_contract[
        "manifest_speaker_aliases"
    ]

    primary_speaker = program_contract[
        "primary_speaker"
    ]

    return (
        bold_value in manifest_aliases
        and manifest_aliases[bold_value]
        == primary_speaker
    )


def is_empty_verified_bold_boundary(
    record,
    lines,
    line_index,
):
    program_id = record["program_id"]

    strategy = parser_contract_v1[
        "programs"
    ][program_id]["parser_strategy"]

    if strategy != "BOLD_ONLY_VERIFIED_MARKERS":
        return False

    current_value = extract_bold_only_value(
        lines[line_index]
    )

    verified_markers = set(
        verified_bold_speaker_markers[
            program_id
        ]
    )

    if current_value not in verified_markers:
        return False

    for next_index in range(
        line_index + 1,
        len(lines),
    ):
        next_line = lines[next_index]

        if not next_line.strip():
            continue

        next_bold_value = extract_bold_only_value(
            next_line
        )

        if next_bold_value in verified_markers:
            return True

        return False

    return True


def is_empty_verified_chehelston_boundary(
    record,
    lines,
    line_index,
):
    relative_path = record["relative_path"]

    strategy = parser_contract_v1[
        "programs"
    ][record["program_id"]][
        "parser_strategy"
    ]

    if (
        strategy
        != "CHEHELSTON_FILE_LEVEL_STRATEGY"
        or relative_path
        in chehelston_contextual_path_set
    ):
        return False

    line = lines[line_index]

    match = plain_colon_pattern.match(line)

    if not match:
        return False

    raw_prefix = match.group(1).strip()
    trailing_text = match.group(2)

    if (
        raw_prefix
        not in final_chehelston_speaker_markers
    ):
        return False

    if trailing_text.strip():
        return False

    for next_index in range(
        line_index + 1,
        len(lines),
    ):
        next_line = lines[next_index]

        if not next_line.strip():
            continue

        next_match = plain_colon_pattern.match(
            next_line
        )

        next_prefix = (
            next_match.group(1).strip()
            if next_match
            else None
        )

        if (
            next_match
            and next_prefix
            in final_chehelston_speaker_markers
        ):
            return True

        return False

    return True


lineage_failures = []
overlap_failures = []
coverage_failures = []
payload_failures = []
instructional_content_gaps = []

structural_skip_counts = Counter()
structural_skip_examples = {}

total_body_nonempty_lines = 0
total_covered_nonempty_lines = 0
total_structural_skips = 0


for relative_path, record in sorted(
    accepted_record_by_path.items()
):
    parsed_file = parsed_file_by_path[
        relative_path
    ]

    file_path = project_root / relative_path

    lines = file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    candidate = boundary_candidate_by_path[
        relative_path
    ]

    body_start_index = (
        candidate["candidate_line_number"] - 1
    )

    coverage_count = Counter()

    for utterance in parsed_file["utterances"]:
        start_line = utterance[
            "source_line_start"
        ]

        end_line = utterance[
            "source_line_end"
        ]

        if not (
            1 <= start_line <= end_line <= len(lines)
        ):
            lineage_failures.append({
                "relative_path": relative_path,
                "utterance_id": utterance[
                    "utterance_id"
                ],
                "reason": "INVALID_SOURCE_RANGE",
                "source_line_start": start_line,
                "source_line_end": end_line,
                "file_line_count": len(lines),
            })
            continue

        expected_source_text = "\n".join(
            lines[
                start_line - 1:
                end_line
            ]
        )

        if (
            utterance["source_text"]
            != expected_source_text
        ):
            lineage_failures.append({
                "relative_path": relative_path,
                "utterance_id": utterance[
                    "utterance_id"
                ],
                "reason": "SOURCE_TEXT_MISMATCH",
                "source_line_start": start_line,
                "source_line_end": end_line,
            })

        for line_number in range(
            start_line,
            end_line + 1,
        ):
            coverage_count[line_number] += 1

    overlapping_lines = sorted(
        line_number
        for line_number, count
        in coverage_count.items()
        if count > 1
    )

    if overlapping_lines:
        overlap_failures.append({
            "relative_path": relative_path,
            "line_numbers": overlapping_lines,
        })

    missing_lines = []

    for index in range(
        body_start_index,
        len(lines),
    ):
        line = lines[index]

        if not line.strip():
            continue

        line_number = index + 1

        total_body_nonempty_lines += 1

        if coverage_count[line_number] > 0:
            total_covered_nonempty_lines += 1
            continue

        skip_type = None

        if is_solo_structural_marker(
            record,
            line,
        ):
            skip_type = "SOLO_MANIFEST_MARKER"

        elif is_empty_verified_bold_boundary(
            record,
            lines,
            index,
        ):
            skip_type = (
                "EMPTY_VERIFIED_BOLD_BOUNDARY"
            )

        elif is_empty_verified_chehelston_boundary(
            record,
            lines,
            index,
        ):
            skip_type = (
                "EMPTY_VERIFIED_CHEHELSTON_BOUNDARY"
            )

        if skip_type is not None:
            structural_skip_counts[
                skip_type
            ] += 1

            total_structural_skips += 1

            structural_skip_examples.setdefault(
                skip_type,
                {
                    "relative_path": relative_path,
                    "line_number": line_number,
                    "raw_line": line,
                },
            )

            continue

        bold_value = extract_bold_only_value(
            line
        )

        if (
            record["program_id"]
            in instructional_bold_content_by_program
            and bold_value
            in instructional_bold_content_by_program[
                record["program_id"]
            ]
        ):
            instructional_content_gaps.append({
                "relative_path": relative_path,
                "line_number": line_number,
                "raw_line": line,
            })

        missing_lines.append({
            "line_number": line_number,
            "raw_line": line,
        })

    if missing_lines:
        coverage_failures.append({
            "relative_path": relative_path,
            "missing_nonempty_lines": missing_lines,
        })

    expected_fragments = expected_payload_fragments(
        record,
        lines,
        body_start_index,
    )

    actual_fragments = actual_payload_fragments(
        parsed_file
    )

    if expected_fragments != actual_fragments:
        first_difference = None

        comparison_length = min(
            len(expected_fragments),
            len(actual_fragments),
        )

        for fragment_index in range(
            comparison_length
        ):
            if (
                expected_fragments[
                    fragment_index
                ]
                != actual_fragments[
                    fragment_index
                ]
            ):
                first_difference = (
                    fragment_index
                )
                break

        if (
            first_difference is None
            and len(expected_fragments)
            != len(actual_fragments)
        ):
            first_difference = (
                comparison_length
            )

        payload_failures.append({
            "relative_path": relative_path,
            "expected_fragment_count": len(
                expected_fragments
            ),
            "actual_fragment_count": len(
                actual_fragments
            ),
            "first_difference_index": (
                first_difference
            ),
            "expected_value": (
                expected_fragments[
                    first_difference
                ]
                if (
                    first_difference is not None
                    and first_difference
                    < len(expected_fragments)
                )
                else None
            ),
            "actual_value": (
                actual_fragments[
                    first_difference
                ]
                if (
                    first_difference is not None
                    and first_difference
                    < len(actual_fragments)
                )
                else None
            ),
        })


print(
    "Files independently revalidated:",
    len(parsed_file_by_path),
)
print(
    "Body non-empty source lines:",
    total_body_nonempty_lines,
)
print(
    "Covered non-empty source lines:",
    total_covered_nonempty_lines,
)
print(
    "Allowed structural skips:",
    total_structural_skips,
)
print()

print("STRUCTURAL SKIP COUNTS")
print("=" * 100)

if structural_skip_counts:
    for skip_type, count in sorted(
        structural_skip_counts.items()
    ):
        print(
            f"- {skip_type}: {count}"
        )

        example = structural_skip_examples[
            skip_type
        ]

        print(
            "  Example:",
            example["relative_path"],
            f"(line {example['line_number']})",
        )
        print(
            " ",
            example["raw_line"],
        )
else:
    print("- NONE")


print()
print("FINAL CONSERVATION FAILURE COUNTS")
print("=" * 100)
print(
    "Lineage failures:",
    len(lineage_failures),
)
print(
    "Overlap failures:",
    len(overlap_failures),
)
print(
    "Coverage failures:",
    len(coverage_failures),
)
print(
    "Instructional-content gaps:",
    len(instructional_content_gaps),
)
print(
    "Payload failures:",
    len(payload_failures),
)


if coverage_failures:
    print()
    print("First coverage failure:")
    print(
        json.dumps(
            coverage_failures[0],
            ensure_ascii=False,
            indent=2,
        )
    )

if payload_failures:
    print()
    print("First payload failure:")
    print(
        json.dumps(
            payload_failures[0],
            ensure_ascii=False,
            indent=2,
        )
    )


print()
print("GLOBAL ACCOUNTING")
print("=" * 100)

accounted_lines = (
    total_covered_nonempty_lines
    + total_structural_skips
)

print(
    "Covered + allowed structural skips:",
    accounted_lines,
)
print(
    "All non-empty body lines:",
    total_body_nonempty_lines,
)
print(
    "Accounting balanced:",
    accounted_lines
    == total_body_nonempty_lines,
)


assert len(parsed_file_by_path) == 183, (
    "Final conservation audit does not cover all parsed files."
)

assert not lineage_failures, (
    "Final source-lineage validation failed."
)

assert not overlap_failures, (
    "At least one source line is consumed by multiple utterances."
)

assert not instructional_content_gaps, (
    "Instructional Bold content was lost."
)

assert not coverage_failures, (
    "At least one non-empty source body line remains unexplained."
)

assert not payload_failures, (
    "Parsed payload does not exactly conserve expected source content."
)

assert (
    accounted_lines
    == total_body_nonempty_lines
), (
    "Global source-line conservation accounting does not balance."
)

assert set(
    structural_skip_counts
).issubset({
    "SOLO_MANIFEST_MARKER",
    "EMPTY_VERIFIED_BOLD_BOUNDARY",
    "EMPTY_VERIFIED_CHEHELSTON_BOUNDARY",
}), (
    "Unexpected structural-skip category found."
)

print()
print(
    "FINAL TRANSCRIPT SOURCE CONSERVATION "
    "AND LINEAGE AUDIT: PASS"
)

Files independently revalidated: 183
Body non-empty source lines: 28894
Covered non-empty source lines: 21571
Allowed structural skips: 7323

STRUCTURAL SKIP COUNTS
- EMPTY_VERIFIED_BOLD_BOUNDARY: 3
  Example: data/transcripts/final/khane-to/episode-03.md (line 53)
  **فرهاد جم**
- SOLO_MANIFEST_MARKER: 7320
  Example: data/transcripts/final/dar-jostojo-final-C/1.md (line 3)
  **دکتر کمیل رودی**

FINAL CONSERVATION FAILURE COUNTS
Lineage failures: 0
Overlap failures: 0
Coverage failures: 0
Instructional-content gaps: 0
Payload failures: 0

GLOBAL ACCOUNTING
Covered + allowed structural skips: 28894
All non-empty body lines: 28894
Accounting balanced: True

FINAL TRANSCRIPT SOURCE CONSERVATION AND LINEAGE AUDIT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### Discovery برای Tiny Utteranceها پیش از هر Merge

Parser پایه اکنون Conservation و Lineage کامل دارد.

مرحله بعد بررسی Utteranceهای بسیار کوتاه است.

هدف این مرحله هنوز Merge کردن نیست.

ابتدا باید مشخص شود:

- Utteranceهای کوتاه در کدام Programها متمرکز شده‌اند.
- طول آن‌ها چقدر است.
- آیا Speaker آن‌ها resolved است یا unresolved.
- آیا Utterance مجاور همان Speaker و همان File را دارد.
- آیا بین دو Utterance یک Speaker Boundary واقعی وجود دارد.
- آیا ادغام احتمالی باعث عبور از Boundary یا ازبین‌رفتن ساختار Source می‌شود.

### اصل محافظه‌کارانه

Tiny بودن متن به‌تنهایی مجوز Merge نیست.

مثلاً عبارت‌هایی مانند:

- `سلام`
- `خب`
- `یعنی چی؟`

ممکن است کوتاه باشند ولی از نظر گفتاری و معنایی یک واحد واقعی باشند.

بنابراین در این مرحله فقط Distribution و Candidateها بررسی می‌شوند.

### طول متن

برای Discovery چند آستانه صرفاً آماری گزارش می‌شود:

- کمتر یا مساوی `10` Character
- کمتر یا مساوی `20` Character
- کمتر یا مساوی `40` Character

این آستانه‌ها هنوز Rule نهایی Merge نیستند.

هیچ Utterance صرفاً به علت قرارگرفتن زیر این Thresholdها تغییر نمی‌کند.

### Merge-safe adjacency

یک جفت فقط زمانی به‌عنوان Candidate ساختاری بررسی می‌شود که:

- هر دو در یک Source File باشند.
- ترتیب Utteranceها متوالی باشد.
- Speaker Canonical یکسان و غیرخالی باشد.
- `speaker_raw` نیز بدون تعارض باشد.
- هیچ Speaker Boundary جدیدی بین آن‌ها وجود نداشته باشد.
- هیچ‌کدام `UNRESOLVED` یا `NARRATIVE_CUE` نباشند.

این Cell فقط Candidateها را شمارش و نمونه‌گیری می‌کند.

هنوز انجام نمی‌شوند:

- Merge
- بازنویسی Text
- Persian normalization
- Deduplication
- ASR cleanup
- Chunking
- Artifact write

**معیار PASS:**

- تمام `15,131` Parsed Utterance بررسی شوند.
- Distribution طول بر اساس Program و Strategy گزارش شود.
- Tiny Utteranceهای resolved و unresolved از هم جدا شوند.
- Candidateهای adjacency فقط با Rule محافظه‌کارانه ساخته شوند.
- هیچ Parsed Utterance تغییر نکند.

</div>

In [22]:
# Code Cell 22

tiny_thresholds = (10, 20, 40)

utterance_length_profiles = []

for item in all_parsed_utterances_v1:
    text_length = len(item["text"].strip())

    utterance_length_profiles.append({
        "utterance_id": item["utterance_id"],
        "program": item["program"],
        "program_id": item["program_id"],
        "relative_path": item["relative_path"],
        "utterance_index": item["utterance_index"],
        "speaker_raw": item["speaker_raw"],
        "speaker_canonical": item["speaker_canonical"],
        "speaker_status": item["speaker_status"],
        "boundary_type": item["boundary_type"],
        "text_length": text_length,
        "text": item["text"],
    })


def structurally_merge_compatible(left, right):
    if left["relative_path"] != right["relative_path"]:
        return False

    if (
        right["utterance_index"]
        != left["utterance_index"] + 1
    ):
        return False

    if left["speaker_status"] in {
        "UNRESOLVED",
        "NARRATIVE_CUE",
    }:
        return False

    if right["speaker_status"] in {
        "UNRESOLVED",
        "NARRATIVE_CUE",
    }:
        return False

    if left["speaker_canonical"] is None:
        return False

    if right["speaker_canonical"] is None:
        return False

    if (
        left["speaker_canonical"]
        != right["speaker_canonical"]
    ):
        return False

    if (
        left["speaker_raw"] is not None
        and right["speaker_raw"] is not None
        and left["speaker_raw"]
        != right["speaker_raw"]
    ):
        return False

    if left["boundary_type"] in {
        "BOLD_ONLY_SPEAKER_BOUNDARY",
        "COLON_SPEAKER_BOUNDARY",
    }:
        return False

    if right["boundary_type"] in {
        "BOLD_ONLY_SPEAKER_BOUNDARY",
        "COLON_SPEAKER_BOUNDARY",
    }:
        return False

    return True


merge_candidate_pairs = []

for parsed_file in parsed_transcript_files_v1:
    utterances = parsed_file["utterances"]

    for index in range(len(utterances) - 1):
        left = utterances[index]
        right = utterances[index + 1]

        if not structurally_merge_compatible(
            left,
            right,
        ):
            continue

        merge_candidate_pairs.append({
            "program": left["program"],
            "program_id": left["program_id"],
            "relative_path": left["relative_path"],
            "left_utterance_id": left["utterance_id"],
            "right_utterance_id": right["utterance_id"],
            "left_length": len(
                left["text"].strip()
            ),
            "right_length": len(
                right["text"].strip()
            ),
            "left_text": left["text"],
            "right_text": right["text"],
            "speaker_raw": left["speaker_raw"],
            "speaker_canonical": (
                left["speaker_canonical"]
            ),
        })


print(
    "Parsed utterances inspected:",
    len(utterance_length_profiles),
)
print(
    "Structurally merge-compatible adjacent pairs:",
    len(merge_candidate_pairs),
)
print()

print("GLOBAL LENGTH DISTRIBUTION")
print("=" * 100)

length_values = [
    item["text_length"]
    for item in utterance_length_profiles
]

sorted_lengths = sorted(length_values)

for threshold in tiny_thresholds:
    count = sum(
        length <= threshold
        for length in length_values
    )

    print(
        f"- <= {threshold:2d} chars: "
        f"{count} utterances"
    )

print(
    "- Minimum length:",
    min(length_values),
)
print(
    "- Maximum length:",
    max(length_values),
)

for percentile in (25, 50, 75, 90, 95):
    position = int(
        round(
            (percentile / 100)
            * (len(sorted_lengths) - 1)
        )
    )

    print(
        f"- P{percentile}:",
        sorted_lengths[position],
    )


print()
print("TINY UTTERANCES BY PROGRAM")
print("=" * 100)

for program_id in sorted(files_by_program):
    program_items = [
        item
        for item in utterance_length_profiles
        if item["program_id"] == program_id
    ]

    program_name = program_items[0][
        "program"
    ]

    print()
    print("Program:", program_name)
    print("Program ID:", program_id)
    print("Utterances:", len(program_items))

    for threshold in tiny_thresholds:
        count = sum(
            item["text_length"] <= threshold
            for item in program_items
        )

        print(
            f"<= {threshold:2d} chars:",
            count,
        )

    unresolved_tiny_40 = sum(
        item["text_length"] <= 40
        and item["speaker_status"]
        in {"UNRESOLVED", "NARRATIVE_CUE"}
        for item in program_items
    )

    resolved_tiny_40 = sum(
        item["text_length"] <= 40
        and item["speaker_status"]
        not in {"UNRESOLVED", "NARRATIVE_CUE"}
        for item in program_items
    )

    print(
        "Resolved <= 40 chars:",
        resolved_tiny_40,
    )
    print(
        "Unresolved/Narrative <= 40 chars:",
        unresolved_tiny_40,
    )

    compatible_pairs = sum(
        pair["program_id"] == program_id
        for pair in merge_candidate_pairs
    )

    print(
        "Structurally compatible adjacent pairs:",
        compatible_pairs,
    )
    print("-" * 100)


print()
print("TINY UTTERANCE STATUS SUMMARY")
print("=" * 100)

for threshold in tiny_thresholds:
    status_counts = Counter(
        item["speaker_status"]
        for item in utterance_length_profiles
        if item["text_length"] <= threshold
    )

    print()
    print(
        f"Threshold <= {threshold} chars"
    )

    for status, count in sorted(
        status_counts.items()
    ):
        print(
            f"- {status}: {count}"
        )


print()
print("REPRESENTATIVE <= 20 CHAR UTTERANCES")
print("=" * 100)

representative_counts = Counter()

for item in utterance_length_profiles:
    if item["text_length"] > 20:
        continue

    key = (
        item["program_id"],
        item["speaker_status"],
    )

    if representative_counts[key] >= 3:
        continue

    representative_counts[key] += 1

    print()
    print("Program:", item["program"])
    print("File:", item["relative_path"])
    print(
        "Utterance index:",
        item["utterance_index"],
    )
    print(
        "Length:",
        item["text_length"],
    )
    print(
        "Speaker raw:",
        item["speaker_raw"],
    )
    print(
        "Speaker canonical:",
        item["speaker_canonical"],
    )
    print(
        "Speaker status:",
        item["speaker_status"],
    )
    print(
        "Boundary type:",
        item["boundary_type"],
    )
    print(
        "Text:",
        repr(item["text"]),
    )
    print("-" * 100)


print()
print("REPRESENTATIVE STRUCTURALLY COMPATIBLE PAIRS")
print("=" * 100)

if merge_candidate_pairs:
    shown_by_program = Counter()

    for pair in merge_candidate_pairs:
        if shown_by_program[
            pair["program_id"]
        ] >= 3:
            continue

        shown_by_program[
            pair["program_id"]
        ] += 1

        print()
        print("Program:", pair["program"])
        print("File:", pair["relative_path"])
        print(
            "Speaker raw:",
            pair["speaker_raw"],
        )
        print(
            "Speaker canonical:",
            pair["speaker_canonical"],
        )
        print(
            "Left length:",
            pair["left_length"],
        )
        print(
            "Left text:",
            repr(pair["left_text"][:220]),
        )
        print(
            "Right length:",
            pair["right_length"],
        )
        print(
            "Right text:",
            repr(pair["right_text"][:220]),
        )
        print("-" * 100)
else:
    print("NONE")


assert len(
    utterance_length_profiles
) == len(all_parsed_utterances_v1), (
    "Tiny-utterance discovery did not inspect all parsed utterances."
)

assert all(
    item["text_length"] > 0
    for item in utterance_length_profiles
), (
    "Empty utterance reached tiny-utterance discovery."
)

assert all(
    pair["speaker_canonical"] is not None
    for pair in merge_candidate_pairs
), (
    "Merge-compatible pair contains unresolved canonical speaker."
)

assert all(
    pair["relative_path"]
    in parsed_file_by_path
    for pair in merge_candidate_pairs
), (
    "Merge-compatible pair references an unknown parsed file."
)

print()
print("TINY UTTERANCE MERGE DISCOVERY: PASS")

Parsed utterances inspected: 15131
Structurally merge-compatible adjacent pairs: 10912

GLOBAL LENGTH DISTRIBUTION
- <= 10 chars: 515 utterances
- <= 20 chars: 2597 utterances
- <= 40 chars: 8063 utterances
- Minimum length: 1
- Maximum length: 10564
- P25: 25
- P50: 38
- P75: 73
- P90: 322
- P95: 598

TINY UTTERANCES BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Utterances: 902
<= 10 chars: 38
<= 20 chars: 95
<= 40 chars: 183
Resolved <= 40 chars: 183
Unresolved/Narrative <= 40 chars: 0
Structurally compatible adjacent pairs: 0
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Utterances: 453
<= 10 chars: 0
<= 20 chars: 1
<= 40 chars: 8
Resolved <= 40 chars: 8
Unresolved/Narrative <= 40 chars: 0
Structurally compatible adjacent pairs: 442
----------------------------------------------------------------------------------------------------

Program: چهلستون
Program ID: 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### بررسی Adjacent Fragmentها با Speaker یکسان پیش از تعریف Merge Rule

Discovery قبلی نشان داد که Tiny Utteranceها در Corpus فراوان‌اند.

اما کوتاه‌بودن متن به‌تنهایی دلیل Merge نیست.

نمونه‌هایی مانند:

- `سلام`
- `از کجا شروع کنیم؟`
- `چرا؟`

می‌توانند Turnهای واقعی و مستقل باشند.

در مقابل، نمونه‌هایی مانند:

- `به نظرتون با`
- `کنید.`

می‌توانند نشانه Fragmentation ناشی از Source Segmentation باشند.

بنابراین قبل از ساخت Merge Rule باید فقط Adjacent Pairهایی را بررسی کنیم که Speaker یکسان آن‌ها با Evidence مستقیم قابل اثبات است.

### Evidence مجاز برای Same Speaker

دو Utterance متوالی فقط زمانی Same-speaker Candidate محسوب می‌شوند که حداقل یکی از این Evidenceها وجود داشته باشد:

- `speaker_raw` هر دو دقیقاً یکسان و غیرخالی باشد.
- `speaker_source_canonical` هر دو دقیقاً یکسان باشد.
- `speaker_canonical` هر دو از Manifest به یک Speaker واحد متصل شده باشد.

هیچ تشابه اسمی یا حدس هویت استفاده نمی‌شود.

### بررسی فاصله Source

بین دو Utterance ممکن است خطی در Parsed Payload حضور نداشته باشد.

این فاصله فقط زمانی Structural محسوب می‌شود که تمام خطوط غیرخالی بین دو Utterance از Markerهای تأییدشده باشند؛ مانند:

- Marker تکراری Solo
- Speaker Marker تأییدشده Bold
- Speaker Marker تأییدشده چهلستون

وجود هر خط محتوایی دیگر در فاصله دو Utterance باعث می‌شود Pair برای Merge ایمن محسوب نشود.

### Tiny Pair

برای Inspection فعلی، Pair زمانی Tiny Candidate است که حداقل یکی از دو طرف:

`<= 40 characters`

باشد.

این Threshold هنوز Merge Rule نیست.

برای Candidateهای بسیار کوتاه‌تر، یعنی زمانی که حداقل یک طرف:

`<= 20 characters`

باشد، نمونه‌های بیشتری نمایش داده می‌شوند.

### نکته مهم درباره Explicit Speaker Markers

تکرار یک Speaker Marker یکسان لزوماً به معنی Speaker Change نیست.

برای مثال ممکن است Source چند Segment متوالی از یک Speaker را با Label یکسان ثبت کرده باشد.

در این مرحله چنین مواردی فقط شناسایی می‌شوند و هنوز ادغام نمی‌شوند.

### هدف این Cell

مشخص‌کردن این است که:

- چه تعداد Tiny Pair واقعاً Same-speaker هستند.
- در کدام Programها متمرکز شده‌اند.
- Evidence Same-speaker آن‌ها چیست.
- آیا بین آن‌ها فقط Structural Marker وجود دارد.
- نمونه‌های Fragment-like چه شکلی دارند.

هنوز انجام نمی‌شوند:

- Merge
- اصلاح غلط املایی
- Persian normalization
- ASR cleanup
- Deduplication
- Chunking
- Artifact write

**معیار PASS:**

- تمام Adjacent Pairهای `15,131` Utterance بررسی شوند.
- Same-speaker Evidence فقط از Metadata تأییدشده ساخته شود.
- Tiny Same-speaker Pairها جداگانه گزارش شوند.
- هر Gap بین دو Utterance از Source اصلی دوباره بررسی شود.
- هیچ Pair دارای خط محتوایی گمشده به‌عنوان Structurally Clean اعلام نشود.
- Parsed Output در این Cell تغییر نکند.

</div>

In [23]:
# Code Cell 23

tiny_pair_threshold = 40
very_tiny_pair_threshold = 20


def same_speaker_evidence(left, right):
    if (
        left["speaker_raw"] is not None
        and right["speaker_raw"] is not None
        and left["speaker_raw"]
        == right["speaker_raw"]
    ):
        return "RAW_EXACT"

    if (
        left["speaker_source_canonical"] is not None
        and right["speaker_source_canonical"] is not None
        and left["speaker_source_canonical"]
        == right["speaker_source_canonical"]
    ):
        return "SOURCE_CANONICAL_EXACT"

    if (
        left["speaker_canonical"] is not None
        and right["speaker_canonical"] is not None
        and left["speaker_canonical"]
        == right["speaker_canonical"]
    ):
        return "MANIFEST_CANONICAL_EXACT"

    return None


def classify_intervening_line(
    record,
    relative_path,
    raw_line,
):
    program_id = record["program_id"]

    if is_solo_structural_marker(
        record,
        raw_line,
    ):
        return "SOLO_MANIFEST_MARKER"

    bold_value = extract_bold_only_value(
        raw_line
    )

    if (
        program_id
        in verified_bold_speaker_markers
        and bold_value
        in set(
            verified_bold_speaker_markers[
                program_id
            ]
        )
    ):
        return "VERIFIED_BOLD_SPEAKER_MARKER"

    if (
        program_id == chehelston_program_id
        and relative_path
        not in chehelston_contextual_path_set
    ):
        match = plain_colon_pattern.match(
            raw_line
        )

        prefix = (
            match.group(1).strip()
            if match
            else None
        )

        if (
            prefix
            in final_chehelston_speaker_markers
        ):
            return (
                "VERIFIED_CHEHELSTON_SPEAKER_MARKER"
            )

    return "OTHER_NONEMPTY"


same_speaker_adjacent_pairs = []
tiny_same_speaker_pairs = []

for parsed_file in parsed_transcript_files_v1:
    relative_path = parsed_file[
        "relative_path"
    ]

    record = accepted_record_by_path[
        relative_path
    ]

    file_path = project_root / relative_path
    lines = file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    utterances = parsed_file[
        "utterances"
    ]

    for index in range(
        len(utterances) - 1
    ):
        left = utterances[index]
        right = utterances[index + 1]

        evidence = same_speaker_evidence(
            left,
            right,
        )

        if evidence is None:
            continue

        gap_items = []

        for source_index in range(
            left["source_line_end"],
            right["source_line_start"] - 1,
        ):
            raw_line = lines[source_index]

            if not raw_line.strip():
                continue

            gap_items.append({
                "line_number": (
                    source_index + 1
                ),
                "raw_line": raw_line,
                "classification": (
                    classify_intervening_line(
                        record,
                        relative_path,
                        raw_line,
                    )
                ),
            })

        gap_is_structural = all(
            item["classification"]
            != "OTHER_NONEMPTY"
            for item in gap_items
        )

        left_length = len(
            left["text"].strip()
        )

        right_length = len(
            right["text"].strip()
        )

        pair = {
            "program": left["program"],
            "program_id": left["program_id"],
            "relative_path": relative_path,
            "left_utterance_index": (
                left["utterance_index"]
            ),
            "right_utterance_index": (
                right["utterance_index"]
            ),
            "left_utterance_id": (
                left["utterance_id"]
            ),
            "right_utterance_id": (
                right["utterance_id"]
            ),
            "same_speaker_evidence": (
                evidence
            ),
            "speaker_raw_left": (
                left["speaker_raw"]
            ),
            "speaker_raw_right": (
                right["speaker_raw"]
            ),
            "speaker_canonical": (
                left["speaker_canonical"]
                if (
                    left["speaker_canonical"]
                    == right["speaker_canonical"]
                )
                else None
            ),
            "speaker_source_canonical": (
                left[
                    "speaker_source_canonical"
                ]
                if (
                    left[
                        "speaker_source_canonical"
                    ]
                    == right[
                        "speaker_source_canonical"
                    ]
                )
                else None
            ),
            "left_boundary_type": (
                left["boundary_type"]
            ),
            "right_boundary_type": (
                right["boundary_type"]
            ),
            "left_length": left_length,
            "right_length": right_length,
            "left_text": left["text"],
            "right_text": right["text"],
            "left_source_lines": (
                left["source_line_start"],
                left["source_line_end"],
            ),
            "right_source_lines": (
                right["source_line_start"],
                right["source_line_end"],
            ),
            "intervening_nonempty_lines": (
                gap_items
            ),
            "gap_is_structural": (
                gap_is_structural
            ),
        }

        same_speaker_adjacent_pairs.append(
            pair
        )

        if (
            min(
                left_length,
                right_length,
            )
            <= tiny_pair_threshold
        ):
            tiny_same_speaker_pairs.append(
                pair
            )


print(
    "Parsed files inspected:",
    len(parsed_transcript_files_v1),
)
print(
    "Parsed utterances:",
    len(all_parsed_utterances_v1),
)
print(
    "Same-speaker adjacent pairs:",
    len(same_speaker_adjacent_pairs),
)
print(
    "Tiny same-speaker pairs "
    f"(one side <= {tiny_pair_threshold} chars):",
    len(tiny_same_speaker_pairs),
)
print(
    "Very-tiny same-speaker pairs "
    f"(one side <= {very_tiny_pair_threshold} chars):",
    sum(
        min(
            pair["left_length"],
            pair["right_length"],
        )
        <= very_tiny_pair_threshold
        for pair in tiny_same_speaker_pairs
    ),
)
print()


print("SAME-SPEAKER EVIDENCE COUNTS")
print("=" * 100)

evidence_counts = Counter(
    pair["same_speaker_evidence"]
    for pair in same_speaker_adjacent_pairs
)

for evidence, count in sorted(
    evidence_counts.items()
):
    print(
        f"- {evidence}: {count}"
    )


print()
print("TINY SAME-SPEAKER PAIRS BY PROGRAM")
print("=" * 100)

for program_id in sorted(files_by_program):
    program_pairs = [
        pair
        for pair in tiny_same_speaker_pairs
        if pair["program_id"]
        == program_id
    ]

    if not program_pairs:
        continue

    program_name = program_pairs[0][
        "program"
    ]

    structurally_clean = sum(
        pair["gap_is_structural"]
        for pair in program_pairs
    )

    both_tiny = sum(
        pair["left_length"]
        <= tiny_pair_threshold
        and pair["right_length"]
        <= tiny_pair_threshold
        for pair in program_pairs
    )

    very_tiny = sum(
        min(
            pair["left_length"],
            pair["right_length"],
        )
        <= very_tiny_pair_threshold
        for pair in program_pairs
    )

    print()
    print("Program:", program_name)
    print("Program ID:", program_id)
    print(
        "Tiny same-speaker pairs:",
        len(program_pairs),
    )
    print(
        "Structurally clean gaps:",
        structurally_clean,
    )
    print(
        "Pairs with both sides <= 40:",
        both_tiny,
    )
    print(
        "Pairs with one side <= 20:",
        very_tiny,
    )

    program_evidence = Counter(
        pair["same_speaker_evidence"]
        for pair in program_pairs
    )

    print(
        "Evidence:",
        dict(
            sorted(
                program_evidence.items()
            )
        ),
    )
    print("-" * 100)


nonstructural_gap_pairs = [
    pair
    for pair in same_speaker_adjacent_pairs
    if not pair["gap_is_structural"]
]


print()
print("INTERVENING GAP VALIDATION")
print("=" * 100)
print(
    "Pairs with non-structural "
    "intervening content:",
    len(nonstructural_gap_pairs),
)

if nonstructural_gap_pairs:
    for pair in nonstructural_gap_pairs[:10]:
        print()
        print("Program:", pair["program"])
        print("File:", pair["relative_path"])
        print(
            "Left utterance:",
            pair["left_utterance_index"],
        )
        print(
            "Right utterance:",
            pair["right_utterance_index"],
        )
        print(
            "Gap lines:",
            json.dumps(
                pair[
                    "intervening_nonempty_lines"
                ],
                ensure_ascii=False,
                indent=2,
            ),
        )


print()
print(
    "REPRESENTATIVE VERY-TINY "
    "SAME-SPEAKER PAIRS"
)
print("=" * 100)

shown_by_program = Counter()

for pair in tiny_same_speaker_pairs:
    if (
        min(
            pair["left_length"],
            pair["right_length"],
        )
        > very_tiny_pair_threshold
    ):
        continue

    if shown_by_program[
        pair["program_id"]
    ] >= 5:
        continue

    shown_by_program[
        pair["program_id"]
    ] += 1

    print()
    print("Program:", pair["program"])
    print("File:", pair["relative_path"])
    print(
        "Same-speaker evidence:",
        pair["same_speaker_evidence"],
    )
    print(
        "Speaker raw left:",
        pair["speaker_raw_left"],
    )
    print(
        "Speaker raw right:",
        pair["speaker_raw_right"],
    )
    print(
        "Speaker canonical:",
        pair["speaker_canonical"],
    )
    print(
        "Left boundary:",
        pair["left_boundary_type"],
    )
    print(
        "Right boundary:",
        pair["right_boundary_type"],
    )
    print(
        "Left source lines:",
        pair["left_source_lines"],
    )
    print(
        "Right source lines:",
        pair["right_source_lines"],
    )
    print(
        "Gap structurally clean:",
        pair["gap_is_structural"],
    )

    if pair[
        "intervening_nonempty_lines"
    ]:
        print("Intervening lines:")

        for gap_item in pair[
            "intervening_nonempty_lines"
        ]:
            print(
                " ",
                gap_item[
                    "line_number"
                ],
                "|",
                gap_item[
                    "classification"
                ],
                "|",
                gap_item[
                    "raw_line"
                ],
            )
    else:
        print(
            "Intervening lines: NONE"
        )

    print(
        "Left:",
        f"({pair['left_length']} chars)",
        repr(
            pair["left_text"][:300]
        ),
    )

    print(
        "Right:",
        f"({pair['right_length']} chars)",
        repr(
            pair["right_text"][:300]
        ),
    )

    print("-" * 100)


assert len(
    all_parsed_utterances_v1
) == 15131, (
    "Parsed utterance population changed before adjacency discovery."
)

assert all(
    pair["relative_path"]
    in parsed_file_by_path
    for pair in same_speaker_adjacent_pairs
), (
    "Same-speaker pair references an unknown parsed file."
)

assert all(
    pair["same_speaker_evidence"]
    in {
        "RAW_EXACT",
        "SOURCE_CANONICAL_EXACT",
        "MANIFEST_CANONICAL_EXACT",
    }
    for pair in same_speaker_adjacent_pairs
), (
    "Unsupported same-speaker evidence found."
)

assert not nonstructural_gap_pairs, (
    "At least one adjacent same-speaker pair "
    "contains unexplained Source content between utterances."
)

print()
print(
    "SAME-SPEAKER TINY ADJACENCY "
    "DISCOVERY: PASS"
)

Parsed files inspected: 183
Parsed utterances: 15131
Same-speaker adjacent pairs: 11041
Tiny same-speaker pairs (one side <= 40 chars): 8963
Very-tiny same-speaker pairs (one side <= 20 chars): 3592

SAME-SPEAKER EVIDENCE COUNTS
- MANIFEST_CANONICAL_EXACT: 3621
- RAW_EXACT: 7420

TINY SAME-SPEAKER PAIRS BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Tiny same-speaker pairs: 29
Structurally clean gaps: 29
Pairs with both sides <= 40: 10
Pairs with one side <= 20: 13
Evidence: {'RAW_EXACT': 29}
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Tiny same-speaker pairs: 15
Structurally clean gaps: 15
Pairs with both sides <= 40: 0
Pairs with one side <= 20: 1
Evidence: {'MANIFEST_CANONICAL_EXACT': 15}
----------------------------------------------------------------------------------------------------

Program: در جستجوی خوشبختی
Program ID: program:dar-jostojo-ye-khoshbakhti

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### تشخیص محافظه‌کارانه Fragment-likelihood پیش از Merge

Discovery قبلی ثابت کرد که Adjacent Pairهای زیادی با Speaker یکسان وجود دارند و هیچ محتوای توضیح‌ناپذیری بین آن‌ها قرار نگرفته است.

اما:

`Same Speaker + Tiny Text`

به‌تنهایی مجوز Merge نیست.

برای مثال:

- `سلام`
- `موفق باشید.`
- `چرا؟`
- `از کجا شروع کنیم؟`

می‌توانند Utteranceهای واقعی و مستقل باشند.

در مقابل نمونه‌هایی مانند:

- `به نظرتون با`
- `و این رو...`
- `یا حتی فاض کارآفرینی`
- `کنید.`

می‌توانند Fragmentهای ناشی از Segmentation Source باشند.

در این مرحله یک Classification محافظه‌کارانه ساخته می‌شود.

### Signalهای Fragment-like

Signalها فقط برای Discovery هستند و هنوز متن را تغییر نمی‌دهند.

#### پایان ناتمام

اگر بخش چپ با نشانه‌هایی مانند:

- `...`
- `…`
- `،`
- `,`
- `:`

تمام شود، احتمال ادامه‌داشتن جمله بیشتر است.

#### واژه آویزان در انتهای بخش چپ

پایان‌هایی مانند:

- `با`
- `از`
- `به`
- `در`
- `برای`
- `که`
- `و`
- `یا`
- `تا`

می‌توانند نشانه قطع‌شدن جمله در نقطه نامناسب باشند.

#### شروع ادامه‌دهنده در بخش راست

شروع‌هایی مانند:

- `و`
- `یا`
- `که`
- `تا`
- `چون`
- `ولی`
- `اما`
- `بنابراین`

در صورتی که بخش قبلی پایان جمله قوی نداشته باشد، می‌توانند Evidence ادامه جمله باشند.

#### Fragment بسیار کوتاه

یک بخش بسیار کوتاه ممکن است انتهای جمله قبلی باشد؛ اما این Signal به‌تنهایی برای Merge کافی نیست.

بنابراین Utteranceهای کوتاهی مانند `سلام` یا `خب` به‌طور خودکار Merge نمی‌شوند.

### سه Classification

هر Tiny Same-speaker Pair به یکی از این گروه‌ها می‌رود:

- `FRAGMENT_LIKELY`
- `AMBIGUOUS_TINY_PAIR`
- `STANDALONE_LIKELY`

هیچ‌کدام در این Cell Merge نمی‌شوند.

### درباره غلط‌های متن

موارد مشکوک املایی یا ASR مانند:

`فاض کارآفرینی`

در این مرحله اصلاح نمی‌شوند.

Raw Source باید دست‌نخورده بماند.

در مراحل Quality Detection، چنین مواردی باید به Correction/Review Ledger منتقل شوند تا:

- Source File
- Line
- Raw Text
- Candidate Correction
- Evidence

قابل Audit باشد.

### هدف این Cell

پیش از اجرای Merge مشخص کنیم:

- چه تعداد Pair واقعاً Fragment-like هستند.
- Signal اصلی هر Candidate چیست.
- این Candidateها در کدام Programها متمرکزند.
- آیا Rule پیشنهادی نمونه‌های واضح Fragment را پیدا می‌کند.
- آیا Turnهای مستقل کوتاه از Candidateهای قوی جدا می‌مانند.

هنوز انجام نمی‌شوند:

- Merge
- Spell correction
- ASR correction
- Persian normalization
- Deduplication
- Chunking
- Artifact write

**معیار PASS:**

- تمام Tiny Same-speaker Pairهای کشف‌شده بررسی شوند.
- Classification برای همه آن‌ها وجود داشته باشد.
- هیچ Text یا Lineage تغییر نکند.
- Fragment Signalها به‌صورت صریح گزارش شوند.
- نمونه‌های هر Classification قابل Inspection باشند.

</div>

In [24]:
# Code Cell 24

fragment_tiny_threshold = 40
ultra_tiny_threshold = 10

continuation_endings = (
    "...",
    "…",
    "،",
    ",",
    ":",
)

hanging_end_tokens = {
    "با",
    "از",
    "به",
    "در",
    "برای",
    "که",
    "و",
    "یا",
    "تا",
    "چون",
    "اگر",
}

continuation_start_tokens = {
    "و",
    "یا",
    "که",
    "تا",
    "چون",
    "ولی",
    "اما",
    "بنابراین",
    "پس",
}

protected_short_standalones = {
    "سلام",
    "خب",
    "بله",
    "نه",
    "حتماً",
    "ممنون",
    "متشکرم",
    "چرا؟",
}


def last_token_raw(text):
    parts = text.strip().split()

    if not parts:
        return ""

    return parts[-1].strip(
        "«»\"'()[]{}."
        "!?؟؛،,:…"
    )


def first_token_raw(text):
    parts = text.strip().split()

    if not parts:
        return ""

    return parts[0].strip(
        "«»\"'()[]{}."
        "!?؟؛،,:…"
    )


def has_strong_terminal(text):
    stripped = text.rstrip()

    while (
        stripped
        and stripped[-1]
        in "»”\"')]} "
    ):
        stripped = stripped[:-1].rstrip()

    return stripped.endswith(
        (
            ".",
            "!",
            "?",
            "؟",
        )
    )


def ends_with_continuation_signal(text):
    stripped = text.rstrip()

    return stripped.endswith(
        continuation_endings
    )


def classify_fragment_pair(pair):
    left_text = pair["left_text"].strip()
    right_text = pair["right_text"].strip()

    left_length = pair["left_length"]
    right_length = pair["right_length"]

    left_last_token = last_token_raw(
        left_text
    )

    right_first_token = first_token_raw(
        right_text
    )

    signals = []

    if (
        left_length <= fragment_tiny_threshold
        and ends_with_continuation_signal(
            left_text
        )
    ):
        signals.append(
            "LEFT_CONTINUATION_PUNCTUATION"
        )

    if (
        left_length <= fragment_tiny_threshold
        and left_last_token
        in hanging_end_tokens
    ):
        signals.append(
            "LEFT_HANGING_CONNECTOR"
        )

    if (
        right_length <= fragment_tiny_threshold
        and right_first_token
        in continuation_start_tokens
        and not has_strong_terminal(
            left_text
        )
    ):
        signals.append(
            "RIGHT_CONTINUATION_START"
        )

    right_is_protected = (
        right_text
        in protected_short_standalones
    )

    left_is_protected = (
        left_text
        in protected_short_standalones
    )

    if (
        right_length <= ultra_tiny_threshold
        and not right_is_protected
        and not has_strong_terminal(
            left_text
        )
    ):
        signals.append(
            "ULTRA_SHORT_RIGHT_AFTER_NONTERMINAL"
        )

    if (
        left_length <= ultra_tiny_threshold
        and not left_is_protected
        and ends_with_continuation_signal(
            left_text
        )
    ):
        signals.append(
            "ULTRA_SHORT_LEFT_CONTINUATION"
        )

    strong_fragment_signals = {
        "LEFT_CONTINUATION_PUNCTUATION",
        "LEFT_HANGING_CONNECTOR",
    }

    supporting_fragment_signals = {
        "RIGHT_CONTINUATION_START",
        "ULTRA_SHORT_RIGHT_AFTER_NONTERMINAL",
        "ULTRA_SHORT_LEFT_CONTINUATION",
    }

    if any(
        signal in strong_fragment_signals
        for signal in signals
    ):
        classification = "FRAGMENT_LIKELY"

    elif len(
        set(signals)
        & supporting_fragment_signals
    ) >= 2:
        classification = "FRAGMENT_LIKELY"

    elif signals:
        classification = "AMBIGUOUS_TINY_PAIR"

    else:
        classification = "STANDALONE_LIKELY"

    return {
        "classification": classification,
        "signals": signals,
        "left_last_token": left_last_token,
        "right_first_token": right_first_token,
        "left_strong_terminal": (
            has_strong_terminal(
                left_text
            )
        ),
        "left_protected_short": (
            left_is_protected
        ),
        "right_protected_short": (
            right_is_protected
        ),
    }


fragment_pair_profiles = []

for pair in tiny_same_speaker_pairs:
    result = classify_fragment_pair(
        pair
    )

    fragment_pair_profiles.append({
        **pair,
        **result,
    })


print(
    "Tiny same-speaker pairs inspected:",
    len(fragment_pair_profiles),
)
print()

print("FRAGMENT CLASSIFICATION COUNTS")
print("=" * 100)

classification_counts = Counter(
    item["classification"]
    for item in fragment_pair_profiles
)

for classification, count in sorted(
    classification_counts.items()
):
    print(
        f"- {classification}: {count}"
    )


print()
print("FRAGMENT SIGNAL COUNTS")
print("=" * 100)

signal_counts = Counter(
    signal
    for item in fragment_pair_profiles
    for signal in item["signals"]
)

if signal_counts:
    for signal, count in sorted(
        signal_counts.items()
    ):
        print(
            f"- {signal}: {count}"
        )
else:
    print("- NONE")


print()
print("FRAGMENT-LIKELY PAIRS BY PROGRAM")
print("=" * 100)

for program_id in sorted(files_by_program):
    program_items = [
        item
        for item in fragment_pair_profiles
        if (
            item["program_id"]
            == program_id
        )
    ]

    if not program_items:
        continue

    program_name = program_items[0][
        "program"
    ]

    program_counts = Counter(
        item["classification"]
        for item in program_items
    )

    print()
    print("Program:", program_name)
    print("Program ID:", program_id)
    print(
        "Tiny pairs:",
        len(program_items),
    )

    for classification in (
        "FRAGMENT_LIKELY",
        "AMBIGUOUS_TINY_PAIR",
        "STANDALONE_LIKELY",
    ):
        print(
            f"{classification}:",
            program_counts[
                classification
            ],
        )

    print("-" * 100)


print()
print("REPRESENTATIVE FRAGMENT-LIKELY PAIRS")
print("=" * 100)

shown_by_program = Counter()

for item in fragment_pair_profiles:
    if (
        item["classification"]
        != "FRAGMENT_LIKELY"
    ):
        continue

    if shown_by_program[
        item["program_id"]
    ] >= 5:
        continue

    shown_by_program[
        item["program_id"]
    ] += 1

    print()
    print("Program:", item["program"])
    print("File:", item["relative_path"])
    print(
        "Same-speaker evidence:",
        item["same_speaker_evidence"],
    )
    print(
        "Signals:",
        item["signals"],
    )
    print(
        "Left length:",
        item["left_length"],
    )
    print(
        "Left:",
        repr(item["left_text"][:300]),
    )
    print(
        "Right length:",
        item["right_length"],
    )
    print(
        "Right:",
        repr(item["right_text"][:300]),
    )
    print(
        "Intervening lines:",
        len(
            item[
                "intervening_nonempty_lines"
            ]
        ),
    )
    print("-" * 100)


print()
print("REPRESENTATIVE AMBIGUOUS TINY PAIRS")
print("=" * 100)

shown_ambiguous_by_program = Counter()

for item in fragment_pair_profiles:
    if (
        item["classification"]
        != "AMBIGUOUS_TINY_PAIR"
    ):
        continue

    if shown_ambiguous_by_program[
        item["program_id"]
    ] >= 3:
        continue

    shown_ambiguous_by_program[
        item["program_id"]
    ] += 1

    print()
    print("Program:", item["program"])
    print("File:", item["relative_path"])
    print(
        "Signals:",
        item["signals"],
    )
    print(
        "Left:",
        repr(item["left_text"][:260]),
    )
    print(
        "Right:",
        repr(item["right_text"][:260]),
    )
    print("-" * 100)


print()
print("PROTECTED SHORT STANDALONE CHECK")
print("=" * 100)

protected_hits = []

for item in fragment_pair_profiles:
    if (
        item["left_protected_short"]
        or item["right_protected_short"]
    ):
        protected_hits.append(
            item
        )

print(
    "Pairs containing protected short standalone:",
    len(protected_hits),
)

protected_fragment_likely = [
    item
    for item in protected_hits
    if (
        item["classification"]
        == "FRAGMENT_LIKELY"
    )
]

print(
    "Protected pairs classified FRAGMENT_LIKELY:",
    len(protected_fragment_likely),
)


assert len(
    fragment_pair_profiles
) == len(tiny_same_speaker_pairs), (
    "Not all tiny same-speaker pairs were classified."
)

assert all(
    item["classification"]
    in {
        "FRAGMENT_LIKELY",
        "AMBIGUOUS_TINY_PAIR",
        "STANDALONE_LIKELY",
    }
    for item in fragment_pair_profiles
), (
    "Unexpected fragment classification found."
)

assert all(
    item["gap_is_structural"]
    for item in fragment_pair_profiles
), (
    "Fragment discovery includes a pair with non-structural gap."
)

assert all(
    item["same_speaker_evidence"]
    in {
        "RAW_EXACT",
        "SOURCE_CANONICAL_EXACT",
        "MANIFEST_CANONICAL_EXACT",
    }
    for item in fragment_pair_profiles
), (
    "Fragment discovery contains unsupported speaker evidence."
)

print()
print(
    "CONSERVATIVE FRAGMENT-LIKELIHOOD "
    "DISCOVERY: PASS"
)

Tiny same-speaker pairs inspected: 8963

FRAGMENT CLASSIFICATION COUNTS
- AMBIGUOUS_TINY_PAIR: 1594
- FRAGMENT_LIKELY: 480
- STANDALONE_LIKELY: 6889

FRAGMENT SIGNAL COUNTS
- LEFT_CONTINUATION_PUNCTUATION: 356
- LEFT_HANGING_CONNECTOR: 97
- RIGHT_CONTINUATION_START: 1472
- ULTRA_SHORT_LEFT_CONTINUATION: 8
- ULTRA_SHORT_RIGHT_AFTER_NONTERMINAL: 237

FRAGMENT-LIKELY PAIRS BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Tiny pairs: 29
FRAGMENT_LIKELY: 9
AMBIGUOUS_TINY_PAIR: 0
STANDALONE_LIKELY: 20
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Tiny pairs: 15
FRAGMENT_LIKELY: 0
AMBIGUOUS_TINY_PAIR: 0
STANDALONE_LIKELY: 15
----------------------------------------------------------------------------------------------------

Program: در جستجوی خوشبختی
Program ID: program:dar-jostojo-ye-khoshbakhti
Tiny pairs: 6278
FRAGMENT_LIKELY: 84
AMBIGUOUS_TINY_PAIR: 1200
STANDALONE_LIKE

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

### ساخت Conservative Merge Plan بدون تغییر Parsed Output

Fragment-likelihood Discovery نشان داد که تعدادی Candidate قوی برای Fragmentation وجود دارد.

اما حتی `FRAGMENT_LIKELY` بودن نیز به‌تنهایی مجوز Merge خودکار نیست.

برای حفظ Speaker Boundaryهای واقعی، Auto-merge در نسخه اول فقط برای Policy زیر مجاز خواهد بود:

`HUMAN_CONFIRMED_SOLO`

در این Programها:

- Speaker اصلی با Evidence انسانی Manifest تأیید شده است.
- `default_body_attribution_allowed = true`
- Unknown-speaker inference ممنوع باقی می‌ماند.
- Merge باعث حدس هویت Speaker نمی‌شود.

### Programهای Explicit

در Programهایی مانند:

- عصر شیرین
- خانه تو
- یونی تهران

حتی اگر دو Turn متوالی `speaker_raw` یکسان داشته باشند، وجود Marker صریح Source یک Boundary واقعی محسوب می‌شود.

بنابراین این Boundaryها در Auto-merge اولیه عبور داده نمی‌شوند.

Candidateهای Fragment-like آن‌ها برای Review قابل نگهداری‌اند، اما Parsed Structure خودکار تغییر نمی‌کند.

### چهلستون و سایر فایل‌های unresolved

هیچ Auto-merge برای موارد زیر انجام نمی‌شود:

- Chehelston Explicit
- Chehelston Contextual
- کتاب باز
- میزان
- اقیانوس آرام
- شراکت

زیرا یا Speaker Boundary صریح و مستقل دارند یا Speaker unresolved است.

### Rule اولیه Auto-merge

یک Pair فقط وارد `merge_plan_v1` می‌شود اگر همه شرایط زیر برقرار باشند:

1. Classification برابر `FRAGMENT_LIKELY` باشد.
2. Program Policy برابر `HUMAN_CONFIRMED_SOLO` باشد.
3. دو Utterance متوالی در همان File باشند.
4. Same-speaker Evidence معتبر وجود داشته باشد.
5. هیچ Hard Explicit Boundary در دو طرف وجود نداشته باشد.
6. Gap بین دو Utterance فقط خالی باشد یا شامل `SOLO_MANIFEST_MARKER` تأییدشده باشد.
7. هیچ محتوای unresolved یا narrative در Pair وجود نداشته باشد.

### Merge Chain

ممکن است چند Pair متوالی به شکل زیر وجود داشته باشند:

`A → B`

`B → C`

در این حالت سه Utterance باید به‌عنوان یک Merge Chain واحد بررسی شوند، نه دو Merge مستقل متداخل.

بنابراین این Cell Candidate Edgeها را به Chainهای متوالی تبدیل می‌کند.

مثلاً:

`A + B + C`

یک Chain با دو Merge Edge محسوب می‌شود.

### نکته مهم

این Cell فقط Plan می‌سازد.

هیچ‌کدام از موارد زیر هنوز تغییر نمی‌کنند:

- `all_parsed_utterances_v1`
- `parsed_transcript_files_v1`
- `text`
- `source_text`
- Source Lineage
- Utterance ID

همچنان انجام نمی‌شوند:

- Actual Merge
- Spell correction
- ASR correction
- Persian normalization
- Deduplication
- Chunking
- Artifact write

**معیار PASS:**

- تمام `480` Fragment-like Pair بررسی شوند.
- Auto-merge Candidate فقط از `HUMAN_CONFIRMED_SOLO` انتخاب شود.
- هیچ Explicit Multi-speaker Boundary وارد Plan نشود.
- Gapهای Plan فقط خالی یا `SOLO_MANIFEST_MARKER` باشند.
- Edgeهای متوالی به Chainهای بدون تداخل تبدیل شوند.
- Parsed Output اصلی بدون تغییر باقی بماند.

</div>

In [25]:
# Code Cell 25

hard_explicit_boundary_types = {
    "BOLD_ONLY_SPEAKER_BOUNDARY",
    "COLON_SPEAKER_BOUNDARY",
    "REGISTERED_NARRATIVE_CUE",
}

allowed_auto_merge_gap_types = {
    "SOLO_MANIFEST_MARKER",
}


def auto_merge_exclusion_reasons(item):
    reasons = []

    program_contract = parser_contract_v1[
        "programs"
    ][item["program_id"]]

    if (
        program_contract["speaker_policy"]
        != "HUMAN_CONFIRMED_SOLO"
    ):
        reasons.append(
            "NON_SOLO_POLICY"
        )

    if (
        item["left_boundary_type"]
        in hard_explicit_boundary_types
        or item["right_boundary_type"]
        in hard_explicit_boundary_types
    ):
        reasons.append(
            "HARD_EXPLICIT_BOUNDARY"
        )

    unexpected_gap_types = sorted({
        gap_item["classification"]
        for gap_item
        in item[
            "intervening_nonempty_lines"
        ]
        if gap_item["classification"]
        not in allowed_auto_merge_gap_types
    })

    if unexpected_gap_types:
        reasons.append(
            "NON_SOLO_STRUCTURAL_GAP"
        )

    if item["same_speaker_evidence"] not in {
        "RAW_EXACT",
        "MANIFEST_CANONICAL_EXACT",
        "SOURCE_CANONICAL_EXACT",
    }:
        reasons.append(
            "UNSUPPORTED_SPEAKER_EVIDENCE"
        )

    return reasons


fragment_likely_pairs = [
    item
    for item in fragment_pair_profiles
    if (
        item["classification"]
        == "FRAGMENT_LIKELY"
    )
]

auto_merge_pair_candidates = []
excluded_fragment_pairs = []

for item in fragment_likely_pairs:
    reasons = auto_merge_exclusion_reasons(
        item
    )

    if reasons:
        excluded_fragment_pairs.append({
            **item,
            "exclusion_reasons": reasons,
        })
    else:
        auto_merge_pair_candidates.append(
            item
        )


candidate_edges_by_file = defaultdict(
    list
)

for item in auto_merge_pair_candidates:
    candidate_edges_by_file[
        item["relative_path"]
    ].append(
        item
    )


merge_chains_v1 = []

for relative_path, edges in sorted(
    candidate_edges_by_file.items()
):
    ordered_edges = sorted(
        edges,
        key=lambda item: (
            item["left_utterance_index"],
            item["right_utterance_index"],
        ),
    )

    current_chain = None

    for edge in ordered_edges:
        left_index = edge[
            "left_utterance_index"
        ]

        right_index = edge[
            "right_utterance_index"
        ]

        assert (
            right_index
            == left_index + 1
        ), (
            "Auto-merge edge is not adjacent."
        )

        if current_chain is None:
            current_chain = {
                "relative_path": relative_path,
                "program": edge["program"],
                "program_id": edge["program_id"],
                "utterance_indices": [
                    left_index,
                    right_index,
                ],
                "edge_count": 1,
                "edges": [edge],
            }

            continue

        if (
            left_index
            == current_chain[
                "utterance_indices"
            ][-1]
        ):
            current_chain[
                "utterance_indices"
            ].append(
                right_index
            )

            current_chain[
                "edge_count"
            ] += 1

            current_chain[
                "edges"
            ].append(
                edge
            )

        else:
            merge_chains_v1.append(
                current_chain
            )

            current_chain = {
                "relative_path": relative_path,
                "program": edge["program"],
                "program_id": edge["program_id"],
                "utterance_indices": [
                    left_index,
                    right_index,
                ],
                "edge_count": 1,
                "edges": [edge],
            }

    if current_chain is not None:
        merge_chains_v1.append(
            current_chain
        )


planned_utterance_ids = set()

chain_overlap_failures = []

for chain in merge_chains_v1:
    parsed_file = parsed_file_by_path[
        chain["relative_path"]
    ]

    utterance_by_index = {
        utterance[
            "utterance_index"
        ]: utterance
        for utterance
        in parsed_file["utterances"]
    }

    chain_ids = [
        utterance_by_index[index][
            "utterance_id"
        ]
        for index
        in chain[
            "utterance_indices"
        ]
    ]

    overlap = (
        set(chain_ids)
        & planned_utterance_ids
    )

    if overlap:
        chain_overlap_failures.append({
            "relative_path": (
                chain["relative_path"]
            ),
            "overlapping_ids": sorted(
                overlap
            ),
        })

    planned_utterance_ids.update(
        chain_ids
    )


print(
    "Fragment-like pairs inspected:",
    len(fragment_likely_pairs),
)
print(
    "Auto-merge candidate edges:",
    len(auto_merge_pair_candidates),
)
print(
    "Excluded fragment-like pairs:",
    len(excluded_fragment_pairs),
)
print(
    "Merge chains:",
    len(merge_chains_v1),
)
print(
    "Unique utterances involved in plan:",
    len(planned_utterance_ids),
)
print()


print("EXCLUSION REASON COUNTS")
print("=" * 100)

exclusion_reason_counts = Counter(
    reason
    for item in excluded_fragment_pairs
    for reason in item[
        "exclusion_reasons"
    ]
)

if exclusion_reason_counts:
    for reason, count in sorted(
        exclusion_reason_counts.items()
    ):
        print(
            f"- {reason}: {count}"
        )
else:
    print("- NONE")


print()
print("AUTO-MERGE PLAN BY PROGRAM")
print("=" * 100)

for program_id in sorted(files_by_program):
    program_edges = [
        item
        for item
        in auto_merge_pair_candidates
        if item["program_id"]
        == program_id
    ]

    program_chains = [
        chain
        for chain in merge_chains_v1
        if chain["program_id"]
        == program_id
    ]

    if not program_edges:
        continue

    program_name = (
        program_edges[0]["program"]
    )

    chain_sizes = [
        len(
            chain[
                "utterance_indices"
            ]
        )
        for chain in program_chains
    ]

    print()
    print("Program:", program_name)
    print("Program ID:", program_id)
    print(
        "Candidate edges:",
        len(program_edges),
    )
    print(
        "Merge chains:",
        len(program_chains),
    )
    print(
        "Utterances in chains:",
        sum(chain_sizes),
    )
    print(
        "Largest chain size:",
        max(chain_sizes),
    )
    print("-" * 100)


print()
print(
    "REPRESENTATIVE AUTO-MERGE CHAINS"
)
print("=" * 100)

shown_by_program = Counter()

for chain in merge_chains_v1:
    if shown_by_program[
        chain["program_id"]
    ] >= 5:
        continue

    shown_by_program[
        chain["program_id"]
    ] += 1

    parsed_file = parsed_file_by_path[
        chain["relative_path"]
    ]

    utterance_by_index = {
        utterance[
            "utterance_index"
        ]: utterance
        for utterance
        in parsed_file["utterances"]
    }

    print()
    print(
        "Program:",
        chain["program"],
    )
    print(
        "File:",
        chain["relative_path"],
    )
    print(
        "Utterance indices:",
        chain["utterance_indices"],
    )
    print(
        "Edge count:",
        chain["edge_count"],
    )

    for index in chain[
        "utterance_indices"
    ]:
        utterance = utterance_by_index[
            index
        ]

        print(
            " ",
            index,
            "|",
            utterance["speaker_raw"],
            "|",
            len(
                utterance[
                    "text"
                ].strip()
            ),
            "chars",
            "|",
            repr(
                utterance[
                    "text"
                ][:240]
            ),
        )

    print(
        "Signals by edge:",
        [
            edge["signals"]
            for edge in chain[
                "edges"
            ]
        ],
    )
    print("-" * 100)


print()
print(
    "EXCLUDED EXPLICIT-BOUNDARY EXAMPLES"
)
print("=" * 100)

explicit_excluded = [
    item
    for item in excluded_fragment_pairs
    if (
        "HARD_EXPLICIT_BOUNDARY"
        in item[
            "exclusion_reasons"
        ]
    )
]

print(
    "Excluded explicit-boundary pairs:",
    len(explicit_excluded),
)

for item in explicit_excluded[:10]:
    print()
    print(
        "Program:",
        item["program"],
    )
    print(
        "File:",
        item["relative_path"],
    )
    print(
        "Reasons:",
        item["exclusion_reasons"],
    )
    print(
        "Left:",
        repr(
            item["left_text"][:220]
        ),
    )
    print(
        "Right:",
        repr(
            item["right_text"][:220]
        ),
    )
    print("-" * 100)


assert len(
    fragment_likely_pairs
) == 480, (
    "Fragment-like population changed unexpectedly."
)

assert (
    len(auto_merge_pair_candidates)
    + len(excluded_fragment_pairs)
    == len(fragment_likely_pairs)
), (
    "Fragment-like pair accounting does not balance."
)

assert all(
    parser_contract_v1[
        "programs"
    ][item["program_id"]][
        "speaker_policy"
    ]
    == "HUMAN_CONFIRMED_SOLO"
    for item
    in auto_merge_pair_candidates
), (
    "Auto-merge plan contains a non-solo program."
)

assert all(
    item["left_boundary_type"]
    not in hard_explicit_boundary_types
    and item["right_boundary_type"]
    not in hard_explicit_boundary_types
    for item
    in auto_merge_pair_candidates
), (
    "Auto-merge plan crosses a hard explicit boundary."
)

assert all(
    all(
        gap_item[
            "classification"
        ]
        in allowed_auto_merge_gap_types
        for gap_item
        in item[
            "intervening_nonempty_lines"
        ]
    )
    for item
    in auto_merge_pair_candidates
), (
    "Auto-merge plan contains an unsupported intervening line."
)

assert not chain_overlap_failures, (
    "Merge chains overlap unexpectedly."
)

assert all(
    chain["edge_count"]
    == (
        len(
            chain[
                "utterance_indices"
            ]
        )
        - 1
    )
    for chain in merge_chains_v1
), (
    "Merge-chain edge accounting is invalid."
)

print()
print(
    "CONSERVATIVE TINY-UTTERANCE "
    "MERGE PLAN: PASS"
)

Fragment-like pairs inspected: 480
Auto-merge candidate edges: 471
Excluded fragment-like pairs: 9
Merge chains: 350
Unique utterances involved in plan: 821

EXCLUSION REASON COUNTS
- HARD_EXPLICIT_BOUNDARY: 9
- NON_SOLO_POLICY: 9

AUTO-MERGE PLAN BY PROGRAM

Program: در جستجوی خوشبختی
Program ID: program:dar-jostojo-ye-khoshbakhti
Candidate edges: 84
Merge chains: 80
Utterances in chains: 164
Largest chain size: 3
----------------------------------------------------------------------------------------------------

Program: فناپ
Program ID: program:fanap
Candidate edges: 1
Merge chains: 1
Utterances in chains: 2
Largest chain size: 2
----------------------------------------------------------------------------------------------------

Program: تربیت مربی
Program ID: program:tarbiat-morabi
Candidate edges: 386
Merge chains: 269
Utterances in chains: 655
Largest chain size: 10
----------------------------------------------------------------------------------------------------

REPRESENTAT

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اجرای Merge محافظه‌کارانه Tiny Fragmentها در حافظه</h3>

<p>
Merge Plan قبلی با موفقیت ساخته شد.
</p>

<p>
در آن Plan، از میان ۴۸۰ Pair دارای Fragment Signal قوی، فقط Candidateهایی وارد Auto-merge شدند که متعلق به Programهای
<code>HUMAN_CONFIRMED_SOLO</code>
بودند.
</p>

<p>
در نتیجه هیچ Speaker Boundary صریح مربوط به Programهای Multi-speaker وارد Merge Plan نشده است.
</p>

<p>
در این Cell برای اولین بار Merge واقعاً اجرا می‌شود، اما خروجی جدید فقط در حافظه Notebook ساخته خواهد شد.
</p>

<p>
Source Fileها و Parsed Output اولیه تغییر نمی‌کنند.
</p>

<h4>قاعده Merge</h4>

<p>
هر Merge Chain به یک Utterance جدید تبدیل می‌شود.
</p>

<p>
برای متن Parsed، متن Utteranceهای عضو Chain با ترتیب اصلی Source و بدون هیچ اصلاح زبانی کنار یکدیگر قرار می‌گیرند.
</p>

<p>
بین Fragmentهای قبلی از جداکننده ساختاری:
<code>\n\n</code>
استفاده می‌شود تا مرزهای اولیه در Payload قابل مشاهده باقی بمانند.
</p>

<p>
بنابراین در این مرحله هیچ واژه، فاصله فارسی، علامت نگارشی یا غلط احتمالی مانند
<code>فاض</code>
اصلاح نمی‌شود.
</p>

<h4>Source Lineage</h4>

<p>
برای Utterance Merge‌شده:
</p>

<ul>
<li><code>source_line_start</code> از اولین عضو Chain گرفته می‌شود.</li>
<li><code>source_line_end</code> از آخرین عضو Chain گرفته می‌شود.</li>
<li><code>source_text</code> دوباره مستقیماً از همان Range در Source File خوانده می‌شود.</li>
</ul>

<p>
به این ترتیب Markerهای ساختاری موجود بین Fragmentها نیز در Lineage خام باقی می‌مانند، حتی اگر در Parsed Text حذف شده باشند.
</p>

<h4>Merge Lineage</h4>

<p>
هر خروجی جدید مشخص می‌کند از کدام Utteranceهای قبل از Merge ساخته شده است.
</p>

<p>
فیلدهای کاری زیر اضافه می‌شوند:
</p>

<ul>
<li><code>merge_status</code></li>
<li><code>merged_from_utterance_ids</code></li>
<li><code>merged_from_utterance_indices</code></li>
<li><code>merge_edge_count</code></li>
<li><code>merge_signals</code></li>
<li><code>premerge_boundary_types</code></li>
</ul>

<p>
Utteranceهایی که Merge نشده‌اند نیز به‌صورت صریح
<code>UNCHANGED</code>
علامت می‌خورند.
</p>

<h4>Speaker</h4>

<p>
تمام اعضای یک Merge Chain باید Speaker Evidence سازگار داشته باشند.
</p>

<p>
اگر داخل یک Chain بیش از یک Speaker Raw، Canonical Speaker، Source-canonical Speaker یا Speaker Status ناسازگار مشاهده شود، اجرای Cell متوقف خواهد شد.
</p>

<h4>شناسه و ترتیب</h4>

<p>
برای Utteranceهای Merge‌شده یک Working ID جدید ساخته می‌شود که Range Utteranceهای اولیه را نشان می‌دهد.
</p>

<p>
پس از Merge، <code>utterance_index</code> در هر File دوباره به‌صورت متوالی از ۱ شماره‌گذاری می‌شود.
</p>

<p>
شناسه‌های اولیه در
<code>merged_from_utterance_ids</code>
حفظ می‌شوند و از بین نمی‌روند.
</p>

<h4>محدودیت مرحله</h4>

<p>
همچنان انجام نمی‌شوند:
</p>

<ul>
<li>اصلاح املایی</li>
<li>اصلاح ASR</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<p>
خروجی اولیه Parser در
<code>all_parsed_utterances_v1</code>
و
<code>parsed_transcript_files_v1</code>
نباید تغییر کند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام Merge Chainهای Plan دقیقاً یک بار اعمال شوند.</li>
<li>هیچ Chain هم‌پوشان وجود نداشته باشد.</li>
<li>کاهش تعداد Utteranceها دقیقاً برابر تعداد Merge Edgeها باشد.</li>
<li>Speaker Metadata داخل هر Chain سازگار باشد.</li>
<li>هیچ Timestamp جدیدی ساخته نشود.</li>
<li>Source Range هر Merge‌شده معتبر باشد.</li>
<li>Parsed Output اولیه بدون تغییر باقی بماند.</li>
<li>هیچ فایل یا Artifact روی Disk نوشته نشود.</li>
</ul>

</div>

In [26]:
# Code Cell 26

original_parsed_state_before_merge = json.dumps(
    all_parsed_utterances_v1,
    ensure_ascii=False,
    sort_keys=True,
)

merge_chain_by_file_and_start = {}
merge_member_indices_by_file = defaultdict(set)

for chain in merge_chains_v1:
    relative_path = chain["relative_path"]
    indices = chain["utterance_indices"]

    assert indices == list(
        range(
            indices[0],
            indices[-1] + 1,
        )
    ), (
        "Merge chain contains non-contiguous utterance indices."
    )

    key = (
        relative_path,
        indices[0],
    )

    assert key not in merge_chain_by_file_and_start, (
        "Duplicate merge-chain start found."
    )

    existing_members = (
        merge_member_indices_by_file[
            relative_path
        ]
    )

    overlap = (
        set(indices)
        & existing_members
    )

    assert not overlap, (
        "Merge chains overlap within a file: "
        f"{relative_path} | {sorted(overlap)}"
    )

    merge_chain_by_file_and_start[
        key
    ] = chain

    existing_members.update(
        indices
    )


def unique_nonnull_value(
    values,
    field_name,
    relative_path,
    indices,
):
    unique_values = {
        value
        for value in values
        if value is not None
    }

    assert len(unique_values) <= 1, (
        "Conflicting values inside merge chain: "
        f"{field_name} | "
        f"{relative_path} | "
        f"{indices} | "
        f"{sorted(unique_values)}"
    )

    if not unique_values:
        return None

    return next(
        iter(unique_values)
    )


def build_merged_chain_utterance(
    record,
    lines,
    chain,
    members,
):
    indices = chain[
        "utterance_indices"
    ]

    first = members[0]
    last = members[-1]

    speaker_raw = unique_nonnull_value(
        [
            item["speaker_raw"]
            for item in members
        ],
        "speaker_raw",
        chain["relative_path"],
        indices,
    )

    speaker_canonical = (
        unique_nonnull_value(
            [
                item[
                    "speaker_canonical"
                ]
                for item in members
            ],
            "speaker_canonical",
            chain["relative_path"],
            indices,
        )
    )

    speaker_source_canonical = (
        unique_nonnull_value(
            [
                item[
                    "speaker_source_canonical"
                ]
                for item in members
            ],
            "speaker_source_canonical",
            chain["relative_path"],
            indices,
        )
    )

    speaker_status_values = {
        item["speaker_status"]
        for item in members
    }

    assert len(
        speaker_status_values
    ) == 1, (
        "Conflicting speaker statuses inside merge chain: "
        f"{chain['relative_path']} | "
        f"{indices} | "
        f"{sorted(speaker_status_values)}"
    )

    speaker_status = next(
        iter(speaker_status_values)
    )

    assert speaker_status == (
        "MANIFEST_ATTRIBUTED"
    ), (
        "Auto-merged chain is not Manifest-attributed."
    )

    assert (
        speaker_canonical is not None
    ), (
        "Auto-merged chain has no canonical speaker."
    )

    source_line_start = first[
        "source_line_start"
    ]

    source_line_end = last[
        "source_line_end"
    ]

    assert (
        1
        <= source_line_start
        <= source_line_end
        <= len(lines)
    ), (
        "Merged source range is invalid."
    )

    source_text = "\n".join(
        lines[
            source_line_start - 1:
            source_line_end
        ]
    )

    merged_text = "\n\n".join(
        item["text"]
        for item in members
    )

    assert merged_text.strip(), (
        "Merged utterance text is empty."
    )

    assert all(
        item["start_time"] is None
        and item["end_time"] is None
        for item in members
    ), (
        "Unexpected timestamp found inside merge chain."
    )

    merge_signals = [
        {
            "left_utterance_index": (
                edge[
                    "left_utterance_index"
                ]
            ),
            "right_utterance_index": (
                edge[
                    "right_utterance_index"
                ]
            ),
            "signals": list(
                edge["signals"]
            ),
        }
        for edge in chain["edges"]
    ]

    return {
        "utterance_id": (
            f"{record['transcript_unit_id']}"
            f":merged:"
            f"{indices[0]:05d}-"
            f"{indices[-1]:05d}"
        ),
        "utterance_index": None,
        "program": first["program"],
        "program_id": first["program_id"],
        "physical_file_id": (
            first["physical_file_id"]
        ),
        "transcript_unit_id": (
            first["transcript_unit_id"]
        ),
        "relative_path": (
            first["relative_path"]
        ),
        "speaker_raw": speaker_raw,
        "speaker_canonical": (
            speaker_canonical
        ),
        "speaker_source_canonical": (
            speaker_source_canonical
        ),
        "speaker_status": (
            speaker_status
        ),
        "boundary_type": (
            "MERGED_SOLO_FRAGMENT_CHAIN"
        ),
        "text": merged_text,
        "source_text": source_text,
        "source_line_start": (
            source_line_start
        ),
        "source_line_end": (
            source_line_end
        ),
        "start_time": None,
        "end_time": None,
        "merge_status": "MERGED",
        "merged_from_utterance_ids": [
            item["utterance_id"]
            for item in members
        ],
        "merged_from_utterance_indices": (
            list(indices)
        ),
        "merge_edge_count": (
            len(members) - 1
        ),
        "merge_signals": merge_signals,
        "premerge_boundary_types": [
            item["boundary_type"]
            for item in members
        ],
    }


def build_unchanged_working_utterance(
    utterance,
):
    working = dict(
        utterance
    )

    original_index = utterance[
        "utterance_index"
    ]

    working.update({
        "merge_status": "UNCHANGED",
        "merged_from_utterance_ids": [
            utterance["utterance_id"]
        ],
        "merged_from_utterance_indices": [
            original_index
        ],
        "merge_edge_count": 0,
        "merge_signals": [],
        "premerge_boundary_types": [
            utterance["boundary_type"]
        ],
    })

    return working


parsed_transcript_files_merged_v1 = []
all_merged_utterances_v1 = []

applied_chain_count = 0
applied_merge_edge_count = 0
applied_premerge_utterance_count = 0

for parsed_file in parsed_transcript_files_v1:
    relative_path = parsed_file[
        "relative_path"
    ]

    record = accepted_record_by_path[
        relative_path
    ]

    file_path = (
        project_root
        / relative_path
    )

    lines = file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    original_utterances = (
        parsed_file["utterances"]
    )

    utterance_by_original_index = {
        item["utterance_index"]: item
        for item in original_utterances
    }

    merged_file_utterances = []

    pointer = 0

    while pointer < len(
        original_utterances
    ):
        current = original_utterances[
            pointer
        ]

        original_index = current[
            "utterance_index"
        ]

        chain = (
            merge_chain_by_file_and_start.get(
                (
                    relative_path,
                    original_index,
                )
            )
        )

        if chain is None:
            assert (
                original_index
                not in merge_member_indices_by_file[
                    relative_path
                ]
            ), (
                "Encountered non-start member of a merge chain "
                "without consuming it from the chain start."
            )

            merged_file_utterances.append(
                build_unchanged_working_utterance(
                    current
                )
            )

            pointer += 1
            continue

        chain_indices = chain[
            "utterance_indices"
        ]

        members = [
            utterance_by_original_index[
                index
            ]
            for index
            in chain_indices
        ]

        expected_pointer_members = (
            original_utterances[
                pointer:
                pointer + len(members)
            ]
        )

        assert [
            item["utterance_index"]
            for item
            in expected_pointer_members
        ] == chain_indices, (
            "Merge chain does not match physical utterance order."
        )

        merged_file_utterances.append(
            build_merged_chain_utterance(
                record=record,
                lines=lines,
                chain=chain,
                members=members,
            )
        )

        applied_chain_count += 1

        applied_merge_edge_count += (
            chain["edge_count"]
        )

        applied_premerge_utterance_count += (
            len(members)
        )

        pointer += len(
            members
        )

    for new_ordinal, utterance in enumerate(
        merged_file_utterances,
        start=1,
    ):
        utterance[
            "utterance_index"
        ] = new_ordinal

    parsed_transcript_files_merged_v1.append({
        **{
            key: value
            for key, value
            in parsed_file.items()
            if key
            not in {
                "utterances",
                "utterance_count",
            }
        },
        "premerge_utterance_count": (
            len(original_utterances)
        ),
        "utterance_count": (
            len(
                merged_file_utterances
            )
        ),
        "merge_chain_count": sum(
            item["merge_status"]
            == "MERGED"
            for item
            in merged_file_utterances
        ),
        "utterances": (
            merged_file_utterances
        ),
    })

    all_merged_utterances_v1.extend(
        merged_file_utterances
    )


expected_merge_edge_count = sum(
    chain["edge_count"]
    for chain in merge_chains_v1
)

expected_postmerge_count = (
    len(all_parsed_utterances_v1)
    - expected_merge_edge_count
)


print(
    "Pre-merge utterances:",
    len(all_parsed_utterances_v1),
)
print(
    "Planned merge chains:",
    len(merge_chains_v1),
)
print(
    "Applied merge chains:",
    applied_chain_count,
)
print(
    "Planned merge edges:",
    expected_merge_edge_count,
)
print(
    "Applied merge edges:",
    applied_merge_edge_count,
)
print(
    "Pre-merge utterances consumed by chains:",
    applied_premerge_utterance_count,
)
print(
    "Post-merge utterances:",
    len(all_merged_utterances_v1),
)
print(
    "Expected post-merge utterances:",
    expected_postmerge_count,
)
print(
    "Net utterance reduction:",
    (
        len(all_parsed_utterances_v1)
        - len(all_merged_utterances_v1)
    ),
)
print()


print("POST-MERGE COUNTS BY PROGRAM")
print("=" * 100)

for program_id in sorted(
    files_by_program
):
    before_items = [
        item
        for item
        in all_parsed_utterances_v1
        if item["program_id"]
        == program_id
    ]

    after_items = [
        item
        for item
        in all_merged_utterances_v1
        if item["program_id"]
        == program_id
    ]

    merged_items = [
        item
        for item
        in after_items
        if item["merge_status"]
        == "MERGED"
    ]

    if not before_items:
        continue

    print()
    print(
        "Program:",
        before_items[0]["program"],
    )
    print(
        "Program ID:",
        program_id,
    )
    print(
        "Before:",
        len(before_items),
    )
    print(
        "After:",
        len(after_items),
    )
    print(
        "Reduction:",
        (
            len(before_items)
            - len(after_items)
        ),
    )
    print(
        "Merged chains:",
        len(merged_items),
    )
    print("-" * 100)


print()
print("REPRESENTATIVE MERGED UTTERANCES")
print("=" * 100)

shown_by_program = Counter()

for item in all_merged_utterances_v1:
    if (
        item["merge_status"]
        != "MERGED"
    ):
        continue

    if shown_by_program[
        item["program_id"]
    ] >= 5:
        continue

    shown_by_program[
        item["program_id"]
    ] += 1

    print()
    print(
        "Program:",
        item["program"],
    )
    print(
        "File:",
        item["relative_path"],
    )
    print(
        "New utterance index:",
        item["utterance_index"],
    )
    print(
        "Merged from indices:",
        item[
            "merged_from_utterance_indices"
        ],
    )
    print(
        "Merge edges:",
        item["merge_edge_count"],
    )
    print(
        "Speaker raw:",
        item["speaker_raw"],
    )
    print(
        "Speaker canonical:",
        item["speaker_canonical"],
    )
    print(
        "Source lines:",
        (
            item["source_line_start"],
            item["source_line_end"],
        ),
    )
    print(
        "Signals:",
        item["merge_signals"],
    )
    print(
        "Merged text:",
        repr(
            item["text"][:500]
        ),
    )
    print("-" * 100)


merged_id_values = [
    item["utterance_id"]
    for item
    in all_merged_utterances_v1
]

assert len(
    parsed_transcript_files_merged_v1
) == len(
    parsed_transcript_files_v1
), (
    "Post-merge file population changed."
)

assert applied_chain_count == len(
    merge_chains_v1
), (
    "Not all planned merge chains were applied."
)

assert applied_merge_edge_count == (
    expected_merge_edge_count
), (
    "Applied merge-edge count does not match plan."
)

assert len(
    all_merged_utterances_v1
) == expected_postmerge_count, (
    "Post-merge utterance count is incorrect."
)

assert (
    len(all_parsed_utterances_v1)
    - len(all_merged_utterances_v1)
) == expected_merge_edge_count, (
    "Net utterance reduction does not equal merge-edge count."
)

assert len(
    merged_id_values
) == len(
    set(merged_id_values)
), (
    "Post-merge utterance IDs are not unique."
)

assert all(
    item["start_time"] is None
    and item["end_time"] is None
    for item
    in all_merged_utterances_v1
), (
    "Timestamp appeared during merge."
)

assert all(
    item["merge_status"]
    in {
        "MERGED",
        "UNCHANGED",
    }
    for item
    in all_merged_utterances_v1
), (
    "Unexpected merge status found."
)

assert all(
    (
        item["merge_edge_count"]
        == len(
            item[
                "merged_from_utterance_ids"
            ]
        ) - 1
    )
    for item
    in all_merged_utterances_v1
), (
    "Merge lineage accounting is invalid."
)

for parsed_file in (
    parsed_transcript_files_merged_v1
):
    expected_indices = list(
        range(
            1,
            parsed_file[
                "utterance_count"
            ] + 1,
        )
    )

    actual_indices = [
        item["utterance_index"]
        for item
        in parsed_file["utterances"]
    ]

    assert (
        actual_indices
        == expected_indices
    ), (
        "Post-merge utterance indices are not sequential: "
        f"{parsed_file['relative_path']}"
    )


original_parsed_state_after_merge = json.dumps(
    all_parsed_utterances_v1,
    ensure_ascii=False,
    sort_keys=True,
)

assert (
    original_parsed_state_before_merge
    == original_parsed_state_after_merge
), (
    "Original parsed utterance population was mutated."
)

print()
print(
    "CONSERVATIVE TINY-UTTERANCE "
    "IN-MEMORY MERGE: PASS"
)

Pre-merge utterances: 15131
Planned merge chains: 350
Applied merge chains: 350
Planned merge edges: 471
Applied merge edges: 471
Pre-merge utterances consumed by chains: 821
Post-merge utterances: 14660
Expected post-merge utterances: 14660
Net utterance reduction: 471

POST-MERGE COUNTS BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Before: 902
After: 902
Reduction: 0
Merged chains: 0
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Before: 453
After: 453
Reduction: 0
Merged chains: 0
----------------------------------------------------------------------------------------------------

Program: چهلستون
Program ID: program:chehelston
Before: 889
After: 889
Reduction: 0
Merged chains: 0
----------------------------------------------------------------------------------------------------

Program: در جستجوی خوشبختی
Program ID: program:dar-jostojo-ye-khoshbakhti
Before: 73

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ممیزی مستقل Post-Merge برای Conservation، Lineage و Coverage</h3>

<p>
Merge محافظه‌کارانه در حافظه با موفقیت اجرا شد و تعداد Utteranceها از
<code>15131</code>
به
<code>14660</code>
رسید.
</p>

<p>
اکنون قبل از ورود به Quality Detection و بررسی ASR Noise باید ثابت شود که Merge باعث حذف، دوباره‌مصرف‌شدن یا جابه‌جایی محتوای Parsed نشده است.
</p>

<p>
این Cell به نتیجه Cell قبلی اعتماد نمی‌کند و خروجی Post-Merge را مستقلاً با دو منبع مقایسه می‌کند:
</p>

<ul>
<li>Parsed Output قبل از Merge</li>
<li>Source File اصلی روی Disk</li>
</ul>

<h4>۱. Pre-Merge Lineage Conservation</h4>

<p>
هر Utterance قبل از Merge باید دقیقاً یک بار در خروجی جدید حضور داشته باشد:
</p>

<ul>
<li>یا به‌صورت <code>UNCHANGED</code></li>
<li>یا به‌عنوان یکی از اعضای <code>MERGED</code></li>
</ul>

<p>
هیچ Pre-Merge Utterance نباید گم شود یا در بیش از یک خروجی استفاده شود.
</p>

<h4>۲. Exact Merge Reconstruction</h4>

<p>
برای هر Utterance Merge‌شده، موارد زیر دوباره از Utteranceهای اولیه بازسازی می‌شوند:
</p>

<ul>
<li>ترتیب اعضا</li>
<li>متن</li>
<li>اولین و آخرین Source Line</li>
<li>Speaker Metadata</li>
</ul>

<p>
متن Merge‌شده باید دقیقاً برابر اتصال متن اعضای اولیه با
<code>\n\n</code>
باشد.
</p>

<p>
هیچ Persian normalization، Spell correction یا ASR correction در این مقایسه انجام نمی‌شود.
</p>

<h4>۳. Exact Source Lineage از Disk</h4>

<p>
برای تمام Utteranceهای Post-Merge، فیلد
<code>source_text</code>
باید دقیقاً برابر Slice ثبت‌شده از Source File اصلی باشد.
</p>

<p>
این موضوع شامل Merge Chainهایی نیز می‌شود که بین اعضای آن‌ها Marker ساختاری Solo قرار داشته است.
</p>

<h4>۴. Source Coverage</h4>

<p>
تمام خطوط غیرخالی Body دوباره حساب می‌شوند.
</p>

<p>
هر خط باید یکی از دو وضعیت را داشته باشد:
</p>

<ul>
<li>داخل Lineage یکی از Utteranceهای Post-Merge پوشش داده شده باشد.</li>
<li>یک Structural Marker مجاز باشد که خارج از Merge باقی مانده است.</li>
</ul>

<p>
از آنجا که Source Range یک Merge Chain می‌تواند Marker ساختاری بین Fragmentها را نیز شامل شود، تعداد Structural Skipهای باقی‌مانده ممکن است نسبت به Audit قبل از Merge کمتر شود.
</p>

<h4>۵. No Overlap</h4>

<p>
هیچ Source Line نباید در Lineage دو Utterance Post-Merge مختلف قرار بگیرد.
</p>

<h4>۶. Timestamp و Speaker Safety</h4>

<p>
Merge نباید:
</p>

<ul>
<li>Timestamp مصنوعی بسازد.</li>
<li>Speaker جدید حدس بزند.</li>
<li>یک Utterance unresolved را وارد Merge کند.</li>
</ul>

<h4>هنوز انجام نمی‌شوند</h4>

<ul>
<li>ASR correction</li>
<li>Spell correction</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>تمام Pre-Merge Utteranceها دقیقاً یک بار در Lineage جدید مصرف شده باشند.</li>
<li>هیچ Pre-Merge ID گم یا تکراری نباشد.</li>
<li>متن تمام Merge Chainها دقیقاً قابل بازسازی باشد.</li>
<li>تمام <code>source_text</code>ها با Disk Exact Match باشند.</li>
<li>هیچ Source-line overlap وجود نداشته باشد.</li>
<li>تمام خطوط غیرخالی Body حساب شده باشند.</li>
<li>هیچ Timestamp جدیدی وجود نداشته باشد.</li>
<li>هیچ Merge دارای Speaker unresolved نباشد.</li>
</ul>

</div>

In [27]:
# Code Cell 27

premerge_by_id = {
    item["utterance_id"]: item
    for item in all_parsed_utterances_v1
}

postmerge_by_id = {
    item["utterance_id"]: item
    for item in all_merged_utterances_v1
}

assert len(
    premerge_by_id
) == len(
    all_parsed_utterances_v1
), (
    "Pre-merge utterance IDs are not unique."
)

assert len(
    postmerge_by_id
) == len(
    all_merged_utterances_v1
), (
    "Post-merge utterance IDs are not unique."
)


lineage_consumption_count = Counter()

merge_reconstruction_failures = []
source_text_failures = []
source_overlap_failures = []
coverage_failures = []
speaker_safety_failures = []
timestamp_failures = []

total_merged_outputs = 0
total_unchanged_outputs = 0

for item in all_merged_utterances_v1:
    source_ids = item[
        "merged_from_utterance_ids"
    ]

    source_indices = item[
        "merged_from_utterance_indices"
    ]

    assert len(source_ids) == len(
        source_indices
    ), (
        "Merge lineage ID/index lengths differ."
    )

    original_members = []

    for source_id in source_ids:
        lineage_consumption_count[
            source_id
        ] += 1

        if source_id not in premerge_by_id:
            merge_reconstruction_failures.append({
                "postmerge_utterance_id": (
                    item["utterance_id"]
                ),
                "reason": (
                    "UNKNOWN_PREMERGE_UTTERANCE_ID"
                ),
                "source_id": source_id,
            })
            continue

        original_members.append(
            premerge_by_id[source_id]
        )

    if len(original_members) != len(
        source_ids
    ):
        continue

    expected_indices = [
        member["utterance_index"]
        for member in original_members
    ]

    if expected_indices != source_indices:
        merge_reconstruction_failures.append({
            "postmerge_utterance_id": (
                item["utterance_id"]
            ),
            "reason": (
                "PREMERGE_INDEX_SEQUENCE_MISMATCH"
            ),
            "expected": expected_indices,
            "actual": source_indices,
        })

    member_paths = {
        member["relative_path"]
        for member in original_members
    }

    if member_paths != {
        item["relative_path"]
    }:
        merge_reconstruction_failures.append({
            "postmerge_utterance_id": (
                item["utterance_id"]
            ),
            "reason": (
                "MULTIPLE_OR_MISMATCHED_SOURCE_FILES"
            ),
            "member_paths": sorted(
                member_paths
            ),
            "postmerge_path": (
                item["relative_path"]
            ),
        })

    if item["merge_status"] == "MERGED":
        total_merged_outputs += 1

        expected_text = "\n\n".join(
            member["text"]
            for member in original_members
        )

        if item["text"] != expected_text:
            merge_reconstruction_failures.append({
                "postmerge_utterance_id": (
                    item["utterance_id"]
                ),
                "reason": (
                    "MERGED_TEXT_RECONSTRUCTION_MISMATCH"
                ),
            })

        expected_start = original_members[0][
            "source_line_start"
        ]

        expected_end = original_members[-1][
            "source_line_end"
        ]

        if (
            item["source_line_start"]
            != expected_start
            or item["source_line_end"]
            != expected_end
        ):
            merge_reconstruction_failures.append({
                "postmerge_utterance_id": (
                    item["utterance_id"]
                ),
                "reason": (
                    "MERGED_SOURCE_RANGE_MISMATCH"
                ),
                "expected": (
                    expected_start,
                    expected_end,
                ),
                "actual": (
                    item["source_line_start"],
                    item["source_line_end"],
                ),
            })

        if len(original_members) < 2:
            merge_reconstruction_failures.append({
                "postmerge_utterance_id": (
                    item["utterance_id"]
                ),
                "reason": (
                    "MERGED_OUTPUT_HAS_FEWER_THAN_TWO_MEMBERS"
                ),
            })

        if (
            item["merge_edge_count"]
            != len(original_members) - 1
        ):
            merge_reconstruction_failures.append({
                "postmerge_utterance_id": (
                    item["utterance_id"]
                ),
                "reason": (
                    "MERGE_EDGE_COUNT_MISMATCH"
                ),
            })

        if item["speaker_status"] in {
            "UNRESOLVED",
            "NARRATIVE_CUE",
        }:
            speaker_safety_failures.append({
                "postmerge_utterance_id": (
                    item["utterance_id"]
                ),
                "reason": (
                    "UNRESOLVED_OR_NARRATIVE_MERGED"
                ),
                "speaker_status": (
                    item["speaker_status"]
                ),
            })

        if (
            item["speaker_canonical"]
            is None
        ):
            speaker_safety_failures.append({
                "postmerge_utterance_id": (
                    item["utterance_id"]
                ),
                "reason": (
                    "MERGED_OUTPUT_HAS_NO_CANONICAL_SPEAKER"
                ),
            })

    elif item["merge_status"] == "UNCHANGED":
        total_unchanged_outputs += 1

        if len(original_members) != 1:
            merge_reconstruction_failures.append({
                "postmerge_utterance_id": (
                    item["utterance_id"]
                ),
                "reason": (
                    "UNCHANGED_OUTPUT_HAS_MULTIPLE_MEMBERS"
                ),
            })

        else:
            original = original_members[0]

            fields_to_match = (
                "text",
                "source_text",
                "source_line_start",
                "source_line_end",
                "speaker_raw",
                "speaker_canonical",
                "speaker_source_canonical",
                "speaker_status",
                "start_time",
                "end_time",
            )

            mismatched_fields = [
                field
                for field in fields_to_match
                if item[field]
                != original[field]
            ]

            if mismatched_fields:
                merge_reconstruction_failures.append({
                    "postmerge_utterance_id": (
                        item["utterance_id"]
                    ),
                    "reason": (
                        "UNCHANGED_OUTPUT_WAS_MODIFIED"
                    ),
                    "mismatched_fields": (
                        mismatched_fields
                    ),
                })

    else:
        merge_reconstruction_failures.append({
            "postmerge_utterance_id": (
                item["utterance_id"]
            ),
            "reason": (
                "UNKNOWN_MERGE_STATUS"
            ),
            "merge_status": (
                item["merge_status"]
            ),
        })

    if (
        item["start_time"] is not None
        or item["end_time"] is not None
    ):
        timestamp_failures.append({
            "postmerge_utterance_id": (
                item["utterance_id"]
            ),
            "start_time": (
                item["start_time"]
            ),
            "end_time": (
                item["end_time"]
            ),
        })


missing_premerge_ids = sorted(
    source_id
    for source_id
    in premerge_by_id
    if lineage_consumption_count[
        source_id
    ] == 0
)

multiply_consumed_premerge_ids = sorted(
    source_id
    for source_id, count
    in lineage_consumption_count.items()
    if count > 1
)

unknown_consumed_ids = sorted(
    source_id
    for source_id
    in lineage_consumption_count
    if source_id not in premerge_by_id
)


total_body_nonempty_lines = 0
total_covered_nonempty_lines = 0
total_allowed_structural_skips = 0

structural_skip_counts_postmerge = Counter()


for parsed_file in (
    parsed_transcript_files_merged_v1
):
    relative_path = parsed_file[
        "relative_path"
    ]

    record = accepted_record_by_path[
        relative_path
    ]

    file_path = (
        project_root
        / relative_path
    )

    lines = file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    candidate = boundary_candidate_by_path[
        relative_path
    ]

    body_start_index = (
        candidate[
            "candidate_line_number"
        ] - 1
    )

    coverage_count = Counter()

    for utterance in parsed_file[
        "utterances"
    ]:
        start_line = utterance[
            "source_line_start"
        ]

        end_line = utterance[
            "source_line_end"
        ]

        if not (
            1
            <= start_line
            <= end_line
            <= len(lines)
        ):
            source_text_failures.append({
                "relative_path": (
                    relative_path
                ),
                "utterance_id": (
                    utterance[
                        "utterance_id"
                    ]
                ),
                "reason": (
                    "INVALID_SOURCE_RANGE"
                ),
                "source_line_start": (
                    start_line
                ),
                "source_line_end": (
                    end_line
                ),
                "file_line_count": (
                    len(lines)
                ),
            })
            continue

        expected_source_text = "\n".join(
            lines[
                start_line - 1:
                end_line
            ]
        )

        if (
            utterance["source_text"]
            != expected_source_text
        ):
            source_text_failures.append({
                "relative_path": (
                    relative_path
                ),
                "utterance_id": (
                    utterance[
                        "utterance_id"
                    ]
                ),
                "reason": (
                    "SOURCE_TEXT_MISMATCH"
                ),
                "source_line_start": (
                    start_line
                ),
                "source_line_end": (
                    end_line
                ),
            })

        for line_number in range(
            start_line,
            end_line + 1,
        ):
            coverage_count[
                line_number
            ] += 1

    overlapping_lines = sorted(
        line_number
        for line_number, count
        in coverage_count.items()
        if count > 1
    )

    if overlapping_lines:
        source_overlap_failures.append({
            "relative_path": (
                relative_path
            ),
            "overlapping_line_numbers": (
                overlapping_lines
            ),
        })

    missing_nonempty_lines = []

    for index in range(
        body_start_index,
        len(lines),
    ):
        raw_line = lines[index]

        if not raw_line.strip():
            continue

        line_number = index + 1

        total_body_nonempty_lines += 1

        if coverage_count[
            line_number
        ] > 0:
            total_covered_nonempty_lines += 1
            continue

        skip_type = None

        if is_solo_structural_marker(
            record,
            raw_line,
        ):
            skip_type = (
                "SOLO_MANIFEST_MARKER"
            )

        elif is_empty_verified_bold_boundary(
            record,
            lines,
            index,
        ):
            skip_type = (
                "EMPTY_VERIFIED_BOLD_BOUNDARY"
            )

        elif is_empty_verified_chehelston_boundary(
            record,
            lines,
            index,
        ):
            skip_type = (
                "EMPTY_VERIFIED_CHEHELSTON_BOUNDARY"
            )

        if skip_type is not None:
            structural_skip_counts_postmerge[
                skip_type
            ] += 1

            total_allowed_structural_skips += 1
            continue

        missing_nonempty_lines.append({
            "line_number": (
                line_number
            ),
            "raw_line": raw_line,
        })

    if missing_nonempty_lines:
        coverage_failures.append({
            "relative_path": (
                relative_path
            ),
            "missing_nonempty_lines": (
                missing_nonempty_lines
            ),
        })


print("PRE-MERGE LINEAGE ACCOUNTING")
print("=" * 100)
print(
    "Pre-merge utterances:",
    len(all_parsed_utterances_v1),
)
print(
    "Post-merge utterances:",
    len(all_merged_utterances_v1),
)
print(
    "Merged outputs:",
    total_merged_outputs,
)
print(
    "Unchanged outputs:",
    total_unchanged_outputs,
)
print(
    "Missing pre-merge IDs:",
    len(missing_premerge_ids),
)
print(
    "Multiply consumed pre-merge IDs:",
    len(
        multiply_consumed_premerge_ids
    ),
)
print(
    "Unknown consumed IDs:",
    len(unknown_consumed_ids),
)


print()
print("POST-MERGE SOURCE ACCOUNTING")
print("=" * 100)
print(
    "Body non-empty source lines:",
    total_body_nonempty_lines,
)
print(
    "Covered non-empty source lines:",
    total_covered_nonempty_lines,
)
print(
    "Allowed structural skips:",
    total_allowed_structural_skips,
)

for skip_type, count in sorted(
    structural_skip_counts_postmerge.items()
):
    print(
        f"- {skip_type}: {count}"
    )

accounted_nonempty_lines = (
    total_covered_nonempty_lines
    + total_allowed_structural_skips
)

print(
    "Covered + allowed skips:",
    accounted_nonempty_lines,
)
print(
    "Accounting balanced:",
    accounted_nonempty_lines
    == total_body_nonempty_lines,
)


print()
print("POST-MERGE FAILURE COUNTS")
print("=" * 100)
print(
    "Merge reconstruction failures:",
    len(
        merge_reconstruction_failures
    ),
)
print(
    "Source-text failures:",
    len(source_text_failures),
)
print(
    "Source-overlap failures:",
    len(source_overlap_failures),
)
print(
    "Coverage failures:",
    len(coverage_failures),
)
print(
    "Speaker-safety failures:",
    len(speaker_safety_failures),
)
print(
    "Timestamp failures:",
    len(timestamp_failures),
)


if merge_reconstruction_failures:
    print()
    print(
        "First merge reconstruction failure:"
    )
    print(
        json.dumps(
            merge_reconstruction_failures[
                0
            ],
            ensure_ascii=False,
            indent=2,
        )
    )

if source_text_failures:
    print()
    print(
        "First source-text failure:"
    )
    print(
        json.dumps(
            source_text_failures[0],
            ensure_ascii=False,
            indent=2,
        )
    )

if source_overlap_failures:
    print()
    print(
        "First source-overlap failure:"
    )
    print(
        json.dumps(
            source_overlap_failures[0],
            ensure_ascii=False,
            indent=2,
        )
    )

if coverage_failures:
    print()
    print(
        "First coverage failure:"
    )
    print(
        json.dumps(
            coverage_failures[0],
            ensure_ascii=False,
            indent=2,
        )
    )

if speaker_safety_failures:
    print()
    print(
        "First speaker-safety failure:"
    )
    print(
        json.dumps(
            speaker_safety_failures[0],
            ensure_ascii=False,
            indent=2,
        )
    )


assert not missing_premerge_ids, (
    "At least one pre-merge utterance disappeared from lineage."
)

assert not multiply_consumed_premerge_ids, (
    "At least one pre-merge utterance was consumed more than once."
)

assert not unknown_consumed_ids, (
    "Post-merge lineage references an unknown pre-merge utterance."
)

assert sum(
    lineage_consumption_count.values()
) == len(
    all_parsed_utterances_v1
), (
    "Pre-merge lineage consumption total is invalid."
)

assert not merge_reconstruction_failures, (
    "Post-merge output cannot be exactly reconstructed from pre-merge utterances."
)

assert not source_text_failures, (
    "Post-merge source_text does not exactly match Source on disk."
)

assert not source_overlap_failures, (
    "At least one Source line is consumed by multiple post-merge utterances."
)

assert not coverage_failures, (
    "At least one non-empty Source line remains unexplained after merge."
)

assert not speaker_safety_failures, (
    "Post-merge speaker-safety validation failed."
)

assert not timestamp_failures, (
    "Timestamp appeared during post-merge processing."
)

assert (
    accounted_nonempty_lines
    == total_body_nonempty_lines
), (
    "Post-merge Source coverage accounting does not balance."
)

assert (
    total_merged_outputs
    == len(merge_chains_v1)
), (
    "Number of merged outputs does not equal planned merge chains."
)

assert (
    total_merged_outputs
    + total_unchanged_outputs
    == len(
        all_merged_utterances_v1
    )
), (
    "Post-merge status accounting does not balance."
)

print()
print(
    "POST-MERGE CONSERVATION, LINEAGE "
    "AND COVERAGE AUDIT: PASS"
)

PRE-MERGE LINEAGE ACCOUNTING
Pre-merge utterances: 15131
Post-merge utterances: 14660
Merged outputs: 350
Unchanged outputs: 14310
Missing pre-merge IDs: 0
Multiply consumed pre-merge IDs: 0
Unknown consumed IDs: 0

POST-MERGE SOURCE ACCOUNTING
Body non-empty source lines: 28894
Covered non-empty source lines: 21655
Allowed structural skips: 7239
- EMPTY_VERIFIED_BOLD_BOUNDARY: 3
- SOLO_MANIFEST_MARKER: 7236
Covered + allowed skips: 28894
Accounting balanced: True

POST-MERGE FAILURE COUNTS
Merge reconstruction failures: 0
Source-text failures: 0
Source-overlap failures: 0
Coverage failures: 0
Speaker-safety failures: 0
Timestamp failures: 0

POST-MERGE CONSERVATION, LINEAGE AND COVERAGE AUDIT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>کشف ASR Repetition و Noise بدون تغییر متن</h3>

<p>
ممیزی Post-Merge با موفقیت کامل شد.
</p>

<p>
تمام <code>15131</code> Utterance قبل از Merge دقیقاً یک بار در Lineage جدید حضور دارند و خروجی Post-Merge شامل <code>14660</code> Utterance است.
</p>

<p>
همچنین تمام <code>28894</code> خط غیرخالی Source دوباره حساب شده‌اند و هیچ Failure در Conservation، Lineage، Coverage، Speaker Safety یا Timestamp وجود ندارد.
</p>

<p>
در این مرحله وارد بخش Quality Detection مربوط به ASR Repetition و Noise می‌شویم.
</p>

<p>
هدف این Cell فقط کشف Candidate است. هیچ متن، Source File یا Parsed Utterance اصلاح یا حذف نمی‌شود.
</p>

<h4>اصل مهم</h4>

<p>
وجود تکرار در Transcript الزاماً خطای ASR نیست.
</p>

<p>
برای مثال عبارت‌هایی مانند:
</p>

<ul>
<li><code>نه، نه. نه. نه.</code></li>
<li>تکرار یک واژه برای تأکید</li>
<li>تکرار بخشی از جمله در گفتار طبیعی</li>
</ul>

<p>
ممکن است کاملاً واقعی باشند.
</p>

<p>
بنابراین تمام موارد این Cell فقط <code>REVIEW CANDIDATE</code> هستند و هیچ Auto-cleanup انجام نمی‌شود.
</p>

<h4>Detectorهای این مرحله</h4>

<p>
چند نوع Signal مستقل بررسی می‌شود.
</p>

<h5>۱. Exact Adjacent Duplicate</h5>

<p>
اگر دو Utterance متوالی در یک فایل دقیقاً Text یکسان داشته باشند، به‌عنوان Candidate ثبت می‌شوند.
</p>

<p>
این Detection به معنی Deduplication نیست و هیچ موردی حذف نمی‌شود.
</p>

<h5>۲. Repeated Token Run</h5>

<p>
اگر یک Token دقیقاً و بدون Persian normalization حداقل سه بار پشت سر هم تکرار شده باشد، Candidate ثبت می‌شود.
</p>

<h5>۳. Repeated N-gram Run</h5>

<p>
اگر یک عبارت دو تا پنج Tokenی دقیقاً و بلافاصله دوباره تکرار شده باشد، Candidate ثبت می‌شود.
</p>

<p>
مقایسه Exact Unicode است و هیچ تبدیل املایی انجام نمی‌شود.
</p>

<h5>۴. Long Character Run</h5>

<p>
تکرار طولانی یک Character نیز بررسی می‌شود.
</p>

<p>
تکرار حروف و اعداد با آستانه محافظه‌کارانه جداگانه ثبت می‌شود و تکرار طولانی علائم نگارشی نیز فقط برای Review گزارش می‌شود.
</p>

<h5>۵. Transcription Marker</h5>

<p>
عبارت‌های داخل کروشه مانند:
</p>

<p>
<code>[گزارش مردمی]</code>
</p>

<p>
یا:
</p>

<p>
<code>[میان‌برنامه]</code>
</p>

<p>
شناسایی می‌شوند.
</p>

<p>
این Markerها به‌طور پیش‌فرض Noise محسوب نمی‌شوند؛ فقط Inventory می‌شوند تا در Quality Report مشخص باشد چه نوع Annotationهایی در Source وجود دارند.
</p>

<h5>۶. Technical Unicode Noise</h5>

<p>
وجود مواردی مانند:
</p>

<ul>
<li>Unicode Replacement Character</li>
<li>NULL</li>
<li>Control Characterهای غیرمنتظره</li>
</ul>

<p>
به‌عنوان Technical Noise بررسی می‌شود.
</p>

<h4>عدم استفاده از Persian Normalization</h4>

<p>
در این Cell:
</p>

<ul>
<li>ی و ک تبدیل نمی‌شوند.</li>
<li>نیم‌فاصله اصلاح نمی‌شود.</li>
<li>املای واژه‌ها اصلاح نمی‌شود.</li>
<li>فاصله‌گذاری فارسی اصلاح نمی‌شود.</li>
<li>هیچ Fuzzy Matching زبانی انجام نمی‌شود.</li>
</ul>

<p>
بنابراین این مرحله با محدودیت Notebook 08 تداخل ندارد.
</p>

<h4>عدم اجرای Deduplication</h4>

<p>
حتی Exact Duplicateهای کشف‌شده نیز فقط Flag می‌شوند.
</p>

<p>
هیچ حذف، Collapse یا Deduplication در Notebook 06 انجام نمی‌شود.
</p>

<h4>خروجی این Cell</h4>

<p>
Candidateهای Quality در متغیر:
</p>

<p>
<code>transcript_quality_candidates_v1</code>
</p>

<p>
در حافظه نگهداری می‌شوند.
</p>

<p>
برای هر Candidate موارد زیر حفظ می‌شوند:
</p>

<ul>
<li>Program</li>
<li>Source File</li>
<li>Utterance ID</li>
<li>Source Line Range</li>
<li>Candidate Type</li>
<li>Evidence</li>
<li>Raw Text</li>
<li>Review Status</li>
</ul>

<p>
هنوز هیچ Artifact روی Disk نوشته نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام <code>14660</code> Utterance Post-Merge بررسی شوند.</li>
<li>تمام Detectorها فقط Detection انجام دهند.</li>
<li>هیچ Text تغییر نکند.</li>
<li>هیچ Candidate خودکار حذف یا اصلاح نشود.</li>
<li>Technical Noise و Repetition Candidateها جدا گزارش شوند.</li>
<li>Annotationهای داخل کروشه جداگانه Inventory شوند.</li>
<li>خروجی Post-Merge قبل و بعد از Cell دقیقاً یکسان باقی بماند.</li>
</ul>

</div>

In [28]:
# Code Cell 28

postmerge_state_before_quality_scan = json.dumps(
    all_merged_utterances_v1,
    ensure_ascii=False,
    sort_keys=True,
)

quality_candidates = []

edge_punctuation = (
    "«»"
    "\"'"
    "()"
    "[]"
    "{}"
    "<>"
    "."
    ","
    "،"
    ":"
    ";"
    "؛"
    "!"
    "?"
    "؟"
    "…"
    "-"
    "–"
    "—"
)


def raw_token_sequence(text):
    tokens = []

    for raw_token in text.split():
        token = raw_token.strip(
            edge_punctuation
        )

        if token:
            tokens.append(
                token
            )

    return tokens


def repeated_token_runs(tokens):
    runs = []

    index = 0

    while index < len(tokens):
        next_index = index + 1

        while (
            next_index < len(tokens)
            and tokens[next_index]
            == tokens[index]
        ):
            next_index += 1

        run_length = (
            next_index - index
        )

        if run_length >= 3:
            runs.append({
                "token": tokens[index],
                "count": run_length,
                "token_start_index": index,
                "token_end_index": (
                    next_index - 1
                ),
            })

        index = next_index

    return runs


def repeated_ngram_runs(
    tokens,
    min_n=2,
    max_n=5,
):
    hits = []
    seen = set()

    token_count = len(tokens)

    for ngram_size in range(
        min_n,
        max_n + 1,
    ):
        max_start = (
            token_count
            - (2 * ngram_size)
        )

        if max_start < 0:
            continue

        for start_index in range(
            max_start + 1
        ):
            left = tuple(
                tokens[
                    start_index:
                    start_index + ngram_size
                ]
            )

            right = tuple(
                tokens[
                    start_index + ngram_size:
                    start_index + (
                        2 * ngram_size
                    )
                ]
            )

            if left != right:
                continue

            key = (
                start_index,
                ngram_size,
                left,
            )

            if key in seen:
                continue

            seen.add(key)

            hits.append({
                "ngram_size": (
                    ngram_size
                ),
                "tokens": list(left),
                "token_start_index": (
                    start_index
                ),
            })

    return hits


def unexpected_control_characters(text):
    hits = []

    allowed_controls = {
        "\n",
        "\r",
        "\t",
    }

    for character_index, char in enumerate(
        text
    ):
        if char in allowed_controls:
            continue

        codepoint = ord(char)

        if (
            codepoint < 32
            or codepoint == 127
        ):
            hits.append({
                "character_index": (
                    character_index
                ),
                "codepoint": (
                    f"U+{codepoint:04X}"
                ),
                "repr": repr(char),
            })

    return hits


bracket_marker_pattern = re.compile(
    r"\[[^\[\]\n]{1,120}\]"
)

letter_or_digit_run_pattern = re.compile(
    r"([^\W_])\1{3,}",
    flags=re.UNICODE,
)

punctuation_run_pattern = re.compile(
    r"([^\w\s])\1{4,}",
    flags=re.UNICODE,
)


def add_quality_candidate(
    candidate_type,
    item,
    evidence,
    review_status="REVIEW_REQUIRED",
):
    quality_candidates.append({
        "candidate_type": (
            candidate_type
        ),
        "review_status": (
            review_status
        ),
        "program": item[
            "program"
        ],
        "program_id": item[
            "program_id"
        ],
        "relative_path": item[
            "relative_path"
        ],
        "utterance_ids": [
            item["utterance_id"]
        ],
        "source_line_start": item[
            "source_line_start"
        ],
        "source_line_end": item[
            "source_line_end"
        ],
        "speaker_raw": item[
            "speaker_raw"
        ],
        "speaker_canonical": item[
            "speaker_canonical"
        ],
        "raw_text": item["text"],
        "evidence": evidence,
    })


for item in all_merged_utterances_v1:
    text = item["text"]

    tokens = raw_token_sequence(
        text
    )

    token_runs = repeated_token_runs(
        tokens
    )

    if token_runs:
        add_quality_candidate(
            candidate_type=(
                "REPEATED_TOKEN_RUN"
            ),
            item=item,
            evidence={
                "runs": token_runs[:20],
                "total_runs": len(
                    token_runs
                ),
                "comparison": (
                    "EXACT_UNICODE_TOKEN"
                ),
            },
        )

    ngram_runs = repeated_ngram_runs(
        tokens
    )

    if ngram_runs:
        add_quality_candidate(
            candidate_type=(
                "REPEATED_NGRAM_RUN"
            ),
            item=item,
            evidence={
                "runs": ngram_runs[:20],
                "total_runs": len(
                    ngram_runs
                ),
                "comparison": (
                    "EXACT_UNICODE_NGRAM"
                ),
            },
        )

    letter_digit_runs = [
        {
            "matched_text": match.group(0),
            "character": match.group(1),
            "start_index": match.start(),
            "end_index": match.end(),
        }
        for match
        in letter_or_digit_run_pattern.finditer(
            text
        )
    ]

    if letter_digit_runs:
        add_quality_candidate(
            candidate_type=(
                "LONG_LETTER_OR_DIGIT_RUN"
            ),
            item=item,
            evidence={
                "runs": (
                    letter_digit_runs[:20]
                ),
                "total_runs": len(
                    letter_digit_runs
                ),
            },
        )

    punctuation_runs = [
        {
            "matched_text": match.group(0),
            "character": match.group(1),
            "start_index": match.start(),
            "end_index": match.end(),
        }
        for match
        in punctuation_run_pattern.finditer(
            text
        )
    ]

    if punctuation_runs:
        add_quality_candidate(
            candidate_type=(
                "LONG_PUNCTUATION_RUN"
            ),
            item=item,
            evidence={
                "runs": (
                    punctuation_runs[:20]
                ),
                "total_runs": len(
                    punctuation_runs
                ),
            },
        )

    bracket_markers = (
        bracket_marker_pattern.findall(
            text
        )
    )

    if bracket_markers:
        add_quality_candidate(
            candidate_type=(
                "BRACKETED_TRANSCRIPTION_MARKER"
            ),
            item=item,
            evidence={
                "markers": (
                    bracket_markers[:50]
                ),
                "total_markers": len(
                    bracket_markers
                ),
            },
            review_status=(
                "STRUCTURAL_MARKER_REVIEW"
            ),
        )

    if "\ufffd" in text:
        add_quality_candidate(
            candidate_type=(
                "UNICODE_REPLACEMENT_CHARACTER"
            ),
            item=item,
            evidence={
                "count": text.count(
                    "\ufffd"
                ),
            },
            review_status=(
                "TECHNICAL_NOISE"
            ),
        )

    if "\x00" in text:
        add_quality_candidate(
            candidate_type=(
                "NULL_CHARACTER"
            ),
            item=item,
            evidence={
                "count": text.count(
                    "\x00"
                ),
            },
            review_status=(
                "TECHNICAL_NOISE"
            ),
        )

    control_hits = (
        unexpected_control_characters(
            text
        )
    )

    if control_hits:
        add_quality_candidate(
            candidate_type=(
                "UNEXPECTED_CONTROL_CHARACTER"
            ),
            item=item,
            evidence={
                "characters": (
                    control_hits[:20]
                ),
                "total_characters": len(
                    control_hits
                ),
            },
            review_status=(
                "TECHNICAL_NOISE"
            ),
        )


exact_adjacent_duplicate_pairs = []

for parsed_file in (
    parsed_transcript_files_merged_v1
):
    utterances = parsed_file[
        "utterances"
    ]

    for index in range(
        len(utterances) - 1
    ):
        left = utterances[index]
        right = utterances[
            index + 1
        ]

        if (
            left["text"]
            != right["text"]
        ):
            continue

        pair = {
            "candidate_type": (
                "EXACT_ADJACENT_DUPLICATE_UTTERANCE"
            ),
            "review_status": (
                "REVIEW_REQUIRED"
            ),
            "program": left[
                "program"
            ],
            "program_id": left[
                "program_id"
            ],
            "relative_path": left[
                "relative_path"
            ],
            "utterance_ids": [
                left[
                    "utterance_id"
                ],
                right[
                    "utterance_id"
                ],
            ],
            "source_line_start": (
                left[
                    "source_line_start"
                ]
            ),
            "source_line_end": (
                right[
                    "source_line_end"
                ]
            ),
            "speaker_raw": (
                left["speaker_raw"]
                if (
                    left["speaker_raw"]
                    == right["speaker_raw"]
                )
                else None
            ),
            "speaker_canonical": (
                left[
                    "speaker_canonical"
                ]
                if (
                    left[
                        "speaker_canonical"
                    ]
                    == right[
                        "speaker_canonical"
                    ]
                )
                else None
            ),
            "raw_text": left[
                "text"
            ],
            "evidence": {
                "comparison": (
                    "EXACT_RAW_TEXT"
                ),
                "left_utterance_index": (
                    left[
                        "utterance_index"
                    ]
                ),
                "right_utterance_index": (
                    right[
                        "utterance_index"
                    ]
                ),
            },
        }

        exact_adjacent_duplicate_pairs.append(
            pair
        )

        quality_candidates.append(
            pair
        )


transcript_quality_candidates_v1 = []

for candidate_index, candidate in enumerate(
    quality_candidates,
    start=1,
):
    transcript_quality_candidates_v1.append({
        "quality_candidate_id": (
            f"transcript-quality:"
            f"{candidate_index:06d}"
        ),
        **candidate,
    })


print(
    "Post-merge utterances inspected:",
    len(all_merged_utterances_v1),
)
print(
    "Quality candidates:",
    len(
        transcript_quality_candidates_v1
    ),
)
print()


print("QUALITY CANDIDATE COUNTS")
print("=" * 100)

candidate_type_counts = Counter(
    item["candidate_type"]
    for item
    in transcript_quality_candidates_v1
)

if candidate_type_counts:
    for candidate_type, count in sorted(
        candidate_type_counts.items()
    ):
        print(
            f"- {candidate_type}: {count}"
        )
else:
    print("- NONE")


print()
print("REVIEW STATUS COUNTS")
print("=" * 100)

review_status_counts = Counter(
    item["review_status"]
    for item
    in transcript_quality_candidates_v1
)

if review_status_counts:
    for status, count in sorted(
        review_status_counts.items()
    ):
        print(
            f"- {status}: {count}"
        )
else:
    print("- NONE")


print()
print("QUALITY CANDIDATES BY PROGRAM")
print("=" * 100)

for program_id in sorted(
    files_by_program
):
    program_candidates = [
        item
        for item
        in transcript_quality_candidates_v1
        if item["program_id"]
        == program_id
    ]

    if not program_candidates:
        continue

    program_name = (
        program_candidates[0][
            "program"
        ]
    )

    program_counts = Counter(
        item["candidate_type"]
        for item
        in program_candidates
    )

    print()
    print(
        "Program:",
        program_name,
    )
    print(
        "Program ID:",
        program_id,
    )
    print(
        "Candidates:",
        len(program_candidates),
    )

    for candidate_type, count in sorted(
        program_counts.items()
    ):
        print(
            f"- {candidate_type}: {count}"
        )

    print("-" * 100)


print()
print("BRACKETED MARKER INVENTORY")
print("=" * 100)

bracket_marker_inventory = Counter()

for candidate in (
    transcript_quality_candidates_v1
):
    if (
        candidate[
            "candidate_type"
        ]
        != "BRACKETED_TRANSCRIPTION_MARKER"
    ):
        continue

    for marker in candidate[
        "evidence"
    ]["markers"]:
        bracket_marker_inventory[
            marker
        ] += 1

if bracket_marker_inventory:
    for marker, count in (
        bracket_marker_inventory.most_common(
            50
        )
    ):
        print(
            f"- {marker}: {count}"
        )
else:
    print("- NONE")


print()
print("REPRESENTATIVE QUALITY CANDIDATES")
print("=" * 100)

shown_by_type = Counter()

for candidate in (
    transcript_quality_candidates_v1
):
    candidate_type = candidate[
        "candidate_type"
    ]

    if shown_by_type[
        candidate_type
    ] >= 5:
        continue

    shown_by_type[
        candidate_type
    ] += 1

    print()
    print(
        "Candidate ID:",
        candidate[
            "quality_candidate_id"
        ],
    )
    print(
        "Type:",
        candidate_type,
    )
    print(
        "Review status:",
        candidate[
            "review_status"
        ],
    )
    print(
        "Program:",
        candidate[
            "program"
        ],
    )
    print(
        "File:",
        candidate[
            "relative_path"
        ],
    )
    print(
        "Utterance IDs:",
        candidate[
            "utterance_ids"
        ],
    )
    print(
        "Source lines:",
        (
            candidate[
                "source_line_start"
            ],
            candidate[
                "source_line_end"
            ],
        ),
    )
    print(
        "Evidence:",
        json.dumps(
            candidate[
                "evidence"
            ],
            ensure_ascii=False,
        ),
    )
    print(
        "Raw text:",
        repr(
            candidate[
                "raw_text"
            ][:500]
        ),
    )
    print("-" * 100)


technical_noise_candidates = [
    item
    for item
    in transcript_quality_candidates_v1
    if item["review_status"]
    == "TECHNICAL_NOISE"
]

repetition_review_candidates = [
    item
    for item
    in transcript_quality_candidates_v1
    if item["candidate_type"]
    in {
        "REPEATED_TOKEN_RUN",
        "REPEATED_NGRAM_RUN",
        "EXACT_ADJACENT_DUPLICATE_UTTERANCE",
    }
]


postmerge_state_after_quality_scan = json.dumps(
    all_merged_utterances_v1,
    ensure_ascii=False,
    sort_keys=True,
)


assert len(
    all_merged_utterances_v1
) == 14660, (
    "Post-merge utterance population changed during quality scan."
)

assert (
    postmerge_state_before_quality_scan
    == postmerge_state_after_quality_scan
), (
    "Post-merge parsed output was mutated by quality detection."
)

assert len({
    item[
        "quality_candidate_id"
    ]
    for item
    in transcript_quality_candidates_v1
}) == len(
    transcript_quality_candidates_v1
), (
    "Quality candidate IDs are not unique."
)

assert all(
    item["review_status"]
    in {
        "REVIEW_REQUIRED",
        "STRUCTURAL_MARKER_REVIEW",
        "TECHNICAL_NOISE",
    }
    for item
    in transcript_quality_candidates_v1
), (
    "Unexpected quality review status found."
)

assert all(
    item["relative_path"]
    in parsed_file_by_path
    for item
    in transcript_quality_candidates_v1
), (
    "Quality candidate references an unknown source file."
)

assert all(
    item[
        "candidate_type"
    ]
    != "EXACT_ADJACENT_DUPLICATE_UTTERANCE"
    or len(
        item["utterance_ids"]
    ) == 2
    for item
    in transcript_quality_candidates_v1
), (
    "Adjacent duplicate candidate does not reference exactly two utterances."
)

print()
print(
    "ASR REPETITION AND RAW-NOISE "
    "DETECTION: PASS"
)

Post-merge utterances inspected: 14660
Quality candidates: 120

QUALITY CANDIDATE COUNTS
- BRACKETED_TRANSCRIPTION_MARKER: 40
- EXACT_ADJACENT_DUPLICATE_UTTERANCE: 4
- REPEATED_NGRAM_RUN: 66
- REPEATED_TOKEN_RUN: 10

REVIEW STATUS COUNTS
- REVIEW_REQUIRED: 80
- STRUCTURAL_MARKER_REVIEW: 40

QUALITY CANDIDATES BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Candidates: 13
- BRACKETED_TRANSCRIPTION_MARKER: 5
- REPEATED_NGRAM_RUN: 7
- REPEATED_TOKEN_RUN: 1
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Candidates: 1
- REPEATED_NGRAM_RUN: 1
----------------------------------------------------------------------------------------------------

Program: چهلستون
Program ID: program:chehelston
Candidates: 18
- BRACKETED_TRANSCRIPTION_MARKER: 17
- REPEATED_NGRAM_RUN: 1
----------------------------------------------------------------------------------------------------

Program: 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت Episode / Speaker / Segment Locator برای Transcriptهای Post-Merge</h3>

<p>
پس از پایان Parsing، Merge محافظه‌کارانه و Quality Detection، اکنون باید برای هر Segment یک Locator قابل Audit ساخته شود.
</p>

<p>
این Locator هنوز Unified Schema مربوط به Notebook 07 نیست.
</p>

<p>
هدف آن فقط این است که در Notebook 06 بتوانیم هر Segment و هر Quality Candidate را بدون جست‌وجوی مبهم دوباره به محل دقیق Source متصل کنیم.
</p>

<h4>Locator هر Segment</h4>

<p>
برای هر یک از <code>14660</code> Utterance Post-Merge اطلاعات زیر ثبت می‌شود:
</p>

<ul>
<li>Program</li>
<li>Program ID</li>
<li>Physical File ID</li>
<li>Transcript Unit ID</li>
<li>Relative Source Path</li>
<li>Post-Merge Utterance Index</li>
<li>Utterance ID</li>
<li>Source Line Start</li>
<li>Source Line End</li>
<li>Speaker Raw</li>
<li>Speaker Canonical</li>
<li>Speaker Source Canonical</li>
<li>Speaker Status</li>
<li>Boundary Type</li>
<li>Merge Status</li>
<li>Start Time</li>
<li>End Time</li>
</ul>

<p>
Timestampها همچنان از Source پیروی می‌کنند و چون Structural Timestamp معتبر در Corpus پیدا نشده است، مقدار آن‌ها <code>None</code> باقی می‌ماند.
</p>

<h4>Episode / Unit Identity</h4>

<p>
شماره Episode یا Unit از متن Filename حدس زده نمی‌شود.
</p>

<p>
Locator از شناسه‌های موجود در Manifest و Parser استفاده می‌کند:
</p>

<ul>
<li><code>transcript_unit_id</code></li>
<li><code>physical_file_id</code></li>
<li><code>relative_path</code></li>
</ul>

<p>
بنابراین هیچ Episode Number جدیدی ساخته یا حدس زده نمی‌شود.
</p>

<h4>Quality Candidate Locator</h4>

<p>
تمام <code>120</code> Quality Candidate نیز به Segment Locator متصل می‌شوند.
</p>

<p>
اگر Candidate مربوط به یک Utterance باشد، Locator همان Segment ثبت می‌شود.
</p>

<p>
اگر Candidate از نوع Exact Adjacent Duplicate باشد و دو Utterance داشته باشد، هر دو Locator حفظ می‌شوند.
</p>

<h4>هدف Review انسانی</h4>

<p>
این ساختار در مرحله بعد برای ساخت Human Review Packet استفاده خواهد شد.
</p>

<p>
Reviewer باید بتواند مستقیماً ببیند:
</p>

<ul>
<li>Candidate در کدام Program است.</li>
<li>در کدام Source File است.</li>
<li>Speaker چه وضعیتی دارد.</li>
<li>دقیقاً روی چه Source Lineهایی قرار گرفته است.</li>
<li>آیا Segment Merge شده یا دست‌نخورده است.</li>
</ul>

<h4>عدم تغییر داده</h4>

<p>
در این Cell هیچ‌کدام از موارد زیر انجام نمی‌شوند:
</p>

<ul>
<li>اصلاح Quality Candidate</li>
<li>حذف Duplicate</li>
<li>Spell correction</li>
<li>ASR correction</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>برای هر <code>14660</code> Segment دقیقاً یک Locator ساخته شود.</li>
<li>Locator IDها یکتا باشند.</li>
<li>تمام Source Pathها به فایل Accepted معتبر متصل باشند.</li>
<li>تمام Source Rangeها معتبر باشند.</li>
<li>تمام <code>120</code> Quality Candidate به Locator معتبر متصل شوند.</li>
<li>هیچ Quality Candidate دارای Utterance ID ناشناخته نباشد.</li>
<li>هیچ Timestamp جدیدی ساخته نشود.</li>
</ul>

</div>

In [29]:
# Code Cell 29

merged_utterance_by_id = {
    item["utterance_id"]: item
    for item
    in all_merged_utterances_v1
}

assert len(
    merged_utterance_by_id
) == len(
    all_merged_utterances_v1
), (
    "Post-merge utterance IDs are not unique."
)


transcript_segment_locator_v1 = []

for item in all_merged_utterances_v1:
    locator_id = (
        f"{item['transcript_unit_id']}"
        f":segment:"
        f"{item['utterance_index']:05d}"
    )

    transcript_segment_locator_v1.append({
        "locator_id": locator_id,
        "program": item["program"],
        "program_id": item["program_id"],
        "physical_file_id": (
            item["physical_file_id"]
        ),
        "transcript_unit_id": (
            item["transcript_unit_id"]
        ),
        "relative_path": (
            item["relative_path"]
        ),
        "utterance_index": (
            item["utterance_index"]
        ),
        "utterance_id": (
            item["utterance_id"]
        ),
        "source_line_start": (
            item["source_line_start"]
        ),
        "source_line_end": (
            item["source_line_end"]
        ),
        "speaker_raw": (
            item["speaker_raw"]
        ),
        "speaker_canonical": (
            item["speaker_canonical"]
        ),
        "speaker_source_canonical": (
            item[
                "speaker_source_canonical"
            ]
        ),
        "speaker_status": (
            item["speaker_status"]
        ),
        "boundary_type": (
            item["boundary_type"]
        ),
        "merge_status": (
            item["merge_status"]
        ),
        "start_time": (
            item["start_time"]
        ),
        "end_time": (
            item["end_time"]
        ),
    })


locator_by_utterance_id = {
    item["utterance_id"]: item
    for item
    in transcript_segment_locator_v1
}

locator_by_locator_id = {
    item["locator_id"]: item
    for item
    in transcript_segment_locator_v1
}


quality_candidate_locator_links_v1 = []
unknown_quality_utterance_ids = []

for candidate in (
    transcript_quality_candidates_v1
):
    linked_locators = []

    for utterance_id in candidate[
        "utterance_ids"
    ]:
        locator = (
            locator_by_utterance_id.get(
                utterance_id
            )
        )

        if locator is None:
            unknown_quality_utterance_ids.append({
                "quality_candidate_id": (
                    candidate[
                        "quality_candidate_id"
                    ]
                ),
                "utterance_id": (
                    utterance_id
                ),
            })
            continue

        linked_locators.append({
            "locator_id": (
                locator["locator_id"]
            ),
            "utterance_id": (
                locator["utterance_id"]
            ),
            "relative_path": (
                locator["relative_path"]
            ),
            "utterance_index": (
                locator["utterance_index"]
            ),
            "source_line_start": (
                locator[
                    "source_line_start"
                ]
            ),
            "source_line_end": (
                locator[
                    "source_line_end"
                ]
            ),
            "speaker_raw": (
                locator["speaker_raw"]
            ),
            "speaker_canonical": (
                locator[
                    "speaker_canonical"
                ]
            ),
            "speaker_status": (
                locator[
                    "speaker_status"
                ]
            ),
            "merge_status": (
                locator[
                    "merge_status"
                ]
            ),
        })

    quality_candidate_locator_links_v1.append({
        "quality_candidate_id": (
            candidate[
                "quality_candidate_id"
            ]
        ),
        "candidate_type": (
            candidate[
                "candidate_type"
            ]
        ),
        "review_status": (
            candidate[
                "review_status"
            ]
        ),
        "program": (
            candidate["program"]
        ),
        "program_id": (
            candidate["program_id"]
        ),
        "relative_path": (
            candidate["relative_path"]
        ),
        "linked_segment_count": (
            len(linked_locators)
        ),
        "segment_locators": (
            linked_locators
        ),
    })


print(
    "Post-merge segments:",
    len(all_merged_utterances_v1),
)
print(
    "Segment locators:",
    len(
        transcript_segment_locator_v1
    ),
)
print(
    "Unique locator IDs:",
    len(locator_by_locator_id),
)
print(
    "Unique linked utterance IDs:",
    len(locator_by_utterance_id),
)
print()

print("LOCATOR COUNTS BY PROGRAM")
print("=" * 100)

for program_id in sorted(
    files_by_program
):
    program_locators = [
        item
        for item
        in transcript_segment_locator_v1
        if item["program_id"]
        == program_id
    ]

    if not program_locators:
        continue

    print()
    print(
        "Program:",
        program_locators[0][
            "program"
        ],
    )
    print(
        "Program ID:",
        program_id,
    )
    print(
        "Segments:",
        len(program_locators),
    )

    speaker_status_counts = Counter(
        item["speaker_status"]
        for item
        in program_locators
    )

    print(
        "Speaker status:",
        dict(
            sorted(
                speaker_status_counts.items()
            )
        ),
    )

    merge_status_counts = Counter(
        item["merge_status"]
        for item
        in program_locators
    )

    print(
        "Merge status:",
        dict(
            sorted(
                merge_status_counts.items()
            )
        ),
    )

    print("-" * 100)


print()
print("QUALITY CANDIDATE LOCATOR LINKING")
print("=" * 100)
print(
    "Quality candidates:",
    len(
        transcript_quality_candidates_v1
    ),
)
print(
    "Candidate locator records:",
    len(
        quality_candidate_locator_links_v1
    ),
)
print(
    "Unknown quality utterance IDs:",
    len(
        unknown_quality_utterance_ids
    ),
)

linked_segment_count_distribution = Counter(
    item["linked_segment_count"]
    for item
    in quality_candidate_locator_links_v1
)

print(
    "Linked segment-count distribution:",
    dict(
        sorted(
            linked_segment_count_distribution.items()
        )
    ),
)


print()
print("REPRESENTATIVE SEGMENT LOCATORS")
print("=" * 100)

shown_by_program = Counter()

for locator in (
    transcript_segment_locator_v1
):
    if shown_by_program[
        locator["program_id"]
    ] >= 2:
        continue

    shown_by_program[
        locator["program_id"]
    ] += 1

    print()
    print(
        "Locator ID:",
        locator["locator_id"],
    )
    print(
        "Program:",
        locator["program"],
    )
    print(
        "File:",
        locator["relative_path"],
    )
    print(
        "Utterance index:",
        locator["utterance_index"],
    )
    print(
        "Utterance ID:",
        locator["utterance_id"],
    )
    print(
        "Source lines:",
        (
            locator[
                "source_line_start"
            ],
            locator[
                "source_line_end"
            ],
        ),
    )
    print(
        "Speaker raw:",
        locator["speaker_raw"],
    )
    print(
        "Speaker canonical:",
        locator[
            "speaker_canonical"
        ],
    )
    print(
        "Speaker status:",
        locator[
            "speaker_status"
        ],
    )
    print(
        "Boundary type:",
        locator[
            "boundary_type"
        ],
    )
    print(
        "Merge status:",
        locator[
            "merge_status"
        ],
    )
    print(
        "Timestamp:",
        (
            locator["start_time"],
            locator["end_time"],
        ),
    )
    print("-" * 100)


print()
print(
    "REPRESENTATIVE QUALITY CANDIDATE LINKS"
)
print("=" * 100)

for item in (
    quality_candidate_locator_links_v1[:10]
):
    print()
    print(
        "Candidate ID:",
        item[
            "quality_candidate_id"
        ],
    )
    print(
        "Type:",
        item["candidate_type"],
    )
    print(
        "Program:",
        item["program"],
    )
    print(
        "Linked segments:",
        item[
            "linked_segment_count"
        ],
    )

    for locator in item[
        "segment_locators"
    ]:
        print(
            " ",
            locator["locator_id"],
            "|",
            locator[
                "relative_path"
            ],
            "| lines",
            (
                locator[
                    "source_line_start"
                ],
                locator[
                    "source_line_end"
                ],
            ),
            "| speaker:",
            locator[
                "speaker_raw"
            ]
            or locator[
                "speaker_canonical"
            ]
            or locator[
                "speaker_status"
            ],
        )

    print("-" * 100)


invalid_source_ranges = []
unknown_source_paths = []

for locator in (
    transcript_segment_locator_v1
):
    relative_path = locator[
        "relative_path"
    ]

    if (
        relative_path
        not in accepted_record_by_path
    ):
        unknown_source_paths.append(
            relative_path
        )
        continue

    file_path = (
        project_root
        / relative_path
    )

    line_count = len(
        file_path.read_text(
            encoding="utf-8"
        ).splitlines()
    )

    if not (
        1
        <= locator[
            "source_line_start"
        ]
        <= locator[
            "source_line_end"
        ]
        <= line_count
    ):
        invalid_source_ranges.append({
            "locator_id": (
                locator["locator_id"]
            ),
            "relative_path": (
                relative_path
            ),
            "source_line_start": (
                locator[
                    "source_line_start"
                ]
            ),
            "source_line_end": (
                locator[
                    "source_line_end"
                ]
            ),
            "file_line_count": (
                line_count
            ),
        })


assert len(
    transcript_segment_locator_v1
) == 14660, (
    "Segment locator does not cover all post-merge utterances."
)

assert len(
    locator_by_locator_id
) == len(
    transcript_segment_locator_v1
), (
    "Locator IDs are not unique."
)

assert len(
    locator_by_utterance_id
) == len(
    transcript_segment_locator_v1
), (
    "Not every post-merge utterance has exactly one locator."
)

assert not unknown_source_paths, (
    "At least one locator references a non-accepted source path."
)

assert not invalid_source_ranges, (
    "At least one locator has an invalid Source range."
)

assert len(
    quality_candidate_locator_links_v1
) == len(
    transcript_quality_candidates_v1
), (
    "Quality candidate locator coverage is incomplete."
)

assert not unknown_quality_utterance_ids, (
    "At least one quality candidate references an unknown post-merge utterance."
)

assert all(
    item["linked_segment_count"]
    in {1, 2}
    for item
    in quality_candidate_locator_links_v1
), (
    "Unexpected quality-candidate segment-link count."
)

assert all(
    item["start_time"] is None
    and item["end_time"] is None
    for item
    in transcript_segment_locator_v1
), (
    "Unexpected timestamp found in segment locator."
)

print()
print(
    "EPISODE / SPEAKER / SEGMENT "
    "LOCATOR BUILD: PASS"
)

Post-merge segments: 14660
Segment locators: 14660
Unique locator IDs: 14660
Unique linked utterance IDs: 14660

LOCATOR COUNTS BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Segments: 902
Speaker status: {'EXPLICIT_SOURCE_MARKER': 902}
Merge status: {'UNCHANGED': 902}
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Segments: 453
Speaker status: {'MANIFEST_ATTRIBUTED': 453}
Merge status: {'UNCHANGED': 453}
----------------------------------------------------------------------------------------------------

Program: چهلستون
Program ID: program:chehelston
Segments: 889
Speaker status: {'EXPLICIT_SOURCE_MARKER': 504, 'UNRESOLVED': 385}
Merge status: {'UNCHANGED': 889}
----------------------------------------------------------------------------------------------------

Program: در جستجوی خوشبختی
Program ID: program:dar-jostojo-ye-khoshbakhti
Segments: 7236
Speaker status:

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت مجموعه Human Review چندبرنامه‌ای برای حداقل ۲۰ Transcript Unit</h3>

<p>
تا اینجا Parsing، Merge، Quality Detection و Segment Locator به‌صورت ماشینی و قابل Audit انجام شده‌اند.
</p>

<p>
اما این موضوع جای Human Review واقعی را نمی‌گیرد.
</p>

<p>
در این مرحله یک مجموعه Review چندبرنامه‌ای و Risk-aware ساخته می‌شود تا در Cell بعد حداقل ۲۰ Transcript Unit واقعاً توسط انسان بررسی شوند.
</p>

<h4>اصل انتخاب</h4>

<p>
Review Set نباید فقط از فایل‌های آسان یا Programهای Solo ساخته شود.
</p>

<p>
بنابراین Selection باید همه ۱۳ Program را پوشش دهد و علاوه بر آن نمونه‌های دارای Risk یا پیچیدگی بیشتر را نیز وارد Review کند.
</p>

<h4>Signalهای مورد استفاده برای اولویت Review</h4>

<ul>
<li>وجود Segmentهای <code>UNRESOLVED</code></li>
<li>وجود <code>NARRATIVE_CUE</code></li>
<li>وجود Speaker Marker صریح</li>
<li>وجود Merge انجام‌شده در Notebook 06</li>
<li>وجود Quality Candidate</li>
<li>تنوع Speaker Status داخل همان Transcript Unit</li>
</ul>

<p>
این Signalها فقط برای اولویت‌بندی Review هستند.
</p>

<p>
هیچ‌کدام باعث اصلاح خودکار Transcript نمی‌شوند.
</p>

<h4>تعداد Review Unitها</h4>

<p>
برای عبور مطمئن از حداقل Roadmap، این Cell دقیقاً ۲۴ Transcript Unit انتخاب می‌کند.
</p>

<p>
ابتدا از هر Program حداقل یک Unit انتخاب می‌شود.
</p>

<p>
سپس تا رسیدن به ۲۴ Unit، Candidateهای Risk بالاتر از کل Corpus اضافه می‌شوند.
</p>

<h4>Unit Identity</h4>

<p>
شماره Episode از Filename حدس زده نمی‌شود.
</p>

<p>
هر Review Unit با شناسه‌های موجود در Pipeline مشخص می‌شود:
</p>

<ul>
<li><code>transcript_unit_id</code></li>
<li><code>physical_file_id</code></li>
<li><code>relative_path</code></li>
</ul>

<h4>موارد خاص که باید در Review قابل حضور باشند</h4>

<p>
با پوشش همه Programها، مجموعه Review شامل Programهای پیچیده و unresolved نیز خواهد بود، از جمله:
</p>

<ul>
<li>کتاب باز</li>
<li>میزان</li>
<li>اقیانوس آرام</li>
<li>شراکت</li>
<li>چهلستون</li>
<li>Programهای Multi-speaker دارای Marker صریح</li>
<li>Programهای Solo که Merge در آن‌ها انجام شده است</li>
</ul>

<h4>محدودیت این Cell</h4>

<p>
این Cell فقط Selection را می‌سازد.
</p>

<p>
هنوز Human Review انجام‌شده تلقی نمی‌شود.
</p>

<p>
در Cell بعد، Packet قابل خواندن برای Reviewer ساخته و نمایش داده خواهد شد.
</p>

<p>
همچنان انجام نمی‌شوند:
</p>

<ul>
<li>اصلاح Transcript</li>
<li>تأیید خودکار Quality Candidate</li>
<li>Spell correction</li>
<li>ASR correction</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>هر ۱۸۳ Source File برای Selection Profile بررسی شوند.</li>
<li>حداقل یک Unit از هر ۱۳ Program انتخاب شود.</li>
<li>دقیقاً ۲۴ Review Unit یکتا ساخته شود.</li>
<li>Selection شامل Unitهای unresolved، explicit-speaker، merged و quality-flagged باشد.</li>
<li>هیچ Transcript یا Quality Candidate تغییر نکند.</li>
</ul>

</div>

In [30]:
# Code Cell 30

quality_candidates_by_path = defaultdict(
    list
)

for candidate in (
    transcript_quality_candidates_v1
):
    quality_candidates_by_path[
        candidate["relative_path"]
    ].append(
        candidate
    )


review_unit_profiles = []

for parsed_file in (
    parsed_transcript_files_merged_v1
):
    relative_path = parsed_file[
        "relative_path"
    ]

    utterances = parsed_file[
        "utterances"
    ]

    assert utterances, (
        "Review selection encountered an empty parsed file."
    )

    first = utterances[0]

    speaker_status_counts = Counter(
        item["speaker_status"]
        for item in utterances
    )

    merge_status_counts = Counter(
        item["merge_status"]
        for item in utterances
    )

    file_quality_candidates = (
        quality_candidates_by_path.get(
            relative_path,
            [],
        )
    )

    quality_type_counts = Counter(
        item["candidate_type"]
        for item
        in file_quality_candidates
    )

    unresolved_count = (
        speaker_status_counts[
            "UNRESOLVED"
        ]
    )

    narrative_count = (
        speaker_status_counts[
            "NARRATIVE_CUE"
        ]
    )

    explicit_count = (
        speaker_status_counts[
            "EXPLICIT_SOURCE_MARKER"
        ]
    )

    manifest_count = (
        speaker_status_counts[
            "MANIFEST_ATTRIBUTED"
        ]
    )

    merged_count = (
        merge_status_counts[
            "MERGED"
        ]
    )

    quality_count = len(
        file_quality_candidates
    )

    speaker_status_diversity = len(
        speaker_status_counts
    )

    risk_score = 0

    if unresolved_count:
        risk_score += 50

    if narrative_count:
        risk_score += 35

    if quality_count:
        risk_score += 30

    if merged_count:
        risk_score += 25

    if explicit_count:
        risk_score += 15

    if speaker_status_diversity > 1:
        risk_score += 15

    risk_score += min(
        unresolved_count,
        10,
    )

    risk_score += min(
        quality_count,
        10,
    )

    risk_score += min(
        merged_count,
        10,
    )

    review_unit_profiles.append({
        "program": first[
            "program"
        ],
        "program_id": first[
            "program_id"
        ],
        "physical_file_id": first[
            "physical_file_id"
        ],
        "transcript_unit_id": first[
            "transcript_unit_id"
        ],
        "relative_path": (
            relative_path
        ),
        "segment_count": len(
            utterances
        ),
        "speaker_status_counts": dict(
            sorted(
                speaker_status_counts.items()
            )
        ),
        "merge_status_counts": dict(
            sorted(
                merge_status_counts.items()
            )
        ),
        "quality_candidate_count": (
            quality_count
        ),
        "quality_candidate_types": dict(
            sorted(
                quality_type_counts.items()
            )
        ),
        "unresolved_segment_count": (
            unresolved_count
        ),
        "narrative_cue_count": (
            narrative_count
        ),
        "explicit_speaker_segment_count": (
            explicit_count
        ),
        "manifest_attributed_segment_count": (
            manifest_count
        ),
        "merged_segment_count": (
            merged_count
        ),
        "speaker_status_diversity": (
            speaker_status_diversity
        ),
        "review_risk_score": (
            risk_score
        ),
    })


assert len(
    review_unit_profiles
) == 183, (
    "Review selection profile does not cover all accepted files."
)


profiles_by_program = defaultdict(
    list
)

for profile in review_unit_profiles:
    profiles_by_program[
        profile["program_id"]
    ].append(
        profile
    )


for program_id in profiles_by_program:
    profiles_by_program[
        program_id
    ].sort(
        key=lambda item: (
            -item[
                "review_risk_score"
            ],
            -item[
                "quality_candidate_count"
            ],
            -item[
                "unresolved_segment_count"
            ],
            -item[
                "merged_segment_count"
            ],
            item[
                "relative_path"
            ],
        )
    )


selected_review_units = []
selected_relative_paths = set()


for program_id in sorted(
    profiles_by_program
):
    highest_priority = (
        profiles_by_program[
            program_id
        ][0]
    )

    selected_review_units.append(
        highest_priority
    )

    selected_relative_paths.add(
        highest_priority[
            "relative_path"
        ]
    )


global_priority_order = sorted(
    review_unit_profiles,
    key=lambda item: (
        -item[
            "review_risk_score"
        ],
        -item[
            "quality_candidate_count"
        ],
        -item[
            "unresolved_segment_count"
        ],
        -item[
            "merged_segment_count"
        ],
        -item[
            "speaker_status_diversity"
        ],
        item[
            "program_id"
        ],
        item[
            "relative_path"
        ],
    ),
)


target_review_unit_count = 24

for profile in global_priority_order:
    if len(
        selected_review_units
    ) >= target_review_unit_count:
        break

    if (
        profile["relative_path"]
        in selected_relative_paths
    ):
        continue

    selected_review_units.append(
        profile
    )

    selected_relative_paths.add(
        profile[
            "relative_path"
        ]
    )


human_review_selection_v1 = []

for selection_index, profile in enumerate(
    selected_review_units,
    start=1,
):
    reasons = []

    if (
        profile[
            "unresolved_segment_count"
        ] > 0
    ):
        reasons.append(
            "HAS_UNRESOLVED_SEGMENTS"
        )

    if (
        profile[
            "narrative_cue_count"
        ] > 0
    ):
        reasons.append(
            "HAS_NARRATIVE_CUES"
        )

    if (
        profile[
            "explicit_speaker_segment_count"
        ] > 0
    ):
        reasons.append(
            "HAS_EXPLICIT_SPEAKER_MARKERS"
        )

    if (
        profile[
            "merged_segment_count"
        ] > 0
    ):
        reasons.append(
            "HAS_MERGED_SEGMENTS"
        )

    if (
        profile[
            "quality_candidate_count"
        ] > 0
    ):
        reasons.append(
            "HAS_QUALITY_CANDIDATES"
        )

    if (
        profile[
            "speaker_status_diversity"
        ] > 1
    ):
        reasons.append(
            "MULTIPLE_SPEAKER_STATUSES"
        )

    if not reasons:
        reasons.append(
            "PROGRAM_BASELINE_REVIEW"
        )

    human_review_selection_v1.append({
        "review_unit_id": (
            f"transcript-review:"
            f"{selection_index:03d}"
        ),
        "selection_index": (
            selection_index
        ),
        "selection_reasons": (
            reasons
        ),
        **profile,
    })


selected_program_counts = Counter(
    item["program_id"]
    for item
    in human_review_selection_v1
)

selected_signal_counts = Counter()

for item in human_review_selection_v1:
    if (
        item[
            "unresolved_segment_count"
        ] > 0
    ):
        selected_signal_counts[
            "HAS_UNRESOLVED_SEGMENTS"
        ] += 1

    if (
        item[
            "narrative_cue_count"
        ] > 0
    ):
        selected_signal_counts[
            "HAS_NARRATIVE_CUES"
        ] += 1

    if (
        item[
            "explicit_speaker_segment_count"
        ] > 0
    ):
        selected_signal_counts[
            "HAS_EXPLICIT_SPEAKER_MARKERS"
        ] += 1

    if (
        item[
            "merged_segment_count"
        ] > 0
    ):
        selected_signal_counts[
            "HAS_MERGED_SEGMENTS"
        ] += 1

    if (
        item[
            "quality_candidate_count"
        ] > 0
    ):
        selected_signal_counts[
            "HAS_QUALITY_CANDIDATES"
        ] += 1


print(
    "Corpus review-unit profiles:",
    len(review_unit_profiles),
)
print(
    "Programs represented in corpus:",
    len(profiles_by_program),
)
print(
    "Selected review units:",
    len(
        human_review_selection_v1
    ),
)
print(
    "Programs represented in selection:",
    len(selected_program_counts),
)
print()


print("SELECTION SIGNAL COVERAGE")
print("=" * 100)

for signal, count in sorted(
    selected_signal_counts.items()
):
    print(
        f"- {signal}: {count}"
    )


print()
print("HUMAN REVIEW SELECTION")
print("=" * 100)

for item in human_review_selection_v1:
    print()
    print(
        "Review unit:",
        item[
            "review_unit_id"
        ],
    )
    print(
        "Program:",
        item["program"],
    )
    print(
        "Program ID:",
        item[
            "program_id"
        ],
    )
    print(
        "Transcript unit ID:",
        item[
            "transcript_unit_id"
        ],
    )
    print(
        "File:",
        item[
            "relative_path"
        ],
    )
    print(
        "Risk score:",
        item[
            "review_risk_score"
        ],
    )
    print(
        "Segments:",
        item[
            "segment_count"
        ],
    )
    print(
        "Speaker status:",
        item[
            "speaker_status_counts"
        ],
    )
    print(
        "Merged segments:",
        item[
            "merged_segment_count"
        ],
    )
    print(
        "Quality candidates:",
        item[
            "quality_candidate_count"
        ],
    )
    print(
        "Quality types:",
        item[
            "quality_candidate_types"
        ],
    )
    print(
        "Selection reasons:",
        item[
            "selection_reasons"
        ],
    )
    print("-" * 100)


selected_relative_path_values = [
    item["relative_path"]
    for item
    in human_review_selection_v1
]

selected_transcript_unit_ids = [
    item["transcript_unit_id"]
    for item
    in human_review_selection_v1
]


assert len(
    human_review_selection_v1
) == 24, (
    "Human review selection must contain exactly 24 units."
)

assert len(
    set(
        selected_relative_path_values
    )
) == 24, (
    "Human review selection contains duplicate source files."
)

assert len(
    set(
        selected_transcript_unit_ids
    )
) == 24, (
    "Human review selection contains duplicate transcript units."
)

assert len(
    selected_program_counts
) == 13, (
    "Human review selection does not cover all 13 programs."
)

assert all(
    count >= 1
    for count in (
        selected_program_counts.values()
    )
), (
    "At least one program has no human-review unit."
)

assert selected_signal_counts[
    "HAS_UNRESOLVED_SEGMENTS"
] >= 1, (
    "Human review selection has no unresolved example."
)

assert selected_signal_counts[
    "HAS_EXPLICIT_SPEAKER_MARKERS"
] >= 1, (
    "Human review selection has no explicit-speaker example."
)

assert selected_signal_counts[
    "HAS_MERGED_SEGMENTS"
] >= 1, (
    "Human review selection has no merged example."
)

assert selected_signal_counts[
    "HAS_QUALITY_CANDIDATES"
] >= 1, (
    "Human review selection has no quality-flagged example."
)

print()
print(
    "STRATIFIED HUMAN REVIEW "
    "SELECTION BUILD: PASS"
)

Corpus review-unit profiles: 183
Programs represented in corpus: 13
Selected review units: 24
Programs represented in selection: 13

SELECTION SIGNAL COVERAGE
- HAS_EXPLICIT_SPEAKER_MARKERS: 13
- HAS_MERGED_SEGMENTS: 3
- HAS_NARRATIVE_CUES: 1
- HAS_QUALITY_CANDIDATES: 21
- HAS_UNRESOLVED_SEGMENTS: 16

HUMAN REVIEW SELECTION

Review unit: transcript-review:001
Program: عصر شیرین
Program ID: program:asre-shirin
Transcript unit ID: program:asre-shirin:episode-001
File: data/transcripts/final/asre_shirin/season-1/episode-01.md
Risk score: 51
Segments: 102
Speaker status: {'EXPLICIT_SOURCE_MARKER': 102}
Merged segments: 0
Quality candidates: 6
Quality types: {'BRACKETED_TRANSCRIPTION_MARKER': 5, 'REPEATED_NGRAM_RUN': 1}
Selection reasons: ['HAS_EXPLICIT_SPEAKER_MARKERS', 'HAS_QUALITY_CANDIDATES']
----------------------------------------------------------------------------------------------------

Review unit: transcript-review:002
Program: بانک ملت
Program ID: program:bank-mellat
Transcript

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت Human Review Packet برای ۲۴ Transcript Unit منتخب</h3>

<p>
Selection مرحله قبل فقط تعیین کرد کدام Transcript Unitها باید وارد Review انسانی شوند.
</p>

<p>
در این Cell هنوز هیچ Unit به‌صورت خودکار <code>HUMAN_REVIEWED</code> یا <code>PASS</code> علامت نمی‌خورد.
</p>

<p>
هدف این مرحله ساخت یک Packet قابل خواندن است تا Boundary، Speaker Attribution، Merge و Quality Flagها به‌صورت انسانی قابل بررسی باشند.
</p>

<h4>تفاوت Validation ماشینی و Human Review</h4>

<p>
Validationهای قبلی ثابت کرده‌اند که:
</p>

<ul>
<li>Source Lineage کامل است.</li>
<li>هیچ محتوایی گم نشده است.</li>
<li>هیچ Source Line دوبار مصرف نشده است.</li>
<li>Mergeها از نظر ساختاری قابل بازسازی‌اند.</li>
<li>Speaker ناشناخته حدس زده نشده است.</li>
</ul>

<p>
اما این Validationها نمی‌توانند به‌تنهایی مشخص کنند که مثلاً:
</p>

<ul>
<li>یک مرز گفتاری از نظر انسانی طبیعی است یا نه.</li>
<li>یک تکرار واقعی گفتار است یا خطای ASR.</li>
<li>یک Annotation داخل کروشه محتوای واقعی، توضیح تدوینی یا Noise است.</li>
<li>یک Merge از نظر معنایی نیز مناسب است یا فقط از نظر ساختاری مجاز بوده است.</li>
</ul>

<p>
این موارد نیازمند Human Review هستند.
</p>

<h4>نمونه‌های انتخاب‌شده داخل هر Unit</h4>

<p>
برای هر یک از ۲۴ Unit، حداکثر چهار Segment با اولویت Risk انتخاب می‌شود.
</p>

<p>
اولویت‌ها به‌ترتیب شامل موارد زیر هستند:
</p>

<ol>
<li>Segment متصل به Quality Candidate</li>
<li>Segment Merge‌شده</li>
<li>Segment دارای <code>UNRESOLVED</code> یا <code>NARRATIVE_CUE</code></li>
<li>Speaker Transition</li>
<li>نمونه ابتدای Unit</li>
<li>نمونه میانی Unit</li>
<li>نمونه انتهای Unit</li>
</ol>

<p>
اگر یک Segment چند ویژگی داشته باشد، تمام Focusهای آن روی همان Sample ثبت می‌شوند.
</p>

<h4>آنچه Reviewer می‌بیند</h4>

<p>
برای هر Sample موارد زیر نمایش داده می‌شوند:
</p>

<ul>
<li>Program</li>
<li>Source File</li>
<li>Segment Index</li>
<li>Source Line Range</li>
<li>Speaker Raw</li>
<li>Speaker Canonical</li>
<li>Speaker Status</li>
<li>Boundary Type</li>
<li>Merge Status</li>
<li>Review Focus</li>
<li>Parsed Text Preview</li>
<li>Raw Source Text Preview</li>
</ul>

<p>
متن کامل هر Sample نیز داخل متغیر
<code>human_review_packet_v1</code>
در حافظه حفظ می‌شود؛ Preview چاپ‌شده فقط برای جلوگیری از خروجی بسیار حجیم محدود می‌شود.
</p>

<h4>معیار بررسی انسانی</h4>

<p>
برای هر Unit باید بررسی شود که:
</p>

<ul>
<li>Speaker Labelهای صریح با Source سازگار باشند.</li>
<li>در فایل‌های unresolved هیچ Speaker حدس زده نشده باشد.</li>
<li>سؤال یا Turn افراد دیگر در فایل‌های Multi-speaker حذف نشده باشد.</li>
<li>Mergeهای نمایش‌داده‌شده از نظر معنایی نیز ادامه طبیعی یک گفتار باشند.</li>
<li>Quality Candidateها واقعاً نیازمند اصلاح هستند یا تکرار/Annotation طبیعی‌اند.</li>
<li>هیچ Segment واضحی به Speaker اشتباه نسبت داده نشده باشد.</li>
</ul>

<h4>نکته درباره Reviewهای قبلی</h4>

<p>
Reviewهای قبلی ویدئوها و Transcriptها ارزشمند هستند، اما این Gate به‌صورت مستقل روی Output همین Pipeline انجام می‌شود.
</p>

<p>
بنابراین تا زمانی که Packet حاضر بررسی نشود، Notebook به‌صورت خودکار ادعای Human Review نخواهد کرد.
</p>

<h4>محدودیت این Cell</h4>

<p>
هنوز انجام نمی‌شوند:
</p>

<ul>
<li>ثبت PASS انسانی</li>
<li>اصلاح Transcript</li>
<li>اصلاح ASR</li>
<li>Spell correction</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<h4>معیار PASS ماشینی این Cell</h4>

<ul>
<li>برای هر ۲۴ Review Unit یک Packet ساخته شود.</li>
<li>هر Packet حداقل یک Sample داشته باشد.</li>
<li>تمام Sampleها به Segment واقعی Post-Merge متصل باشند.</li>
<li>تمام Source Textها دوباره از Disk قابل تأیید باشند.</li>
<li>هیچ Transcript یا Quality Candidate تغییر نکند.</li>
</ul>

</div>

In [31]:
# Code Cell 31

merged_file_by_path = {
    item["relative_path"]: item
    for item
    in parsed_transcript_files_merged_v1
}

quality_links_by_path = defaultdict(list)

for link in quality_candidate_locator_links_v1:
    quality_links_by_path[
        link["relative_path"]
    ].append(
        link
    )


def preview_text(
    text,
    limit=420,
):
    flattened = text.replace(
        "\n",
        " ⏎ ",
    )

    if len(flattened) <= limit:
        return flattened

    return (
        flattened[:limit]
        + " … [TRUNCATED]"
    )


human_review_packet_v1 = []

packet_source_failures = []
packet_unknown_segment_failures = []

for review_unit in (
    human_review_selection_v1
):
    relative_path = review_unit[
        "relative_path"
    ]

    parsed_file = merged_file_by_path[
        relative_path
    ]

    utterances = parsed_file[
        "utterances"
    ]

    utterance_by_index = {
        item["utterance_index"]: item
        for item in utterances
    }

    focus_by_index = defaultdict(set)
    ordered_priority_indices = []


    def add_review_focus(
        utterance_index,
        focus,
    ):
        if (
            utterance_index
            not in utterance_by_index
        ):
            packet_unknown_segment_failures.append({
                "review_unit_id": (
                    review_unit[
                        "review_unit_id"
                    ]
                ),
                "relative_path": (
                    relative_path
                ),
                "utterance_index": (
                    utterance_index
                ),
                "focus": focus,
            })
            return

        focus_by_index[
            utterance_index
        ].add(
            focus
        )

        if (
            utterance_index
            not in ordered_priority_indices
        ):
            ordered_priority_indices.append(
                utterance_index
            )


    file_quality_links = (
        quality_links_by_path.get(
            relative_path,
            [],
        )
    )

    for quality_link in (
        file_quality_links[:3]
    ):
        for locator in quality_link[
            "segment_locators"
        ]:
            add_review_focus(
                locator[
                    "utterance_index"
                ],
                (
                    "QUALITY:"
                    + quality_link[
                        "candidate_type"
                    ]
                ),
            )


    merged_items = [
        item
        for item in utterances
        if item["merge_status"]
        == "MERGED"
    ]

    for item in merged_items[:2]:
        add_review_focus(
            item["utterance_index"],
            "MERGED_SEGMENT",
        )


    risk_status_items = [
        item
        for item in utterances
        if item["speaker_status"]
        in {
            "UNRESOLVED",
            "NARRATIVE_CUE",
        }
    ]

    for item in risk_status_items[:2]:
        add_review_focus(
            item["utterance_index"],
            (
                "SPEAKER_STATUS:"
                + item[
                    "speaker_status"
                ]
            ),
        )


    transition_count = 0

    for index in range(
        len(utterances) - 1
    ):
        left = utterances[index]
        right = utterances[
            index + 1
        ]

        left_identity = (
            left["speaker_raw"],
            left["speaker_canonical"],
            left["speaker_status"],
        )

        right_identity = (
            right["speaker_raw"],
            right["speaker_canonical"],
            right["speaker_status"],
        )

        if (
            left_identity
            == right_identity
        ):
            continue

        add_review_focus(
            left[
                "utterance_index"
            ],
            "SPEAKER_TRANSITION_LEFT",
        )

        add_review_focus(
            right[
                "utterance_index"
            ],
            "SPEAKER_TRANSITION_RIGHT",
        )

        transition_count += 1

        if transition_count >= 1:
            break


    baseline_indices = [
        1,
        (
            (len(utterances) + 1)
            // 2
        ),
        len(utterances),
    ]

    baseline_labels = [
        "BASELINE_START",
        "BASELINE_MIDDLE",
        "BASELINE_END",
    ]

    for utterance_index, label in zip(
        baseline_indices,
        baseline_labels,
    ):
        add_review_focus(
            utterance_index,
            label,
        )


    selected_indices = (
        ordered_priority_indices[:4]
    )

    samples = []

    source_file_path = (
        project_root
        / relative_path
    )

    source_lines = source_file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    for sample_index, utterance_index in enumerate(
        selected_indices,
        start=1,
    ):
        item = utterance_by_index[
            utterance_index
        ]

        expected_source_text = "\n".join(
            source_lines[
                item[
                    "source_line_start"
                ] - 1:
                item[
                    "source_line_end"
                ]
            ]
        )

        if (
            expected_source_text
            != item["source_text"]
        ):
            packet_source_failures.append({
                "review_unit_id": (
                    review_unit[
                        "review_unit_id"
                    ]
                ),
                "utterance_id": (
                    item[
                        "utterance_id"
                    ]
                ),
                "relative_path": (
                    relative_path
                ),
            })

        related_quality_ids = []

        for quality_link in file_quality_links:
            linked_ids = {
                locator[
                    "utterance_id"
                ]
                for locator
                in quality_link[
                    "segment_locators"
                ]
            }

            if (
                item["utterance_id"]
                in linked_ids
            ):
                related_quality_ids.append(
                    quality_link[
                        "quality_candidate_id"
                    ]
                )

        samples.append({
            "sample_index": (
                sample_index
            ),
            "review_focus": sorted(
                focus_by_index[
                    utterance_index
                ]
            ),
            "utterance_index": (
                utterance_index
            ),
            "utterance_id": (
                item["utterance_id"]
            ),
            "source_line_start": (
                item[
                    "source_line_start"
                ]
            ),
            "source_line_end": (
                item[
                    "source_line_end"
                ]
            ),
            "speaker_raw": (
                item["speaker_raw"]
            ),
            "speaker_canonical": (
                item[
                    "speaker_canonical"
                ]
            ),
            "speaker_source_canonical": (
                item[
                    "speaker_source_canonical"
                ]
            ),
            "speaker_status": (
                item["speaker_status"]
            ),
            "boundary_type": (
                item["boundary_type"]
            ),
            "merge_status": (
                item["merge_status"]
            ),
            "merged_from_utterance_ids": (
                item[
                    "merged_from_utterance_ids"
                ]
            ),
            "related_quality_candidate_ids": (
                related_quality_ids
            ),
            "parsed_text": (
                item["text"]
            ),
            "source_text": (
                item["source_text"]
            ),
        })


    human_review_packet_v1.append({
        "review_unit_id": (
            review_unit[
                "review_unit_id"
            ]
        ),
        "selection_index": (
            review_unit[
                "selection_index"
            ]
        ),
        "program": (
            review_unit["program"]
        ),
        "program_id": (
            review_unit[
                "program_id"
            ]
        ),
        "transcript_unit_id": (
            review_unit[
                "transcript_unit_id"
            ]
        ),
        "physical_file_id": (
            review_unit[
                "physical_file_id"
            ]
        ),
        "relative_path": (
            relative_path
        ),
        "segment_count": (
            len(utterances)
        ),
        "quality_candidate_count": (
            review_unit[
                "quality_candidate_count"
            ]
        ),
        "merged_segment_count": (
            review_unit[
                "merged_segment_count"
            ]
        ),
        "speaker_status_counts": (
            review_unit[
                "speaker_status_counts"
            ]
        ),
        "selection_reasons": (
            review_unit[
                "selection_reasons"
            ]
        ),
        "human_review_status": (
            "PENDING"
        ),
        "samples": samples,
    })


print(
    "Human review units:",
    len(human_review_packet_v1),
)
print(
    "Total review samples:",
    sum(
        len(
            item["samples"]
        )
        for item
        in human_review_packet_v1
    ),
)
print(
    "Source verification failures:",
    len(packet_source_failures),
)
print(
    "Unknown sample segment failures:",
    len(
        packet_unknown_segment_failures
    ),
)


print()
print("HUMAN REVIEW PACKET")
print("=" * 110)

for unit in human_review_packet_v1:
    print()
    print(
        "Review unit:",
        unit[
            "review_unit_id"
        ],
    )
    print(
        "Program:",
        unit["program"],
    )
    print(
        "Transcript unit:",
        unit[
            "transcript_unit_id"
        ],
    )
    print(
        "File:",
        unit["relative_path"],
    )
    print(
        "Segments:",
        unit["segment_count"],
    )
    print(
        "Speaker status:",
        unit[
            "speaker_status_counts"
        ],
    )
    print(
        "Merged segments:",
        unit[
            "merged_segment_count"
        ],
    )
    print(
        "Quality candidates:",
        unit[
            "quality_candidate_count"
        ],
    )
    print(
        "Selection reasons:",
        unit[
            "selection_reasons"
        ],
    )
    print(
        "Human review status:",
        unit[
            "human_review_status"
        ],
    )

    for sample in unit[
        "samples"
    ]:
        print()
        print(
            "  Sample:",
            sample[
                "sample_index"
            ],
        )
        print(
            "  Focus:",
            sample[
                "review_focus"
            ],
        )
        print(
            "  Segment index:",
            sample[
                "utterance_index"
            ],
        )
        print(
            "  Source lines:",
            (
                sample[
                    "source_line_start"
                ],
                sample[
                    "source_line_end"
                ],
            ),
        )
        print(
            "  Speaker raw:",
            sample[
                "speaker_raw"
            ],
        )
        print(
            "  Speaker canonical:",
            sample[
                "speaker_canonical"
            ],
        )
        print(
            "  Speaker status:",
            sample[
                "speaker_status"
            ],
        )
        print(
            "  Boundary:",
            sample[
                "boundary_type"
            ],
        )
        print(
            "  Merge status:",
            sample[
                "merge_status"
            ],
        )
        print(
            "  Quality IDs:",
            sample[
                "related_quality_candidate_ids"
            ],
        )
        print(
            "  PARSED:",
            preview_text(
                sample[
                    "parsed_text"
                ]
            ),
        )
        print(
            "  SOURCE:",
            preview_text(
                sample[
                    "source_text"
                ]
            ),
        )
        print(
            "  " + "-" * 100
        )

    print("=" * 110)


assert len(
    human_review_packet_v1
) == 24, (
    "Human review packet does not contain 24 units."
)

assert all(
    item[
        "human_review_status"
    ] == "PENDING"
    for item
    in human_review_packet_v1
), (
    "Human review status was assigned automatically."
)

assert all(
    len(item["samples"]) >= 1
    for item
    in human_review_packet_v1
), (
    "At least one review unit has no samples."
)

assert not packet_source_failures, (
    "Human review packet contains a Source mismatch."
)

assert not packet_unknown_segment_failures, (
    "Human review packet references an unknown segment."
)

assert {
    item["program_id"]
    for item
    in human_review_packet_v1
} == set(
    profiles_by_program
), (
    "Human review packet lost program coverage."
)

print()
print(
    "HUMAN REVIEW PACKET BUILD: PASS"
)
print(
    "HUMAN REVIEW STATUS: PENDING"
)

Human review units: 24
Total review samples: 96
Source verification failures: 0
Unknown sample segment failures: 0

HUMAN REVIEW PACKET

Review unit: transcript-review:001
Program: عصر شیرین
Transcript unit: program:asre-shirin:episode-001
File: data/transcripts/final/asre_shirin/season-1/episode-01.md
Segments: 102
Speaker status: {'EXPLICIT_SOURCE_MARKER': 102}
Merged segments: 0
Quality candidates: 6
Selection reasons: ['HAS_EXPLICIT_SPEAKER_MARKERS', 'HAS_QUALITY_CANDIDATES']
Human review status: PENDING

  Sample: 1
  Focus: ['QUALITY:REPEATED_NGRAM_RUN']
  Segment index: 64
  Source lines: (262, 264)
  Speaker raw: دکتر کمیل رودی
  Speaker canonical: None
  Speaker status: EXPLICIT_SOURCE_MARKER
  Boundary: BOLD_ONLY_SPEAKER_BOUNDARY
  Merge status: UNCHANGED
  Quality IDs: ['transcript-quality:000001']
  PARSED: که برای همه لازمه؛ برای همه. یعنی اهمیتش، اهمیت زندگیه. ما داریم زندگی می‌کنیم و این یک مهارت زندگیه، مثل مهارت ارتباط داشتن، مهارت گفت‌وگو کردن؛ یک مهارتیه که همه باید 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ممیزی مرزهای داخلی بدون Label در فایل‌های Explicit چهلستون</h3>

<p>
Human Review Packet نشان داد که بعضی Segmentهای چهلستون با یک Speaker Marker صریح شروع می‌شوند، اما داخل همان Segment چند Paragraph بدون Speaker Marker قرار گرفته‌اند.
</p>

<p>
در بعضی نمونه‌ها Paragraph اول شامل سؤال مجری است و Paragraphهای بعدی ممکن است ادامه همان Speaker یا پاسخ فرد دیگری باشند.
</p>

<p>
از آنجا که Speaker Label صریح برای Paragraphهای بعدی وجود ندارد، نسبت‌دادن خودکار همه آن‌ها به Speaker اولیه می‌تواند Attribution بیش از Evidence باشد.
</p>

<h4>اصل این Diagnostic</h4>

<p>
این Cell هیچ Speaker جدیدی حدس نمی‌زند و هیچ Segmentی را تغییر نمی‌دهد.
</p>

<p>
فقط بررسی می‌کند در فایل‌های Explicit چهلستون:
</p>

<ul>
<li>هر Segment چند Paragraph محتوایی دارد.</li>
<li>چند Segment بیش از یک Paragraph بدون Label داخلی دارند.</li>
<li>Speaker Marker اولیه چه بوده است.</li>
<li>آیا Paragraph اول دارای علامت سؤال است.</li>
<li>آیا بعد از Paragraph سؤال، Paragraphهای بدون Label دیگری داخل همان Segment وجود دارند.</li>
<li>آیا Segmentهای دارای چند Paragraph در Human Review Packet نیز دیده شده‌اند.</li>
</ul>

<h4>چرا علامت سؤال فقط Signal است</h4>

<p>
وجود علامت سؤال به‌تنهایی اثبات Speaker Change نیست.
</p>

<p>
یک Speaker می‌تواند سؤال بلاغی بپرسد و خودش ادامه بدهد.
</p>

<p>
بنابراین Classification این Cell فقط برای Review است و هیچ Attribution جدیدی ایجاد نمی‌کند.
</p>

<h4>Classificationهای Diagnostic</h4>

<ul>
<li><code>SINGLE_PARAGRAPH_EXPLICIT</code>: فقط یک Paragraph محتوایی زیر Marker وجود دارد.</li>
<li><code>MULTI_PARAGRAPH_EXPLICIT</code>: چند Paragraph زیر یک Marker وجود دارد ولی Signal قوی‌تری نداریم.</li>
<li><code>QUESTION_THEN_UNLABELED_PARAGRAPHS</code>: Paragraph اول دارای سؤال است و بعد از آن Paragraph بدون Label دیگری وجود دارد.</li>
</ul>

<p>
گروه سوم به معنی Speaker Change قطعی نیست؛ فقط نشان می‌دهد Attribution همه Paragraphها به Speaker اولیه نیازمند Human Evidence است.
</p>

<h4>محدودیت</h4>

<p>
در این Cell انجام نمی‌شوند:
</p>

<ul>
<li>تغییر Parser</li>
<li>Split کردن Segment</li>
<li>حدس Speaker</li>
<li>Canonicalization جدید</li>
<li>اصلاح متن</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>تمام Segmentهای Explicit چهلستون بررسی شوند.</li>
<li>تمام Paragraphهای داخلی بدون تغییر شمارش شوند.</li>
<li>Segmentهای چندپاراگرافی جدا گزارش شوند.</li>
<li>Question-then-unlabeled فقط به‌عنوان Review Signal ثبت شود.</li>
<li>هیچ Speaker جدیدی ساخته یا استنتاج نشود.</li>
<li>خروجی Post-Merge تغییر نکند.</li>
</ul>

</div>

In [32]:
# Code Cell 32

postmerge_state_before_chehelston_boundary_audit = json.dumps(
    all_merged_utterances_v1,
    ensure_ascii=False,
    sort_keys=True,
)

chehelston_explicit_boundary_audit_v1 = []

chehelston_explicit_segments = [
    item
    for item in all_merged_utterances_v1
    if (
        item["program_id"]
        == chehelston_program_id
        and item["speaker_status"]
        == "EXPLICIT_SOURCE_MARKER"
        and item["boundary_type"]
        == "COLON_SPEAKER_BOUNDARY"
    )
]


def split_nonempty_paragraphs(text):
    return [
        paragraph.strip()
        for paragraph in re.split(
            r"\n\s*\n",
            text,
        )
        if paragraph.strip()
    ]


def paragraph_has_question(text):
    return (
        "؟" in text
        or "?" in text
    )


for item in chehelston_explicit_segments:
    paragraphs = split_nonempty_paragraphs(
        item["text"]
    )

    first_paragraph = (
        paragraphs[0]
        if paragraphs
        else ""
    )

    remaining_paragraphs = (
        paragraphs[1:]
        if len(paragraphs) > 1
        else []
    )

    first_has_question = (
        paragraph_has_question(
            first_paragraph
        )
    )

    if len(paragraphs) == 1:
        classification = (
            "SINGLE_PARAGRAPH_EXPLICIT"
        )

    elif (
        first_has_question
        and remaining_paragraphs
    ):
        classification = (
            "QUESTION_THEN_UNLABELED_PARAGRAPHS"
        )

    else:
        classification = (
            "MULTI_PARAGRAPH_EXPLICIT"
        )

    chehelston_explicit_boundary_audit_v1.append({
        "program": item["program"],
        "program_id": item["program_id"],
        "relative_path": item[
            "relative_path"
        ],
        "utterance_id": item[
            "utterance_id"
        ],
        "utterance_index": item[
            "utterance_index"
        ],
        "source_line_start": item[
            "source_line_start"
        ],
        "source_line_end": item[
            "source_line_end"
        ],
        "speaker_raw": item[
            "speaker_raw"
        ],
        "speaker_canonical": item[
            "speaker_canonical"
        ],
        "paragraph_count": len(
            paragraphs
        ),
        "first_paragraph_has_question": (
            first_has_question
        ),
        "classification": (
            classification
        ),
        "paragraphs": paragraphs,
    })


classification_counts = Counter(
    item["classification"]
    for item
    in chehelston_explicit_boundary_audit_v1
)

speaker_classification_counts = Counter(
    (
        item["speaker_raw"],
        item["classification"],
    )
    for item
    in chehelston_explicit_boundary_audit_v1
)

multi_paragraph_items = [
    item
    for item
    in chehelston_explicit_boundary_audit_v1
    if item["paragraph_count"] > 1
]

question_then_unlabeled_items = [
    item
    for item
    in chehelston_explicit_boundary_audit_v1
    if (
        item["classification"]
        == "QUESTION_THEN_UNLABELED_PARAGRAPHS"
    )
]


review_packet_paths = {
    item["relative_path"]
    for item
    in human_review_packet_v1
}

review_overlap_items = [
    item
    for item
    in question_then_unlabeled_items
    if item["relative_path"]
    in review_packet_paths
]


print(
    "Chehelston explicit segments inspected:",
    len(
        chehelston_explicit_boundary_audit_v1
    ),
)
print(
    "Multi-paragraph explicit segments:",
    len(multi_paragraph_items),
)
print(
    "Question then unlabeled-paragraph signals:",
    len(
        question_then_unlabeled_items
    ),
)
print(
    "Question-signal segments in current human-review files:",
    len(review_overlap_items),
)
print()


print("CLASSIFICATION COUNTS")
print("=" * 100)

for classification, count in sorted(
    classification_counts.items()
):
    print(
        f"- {classification}: {count}"
    )


print()
print("CLASSIFICATION BY SOURCE SPEAKER")
print("=" * 100)

for (
    speaker_raw,
    classification,
), count in sorted(
    speaker_classification_counts.items(),
    key=lambda item: (
        str(item[0][0]),
        item[0][1],
    ),
):
    print(
        f"- {speaker_raw!r} | "
        f"{classification}: {count}"
    )


print()
print(
    "QUESTION THEN UNLABELED PARAGRAPHS BY FILE"
)
print("=" * 100)

question_signal_by_file = Counter(
    item["relative_path"]
    for item
    in question_then_unlabeled_items
)

if question_signal_by_file:
    for relative_path, count in (
        question_signal_by_file.most_common()
    ):
        print(
            f"- {relative_path}: {count}"
        )
else:
    print("- NONE")


print()
print(
    "REPRESENTATIVE QUESTION-THEN-UNLABELED SEGMENTS"
)
print("=" * 100)

shown_by_file = Counter()

for item in question_then_unlabeled_items:
    if shown_by_file[
        item["relative_path"]
    ] >= 3:
        continue

    shown_by_file[
        item["relative_path"]
    ] += 1

    print()
    print(
        "File:",
        item["relative_path"],
    )
    print(
        "Utterance index:",
        item["utterance_index"],
    )
    print(
        "Utterance ID:",
        item["utterance_id"],
    )
    print(
        "Source lines:",
        (
            item["source_line_start"],
            item["source_line_end"],
        ),
    )
    print(
        "Speaker raw:",
        item["speaker_raw"],
    )
    print(
        "Speaker canonical:",
        item[
            "speaker_canonical"
        ],
    )
    print(
        "Paragraph count:",
        item["paragraph_count"],
    )

    for paragraph_index, paragraph in enumerate(
        item["paragraphs"],
        start=1,
    ):
        print()
        print(
            f"  Paragraph {paragraph_index}:"
        )
        print(
            " ",
            preview_text(
                paragraph,
                limit=500,
            ),
        )

    print("-" * 100)


print()
print(
    "CURRENT HUMAN-REVIEW OVERLAP EXAMPLES"
)
print("=" * 100)

for item in review_overlap_items[:20]:
    print()
    print(
        "File:",
        item["relative_path"],
    )
    print(
        "Segment:",
        item["utterance_index"],
    )
    print(
        "Speaker raw:",
        item["speaker_raw"],
    )
    print(
        "Paragraphs:",
        item["paragraph_count"],
    )
    print(
        "First paragraph:",
        preview_text(
            item["paragraphs"][0],
            limit=350,
        ),
    )

    if len(
        item["paragraphs"]
    ) > 1:
        print(
            "Second paragraph:",
            preview_text(
                item["paragraphs"][1],
                limit=350,
            ),
        )

    print("-" * 100)


postmerge_state_after_chehelston_boundary_audit = json.dumps(
    all_merged_utterances_v1,
    ensure_ascii=False,
    sort_keys=True,
)


assert len(
    chehelston_explicit_boundary_audit_v1
) == sum(
    1
    for item in all_merged_utterances_v1
    if (
        item["program_id"]
        == chehelston_program_id
        and item["speaker_status"]
        == "EXPLICIT_SOURCE_MARKER"
        and item["boundary_type"]
        == "COLON_SPEAKER_BOUNDARY"
    )
), (
    "Chehelston explicit boundary audit coverage is incomplete."
)

assert all(
    item["paragraph_count"] >= 1
    for item
    in chehelston_explicit_boundary_audit_v1
), (
    "Empty explicit Chehelston segment reached the audit."
)

assert all(
    item["classification"]
    in {
        "SINGLE_PARAGRAPH_EXPLICIT",
        "MULTI_PARAGRAPH_EXPLICIT",
        "QUESTION_THEN_UNLABELED_PARAGRAPHS",
    }
    for item
    in chehelston_explicit_boundary_audit_v1
), (
    "Unexpected Chehelston boundary classification found."
)

assert all(
    item["speaker_raw"] is not None
    for item
    in chehelston_explicit_boundary_audit_v1
), (
    "Explicit Chehelston audit contains a missing raw speaker."
)

assert (
    postmerge_state_before_chehelston_boundary_audit
    == postmerge_state_after_chehelston_boundary_audit
), (
    "Chehelston diagnostic mutated post-merge output."
)

print()
print(
    "CHEHELSTON EMBEDDED UNLABELED-BOUNDARY "
    "DIAGNOSTIC: PASS"
)

Chehelston explicit segments inspected: 504
Multi-paragraph explicit segments: 292
Question then unlabeled-paragraph signals: 25
Question-signal segments in current human-review files: 17

CLASSIFICATION COUNTS
- MULTI_PARAGRAPH_EXPLICIT: 267
- QUESTION_THEN_UNLABELED_PARAGRAPHS: 25
- SINGLE_PARAGRAPH_EXPLICIT: 212

CLASSIFICATION BY SOURCE SPEAKER
- 'آقای تهمتن' | MULTI_PARAGRAPH_EXPLICIT: 4
- 'آقای تهمتن' | QUESTION_THEN_UNLABELED_PARAGRAPHS: 1
- 'آقای تهمتن' | SINGLE_PARAGRAPH_EXPLICIT: 1
- 'آقای داودی' | MULTI_PARAGRAPH_EXPLICIT: 6
- 'آقای داودی' | QUESTION_THEN_UNLABELED_PARAGRAPHS: 1
- 'آقای دکتر رودی' | QUESTION_THEN_UNLABELED_PARAGRAPHS: 1
- 'افسر موموندی' | MULTI_PARAGRAPH_EXPLICIT: 3
- 'حامد زحمتکش' | MULTI_PARAGRAPH_EXPLICIT: 1
- 'خانم تختی\u200cپور' | MULTI_PARAGRAPH_EXPLICIT: 3
- 'خانم دکتر جلالی' | MULTI_PARAGRAPH_EXPLICIT: 4
- 'خانم کاظمی' | MULTI_PARAGRAPH_EXPLICIT: 8
- 'خانم کاظمی' | SINGLE_PARAGRAPH_EXPLICIT: 1
- 'دکتر اخوان' | MULTI_PARAGRAPH_EXPLICIT: 3
- 'دکتر افرا

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ترمیم Evidence-Conservative مرز Speaker در فایل‌های Explicit چهلستون</h3>

<p>
Diagnostic قبلی نشان داد که Parser اولیه چهلستون در بعضی Sourceها یک Speaker Marker صریح را تا چند Paragraph بعدی ادامه داده است.
</p>

<p>
این رفتار از نظر حفظ متن مشکلی ایجاد نکرده بود، اما از نظر Attribution بیش از Evidence موجود در Source عمل می‌کرد.
</p>

<p>
برای مثال ممکن است Source به این شکل باشد:
</p>

<pre>
مجری: سؤال ...

پاسخ بدون Label ...

ادامه پاسخ بدون Label ...
</pre>

<p>
در چنین ساختاری فقط Paragraph اول Evidence صریح برای Speaker
<code>مجری</code>
دارد.
</p>

<p>
Paragraphهای بعدی ممکن است پاسخ مهمان، ادامه مجری یا Speaker دیگری باشند؛ بنابراین تا زمانی که Source Evidence صریح وجود نداشته باشد، هویت Speaker آنها نباید استنتاج شود.
</p>

<h4>قاعده جدید چهلستون</h4>

<p>
در ۳۵ فایل Explicit چهلستون:
</p>

<ul>
<li>هر Paragraph که با Marker معتبر <code>Speaker:</code> شروع می‌شود، یک Utterance با <code>EXPLICIT_SOURCE_MARKER</code> می‌سازد.</li>
<li>Label Speaker فقط به همان Paragraph متصل می‌شود.</li>
<li>هر Paragraph بعدی که Marker صریح ندارد، یک Utterance مستقل با <code>UNRESOLVED</code> می‌شود.</li>
<li>Speaker قبلی به Paragraph بدون Label منتقل نمی‌شود.</li>
<li>هیچ Speaker از متن سؤال، پاسخ، ترتیب مکالمه یا نام‌های داخل جمله استنتاج نمی‌شود.</li>
</ul>

<h4>فایل‌های Contextual چهلستون</h4>

<p>
سه فایل Contextual قبلی:
</p>

<ul>
<li>چهلستون ۱</li>
<li>چهلستون ۲</li>
<li>چهلستون ۳۸</li>
</ul>

<p>
در این Repair تغییر نمی‌کنند و همان Policy محافظه‌کارانه قبلی آنها حفظ می‌شود.
</p>

<h4>Raw Source</h4>

<p>
هیچ Source File تغییر نمی‌کند.
</p>

<p>
برای هر Utterance جدید:
</p>

<ul>
<li><code>source_text</code> مستقیماً از Disk گرفته می‌شود.</li>
<li><code>source_line_start</code> و <code>source_line_end</code> دقیق حفظ می‌شوند.</li>
<li>هیچ Persian normalization انجام نمی‌شود.</li>
<li>هیچ Spell یا ASR correction انجام نمی‌شود.</li>
</ul>

<h4>شناسه‌ها</h4>

<p>
بعد از Split محافظه‌کارانه، Utterance IDهای فایل‌های Explicit چهلستون دوباره به شکل Deterministic و بر اساس ترتیب Source ساخته می‌شوند.
</p>

<p>
این تغییر لازم است چون تعداد Utteranceهای چهلستون بعد از Repair تغییر خواهد کرد.
</p>

<h4>نسخه خروجی</h4>

<p>
خروجی قبلی Parser حذف نمی‌شود و برای Audit تاریخی باقی می‌ماند.
</p>

<p>
Stream اصلاح‌شده در متغیرهای جدید زیر ساخته می‌شود:
</p>

<ul>
<li><code>parsed_transcript_files_repaired_v1</code></li>
<li><code>all_parsed_utterances_repaired_v1</code></li>
<li><code>parsed_file_repaired_by_path</code></li>
</ul>

<p>
از این نقطه به بعد Final Pipeline باید از Stream اصلاح‌شده استفاده کند.
</p>

<p>
Merge، Quality Detection، Locator و Human Review قبلی برای Final Artifact دوباره از روی این Stream ساخته خواهند شد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هر ۱۸۳ Source File همچنان در Stream جدید حضور داشته باشد.</li>
<li>فقط ۳۵ فایل Explicit چهلستون Reparse شوند.</li>
<li>سه فایل Contextual چهلستون بدون تغییر باقی بمانند.</li>
<li>هیچ Paragraph بدون Label در فایل Explicit به Speaker قبلی نسبت داده نشود.</li>
<li>تمام Paragraphهای دارای Marker معتبر Speaker صریح خود را حفظ کنند.</li>
<li>تمام Source Lineهای غیرخالی Body دقیقاً یک بار حساب شوند، به‌جز Marker صریح بدون Payload در صورت وجود.</li>
<li>هیچ Source Line overlap وجود نداشته باشد.</li>
<li>هیچ Timestamp ساخته نشود.</li>
<li>هیچ Source File روی Disk تغییر نکند.</li>
</ul>

</div>

In [33]:
# Code Cell 33

chehelston_explicit_paths = sorted(
    record["relative_path"]
    for record in accepted_files
    if (
        record["program_id"]
        == chehelston_program_id
        and record["relative_path"]
        not in chehelston_contextual_path_set
    )
)

assert len(
    chehelston_explicit_paths
) == 35, (
    "Expected 35 explicit Chehelston files."
)


old_chehelston_explicit_utterances = [
    item
    for item in all_parsed_utterances_v1
    if (
        item["program_id"]
        == chehelston_program_id
        and item["relative_path"]
        in chehelston_explicit_paths
        and item["speaker_status"]
        == "EXPLICIT_SOURCE_MARKER"
    )
]


label_metadata_candidates = defaultdict(set)

for item in old_chehelston_explicit_utterances:
    label_metadata_candidates[
        item["speaker_raw"]
    ].add(
        (
            item[
                "speaker_canonical"
            ],
            item[
                "speaker_source_canonical"
            ],
        )
    )


chehelston_label_metadata = {}

for raw_label, metadata_values in (
    label_metadata_candidates.items()
):
    assert len(
        metadata_values
    ) == 1, (
        "Conflicting existing speaker metadata "
        f"for Chehelston label: {raw_label!r}"
    )

    chehelston_label_metadata[
        raw_label
    ] = next(
        iter(metadata_values)
    )


chehelston_explicit_line_pattern = re.compile(
    r"^\s*([^:\n]{1,160})\s*:\s*(.*)$"
)


def match_verified_chehelston_marker(
    raw_line,
):
    match = (
        chehelston_explicit_line_pattern.match(
            raw_line
        )
    )

    if not match:
        return None

    raw_label = (
        match.group(1).strip()
    )

    if (
        raw_label
        not in final_chehelston_speaker_markers
    ):
        return None

    tail = match.group(2)

    return {
        "raw_label": raw_label,
        "tail": tail,
    }


def build_chehelston_repaired_utterance(
    record,
    program_name,
    utterance_index,
    speaker_raw,
    speaker_canonical,
    speaker_source_canonical,
    speaker_status,
    boundary_type,
    text,
    source_text,
    source_line_start,
    source_line_end,
):
    assert text.strip(), (
        "Cannot build an empty repaired utterance."
    )

    return {
        "utterance_id": (
            f"{record['transcript_unit_id']}"
            f":utterance:"
            f"{utterance_index:05d}"
        ),
        "utterance_index": (
            utterance_index
        ),
        "program": (
            program_name
        ),
        "program_id": (
            record["program_id"]
        ),
        "physical_file_id": (
            record["physical_file_id"]
        ),
        "transcript_unit_id": (
            record["transcript_unit_id"]
        ),
        "relative_path": (
            record["relative_path"]
        ),
        "speaker_raw": (
            speaker_raw
        ),
        "speaker_canonical": (
            speaker_canonical
        ),
        "speaker_source_canonical": (
            speaker_source_canonical
        ),
        "speaker_status": (
            speaker_status
        ),
        "boundary_type": (
            boundary_type
        ),
        "text": text,
        "source_text": source_text,
        "source_line_start": (
            source_line_start
        ),
        "source_line_end": (
            source_line_end
        ),
        "start_time": None,
        "end_time": None,
    }


def parse_chehelston_explicit_conservative(
    record,
    existing_parsed_file,
):
    relative_path = record[
        "relative_path"
    ]

    file_path = (
        project_root
        / relative_path
    )

    lines = file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    body_start_index = (
        boundary_candidate_by_path[
            relative_path
        ]["candidate_line_number"]
        - 1
    )

    existing_utterances = (
        existing_parsed_file[
            "utterances"
        ]
    )

    assert existing_utterances, (
        "Existing parsed Chehelston file is empty."
    )

    program_name = (
        existing_utterances[0][
            "program"
        ]
    )

    repaired = []

    boundary_only_markers = []

    index = body_start_index

    while index < len(lines):
        raw_line = lines[index]

        if not raw_line.strip():
            index += 1
            continue

        marker = (
            match_verified_chehelston_marker(
                raw_line
            )
        )

        if marker is not None:
            paragraph_start = index
            paragraph_end = index + 1

            while paragraph_end < len(
                lines
            ):
                candidate_line = lines[
                    paragraph_end
                ]

                if not candidate_line.strip():
                    break

                if (
                    match_verified_chehelston_marker(
                        candidate_line
                    )
                    is not None
                ):
                    break

                paragraph_end += 1

            source_slice = lines[
                paragraph_start:
                paragraph_end
            ]

            source_text = "\n".join(
                source_slice
            )

            payload_lines = [
                marker["tail"],
                *source_slice[1:],
            ]

            payload_text = "\n".join(
                payload_lines
            ).strip()

            if not payload_text:
                boundary_only_markers.append({
                    "relative_path": (
                        relative_path
                    ),
                    "source_line": (
                        paragraph_start
                        + 1
                    ),
                    "raw_label": (
                        marker[
                            "raw_label"
                        ]
                    ),
                    "raw_line": (
                        raw_line
                    ),
                })

                index = (
                    paragraph_end
                )
                continue

            raw_label = marker[
                "raw_label"
            ]

            assert (
                raw_label
                in chehelston_label_metadata
            ), (
                "Verified Chehelston marker has no "
                "previously observed metadata: "
                f"{raw_label!r}"
            )

            (
                speaker_canonical,
                speaker_source_canonical,
            ) = (
                chehelston_label_metadata[
                    raw_label
                ]
            )

            repaired.append(
                build_chehelston_repaired_utterance(
                    record=record,
                    program_name=program_name,
                    utterance_index=(
                        len(repaired)
                        + 1
                    ),
                    speaker_raw=(
                        raw_label
                    ),
                    speaker_canonical=(
                        speaker_canonical
                    ),
                    speaker_source_canonical=(
                        speaker_source_canonical
                    ),
                    speaker_status=(
                        "EXPLICIT_SOURCE_MARKER"
                    ),
                    boundary_type=(
                        "COLON_SPEAKER_BOUNDARY"
                    ),
                    text=(
                        payload_text
                    ),
                    source_text=(
                        source_text
                    ),
                    source_line_start=(
                        paragraph_start
                        + 1
                    ),
                    source_line_end=(
                        paragraph_end
                    ),
                )
            )

            index = paragraph_end
            continue

        paragraph_start = index
        paragraph_end = index + 1

        while paragraph_end < len(
            lines
        ):
            candidate_line = lines[
                paragraph_end
            ]

            if not candidate_line.strip():
                break

            if (
                match_verified_chehelston_marker(
                    candidate_line
                )
                is not None
            ):
                break

            paragraph_end += 1

        source_slice = lines[
            paragraph_start:
            paragraph_end
        ]

        source_text = "\n".join(
            source_slice
        )

        unresolved_text = (
            source_text.strip()
        )

        assert unresolved_text, (
            "Unexpected empty unlabeled paragraph."
        )

        repaired.append(
            build_chehelston_repaired_utterance(
                record=record,
                program_name=program_name,
                utterance_index=(
                    len(repaired)
                    + 1
                ),
                speaker_raw=None,
                speaker_canonical=None,
                speaker_source_canonical=None,
                speaker_status=(
                    "UNRESOLVED"
                ),
                boundary_type=(
                    "CHEHELSTON_UNLABELED_"
                    "PARAGRAPH_UNRESOLVED"
                ),
                text=(
                    unresolved_text
                ),
                source_text=(
                    source_text
                ),
                source_line_start=(
                    paragraph_start
                    + 1
                ),
                source_line_end=(
                    paragraph_end
                ),
            )
        )

        index = paragraph_end

    return (
        repaired,
        boundary_only_markers,
    )


repaired_chehelston_files = {}

all_boundary_only_chehelston_markers = []

for relative_path in (
    chehelston_explicit_paths
):
    record = (
        accepted_record_by_path[
            relative_path
        ]
    )

    existing_parsed_file = (
        parsed_file_by_path[
            relative_path
        ]
    )

    (
        repaired_utterances,
        boundary_only_markers,
    ) = (
        parse_chehelston_explicit_conservative(
            record=record,
            existing_parsed_file=(
                existing_parsed_file
            ),
        )
    )

    repaired_chehelston_files[
        relative_path
    ] = {
        **{
            key: value
            for key, value
            in existing_parsed_file.items()
            if key
            not in {
                "utterances",
                "utterance_count",
            }
        },
        "utterance_count": len(
            repaired_utterances
        ),
        "utterances": (
            repaired_utterances
        ),
        "chehelston_boundary_policy": (
            "EXPLICIT_LABEL_ONLY_"
            "SAME_PARAGRAPH"
        ),
    }

    all_boundary_only_chehelston_markers.extend(
        boundary_only_markers
    )


parsed_transcript_files_repaired_v1 = []

for parsed_file in (
    parsed_transcript_files_v1
):
    relative_path = parsed_file[
        "relative_path"
    ]

    if (
        relative_path
        in repaired_chehelston_files
    ):
        parsed_transcript_files_repaired_v1.append(
            repaired_chehelston_files[
                relative_path
            ]
        )
    else:
        parsed_transcript_files_repaired_v1.append(
            parsed_file
        )


parsed_file_repaired_by_path = {
    item["relative_path"]: item
    for item
    in parsed_transcript_files_repaired_v1
}

all_parsed_utterances_repaired_v1 = [
    utterance
    for parsed_file
    in parsed_transcript_files_repaired_v1
    for utterance
    in parsed_file["utterances"]
]


old_chehelston_explicit_count = sum(
    1
    for item
    in all_parsed_utterances_v1
    if (
        item["program_id"]
        == chehelston_program_id
        and item["relative_path"]
        in chehelston_explicit_paths
    )
)

new_chehelston_explicit_file_utterances = [
    item
    for item
    in all_parsed_utterances_repaired_v1
    if (
        item["program_id"]
        == chehelston_program_id
        and item["relative_path"]
        in chehelston_explicit_paths
    )
]

new_chehelston_explicit_count = len(
    new_chehelston_explicit_file_utterances
)

new_chehelston_explicit_speaker_count = sum(
    item["speaker_status"]
    == "EXPLICIT_SOURCE_MARKER"
    for item
    in new_chehelston_explicit_file_utterances
)

new_chehelston_unresolved_count = sum(
    item["speaker_status"]
    == "UNRESOLVED"
    for item
    in new_chehelston_explicit_file_utterances
)


contextual_state_before = {
    path: json.dumps(
        parsed_file_by_path[
            path
        ]["utterances"],
        ensure_ascii=False,
        sort_keys=True,
    )
    for path
    in sorted(
        chehelston_contextual_path_set
    )
}

contextual_state_after = {
    path: json.dumps(
        parsed_file_repaired_by_path[
            path
        ]["utterances"],
        ensure_ascii=False,
        sort_keys=True,
    )
    for path
    in sorted(
        chehelston_contextual_path_set
    )
}


non_chehelston_change_failures = []

for relative_path, old_file in (
    parsed_file_by_path.items()
):
    if (
        relative_path
        in chehelston_explicit_paths
    ):
        continue

    new_file = (
        parsed_file_repaired_by_path[
            relative_path
        ]
    )

    if json.dumps(
        old_file["utterances"],
        ensure_ascii=False,
        sort_keys=True,
    ) != json.dumps(
        new_file["utterances"],
        ensure_ascii=False,
        sort_keys=True,
    ):
        non_chehelston_change_failures.append(
            relative_path
        )


coverage_failures = []
overlap_failures = []

total_explicit_body_nonempty = 0
total_explicit_covered_nonempty = 0
total_boundary_only_markers = len(
    all_boundary_only_chehelston_markers
)


for relative_path in (
    chehelston_explicit_paths
):
    file_path = (
        project_root
        / relative_path
    )

    lines = file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    body_start_index = (
        boundary_candidate_by_path[
            relative_path
        ]["candidate_line_number"]
        - 1
    )

    repaired_utterances = (
        repaired_chehelston_files[
            relative_path
        ]["utterances"]
    )

    covered_line_counts = Counter()

    for utterance in repaired_utterances:
        for line_number in range(
            utterance[
                "source_line_start"
            ],
            utterance[
                "source_line_end"
            ] + 1,
        ):
            covered_line_counts[
                line_number
            ] += 1

    overlapping = sorted(
        line_number
        for line_number, count
        in covered_line_counts.items()
        if count > 1
    )

    if overlapping:
        overlap_failures.append({
            "relative_path": (
                relative_path
            ),
            "lines": overlapping,
        })

    boundary_only_line_numbers = {
        item["source_line"]
        for item
        in all_boundary_only_chehelston_markers
        if item["relative_path"]
        == relative_path
    }

    missing_lines = []

    for source_index in range(
        body_start_index,
        len(lines),
    ):
        raw_line = lines[
            source_index
        ]

        if not raw_line.strip():
            continue

        line_number = (
            source_index + 1
        )

        total_explicit_body_nonempty += 1

        if (
            covered_line_counts[
                line_number
            ] == 1
        ):
            total_explicit_covered_nonempty += 1
            continue

        if (
            line_number
            in boundary_only_line_numbers
        ):
            continue

        missing_lines.append({
            "line_number": (
                line_number
            ),
            "raw_line": (
                raw_line
            ),
        })

    if missing_lines:
        coverage_failures.append({
            "relative_path": (
                relative_path
            ),
            "missing_lines": (
                missing_lines
            ),
        })


attributed_unlabeled_failures = []

for item in (
    new_chehelston_explicit_file_utterances
):
    if (
        item["boundary_type"]
        == (
            "CHEHELSTON_UNLABELED_"
            "PARAGRAPH_UNRESOLVED"
        )
    ):
        if any(
            value is not None
            for value in (
                item["speaker_raw"],
                item[
                    "speaker_canonical"
                ],
                item[
                    "speaker_source_canonical"
                ],
            )
        ):
            attributed_unlabeled_failures.append(
                item["utterance_id"]
            )


print(
    "Accepted files in repaired stream:",
    len(
        parsed_transcript_files_repaired_v1
    ),
)
print(
    "Total utterances before repair:",
    len(
        all_parsed_utterances_v1
    ),
)
print(
    "Total utterances after repair:",
    len(
        all_parsed_utterances_repaired_v1
    ),
)
print()

print("CHEHELSTON EXPLICIT REPAIR")
print("=" * 100)
print(
    "Explicit files reparsed:",
    len(
        chehelston_explicit_paths
    ),
)
print(
    "Old utterances in explicit files:",
    old_chehelston_explicit_count,
)
print(
    "New utterances in explicit files:",
    new_chehelston_explicit_count,
)
print(
    "Explicit speaker paragraphs:",
    new_chehelston_explicit_speaker_count,
)
print(
    "Unlabeled unresolved paragraphs:",
    new_chehelston_unresolved_count,
)
print(
    "Boundary-only speaker markers:",
    total_boundary_only_markers,
)
print()

print("SOURCE ACCOUNTING")
print("=" * 100)
print(
    "Non-empty body lines:",
    total_explicit_body_nonempty,
)
print(
    "Covered non-empty body lines:",
    total_explicit_covered_nonempty,
)
print(
    "Boundary-only marker lines:",
    total_boundary_only_markers,
)
print(
    "Coverage failures:",
    len(coverage_failures),
)
print(
    "Overlap failures:",
    len(overlap_failures),
)
print()

print("REPAIR SAFETY")
print("=" * 100)
print(
    "Non-target files changed:",
    len(
        non_chehelston_change_failures
    ),
)
print(
    "Contextual Chehelston files unchanged:",
    (
        contextual_state_before
        == contextual_state_after
    ),
)
print(
    "Attributed unlabeled paragraphs:",
    len(
        attributed_unlabeled_failures
    ),
)
print()

print(
    "SPEAKER STATUS COUNTS IN REPAIRED "
    "CHEHELSTON EXPLICIT FILES"
)
print("=" * 100)

repaired_status_counts = Counter(
    item["speaker_status"]
    for item
    in new_chehelston_explicit_file_utterances
)

for status, count in sorted(
    repaired_status_counts.items()
):
    print(
        f"- {status}: {count}"
    )


print()
print(
    "REPRESENTATIVE REPAIRED QUESTION / RESPONSE"
)
print("=" * 100)

shown = 0

for old_item in (
    question_then_unlabeled_items
):
    if shown >= 10:
        break

    relative_path = old_item[
        "relative_path"
    ]

    repaired_items = [
        item
        for item
        in repaired_chehelston_files[
            relative_path
        ]["utterances"]
        if (
            item[
                "source_line_start"
            ]
            >= old_item[
                "source_line_start"
            ]
            and item[
                "source_line_end"
            ]
            <= old_item[
                "source_line_end"
            ]
        )
    ]

    if len(repaired_items) < 2:
        continue

    shown += 1

    print()
    print(
        "File:",
        relative_path,
    )
    print(
        "Old source range:",
        (
            old_item[
                "source_line_start"
            ],
            old_item[
                "source_line_end"
            ],
        ),
    )

    for repaired_item in (
        repaired_items[:5]
    ):
        print(
            " ",
            repaired_item[
                "utterance_index"
            ],
            "|",
            repaired_item[
                "speaker_status"
            ],
            "| speaker:",
            repaired_item[
                "speaker_raw"
            ],
            "| lines:",
            (
                repaired_item[
                    "source_line_start"
                ],
                repaired_item[
                    "source_line_end"
                ],
            ),
            "|",
            preview_text(
                repaired_item[
                    "text"
                ],
                limit=260,
            ),
        )

    print("-" * 100)


assert len(
    parsed_transcript_files_repaired_v1
) == 183, (
    "Repaired stream lost accepted files."
)

assert len(
    parsed_file_repaired_by_path
) == 183, (
    "Repaired file paths are not unique."
)

assert not (
    non_chehelston_change_failures
), (
    "A non-target file changed during Chehelston repair."
)

assert (
    contextual_state_before
    == contextual_state_after
), (
    "Contextual Chehelston files changed during explicit repair."
)

assert not coverage_failures, (
    "Repaired Chehelston explicit Source coverage failed."
)

assert not overlap_failures, (
    "Repaired Chehelston explicit Source ranges overlap."
)

assert not (
    attributed_unlabeled_failures
), (
    "An unlabeled Chehelston paragraph retained speaker attribution."
)

assert all(
    item["start_time"] is None
    and item["end_time"] is None
    for item
    in all_parsed_utterances_repaired_v1
), (
    "Unexpected timestamp appeared during Chehelston repair."
)

assert all(
    item["speaker_status"]
    != "UNRESOLVED"
    or (
        item["speaker_raw"] is None
        and item[
            "speaker_canonical"
        ] is None
        and item[
            "speaker_source_canonical"
        ] is None
    )
    for item
    in new_chehelston_explicit_file_utterances
), (
    "Unresolved Chehelston paragraph contains speaker attribution."
)

print()
print(
    "CHEHELSTON EVIDENCE-CONSERVATIVE "
    "SPEAKER-BOUNDARY REPAIR: PASS"
)

Accepted files in repaired stream: 183
Total utterances before repair: 15131
Total utterances after repair: 16697

CHEHELSTON EXPLICIT REPAIR
Explicit files reparsed: 35
Old utterances in explicit files: 539
New utterances in explicit files: 2105
Explicit speaker paragraphs: 504
Unlabeled unresolved paragraphs: 1601
Boundary-only speaker markers: 0

SOURCE ACCOUNTING
Non-empty body lines: 2105
Covered non-empty body lines: 2105
Boundary-only marker lines: 0
Coverage failures: 0
Overlap failures: 0

REPAIR SAFETY
Non-target files changed: 0
Contextual Chehelston files unchanged: True
Attributed unlabeled paragraphs: 0

SPEAKER STATUS COUNTS IN REPAIRED CHEHELSTON EXPLICIT FILES
- EXPLICIT_SOURCE_MARKER: 504
- UNRESOLVED: 1601

REPRESENTATIVE REPAIRED QUESTION / RESPONSE

File: data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 10.md
Old source range: (125, 135)
  53 | EXPLICIT_SOURCE_MARKER | speaker: آقای داودی | lines: (125, 125) | اول باید ریشه مسئله را دید. نیاز و 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازسازی Post-Merge Stream بر پایه Parser اصلاح‌شده</h3>

<p>
پس از Repair چهلستون، تمام خروجی‌های پایین‌دستی باید از Stream اصلاح‌شده ساخته شوند.
</p>

<p>
Merge Plan قبلی فقط روی سه Program دارای Policy
<code>HUMAN_CONFIRMED_SOLO</code>
اعمال شده بود:
</p>

<ul>
<li>در جستجوی خوشبختی</li>
<li>فناپ</li>
<li>تربیت مربی</li>
</ul>

<p>
بنابراین Repair چهلستون نباید هیچ Merge Candidate یا Merge Chain قبلی را تغییر دهد.
</p>

<p>
در این Cell ابتدا این موضوع به‌صورت صریح بررسی می‌شود.
</p>

<h4>قاعده بازسازی</h4>

<p>
Stream جدید از
<code>all_parsed_utterances_repaired_v1</code>
شروع می‌شود.
</p>

<p>
برای فایل‌های Solo قبلی، همان Merge Chainهای تأییدشده دوباره اجرا می‌شوند.
</p>

<p>
برای تمام فایل‌های چهلستون، از جمله ۳۵ فایل Repair‌شده، هیچ Merge انجام نمی‌شود.
</p>

<h4>چرا Merge Discovery را دوباره اجرا نمی‌کنیم؟</h4>

<p>
Repair فقط ساختار Speaker Attribution چهلستون را تغییر داده است.
</p>

<p>
هیچ‌یک از فایل‌هایی که قبلاً وارد Auto-merge شده بودند تغییر نکرده‌اند.
</p>

<p>
بنابراین اگر ثابت شود هیچ Merge Chain به چهلستون تعلق ندارد و تمام Source Fileهای Merge‌شده نسبت به Stream قبلی Byte-for-byte در Parsed Representation یکسان مانده‌اند، همان Plan قبلی قابل استفاده است.
</p>

<h4>Lineage</h4>

<p>
همان قواعد قبلی حفظ می‌شوند:
</p>

<ul>
<li>هر Utterance Merge‌شده فهرست کامل <code>merged_from_utterance_ids</code> را نگه می‌دارد.</li>
<li>Source Range از اولین عضو تا آخرین عضو Chain حفظ می‌شود.</li>
<li><code>source_text</code> مستقیماً از Disk بازسازی می‌شود.</li>
<li>هیچ Timestamp جدیدی ساخته نمی‌شود.</li>
</ul>

<h4>ممیزی مستقل</h4>

<p>
پس از بازسازی، موارد زیر دوباره بررسی می‌شوند:
</p>

<ul>
<li>تمام Pre-Merge Utteranceهای Stream اصلاح‌شده دقیقاً یک بار مصرف شده باشند.</li>
<li>کاهش تعداد Utteranceها دقیقاً برابر تعداد Merge Edgeهای قبلی باشد.</li>
<li>هیچ Merge Chain مربوط به چهلستون نباشد.</li>
<li>تمام Source Lineهای غیرخالی Corpus حساب شده باشند.</li>
<li>هیچ Source Line overlap وجود نداشته باشد.</li>
<li>هیچ Speaker unresolved وارد Merge نشده باشد.</li>
</ul>

<h4>نسخه خروجی</h4>

<p>
خروجی جدید در متغیرهای زیر ساخته می‌شود:
</p>

<ul>
<li><code>parsed_transcript_files_merged_repaired_v1</code></li>
<li><code>all_merged_utterances_repaired_v1</code></li>
</ul>

<p>
از این نقطه به بعد Quality Detection، Locator و Human Review قبلی دیگر Final محسوب نمی‌شوند و باید بر اساس این Stream جدید بازسازی شوند.
</p>

<h4>هنوز انجام نمی‌شوند</h4>

<ul>
<li>Spell correction</li>
<li>ASR correction</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>هیچ Merge Chain متعلق به چهلستون نباشد.</li>
<li>تمام فایل‌های درگیر Merge نسبت به Parser قبل از Repair بدون تغییر باشند.</li>
<li>تمام ۳۵۰ Merge Chain دوباره دقیقاً یک بار اجرا شوند.</li>
<li>کاهش تعداد Utteranceها برابر ۴۷۱ Merge Edge باشد.</li>
<li>تمام Pre-Merge Utteranceهای Stream اصلاح‌شده دقیقاً یک بار در Lineage جدید حضور داشته باشند.</li>
<li>Source Coverage کل Corpus کامل باشد.</li>
<li>هیچ Source overlap، Speaker-safety failure یا Timestamp failure وجود نداشته باشد.</li>
</ul>

</div>

In [34]:
# Code Cell 34

merge_chain_paths = {
    chain["relative_path"]
    for chain in merge_chains_v1
}

chehelston_merge_paths = sorted(
    merge_chain_paths
    & set(
        path
        for path in accepted_record_by_path
        if accepted_record_by_path[
            path
        ]["program_id"]
        == chehelston_program_id
    )
)

merge_source_change_failures = []

for relative_path in sorted(
    merge_chain_paths
):
    old_file = parsed_file_by_path[
        relative_path
    ]

    repaired_file = (
        parsed_file_repaired_by_path[
            relative_path
        ]
    )

    old_state = json.dumps(
        old_file["utterances"],
        ensure_ascii=False,
        sort_keys=True,
    )

    repaired_state = json.dumps(
        repaired_file["utterances"],
        ensure_ascii=False,
        sort_keys=True,
    )

    if old_state != repaired_state:
        merge_source_change_failures.append(
            relative_path
        )


assert not chehelston_merge_paths, (
    "A previous merge chain unexpectedly belongs "
    "to Chehelston."
)

assert not merge_source_change_failures, (
    "At least one file used by the existing merge plan "
    "changed during Chehelston repair."
)


parsed_transcript_files_merged_repaired_v1 = []
all_merged_utterances_repaired_v1 = []

applied_repaired_chain_count = 0
applied_repaired_edge_count = 0


for parsed_file in (
    parsed_transcript_files_repaired_v1
):
    relative_path = parsed_file[
        "relative_path"
    ]

    record = accepted_record_by_path[
        relative_path
    ]

    file_path = (
        project_root
        / relative_path
    )

    lines = file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    original_utterances = (
        parsed_file["utterances"]
    )

    utterance_by_original_index = {
        item["utterance_index"]: item
        for item in original_utterances
    }

    merged_file_utterances = []

    pointer = 0

    while pointer < len(
        original_utterances
    ):
        current = original_utterances[
            pointer
        ]

        original_index = current[
            "utterance_index"
        ]

        chain = (
            merge_chain_by_file_and_start.get(
                (
                    relative_path,
                    original_index,
                )
            )
        )

        if chain is None:
            assert (
                original_index
                not in merge_member_indices_by_file[
                    relative_path
                ]
            ), (
                "Encountered a non-start merge member "
                "without consuming its chain."
            )

            merged_file_utterances.append(
                build_unchanged_working_utterance(
                    current
                )
            )

            pointer += 1
            continue

        chain_indices = chain[
            "utterance_indices"
        ]

        members = [
            utterance_by_original_index[
                index
            ]
            for index
            in chain_indices
        ]

        expected_members = (
            original_utterances[
                pointer:
                pointer + len(members)
            ]
        )

        assert [
            item["utterance_index"]
            for item
            in expected_members
        ] == chain_indices, (
            "Merge chain no longer matches "
            "the repaired stream order."
        )

        merged_file_utterances.append(
            build_merged_chain_utterance(
                record=record,
                lines=lines,
                chain=chain,
                members=members,
            )
        )

        applied_repaired_chain_count += 1
        applied_repaired_edge_count += (
            chain["edge_count"]
        )

        pointer += len(
            members
        )

    for new_index, item in enumerate(
        merged_file_utterances,
        start=1,
    ):
        item[
            "utterance_index"
        ] = new_index

    parsed_transcript_files_merged_repaired_v1.append({
        **{
            key: value
            for key, value
            in parsed_file.items()
            if key
            not in {
                "utterances",
                "utterance_count",
            }
        },
        "premerge_utterance_count": (
            len(original_utterances)
        ),
        "utterance_count": (
            len(
                merged_file_utterances
            )
        ),
        "merge_chain_count": sum(
            item["merge_status"]
            == "MERGED"
            for item
            in merged_file_utterances
        ),
        "utterances": (
            merged_file_utterances
        ),
    })

    all_merged_utterances_repaired_v1.extend(
        merged_file_utterances
    )


expected_repaired_postmerge_count = (
    len(
        all_parsed_utterances_repaired_v1
    )
    - sum(
        chain["edge_count"]
        for chain
        in merge_chains_v1
    )
)


premerge_repaired_by_id = {
    item["utterance_id"]: item
    for item
    in all_parsed_utterances_repaired_v1
}

repaired_lineage_consumption = Counter()

for item in (
    all_merged_utterances_repaired_v1
):
    for source_id in item[
        "merged_from_utterance_ids"
    ]:
        repaired_lineage_consumption[
            source_id
        ] += 1


missing_repaired_premerge_ids = [
    source_id
    for source_id
    in premerge_repaired_by_id
    if repaired_lineage_consumption[
        source_id
    ] == 0
]

multiply_consumed_repaired_ids = [
    source_id
    for source_id, count
    in repaired_lineage_consumption.items()
    if count > 1
]

unknown_repaired_lineage_ids = [
    source_id
    for source_id
    in repaired_lineage_consumption
    if source_id
    not in premerge_repaired_by_id
]


repaired_merged_file_by_path = {
    item["relative_path"]: item
    for item
    in parsed_transcript_files_merged_repaired_v1
}


total_body_nonempty_repaired = 0
total_covered_nonempty_repaired = 0
total_allowed_skips_repaired = 0

repaired_skip_counts = Counter()

repaired_coverage_failures = []
repaired_overlap_failures = []
repaired_source_text_failures = []


for parsed_file in (
    parsed_transcript_files_merged_repaired_v1
):
    relative_path = parsed_file[
        "relative_path"
    ]

    record = accepted_record_by_path[
        relative_path
    ]

    file_path = (
        project_root
        / relative_path
    )

    lines = file_path.read_text(
        encoding="utf-8"
    ).splitlines()

    body_start_index = (
        boundary_candidate_by_path[
            relative_path
        ]["candidate_line_number"]
        - 1
    )

    coverage_count = Counter()

    for utterance in parsed_file[
        "utterances"
    ]:
        start_line = utterance[
            "source_line_start"
        ]

        end_line = utterance[
            "source_line_end"
        ]

        expected_source_text = "\n".join(
            lines[
                start_line - 1:
                end_line
            ]
        )

        if (
            utterance["source_text"]
            != expected_source_text
        ):
            repaired_source_text_failures.append({
                "relative_path": (
                    relative_path
                ),
                "utterance_id": (
                    utterance[
                        "utterance_id"
                    ]
                ),
            })

        for line_number in range(
            start_line,
            end_line + 1,
        ):
            coverage_count[
                line_number
            ] += 1

    overlapping_lines = sorted(
        line_number
        for line_number, count
        in coverage_count.items()
        if count > 1
    )

    if overlapping_lines:
        repaired_overlap_failures.append({
            "relative_path": (
                relative_path
            ),
            "lines": overlapping_lines,
        })

    missing_lines = []

    for source_index in range(
        body_start_index,
        len(lines),
    ):
        raw_line = lines[
            source_index
        ]

        if not raw_line.strip():
            continue

        line_number = (
            source_index + 1
        )

        total_body_nonempty_repaired += 1

        if coverage_count[
            line_number
        ] > 0:
            total_covered_nonempty_repaired += 1
            continue

        skip_type = None

        if is_solo_structural_marker(
            record,
            raw_line,
        ):
            skip_type = (
                "SOLO_MANIFEST_MARKER"
            )

        elif is_empty_verified_bold_boundary(
            record,
            lines,
            source_index,
        ):
            skip_type = (
                "EMPTY_VERIFIED_BOLD_BOUNDARY"
            )

        elif is_empty_verified_chehelston_boundary(
            record,
            lines,
            source_index,
        ):
            skip_type = (
                "EMPTY_VERIFIED_CHEHELSTON_BOUNDARY"
            )

        if skip_type is not None:
            repaired_skip_counts[
                skip_type
            ] += 1

            total_allowed_skips_repaired += 1
            continue

        missing_lines.append({
            "line_number": (
                line_number
            ),
            "raw_line": (
                raw_line
            ),
        })

    if missing_lines:
        repaired_coverage_failures.append({
            "relative_path": (
                relative_path
            ),
            "missing_lines": (
                missing_lines
            ),
        })


speaker_safety_failures_repaired = [
    item["utterance_id"]
    for item
    in all_merged_utterances_repaired_v1
    if (
        item["merge_status"]
        == "MERGED"
        and (
            item["speaker_status"]
            in {
                "UNRESOLVED",
                "NARRATIVE_CUE",
            }
            or item[
                "speaker_canonical"
            ] is None
        )
    )
]

timestamp_failures_repaired = [
    item["utterance_id"]
    for item
    in all_merged_utterances_repaired_v1
    if (
        item["start_time"]
        is not None
        or item["end_time"]
        is not None
    )
]


print("MERGE-PLAN REUSE SAFETY")
print("=" * 100)
print(
    "Merge-chain files:",
    len(merge_chain_paths),
)
print(
    "Chehelston merge-chain files:",
    len(
        chehelston_merge_paths
    ),
)
print(
    "Changed files used by merge plan:",
    len(
        merge_source_change_failures
    ),
)
print()

print("REPAIRED POST-MERGE ACCOUNTING")
print("=" * 100)
print(
    "Repaired pre-merge utterances:",
    len(
        all_parsed_utterances_repaired_v1
    ),
)
print(
    "Planned merge chains:",
    len(merge_chains_v1),
)
print(
    "Applied merge chains:",
    applied_repaired_chain_count,
)
print(
    "Planned merge edges:",
    sum(
        chain["edge_count"]
        for chain
        in merge_chains_v1
    ),
)
print(
    "Applied merge edges:",
    applied_repaired_edge_count,
)
print(
    "Repaired post-merge utterances:",
    len(
        all_merged_utterances_repaired_v1
    ),
)
print(
    "Expected repaired post-merge utterances:",
    expected_repaired_postmerge_count,
)
print(
    "Net reduction:",
    (
        len(
            all_parsed_utterances_repaired_v1
        )
        - len(
            all_merged_utterances_repaired_v1
        )
    ),
)
print()

print("REPAIRED LINEAGE")
print("=" * 100)
print(
    "Missing pre-merge IDs:",
    len(
        missing_repaired_premerge_ids
    ),
)
print(
    "Multiply consumed IDs:",
    len(
        multiply_consumed_repaired_ids
    ),
)
print(
    "Unknown lineage IDs:",
    len(
        unknown_repaired_lineage_ids
    ),
)
print()

print("REPAIRED SOURCE ACCOUNTING")
print("=" * 100)
print(
    "Body non-empty source lines:",
    total_body_nonempty_repaired,
)
print(
    "Covered non-empty source lines:",
    total_covered_nonempty_repaired,
)
print(
    "Allowed structural skips:",
    total_allowed_skips_repaired,
)

for skip_type, count in sorted(
    repaired_skip_counts.items()
):
    print(
        f"- {skip_type}: {count}"
    )

repaired_accounted_total = (
    total_covered_nonempty_repaired
    + total_allowed_skips_repaired
)

print(
    "Covered + allowed skips:",
    repaired_accounted_total,
)
print(
    "Accounting balanced:",
    (
        repaired_accounted_total
        == total_body_nonempty_repaired
    ),
)
print()

print("REPAIRED FAILURE COUNTS")
print("=" * 100)
print(
    "Source-text failures:",
    len(
        repaired_source_text_failures
    ),
)
print(
    "Source-overlap failures:",
    len(
        repaired_overlap_failures
    ),
)
print(
    "Coverage failures:",
    len(
        repaired_coverage_failures
    ),
)
print(
    "Speaker-safety failures:",
    len(
        speaker_safety_failures_repaired
    ),
)
print(
    "Timestamp failures:",
    len(
        timestamp_failures_repaired
    ),
)


assert applied_repaired_chain_count == len(
    merge_chains_v1
), (
    "Not all merge chains were applied "
    "to the repaired stream."
)

assert applied_repaired_edge_count == sum(
    chain["edge_count"]
    for chain
    in merge_chains_v1
), (
    "Merge-edge count changed after repair."
)

assert len(
    all_merged_utterances_repaired_v1
) == expected_repaired_postmerge_count, (
    "Repaired post-merge utterance count is invalid."
)

assert not (
    missing_repaired_premerge_ids
), (
    "A repaired pre-merge utterance disappeared."
)

assert not (
    multiply_consumed_repaired_ids
), (
    "A repaired pre-merge utterance was consumed more than once."
)

assert not (
    unknown_repaired_lineage_ids
), (
    "Repaired merge lineage contains an unknown source utterance."
)

assert not (
    repaired_source_text_failures
), (
    "Repaired post-merge source_text mismatch detected."
)

assert not (
    repaired_overlap_failures
), (
    "Repaired post-merge Source overlap detected."
)

assert not (
    repaired_coverage_failures
), (
    "Repaired post-merge Source coverage failed."
)

assert (
    repaired_accounted_total
    == total_body_nonempty_repaired
), (
    "Repaired Source accounting does not balance."
)

assert not (
    speaker_safety_failures_repaired
), (
    "An unresolved or unattributed segment was merged."
)

assert not (
    timestamp_failures_repaired
), (
    "Unexpected timestamp appeared in repaired stream."
)

print()
print(
    "REPAIRED POST-MERGE STREAM "
    "REBUILD AND AUDIT: PASS"
)

MERGE-PLAN REUSE SAFETY
Merge-chain files: 37
Chehelston merge-chain files: 0
Changed files used by merge plan: 0

REPAIRED POST-MERGE ACCOUNTING
Repaired pre-merge utterances: 16697
Planned merge chains: 350
Applied merge chains: 350
Planned merge edges: 471
Applied merge edges: 471
Repaired post-merge utterances: 16226
Expected repaired post-merge utterances: 16226
Net reduction: 471

REPAIRED LINEAGE
Missing pre-merge IDs: 0
Multiply consumed IDs: 0
Unknown lineage IDs: 0

REPAIRED SOURCE ACCOUNTING
Body non-empty source lines: 28894
Covered non-empty source lines: 21655
Allowed structural skips: 7239
- EMPTY_VERIFIED_BOLD_BOUNDARY: 3
- SOLO_MANIFEST_MARKER: 7236
Covered + allowed skips: 28894
Accounting balanced: True

REPAIRED FAILURE COUNTS
Source-text failures: 0
Source-overlap failures: 0
Coverage failures: 0
Speaker-safety failures: 0
Timestamp failures: 0

REPAIRED POST-MERGE STREAM REBUILD AND AUDIT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازسازی Quality Detection روی Stream اصلاح‌شده</h3>

<p>
Quality Detection قبلی روی Stream قبل از Repair چهلستون اجرا شده بود و دیگر برای Final Output معتبر نیست.
</p>

<p>
در این Cell همان Detectorهای محافظه‌کارانه دوباره روی
<code>all_merged_utterances_repaired_v1</code>
اجرا می‌شوند.
</p>

<p>
هیچ Detector جدید زبانی اضافه نمی‌شود و هیچ متن یا Candidate قبلی به‌صورت خودکار اصلاح نمی‌شود.
</p>

<h4>Detectorهای فعال</h4>

<ul>
<li><code>REPEATED_TOKEN_RUN</code></li>
<li><code>REPEATED_NGRAM_RUN</code></li>
<li><code>LONG_LETTER_OR_DIGIT_RUN</code></li>
<li><code>LONG_PUNCTUATION_RUN</code></li>
<li><code>BRACKETED_TRANSCRIPTION_MARKER</code></li>
<li><code>UNICODE_REPLACEMENT_CHARACTER</code></li>
<li><code>NULL_CHARACTER</code></li>
<li><code>UNEXPECTED_CONTROL_CHARACTER</code></li>
<li><code>EXACT_ADJACENT_DUPLICATE_UTTERANCE</code></li>
</ul>

<h4>اثر Repair چهلستون</h4>

<p>
چون Paragraphهای چهلستون اکنون Segmentهای مستقل شده‌اند، تعداد و محل بعضی Quality Candidateها می‌تواند نسبت به Scan قبلی تغییر کند.
</p>

<p>
این تغییر طبیعی است و نباید با Regression اشتباه گرفته شود.
</p>

<p>
برای مثال یک Repeated N-gram که قبلاً داخل یک Segment بسیار بلند دیده می‌شد، ممکن است بعد از Split دیگر در همان Segment وجود نداشته باشد.
</p>

<h4>اصل محافظه‌کاری</h4>

<p>
تمام Candidateها فقط برای Review هستند.
</p>

<p>
حتی Exact Duplicate نیز در NB06 حذف نمی‌شود.
</p>

<p>
Bracketed Annotation نیز به‌صورت پیش‌فرض Noise محسوب نمی‌شود.
</p>

<h4>خروجی جدید</h4>

<p>
خروجی معتبر جدید در متغیر:
</p>

<p>
<code>transcript_quality_candidates_repaired_v1</code>
</p>

<p>
نگهداری می‌شود.
</p>

<p>
Quality Candidateهای قبلی فقط سابقه Diagnostic هستند و از این نقطه به بعد برای Artifact نهایی استفاده نمی‌شوند.
</p>

<h4>هنوز انجام نمی‌شوند</h4>

<ul>
<li>اصلاح ASR</li>
<li>Spell correction</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>تمام <code>16226</code> Utterance اصلاح‌شده بررسی شوند.</li>
<li>هیچ Utterance تغییر نکند.</li>
<li>تمام Candidate IDها یکتا باشند.</li>
<li>Exact Duplicateها فقط Flag شوند.</li>
<li>Technical Noise جداگانه گزارش شود.</li>
<li>هیچ Candidate به Utterance خارج از Stream اصلاح‌شده اشاره نکند.</li>
</ul>

</div>

In [35]:
# Code Cell 35

repaired_postmerge_state_before_quality_scan = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)

repaired_quality_candidates = []


def add_repaired_quality_candidate(
    candidate_type,
    item,
    evidence,
    review_status="REVIEW_REQUIRED",
):
    repaired_quality_candidates.append({
        "candidate_type": (
            candidate_type
        ),
        "review_status": (
            review_status
        ),
        "program": item[
            "program"
        ],
        "program_id": item[
            "program_id"
        ],
        "relative_path": item[
            "relative_path"
        ],
        "utterance_ids": [
            item["utterance_id"]
        ],
        "source_line_start": item[
            "source_line_start"
        ],
        "source_line_end": item[
            "source_line_end"
        ],
        "speaker_raw": item[
            "speaker_raw"
        ],
        "speaker_canonical": item[
            "speaker_canonical"
        ],
        "speaker_status": item[
            "speaker_status"
        ],
        "raw_text": item[
            "text"
        ],
        "evidence": evidence,
    })


for item in (
    all_merged_utterances_repaired_v1
):
    text = item["text"]

    tokens = raw_token_sequence(
        text
    )

    token_runs = repeated_token_runs(
        tokens
    )

    if token_runs:
        add_repaired_quality_candidate(
            candidate_type=(
                "REPEATED_TOKEN_RUN"
            ),
            item=item,
            evidence={
                "runs": token_runs[:20],
                "total_runs": len(
                    token_runs
                ),
                "comparison": (
                    "EXACT_UNICODE_TOKEN"
                ),
            },
        )

    ngram_runs = repeated_ngram_runs(
        tokens
    )

    if ngram_runs:
        add_repaired_quality_candidate(
            candidate_type=(
                "REPEATED_NGRAM_RUN"
            ),
            item=item,
            evidence={
                "runs": ngram_runs[:20],
                "total_runs": len(
                    ngram_runs
                ),
                "comparison": (
                    "EXACT_UNICODE_NGRAM"
                ),
            },
        )

    letter_digit_runs = [
        {
            "matched_text": match.group(0),
            "character": match.group(1),
            "start_index": match.start(),
            "end_index": match.end(),
        }
        for match
        in letter_or_digit_run_pattern.finditer(
            text
        )
    ]

    if letter_digit_runs:
        add_repaired_quality_candidate(
            candidate_type=(
                "LONG_LETTER_OR_DIGIT_RUN"
            ),
            item=item,
            evidence={
                "runs": (
                    letter_digit_runs[:20]
                ),
                "total_runs": len(
                    letter_digit_runs
                ),
            },
        )

    punctuation_runs = [
        {
            "matched_text": match.group(0),
            "character": match.group(1),
            "start_index": match.start(),
            "end_index": match.end(),
        }
        for match
        in punctuation_run_pattern.finditer(
            text
        )
    ]

    if punctuation_runs:
        add_repaired_quality_candidate(
            candidate_type=(
                "LONG_PUNCTUATION_RUN"
            ),
            item=item,
            evidence={
                "runs": (
                    punctuation_runs[:20]
                ),
                "total_runs": len(
                    punctuation_runs
                ),
            },
        )

    bracket_markers = (
        bracket_marker_pattern.findall(
            text
        )
    )

    if bracket_markers:
        add_repaired_quality_candidate(
            candidate_type=(
                "BRACKETED_TRANSCRIPTION_MARKER"
            ),
            item=item,
            evidence={
                "markers": (
                    bracket_markers[:50]
                ),
                "total_markers": len(
                    bracket_markers
                ),
            },
            review_status=(
                "STRUCTURAL_MARKER_REVIEW"
            ),
        )

    if "\ufffd" in text:
        add_repaired_quality_candidate(
            candidate_type=(
                "UNICODE_REPLACEMENT_CHARACTER"
            ),
            item=item,
            evidence={
                "count": text.count(
                    "\ufffd"
                ),
            },
            review_status=(
                "TECHNICAL_NOISE"
            ),
        )

    if "\x00" in text:
        add_repaired_quality_candidate(
            candidate_type=(
                "NULL_CHARACTER"
            ),
            item=item,
            evidence={
                "count": text.count(
                    "\x00"
                ),
            },
            review_status=(
                "TECHNICAL_NOISE"
            ),
        )

    control_hits = (
        unexpected_control_characters(
            text
        )
    )

    if control_hits:
        add_repaired_quality_candidate(
            candidate_type=(
                "UNEXPECTED_CONTROL_CHARACTER"
            ),
            item=item,
            evidence={
                "characters": (
                    control_hits[:20]
                ),
                "total_characters": len(
                    control_hits
                ),
            },
            review_status=(
                "TECHNICAL_NOISE"
            ),
        )


repaired_exact_adjacent_duplicates = []

for parsed_file in (
    parsed_transcript_files_merged_repaired_v1
):
    utterances = parsed_file[
        "utterances"
    ]

    for index in range(
        len(utterances) - 1
    ):
        left = utterances[
            index
        ]

        right = utterances[
            index + 1
        ]

        if (
            left["text"]
            != right["text"]
        ):
            continue

        pair = {
            "candidate_type": (
                "EXACT_ADJACENT_DUPLICATE_UTTERANCE"
            ),
            "review_status": (
                "REVIEW_REQUIRED"
            ),
            "program": left[
                "program"
            ],
            "program_id": left[
                "program_id"
            ],
            "relative_path": left[
                "relative_path"
            ],
            "utterance_ids": [
                left[
                    "utterance_id"
                ],
                right[
                    "utterance_id"
                ],
            ],
            "source_line_start": (
                left[
                    "source_line_start"
                ]
            ),
            "source_line_end": (
                right[
                    "source_line_end"
                ]
            ),
            "speaker_raw": (
                left["speaker_raw"]
                if (
                    left["speaker_raw"]
                    == right["speaker_raw"]
                )
                else None
            ),
            "speaker_canonical": (
                left[
                    "speaker_canonical"
                ]
                if (
                    left[
                        "speaker_canonical"
                    ]
                    == right[
                        "speaker_canonical"
                    ]
                )
                else None
            ),
            "speaker_status": (
                left[
                    "speaker_status"
                ]
                if (
                    left[
                        "speaker_status"
                    ]
                    == right[
                        "speaker_status"
                    ]
                )
                else "MIXED"
            ),
            "raw_text": left[
                "text"
            ],
            "evidence": {
                "comparison": (
                    "EXACT_RAW_TEXT"
                ),
                "left_utterance_index": (
                    left[
                        "utterance_index"
                    ]
                ),
                "right_utterance_index": (
                    right[
                        "utterance_index"
                    ]
                ),
            },
        }

        repaired_exact_adjacent_duplicates.append(
            pair
        )

        repaired_quality_candidates.append(
            pair
        )


transcript_quality_candidates_repaired_v1 = []

for candidate_index, candidate in enumerate(
    repaired_quality_candidates,
    start=1,
):
    transcript_quality_candidates_repaired_v1.append({
        "quality_candidate_id": (
            f"transcript-quality-repaired:"
            f"{candidate_index:06d}"
        ),
        **candidate,
    })


repaired_utterance_ids = {
    item["utterance_id"]
    for item
    in all_merged_utterances_repaired_v1
}

unknown_candidate_utterance_ids = []

for candidate in (
    transcript_quality_candidates_repaired_v1
):
    for utterance_id in candidate[
        "utterance_ids"
    ]:
        if (
            utterance_id
            not in repaired_utterance_ids
        ):
            unknown_candidate_utterance_ids.append({
                "quality_candidate_id": (
                    candidate[
                        "quality_candidate_id"
                    ]
                ),
                "utterance_id": (
                    utterance_id
                ),
            })


print(
    "Repaired post-merge utterances inspected:",
    len(
        all_merged_utterances_repaired_v1
    ),
)
print(
    "Repaired quality candidates:",
    len(
        transcript_quality_candidates_repaired_v1
    ),
)
print()


print("REPAIRED QUALITY CANDIDATE COUNTS")
print("=" * 100)

repaired_candidate_type_counts = Counter(
    item["candidate_type"]
    for item
    in transcript_quality_candidates_repaired_v1
)

if repaired_candidate_type_counts:
    for candidate_type, count in sorted(
        repaired_candidate_type_counts.items()
    ):
        print(
            f"- {candidate_type}: {count}"
        )
else:
    print("- NONE")


print()
print("REPAIRED REVIEW STATUS COUNTS")
print("=" * 100)

repaired_review_status_counts = Counter(
    item["review_status"]
    for item
    in transcript_quality_candidates_repaired_v1
)

if repaired_review_status_counts:
    for status, count in sorted(
        repaired_review_status_counts.items()
    ):
        print(
            f"- {status}: {count}"
        )
else:
    print("- NONE")


print()
print("QUALITY CANDIDATES BY PROGRAM")
print("=" * 100)

for program_id in sorted(
    files_by_program
):
    program_candidates = [
        item
        for item
        in transcript_quality_candidates_repaired_v1
        if item["program_id"]
        == program_id
    ]

    if not program_candidates:
        continue

    print()
    print(
        "Program:",
        program_candidates[0][
            "program"
        ],
    )
    print(
        "Program ID:",
        program_id,
    )
    print(
        "Candidates:",
        len(
            program_candidates
        ),
    )

    program_counts = Counter(
        item["candidate_type"]
        for item
        in program_candidates
    )

    for candidate_type, count in sorted(
        program_counts.items()
    ):
        print(
            f"- {candidate_type}: {count}"
        )

    print("-" * 100)


print()
print("REPAIRED BRACKETED MARKER INVENTORY")
print("=" * 100)

repaired_bracket_inventory = Counter()

for candidate in (
    transcript_quality_candidates_repaired_v1
):
    if (
        candidate[
            "candidate_type"
        ]
        != "BRACKETED_TRANSCRIPTION_MARKER"
    ):
        continue

    for marker in candidate[
        "evidence"
    ]["markers"]:
        repaired_bracket_inventory[
            marker
        ] += 1

if repaired_bracket_inventory:
    for marker, count in (
        repaired_bracket_inventory.most_common(
            50
        )
    ):
        print(
            f"- {marker}: {count}"
        )
else:
    print("- NONE")


print()
print("REPRESENTATIVE REPAIRED QUALITY CANDIDATES")
print("=" * 100)

shown_by_type = Counter()

for candidate in (
    transcript_quality_candidates_repaired_v1
):
    candidate_type = candidate[
        "candidate_type"
    ]

    if shown_by_type[
        candidate_type
    ] >= 5:
        continue

    shown_by_type[
        candidate_type
    ] += 1

    print()
    print(
        "Candidate ID:",
        candidate[
            "quality_candidate_id"
        ],
    )
    print(
        "Type:",
        candidate_type,
    )
    print(
        "Review status:",
        candidate[
            "review_status"
        ],
    )
    print(
        "Program:",
        candidate[
            "program"
        ],
    )
    print(
        "File:",
        candidate[
            "relative_path"
        ],
    )
    print(
        "Speaker raw:",
        candidate[
            "speaker_raw"
        ],
    )
    print(
        "Speaker status:",
        candidate[
            "speaker_status"
        ],
    )
    print(
        "Utterance IDs:",
        candidate[
            "utterance_ids"
        ],
    )
    print(
        "Source lines:",
        (
            candidate[
                "source_line_start"
            ],
            candidate[
                "source_line_end"
            ],
        ),
    )
    print(
        "Evidence:",
        json.dumps(
            candidate[
                "evidence"
            ],
            ensure_ascii=False,
        ),
    )
    print(
        "Raw text:",
        repr(
            candidate[
                "raw_text"
            ][:500]
        ),
    )
    print("-" * 100)


repaired_technical_noise_candidates = [
    item
    for item
    in transcript_quality_candidates_repaired_v1
    if item[
        "review_status"
    ] == "TECHNICAL_NOISE"
]

repaired_repetition_candidates = [
    item
    for item
    in transcript_quality_candidates_repaired_v1
    if item["candidate_type"]
    in {
        "REPEATED_TOKEN_RUN",
        "REPEATED_NGRAM_RUN",
        "EXACT_ADJACENT_DUPLICATE_UTTERANCE",
    }
]


repaired_postmerge_state_after_quality_scan = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


assert len(
    all_merged_utterances_repaired_v1
) == 16226, (
    "Repaired post-merge utterance population changed."
)

assert (
    repaired_postmerge_state_before_quality_scan
    == repaired_postmerge_state_after_quality_scan
), (
    "Quality scan mutated repaired post-merge output."
)

assert len({
    item[
        "quality_candidate_id"
    ]
    for item
    in transcript_quality_candidates_repaired_v1
}) == len(
    transcript_quality_candidates_repaired_v1
), (
    "Repaired quality candidate IDs are not unique."
)

assert not (
    unknown_candidate_utterance_ids
), (
    "A repaired quality candidate references "
    "an unknown utterance."
)

assert all(
    item["review_status"]
    in {
        "REVIEW_REQUIRED",
        "STRUCTURAL_MARKER_REVIEW",
        "TECHNICAL_NOISE",
    }
    for item
    in transcript_quality_candidates_repaired_v1
), (
    "Unexpected repaired quality review status."
)

assert all(
    item[
        "candidate_type"
    ]
    != "EXACT_ADJACENT_DUPLICATE_UTTERANCE"
    or len(
        item["utterance_ids"]
    ) == 2
    for item
    in transcript_quality_candidates_repaired_v1
), (
    "Exact adjacent duplicate does not reference two utterances."
)

print()
print(
    "REPAIRED ASR REPETITION AND "
    "RAW-NOISE DETECTION: PASS"
)

Repaired post-merge utterances inspected: 16226
Repaired quality candidates: 133

REPAIRED QUALITY CANDIDATE COUNTS
- BRACKETED_TRANSCRIPTION_MARKER: 53
- EXACT_ADJACENT_DUPLICATE_UTTERANCE: 4
- REPEATED_NGRAM_RUN: 66
- REPEATED_TOKEN_RUN: 10

REPAIRED REVIEW STATUS COUNTS
- REVIEW_REQUIRED: 80
- STRUCTURAL_MARKER_REVIEW: 53

QUALITY CANDIDATES BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Candidates: 13
- BRACKETED_TRANSCRIPTION_MARKER: 5
- REPEATED_NGRAM_RUN: 7
- REPEATED_TOKEN_RUN: 1
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Candidates: 1
- REPEATED_NGRAM_RUN: 1
----------------------------------------------------------------------------------------------------

Program: چهلستون
Program ID: program:chehelston
Candidates: 31
- BRACKETED_TRANSCRIPTION_MARKER: 30
- REPEATED_NGRAM_RUN: 1
---------------------------------------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازسازی Segment Locator و Quality Link روی Stream اصلاح‌شده</h3>

<p>
Segment Locator قبلی قبل از Repair چهلستون ساخته شده بود و دیگر برای Final Pipeline معتبر نیست.
</p>

<p>
در این Cell Locatorها از ابتدا بر اساس
<code>all_merged_utterances_repaired_v1</code>
بازسازی می‌شوند.
</p>

<h4>مبنای جدید</h4>

<p>
Stream معتبر فعلی شامل:
</p>

<ul>
<li><code>16226</code> Segment Post-Merge</li>
<li><code>133</code> Quality Candidate</li>
<li>Speaker Boundary اصلاح‌شده برای ۳۵ فایل Explicit چهلستون</li>
</ul>

<h4>Locator هر Segment</h4>

<p>
برای هر Segment اطلاعات زیر حفظ می‌شود:
</p>

<ul>
<li>Program و Program ID</li>
<li>Physical File ID</li>
<li>Transcript Unit ID</li>
<li>Relative Path</li>
<li>Post-Merge Segment Index</li>
<li>Utterance ID</li>
<li>Source Line Range</li>
<li>Speaker Raw</li>
<li>Speaker Canonical</li>
<li>Speaker Source Canonical</li>
<li>Speaker Status</li>
<li>Boundary Type</li>
<li>Merge Status</li>
<li>Timestamp</li>
</ul>

<h4>چهلستون اصلاح‌شده</h4>

<p>
در Locator جدید، Paragraphهای بدون Label چهلستون باید با:
</p>

<p>
<code>speaker_status = UNRESOLVED</code>
</p>

<p>
و بدون هیچ Speaker Raw یا Canonical ظاهر شوند.
</p>

<p>
این موضوع به‌صورت مستقل Assert خواهد شد.
</p>

<h4>Quality Candidate Linking</h4>

<p>
تمام <code>133</code> Quality Candidate جدید به Locatorهای جدید متصل می‌شوند.
</p>

<p>
Candidateهای Single-segment باید یک Locator و Exact Adjacent Duplicateها باید دقیقاً دو Locator داشته باشند.
</p>

<h4>عدم استفاده از Locator قدیمی</h4>

<p>
متغیرهای Locator قبلی برای Audit تاریخی باقی می‌مانند، اما از این مرحله به بعد Final Pipeline فقط از نسخه اصلاح‌شده استفاده خواهد کرد.
</p>

<h4>خروجی جدید</h4>

<ul>
<li><code>transcript_segment_locator_repaired_v1</code></li>
<li><code>locator_repaired_by_utterance_id</code></li>
<li><code>quality_candidate_locator_links_repaired_v1</code></li>
</ul>

<h4>هنوز انجام نمی‌شوند</h4>

<ul>
<li>Human Review PASS</li>
<li>اصلاح ASR</li>
<li>Spell correction</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>برای هر <code>16226</code> Segment دقیقاً یک Locator ساخته شود.</li>
<li>Locator IDها یکتا باشند.</li>
<li>تمام Source Path و Source Rangeها معتبر باشند.</li>
<li>تمام <code>133</code> Quality Candidate به Locator معتبر متصل شوند.</li>
<li>هیچ Quality Candidate دارای Utterance ID ناشناخته نباشد.</li>
<li>Paragraphهای unlabeled چهلستون هیچ Attribution نداشته باشند.</li>
<li>هیچ Timestamp جدیدی ساخته نشود.</li>
</ul>

</div>

In [36]:
# Code Cell 36

repaired_merged_utterance_by_id = {
    item["utterance_id"]: item
    for item
    in all_merged_utterances_repaired_v1
}

assert len(
    repaired_merged_utterance_by_id
) == len(
    all_merged_utterances_repaired_v1
), (
    "Repaired post-merge utterance IDs are not unique."
)


transcript_segment_locator_repaired_v1 = []

for item in (
    all_merged_utterances_repaired_v1
):
    locator_id = (
        f"{item['transcript_unit_id']}"
        f":segment:"
        f"{item['utterance_index']:05d}"
    )

    transcript_segment_locator_repaired_v1.append({
        "locator_id": (
            locator_id
        ),
        "program": (
            item["program"]
        ),
        "program_id": (
            item["program_id"]
        ),
        "physical_file_id": (
            item["physical_file_id"]
        ),
        "transcript_unit_id": (
            item["transcript_unit_id"]
        ),
        "relative_path": (
            item["relative_path"]
        ),
        "utterance_index": (
            item["utterance_index"]
        ),
        "utterance_id": (
            item["utterance_id"]
        ),
        "source_line_start": (
            item["source_line_start"]
        ),
        "source_line_end": (
            item["source_line_end"]
        ),
        "speaker_raw": (
            item["speaker_raw"]
        ),
        "speaker_canonical": (
            item["speaker_canonical"]
        ),
        "speaker_source_canonical": (
            item[
                "speaker_source_canonical"
            ]
        ),
        "speaker_status": (
            item["speaker_status"]
        ),
        "boundary_type": (
            item["boundary_type"]
        ),
        "merge_status": (
            item["merge_status"]
        ),
        "start_time": (
            item["start_time"]
        ),
        "end_time": (
            item["end_time"]
        ),
    })


locator_repaired_by_utterance_id = {
    item["utterance_id"]: item
    for item
    in transcript_segment_locator_repaired_v1
}

locator_repaired_by_locator_id = {
    item["locator_id"]: item
    for item
    in transcript_segment_locator_repaired_v1
}


quality_candidate_locator_links_repaired_v1 = []

unknown_repaired_quality_utterance_ids = []

for candidate in (
    transcript_quality_candidates_repaired_v1
):
    linked_locators = []

    for utterance_id in candidate[
        "utterance_ids"
    ]:
        locator = (
            locator_repaired_by_utterance_id.get(
                utterance_id
            )
        )

        if locator is None:
            unknown_repaired_quality_utterance_ids.append({
                "quality_candidate_id": (
                    candidate[
                        "quality_candidate_id"
                    ]
                ),
                "utterance_id": (
                    utterance_id
                ),
            })
            continue

        linked_locators.append({
            "locator_id": (
                locator["locator_id"]
            ),
            "utterance_id": (
                locator["utterance_id"]
            ),
            "relative_path": (
                locator["relative_path"]
            ),
            "utterance_index": (
                locator["utterance_index"]
            ),
            "source_line_start": (
                locator[
                    "source_line_start"
                ]
            ),
            "source_line_end": (
                locator[
                    "source_line_end"
                ]
            ),
            "speaker_raw": (
                locator["speaker_raw"]
            ),
            "speaker_canonical": (
                locator[
                    "speaker_canonical"
                ]
            ),
            "speaker_source_canonical": (
                locator[
                    "speaker_source_canonical"
                ]
            ),
            "speaker_status": (
                locator[
                    "speaker_status"
                ]
            ),
            "boundary_type": (
                locator[
                    "boundary_type"
                ]
            ),
            "merge_status": (
                locator[
                    "merge_status"
                ]
            ),
        })

    quality_candidate_locator_links_repaired_v1.append({
        "quality_candidate_id": (
            candidate[
                "quality_candidate_id"
            ]
        ),
        "candidate_type": (
            candidate[
                "candidate_type"
            ]
        ),
        "review_status": (
            candidate[
                "review_status"
            ]
        ),
        "program": (
            candidate["program"]
        ),
        "program_id": (
            candidate["program_id"]
        ),
        "relative_path": (
            candidate["relative_path"]
        ),
        "linked_segment_count": (
            len(linked_locators)
        ),
        "segment_locators": (
            linked_locators
        ),
    })


invalid_repaired_locator_paths = []
invalid_repaired_source_ranges = []

for locator in (
    transcript_segment_locator_repaired_v1
):
    relative_path = locator[
        "relative_path"
    ]

    if (
        relative_path
        not in accepted_record_by_path
    ):
        invalid_repaired_locator_paths.append(
            relative_path
        )
        continue

    file_path = (
        project_root
        / relative_path
    )

    line_count = len(
        file_path.read_text(
            encoding="utf-8"
        ).splitlines()
    )

    if not (
        1
        <= locator[
            "source_line_start"
        ]
        <= locator[
            "source_line_end"
        ]
        <= line_count
    ):
        invalid_repaired_source_ranges.append({
            "locator_id": (
                locator[
                    "locator_id"
                ]
            ),
            "relative_path": (
                relative_path
            ),
            "source_line_start": (
                locator[
                    "source_line_start"
                ]
            ),
            "source_line_end": (
                locator[
                    "source_line_end"
                ]
            ),
            "file_line_count": (
                line_count
            ),
        })


repaired_chehelston_unlabeled_locators = [
    item
    for item
    in transcript_segment_locator_repaired_v1
    if (
        item["program_id"]
        == chehelston_program_id
        and item[
            "boundary_type"
        ]
        == (
            "CHEHELSTON_UNLABELED_"
            "PARAGRAPH_UNRESOLVED"
        )
    )
]

invalid_chehelston_unlabeled_attribution = [
    item
    for item
    in repaired_chehelston_unlabeled_locators
    if (
        item["speaker_status"]
        != "UNRESOLVED"
        or item["speaker_raw"]
        is not None
        or item[
            "speaker_canonical"
        ] is not None
        or item[
            "speaker_source_canonical"
        ] is not None
    )
]


print(
    "Repaired post-merge segments:",
    len(
        all_merged_utterances_repaired_v1
    ),
)
print(
    "Repaired segment locators:",
    len(
        transcript_segment_locator_repaired_v1
    ),
)
print(
    "Unique repaired locator IDs:",
    len(
        locator_repaired_by_locator_id
    ),
)
print(
    "Unique repaired linked utterance IDs:",
    len(
        locator_repaired_by_utterance_id
    ),
)
print()


print("REPAIRED LOCATOR COUNTS BY PROGRAM")
print("=" * 100)

for program_id in sorted(
    files_by_program
):
    program_locators = [
        item
        for item
        in transcript_segment_locator_repaired_v1
        if item["program_id"]
        == program_id
    ]

    if not program_locators:
        continue

    print()
    print(
        "Program:",
        program_locators[0][
            "program"
        ],
    )
    print(
        "Program ID:",
        program_id,
    )
    print(
        "Segments:",
        len(
            program_locators
        ),
    )

    speaker_status_counts = Counter(
        item["speaker_status"]
        for item
        in program_locators
    )

    print(
        "Speaker status:",
        dict(
            sorted(
                speaker_status_counts.items()
            )
        ),
    )

    merge_status_counts = Counter(
        item["merge_status"]
        for item
        in program_locators
    )

    print(
        "Merge status:",
        dict(
            sorted(
                merge_status_counts.items()
            )
        ),
    )

    print("-" * 100)


print()
print(
    "REPAIRED QUALITY CANDIDATE LOCATOR LINKING"
)
print("=" * 100)
print(
    "Quality candidates:",
    len(
        transcript_quality_candidates_repaired_v1
    ),
)
print(
    "Candidate locator records:",
    len(
        quality_candidate_locator_links_repaired_v1
    ),
)
print(
    "Unknown quality utterance IDs:",
    len(
        unknown_repaired_quality_utterance_ids
    ),
)

repaired_link_count_distribution = Counter(
    item["linked_segment_count"]
    for item
    in quality_candidate_locator_links_repaired_v1
)

print(
    "Linked segment-count distribution:",
    dict(
        sorted(
            repaired_link_count_distribution.items()
        )
    ),
)


print()
print(
    "CHEHELSTON UNLABELED LOCATOR SAFETY"
)
print("=" * 100)
print(
    "Unlabeled repaired Chehelston segments:",
    len(
        repaired_chehelston_unlabeled_locators
    ),
)
print(
    "Invalid unlabeled speaker attributions:",
    len(
        invalid_chehelston_unlabeled_attribution
    ),
)


print()
print(
    "REPRESENTATIVE REPAIRED CHEHELSTON LOCATORS"
)
print("=" * 100)

shown_explicit = 0
shown_unresolved = 0

for locator in (
    transcript_segment_locator_repaired_v1
):
    if (
        locator["program_id"]
        != chehelston_program_id
    ):
        continue

    if (
        locator["speaker_status"]
        == "EXPLICIT_SOURCE_MARKER"
        and shown_explicit < 5
    ):
        shown_explicit += 1

    elif (
        locator["boundary_type"]
        == (
            "CHEHELSTON_UNLABELED_"
            "PARAGRAPH_UNRESOLVED"
        )
        and shown_unresolved < 5
    ):
        shown_unresolved += 1

    else:
        continue

    print()
    print(
        "Locator ID:",
        locator["locator_id"],
    )
    print(
        "File:",
        locator["relative_path"],
    )
    print(
        "Segment index:",
        locator["utterance_index"],
    )
    print(
        "Source lines:",
        (
            locator[
                "source_line_start"
            ],
            locator[
                "source_line_end"
            ],
        ),
    )
    print(
        "Speaker raw:",
        locator["speaker_raw"],
    )
    print(
        "Speaker canonical:",
        locator[
            "speaker_canonical"
        ],
    )
    print(
        "Speaker status:",
        locator[
            "speaker_status"
        ],
    )
    print(
        "Boundary:",
        locator[
            "boundary_type"
        ],
    )
    print("-" * 100)

    if (
        shown_explicit >= 5
        and shown_unresolved >= 5
    ):
        break


print()
print(
    "REPRESENTATIVE REPAIRED QUALITY LINKS"
)
print("=" * 100)

for item in (
    quality_candidate_locator_links_repaired_v1[
        :10
    ]
):
    print()
    print(
        "Candidate ID:",
        item[
            "quality_candidate_id"
        ],
    )
    print(
        "Type:",
        item[
            "candidate_type"
        ],
    )
    print(
        "Program:",
        item[
            "program"
        ],
    )
    print(
        "Linked segments:",
        item[
            "linked_segment_count"
        ],
    )

    for locator in item[
        "segment_locators"
    ]:
        print(
            " ",
            locator["locator_id"],
            "|",
            locator[
                "relative_path"
            ],
            "| lines",
            (
                locator[
                    "source_line_start"
                ],
                locator[
                    "source_line_end"
                ],
            ),
            "| speaker status:",
            locator[
                "speaker_status"
            ],
            "| speaker:",
            (
                locator[
                    "speaker_raw"
                ]
                or locator[
                    "speaker_canonical"
                ]
                or "NONE"
            ),
        )

    print("-" * 100)


assert len(
    transcript_segment_locator_repaired_v1
) == 16226, (
    "Repaired locator does not cover "
    "all post-merge segments."
)

assert len(
    locator_repaired_by_locator_id
) == len(
    transcript_segment_locator_repaired_v1
), (
    "Repaired locator IDs are not unique."
)

assert len(
    locator_repaired_by_utterance_id
) == len(
    transcript_segment_locator_repaired_v1
), (
    "Not every repaired utterance has "
    "exactly one locator."
)

assert not (
    invalid_repaired_locator_paths
), (
    "A repaired locator references "
    "an unknown Source path."
)

assert not (
    invalid_repaired_source_ranges
), (
    "A repaired locator has an invalid "
    "Source line range."
)

assert len(
    quality_candidate_locator_links_repaired_v1
) == len(
    transcript_quality_candidates_repaired_v1
), (
    "Repaired quality locator coverage "
    "is incomplete."
)

assert not (
    unknown_repaired_quality_utterance_ids
), (
    "A repaired quality candidate references "
    "an unknown post-merge utterance."
)

assert all(
    item["linked_segment_count"]
    in {
        1,
        2,
    }
    for item
    in quality_candidate_locator_links_repaired_v1
), (
    "Unexpected repaired quality link count."
)

assert repaired_link_count_distribution[
    2
] == len(
    repaired_exact_adjacent_duplicates
), (
    "Two-segment quality links do not match "
    "exact adjacent duplicate candidates."
)

assert not (
    invalid_chehelston_unlabeled_attribution
), (
    "A repaired unlabeled Chehelston segment "
    "has speaker attribution."
)

assert all(
    item["start_time"] is None
    and item["end_time"] is None
    for item
    in transcript_segment_locator_repaired_v1
), (
    "Unexpected timestamp found "
    "in repaired locator."
)

print()
print(
    "REPAIRED EPISODE / SPEAKER / SEGMENT "
    "LOCATOR BUILD: PASS"
)

Repaired post-merge segments: 16226
Repaired segment locators: 16226
Unique repaired locator IDs: 16226
Unique repaired linked utterance IDs: 16226

REPAIRED LOCATOR COUNTS BY PROGRAM

Program: عصر شیرین
Program ID: program:asre-shirin
Segments: 902
Speaker status: {'EXPLICIT_SOURCE_MARKER': 902}
Merge status: {'UNCHANGED': 902}
----------------------------------------------------------------------------------------------------

Program: بانک ملت
Program ID: program:bank-mellat
Segments: 453
Speaker status: {'MANIFEST_ATTRIBUTED': 453}
Merge status: {'UNCHANGED': 453}
----------------------------------------------------------------------------------------------------

Program: چهلستون
Program ID: program:chehelston
Segments: 2455
Speaker status: {'EXPLICIT_SOURCE_MARKER': 504, 'UNRESOLVED': 1951}
Merge status: {'UNCHANGED': 2455}
----------------------------------------------------------------------------------------------------

Program: در جستجوی خوشبختی
Program ID: program:dar-josto

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازسازی Stratified Human Review Selection روی Stream اصلاح‌شده</h3>

<p>
Human Review Selection قبلی بر اساس Parser قبل از Repair چهلستون ساخته شده بود و دیگر مبنای نهایی Notebook 06 نیست.
</p>

<p>
در این Cell مجموعه Review از ابتدا روی Stream اصلاح‌شده ساخته می‌شود.
</p>

<h4>هدف</h4>

<p>
حداقل Roadmap برابر ۲۰ Transcript Unit است.
</p>

<p>
برای پوشش مطمئن‌تر، همان هدف <code>24</code> Unit حفظ می‌شود، اما Selection جدید علاوه بر Risk باید از تمرکز بیش از حد روی یک Program جلوگیری کند.
</p>

<h4>اصل Stratification</h4>

<p>
ابتدا از هر ۱۳ Program حداقل یک Unit انتخاب می‌شود.
</p>

<p>
سپس Unitهای Risk بالاتر به مجموعه اضافه می‌شوند، اما برای جلوگیری از تمرکز بیش از حد، هر Program حداکثر سه Unit در Selection خواهد داشت.
</p>

<h4>Signalهای Risk</h4>

<ul>
<li>وجود Segmentهای <code>UNRESOLVED</code></li>
<li>وجود <code>NARRATIVE_CUE</code></li>
<li>وجود Explicit Speaker Marker</li>
<li>وجود Merge</li>
<li>وجود Quality Candidate</li>
<li>وجود چند Speaker Status در یک Unit</li>
</ul>

<h4>پوشش خاص چهلستون</h4>

<p>
چون Repair اصلی این Notebook روی چهلستون انجام شد، Selection باید هر دو گروه زیر را شامل شود:
</p>

<ul>
<li>حداقل یک فایل Explicit Repair‌شده</li>
<li>حداقل یک فایل Contextual که همچنان unresolved باقی مانده است</li>
</ul>

<p>
این دو گروه عمداً جدا کنترل می‌شوند.
</p>

<h4>عدم ادعای Human Review</h4>

<p>
این Cell فقط Selection را می‌سازد.
</p>

<p>
هیچ Unit به‌صورت خودکار <code>HUMAN_REVIEWED</code> یا <code>PASS</code> نمی‌شود.
</p>

<h4>نسخه خروجی</h4>

<p>
Selection معتبر جدید در:
</p>

<p>
<code>human_review_selection_repaired_v1</code>
</p>

<p>
نگهداری می‌شود.
</p>

<h4>هنوز انجام نمی‌شوند</h4>

<ul>
<li>ثبت Human Review PASS</li>
<li>اصلاح Transcript</li>
<li>اصلاح ASR</li>
<li>Spell correction</li>
<li>Persian normalization</li>
<li>Deduplication</li>
<li>Chunking</li>
<li>Artifact write</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>هر ۱۸۳ Source File در Profile جدید حضور داشته باشد.</li>
<li>دقیقاً ۲۴ Transcript Unit یکتا انتخاب شود.</li>
<li>هر ۱۳ Program حداقل یک Unit داشته باشند.</li>
<li>هیچ Program بیش از سه Unit نداشته باشد.</li>
<li>حداقل یک Unit دارای Merge باشد.</li>
<li>حداقل یک Unit دارای Quality Candidate باشد.</li>
<li>حداقل یک Unit دارای unresolved باشد.</li>
<li>حداقل یک Unit دارای Narrative Cue باشد.</li>
<li>حداقل یک چهلستون Explicit Repair‌شده و یک چهلستون Contextual انتخاب شوند.</li>
</ul>

</div>

In [37]:
# Code Cell 37

repaired_quality_candidates_by_path = defaultdict(
    list
)

for candidate in (
    transcript_quality_candidates_repaired_v1
):
    repaired_quality_candidates_by_path[
        candidate["relative_path"]
    ].append(
        candidate
    )


review_unit_profiles_repaired = []

for parsed_file in (
    parsed_transcript_files_merged_repaired_v1
):
    relative_path = parsed_file[
        "relative_path"
    ]

    utterances = parsed_file[
        "utterances"
    ]

    assert utterances, (
        "Repaired review profile encountered an empty file."
    )

    first = utterances[0]

    speaker_status_counts = Counter(
        item["speaker_status"]
        for item
        in utterances
    )

    merge_status_counts = Counter(
        item["merge_status"]
        for item
        in utterances
    )

    file_quality_candidates = (
        repaired_quality_candidates_by_path.get(
            relative_path,
            [],
        )
    )

    quality_type_counts = Counter(
        item["candidate_type"]
        for item
        in file_quality_candidates
    )

    unresolved_count = (
        speaker_status_counts[
            "UNRESOLVED"
        ]
    )

    narrative_count = (
        speaker_status_counts[
            "NARRATIVE_CUE"
        ]
    )

    explicit_count = (
        speaker_status_counts[
            "EXPLICIT_SOURCE_MARKER"
        ]
    )

    manifest_count = (
        speaker_status_counts[
            "MANIFEST_ATTRIBUTED"
        ]
    )

    merged_count = (
        merge_status_counts[
            "MERGED"
        ]
    )

    quality_count = len(
        file_quality_candidates
    )

    speaker_status_diversity = len(
        speaker_status_counts
    )

    is_chehelston_contextual = (
        relative_path
        in chehelston_contextual_path_set
    )

    is_chehelston_repaired_explicit = (
        relative_path
        in chehelston_explicit_paths
    )

    risk_score = 0

    if unresolved_count:
        risk_score += 50

    if narrative_count:
        risk_score += 40

    if quality_count:
        risk_score += 30

    if merged_count:
        risk_score += 25

    if explicit_count:
        risk_score += 15

    if speaker_status_diversity > 1:
        risk_score += 15

    if is_chehelston_repaired_explicit:
        risk_score += 20

    if is_chehelston_contextual:
        risk_score += 20

    risk_score += min(
        unresolved_count,
        10,
    )

    risk_score += min(
        quality_count,
        10,
    )

    risk_score += min(
        merged_count,
        10,
    )

    review_unit_profiles_repaired.append({
        "program": (
            first["program"]
        ),
        "program_id": (
            first["program_id"]
        ),
        "physical_file_id": (
            first["physical_file_id"]
        ),
        "transcript_unit_id": (
            first["transcript_unit_id"]
        ),
        "relative_path": (
            relative_path
        ),
        "segment_count": (
            len(utterances)
        ),
        "speaker_status_counts": dict(
            sorted(
                speaker_status_counts.items()
            )
        ),
        "merge_status_counts": dict(
            sorted(
                merge_status_counts.items()
            )
        ),
        "quality_candidate_count": (
            quality_count
        ),
        "quality_candidate_types": dict(
            sorted(
                quality_type_counts.items()
            )
        ),
        "unresolved_segment_count": (
            unresolved_count
        ),
        "narrative_cue_count": (
            narrative_count
        ),
        "explicit_speaker_segment_count": (
            explicit_count
        ),
        "manifest_attributed_segment_count": (
            manifest_count
        ),
        "merged_segment_count": (
            merged_count
        ),
        "speaker_status_diversity": (
            speaker_status_diversity
        ),
        "is_chehelston_contextual": (
            is_chehelston_contextual
        ),
        "is_chehelston_repaired_explicit": (
            is_chehelston_repaired_explicit
        ),
        "review_risk_score": (
            risk_score
        ),
    })


assert len(
    review_unit_profiles_repaired
) == 183, (
    "Repaired review profiles do not cover all accepted files."
)


profiles_repaired_by_program = defaultdict(
    list
)

for profile in (
    review_unit_profiles_repaired
):
    profiles_repaired_by_program[
        profile["program_id"]
    ].append(
        profile
    )


def repaired_review_priority(item):
    return (
        -item[
            "review_risk_score"
        ],
        -item[
            "quality_candidate_count"
        ],
        -item[
            "unresolved_segment_count"
        ],
        -item[
            "merged_segment_count"
        ],
        -item[
            "speaker_status_diversity"
        ],
        item[
            "relative_path"
        ],
    )


for program_id in (
    profiles_repaired_by_program
):
    profiles_repaired_by_program[
        program_id
    ].sort(
        key=repaired_review_priority
    )


selected_repaired_profiles = []
selected_repaired_paths = set()
selected_repaired_program_counts = Counter()


def add_repaired_review_profile(
    profile,
):
    relative_path = profile[
        "relative_path"
    ]

    program_id = profile[
        "program_id"
    ]

    if (
        relative_path
        in selected_repaired_paths
    ):
        return False

    if (
        selected_repaired_program_counts[
            program_id
        ] >= 3
    ):
        return False

    selected_repaired_profiles.append(
        profile
    )

    selected_repaired_paths.add(
        relative_path
    )

    selected_repaired_program_counts[
        program_id
    ] += 1

    return True


for program_id in sorted(
    profiles_repaired_by_program
):
    add_repaired_review_profile(
        profiles_repaired_by_program[
            program_id
        ][0]
    )


selected_has_chehelston_explicit = any(
    item[
        "is_chehelston_repaired_explicit"
    ]
    for item
    in selected_repaired_profiles
)

selected_has_chehelston_contextual = any(
    item[
        "is_chehelston_contextual"
    ]
    for item
    in selected_repaired_profiles
)


if not selected_has_chehelston_explicit:
    explicit_candidates = sorted(
        [
            item
            for item
            in review_unit_profiles_repaired
            if item[
                "is_chehelston_repaired_explicit"
            ]
        ],
        key=repaired_review_priority,
    )

    assert explicit_candidates, (
        "No repaired explicit Chehelston profile found."
    )

    added = False

    for profile in explicit_candidates:
        if add_repaired_review_profile(
            profile
        ):
            added = True
            break

    assert added, (
        "Could not add repaired explicit Chehelston review unit."
    )


if not selected_has_chehelston_contextual:
    contextual_candidates = sorted(
        [
            item
            for item
            in review_unit_profiles_repaired
            if item[
                "is_chehelston_contextual"
            ]
        ],
        key=repaired_review_priority,
    )

    assert contextual_candidates, (
        "No contextual Chehelston profile found."
    )

    added = False

    for profile in contextual_candidates:
        if add_repaired_review_profile(
            profile
        ):
            added = True
            break

    assert added, (
        "Could not add contextual Chehelston review unit."
    )


global_repaired_priority = sorted(
    review_unit_profiles_repaired,
    key=lambda item: (
        -item[
            "review_risk_score"
        ],
        -item[
            "quality_candidate_count"
        ],
        -item[
            "unresolved_segment_count"
        ],
        -item[
            "merged_segment_count"
        ],
        -item[
            "speaker_status_diversity"
        ],
        item[
            "program_id"
        ],
        item[
            "relative_path"
        ],
    ),
)


target_repaired_review_count = 24

for profile in (
    global_repaired_priority
):
    if len(
        selected_repaired_profiles
    ) >= target_repaired_review_count:
        break

    add_repaired_review_profile(
        profile
    )


assert len(
    selected_repaired_profiles
) == target_repaired_review_count, (
    "Could not build 24 repaired review units "
    "under the per-program cap."
)


human_review_selection_repaired_v1 = []

for selection_index, profile in enumerate(
    selected_repaired_profiles,
    start=1,
):
    reasons = []

    if (
        profile[
            "unresolved_segment_count"
        ] > 0
    ):
        reasons.append(
            "HAS_UNRESOLVED_SEGMENTS"
        )

    if (
        profile[
            "narrative_cue_count"
        ] > 0
    ):
        reasons.append(
            "HAS_NARRATIVE_CUES"
        )

    if (
        profile[
            "explicit_speaker_segment_count"
        ] > 0
    ):
        reasons.append(
            "HAS_EXPLICIT_SPEAKER_MARKERS"
        )

    if (
        profile[
            "merged_segment_count"
        ] > 0
    ):
        reasons.append(
            "HAS_MERGED_SEGMENTS"
        )

    if (
        profile[
            "quality_candidate_count"
        ] > 0
    ):
        reasons.append(
            "HAS_QUALITY_CANDIDATES"
        )

    if (
        profile[
            "speaker_status_diversity"
        ] > 1
    ):
        reasons.append(
            "MULTIPLE_SPEAKER_STATUSES"
        )

    if (
        profile[
            "is_chehelston_repaired_explicit"
        ]
    ):
        reasons.append(
            "CHEHELSTON_REPAIRED_EXPLICIT"
        )

    if (
        profile[
            "is_chehelston_contextual"
        ]
    ):
        reasons.append(
            "CHEHELSTON_CONTEXTUAL_UNRESOLVED"
        )

    if not reasons:
        reasons.append(
            "PROGRAM_BASELINE_REVIEW"
        )

    human_review_selection_repaired_v1.append({
        "review_unit_id": (
            f"transcript-review-repaired:"
            f"{selection_index:03d}"
        ),
        "selection_index": (
            selection_index
        ),
        "selection_reasons": (
            reasons
        ),
        **profile,
    })


final_program_counts = Counter(
    item["program_id"]
    for item
    in human_review_selection_repaired_v1
)

selection_signal_counts = Counter()

for item in (
    human_review_selection_repaired_v1
):
    if (
        item[
            "unresolved_segment_count"
        ] > 0
    ):
        selection_signal_counts[
            "HAS_UNRESOLVED_SEGMENTS"
        ] += 1

    if (
        item[
            "narrative_cue_count"
        ] > 0
    ):
        selection_signal_counts[
            "HAS_NARRATIVE_CUES"
        ] += 1

    if (
        item[
            "explicit_speaker_segment_count"
        ] > 0
    ):
        selection_signal_counts[
            "HAS_EXPLICIT_SPEAKER_MARKERS"
        ] += 1

    if (
        item[
            "merged_segment_count"
        ] > 0
    ):
        selection_signal_counts[
            "HAS_MERGED_SEGMENTS"
        ] += 1

    if (
        item[
            "quality_candidate_count"
        ] > 0
    ):
        selection_signal_counts[
            "HAS_QUALITY_CANDIDATES"
        ] += 1

    if (
        item[
            "is_chehelston_repaired_explicit"
        ]
    ):
        selection_signal_counts[
            "CHEHELSTON_REPAIRED_EXPLICIT"
        ] += 1

    if (
        item[
            "is_chehelston_contextual"
        ]
    ):
        selection_signal_counts[
            "CHEHELSTON_CONTEXTUAL_UNRESOLVED"
        ] += 1


print(
    "Corpus review-unit profiles:",
    len(
        review_unit_profiles_repaired
    ),
)
print(
    "Programs represented in corpus:",
    len(
        profiles_repaired_by_program
    ),
)
print(
    "Selected repaired review units:",
    len(
        human_review_selection_repaired_v1
    ),
)
print(
    "Programs represented in selection:",
    len(
        final_program_counts
    ),
)
print(
    "Maximum units from one program:",
    max(
        final_program_counts.values()
    ),
)
print()


print("REPAIRED SELECTION SIGNAL COVERAGE")
print("=" * 100)

for signal, count in sorted(
    selection_signal_counts.items()
):
    print(
        f"- {signal}: {count}"
    )


print()
print("PROGRAM DISTRIBUTION")
print("=" * 100)

for program_id, count in sorted(
    final_program_counts.items()
):
    program_name = next(
        item["program"]
        for item
        in human_review_selection_repaired_v1
        if item["program_id"]
        == program_id
    )

    print(
        f"- {program_name} "
        f"({program_id}): {count}"
    )


print()
print("REPAIRED HUMAN REVIEW SELECTION")
print("=" * 100)

for item in (
    human_review_selection_repaired_v1
):
    print()
    print(
        "Review unit:",
        item[
            "review_unit_id"
        ],
    )
    print(
        "Program:",
        item["program"],
    )
    print(
        "Transcript unit ID:",
        item[
            "transcript_unit_id"
        ],
    )
    print(
        "File:",
        item[
            "relative_path"
        ],
    )
    print(
        "Risk score:",
        item[
            "review_risk_score"
        ],
    )
    print(
        "Segments:",
        item[
            "segment_count"
        ],
    )
    print(
        "Speaker status:",
        item[
            "speaker_status_counts"
        ],
    )
    print(
        "Merged segments:",
        item[
            "merged_segment_count"
        ],
    )
    print(
        "Quality candidates:",
        item[
            "quality_candidate_count"
        ],
    )
    print(
        "Quality types:",
        item[
            "quality_candidate_types"
        ],
    )
    print(
        "Selection reasons:",
        item[
            "selection_reasons"
        ],
    )
    print("-" * 100)


selected_repaired_relative_paths = [
    item["relative_path"]
    for item
    in human_review_selection_repaired_v1
]

selected_repaired_unit_ids = [
    item["transcript_unit_id"]
    for item
    in human_review_selection_repaired_v1
]


assert len(
    human_review_selection_repaired_v1
) == 24, (
    "Repaired human review selection "
    "must contain exactly 24 units."
)

assert len(
    set(
        selected_repaired_relative_paths
    )
) == 24, (
    "Repaired review selection contains duplicate files."
)

assert len(
    set(
        selected_repaired_unit_ids
    )
) == 24, (
    "Repaired review selection contains duplicate units."
)

assert len(
    final_program_counts
) == 13, (
    "Repaired review selection does not "
    "cover all 13 programs."
)

assert max(
    final_program_counts.values()
) <= 3, (
    "A program exceeds the three-unit review cap."
)

assert selection_signal_counts[
    "HAS_UNRESOLVED_SEGMENTS"
] >= 1, (
    "No unresolved review unit selected."
)

assert selection_signal_counts[
    "HAS_NARRATIVE_CUES"
] >= 1, (
    "No narrative-cue review unit selected."
)

assert selection_signal_counts[
    "HAS_EXPLICIT_SPEAKER_MARKERS"
] >= 1, (
    "No explicit-speaker review unit selected."
)

assert selection_signal_counts[
    "HAS_MERGED_SEGMENTS"
] >= 1, (
    "No merged review unit selected."
)

assert selection_signal_counts[
    "HAS_QUALITY_CANDIDATES"
] >= 1, (
    "No quality-flagged review unit selected."
)

assert selection_signal_counts[
    "CHEHELSTON_REPAIRED_EXPLICIT"
] >= 1, (
    "No repaired explicit Chehelston unit selected."
)

assert selection_signal_counts[
    "CHEHELSTON_CONTEXTUAL_UNRESOLVED"
] >= 1, (
    "No contextual Chehelston unit selected."
)

print()
print(
    "REPAIRED STRATIFIED HUMAN REVIEW "
    "SELECTION BUILD: PASS"
)

Corpus review-unit profiles: 183
Programs represented in corpus: 13
Selected repaired review units: 24
Programs represented in selection: 13
Maximum units from one program: 3

REPAIRED SELECTION SIGNAL COVERAGE
- CHEHELSTON_CONTEXTUAL_UNRESOLVED: 1
- CHEHELSTON_REPAIRED_EXPLICIT: 2
- HAS_EXPLICIT_SPEAKER_MARKERS: 10
- HAS_MERGED_SEGMENTS: 6
- HAS_NARRATIVE_CUES: 1
- HAS_QUALITY_CANDIDATES: 20
- HAS_UNRESOLVED_SEGMENTS: 8

PROGRAM DISTRIBUTION
- عصر شیرین (program:asre-shirin): 2
- بانک ملت (program:bank-mellat): 1
- چهلستون (program:chehelston): 3
- در جستجوی خوشبختی (program:dar-jostojo-ye-khoshbakhti): 3
- فناپ (program:fanap): 1
- کتاب باز (program:ketab-baz): 1
- خانه تو (program:khane-to): 3
- میزان (program:mizan): 1
- ام کیو (program:mq): 1
- اقیانوس آرام (program:oghyanous-aram): 2
- شراکت (program:sherakat): 1
- تربیت مربی (program:tarbiat-morabi): 2
- یونی تهران (program:uni-tehran): 3

REPAIRED HUMAN REVIEW SELECTION

Review unit: transcript-review-repaired:001
Program: عصر 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اصلاح Human Review Packet و الزام نمایش هر دو سمت مرز Repair‌شده چهلستون</h3>

<p>
نسخه قبلی Human Review Packet یک نقص در منطق Sample Selection داشت.
</p>

<p>
در فایل‌های Explicit چهلستون، Pair مربوط به مرز:
</p>

<pre>
EXPLICIT_SOURCE_MARKER
↓
CHEHELSTON_UNLABELED_PARAGRAPH_UNRESOLVED
</pre>

<p>
در حافظه شناسایی می‌شد، اما چون Quality Candidateها زودتر وارد چهار Sample نهایی می‌شدند، ممکن بود فقط سمت Explicit مرز نمایش داده شود و Paragraph بعدی که باید <code>UNRESOLVED</code> باشد از Packet نهایی حذف شود.
</p>

<p>
در نتیجه شمارنده قبلی وجود Pair را ثابت می‌کرد، اما Human Reviewer الزاماً هر دو سمت آن را مشاهده نمی‌کرد.
</p>

<h4>قاعده اصلاح‌شده Selection</h4>

<p>
برای هر Unit دارای Risk ویژه، بعضی Sampleها Mandatory خواهند بود.
</p>

<ul>
<li>در هر فایل Explicit Repair‌شده چهلستون، هر دو سمت یک مرز Explicit → Unresolved اجباری هستند.</li>
<li>در هر Unit دارای Merge، حداقل یک Segment Merge‌شده اجباری است.</li>
<li>در هر Unit دارای Quality Candidate، حداقل یک Quality Sample باید حضور داشته باشد.</li>
<li>ظرفیت باقی‌مانده از چهار Sample با Risk، Speaker Transition و Baseline تکمیل می‌شود.</li>
</ul>

<h4>Human Review چهلستون</h4>

<p>
Reviewer باید در Packet جدید بتواند پشت سر هم ببیند:
</p>

<ul>
<li>Paragraph دارای Speaker Marker صریح</li>
<li>Paragraph بعدی بدون Label</li>
<li>Speaker صریح فقط روی Paragraph اول</li>
<li>مقادیر Speaker در Paragraph دوم همگی <code>None</code></li>
<li><code>speaker_status = UNRESOLVED</code> برای Paragraph دوم</li>
</ul>

<h4>محدودیت</h4>

<p>
این Repair فقط Packet Review را اصلاح می‌کند.
</p>

<p>
Parser، Post-Merge Stream، Quality Candidateها و Locatorها تغییر نمی‌کنند.
</p>

<p>
همچنان هیچ Human Review PASS به‌صورت خودکار ثبت نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>۲۴ Review Unit و حداکثر چهار Sample برای هر Unit حفظ شوند.</li>
<li>هر Unit دارای Merge حداقل یک Sample Merge داشته باشد.</li>
<li>هر Unit دارای Quality Candidate حداقل یک Quality Sample داشته باشد.</li>
<li>برای تمام Unitهای Explicit Repair‌شده چهلستون، هر دو سمت یک Pair Explicit → Unresolved در Sampleهای نهایی حضور داشته باشند.</li>
<li>سمت Unresolved هیچ Speaker Attribution نداشته باشد.</li>
<li>تمام Source Textها دوباره از Disk تأیید شوند.</li>
<li>هیچ Human Review PASS خودکار ثبت نشود.</li>
<li>هیچ Transcript یا Output پایین‌دستی تغییر نکند.</li>
</ul>

</div>

In [38]:
# Code Cell 38

repaired_packet_stream_state_before = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)

repaired_quality_links_by_path = defaultdict(
    list
)

for link in (
    quality_candidate_locator_links_repaired_v1
):
    repaired_quality_links_by_path[
        link["relative_path"]
    ].append(
        link
    )


human_review_packet_repaired_v1 = []

repaired_packet_source_failures = []
repaired_packet_unknown_segment_failures = []
repaired_packet_merge_sampling_failures = []
repaired_packet_quality_sampling_failures = []
repaired_packet_chehelston_pair_failures = []

selected_chehelston_pair_count = 0


for review_unit in (
    human_review_selection_repaired_v1
):
    relative_path = review_unit[
        "relative_path"
    ]

    parsed_file = (
        repaired_merged_file_by_path[
            relative_path
        ]
    )

    utterances = parsed_file[
        "utterances"
    ]

    utterance_by_index = {
        item["utterance_index"]: item
        for item
        in utterances
    }

    focus_by_index = defaultdict(
        set
    )

    mandatory_indices = []
    quality_indices = []
    secondary_priority_indices = []
    transition_indices = []
    baseline_priority_indices = []


    def register_focus(
        utterance_index,
        focus,
        bucket=None,
    ):
        if (
            utterance_index
            not in utterance_by_index
        ):
            repaired_packet_unknown_segment_failures.append({
                "review_unit_id": (
                    review_unit[
                        "review_unit_id"
                    ]
                ),
                "relative_path": (
                    relative_path
                ),
                "utterance_index": (
                    utterance_index
                ),
                "focus": (
                    focus
                ),
            })
            return

        focus_by_index[
            utterance_index
        ].add(
            focus
        )

        if (
            bucket is not None
            and utterance_index
            not in bucket
        ):
            bucket.append(
                utterance_index
            )


    required_chehelston_pair = None

    if review_unit[
        "is_chehelston_repaired_explicit"
    ]:
        for index in range(
            len(utterances) - 1
        ):
            left = utterances[
                index
            ]

            right = utterances[
                index + 1
            ]

            if not (
                left[
                    "speaker_status"
                ]
                == "EXPLICIT_SOURCE_MARKER"
                and right[
                    "boundary_type"
                ]
                == (
                    "CHEHELSTON_UNLABELED_"
                    "PARAGRAPH_UNRESOLVED"
                )
            ):
                continue

            required_chehelston_pair = (
                left[
                    "utterance_index"
                ],
                right[
                    "utterance_index"
                ],
            )

            register_focus(
                left[
                    "utterance_index"
                ],
                (
                    "CHEHELSTON_"
                    "EXPLICIT_BOUNDARY"
                ),
                mandatory_indices,
            )

            register_focus(
                right[
                    "utterance_index"
                ],
                (
                    "CHEHELSTON_"
                    "UNLABELED_AFTER_EXPLICIT"
                ),
                mandatory_indices,
            )

            break

        if (
            required_chehelston_pair
            is None
        ):
            repaired_packet_chehelston_pair_failures.append({
                "review_unit_id": (
                    review_unit[
                        "review_unit_id"
                    ]
                ),
                "relative_path": (
                    relative_path
                ),
                "reason": (
                    "NO_EXPLICIT_TO_UNRESOLVED_PAIR_FOUND"
                ),
            })


    merged_items = [
        item
        for item
        in utterances
        if item[
            "merge_status"
        ] == "MERGED"
    ]

    if merged_items:
        register_focus(
            merged_items[0][
                "utterance_index"
            ],
            "MERGED_SEGMENT",
            mandatory_indices,
        )

        for item in merged_items[
            1:3
        ]:
            register_focus(
                item[
                    "utterance_index"
                ],
                "MERGED_SEGMENT",
                secondary_priority_indices,
            )


    file_quality_links = (
        repaired_quality_links_by_path.get(
            relative_path,
            [],
        )
    )

    for quality_link in (
        file_quality_links
    ):
        for locator in quality_link[
            "segment_locators"
        ]:
            register_focus(
                locator[
                    "utterance_index"
                ],
                (
                    "QUALITY:"
                    + quality_link[
                        "candidate_type"
                    ]
                ),
                quality_indices,
            )


    risk_status_items = [
        item
        for item
        in utterances
        if item[
            "speaker_status"
        ]
        in {
            "UNRESOLVED",
            "NARRATIVE_CUE",
        }
    ]

    for item in risk_status_items[
        :3
    ]:
        register_focus(
            item[
                "utterance_index"
            ],
            (
                "SPEAKER_STATUS:"
                + item[
                    "speaker_status"
                ]
            ),
            secondary_priority_indices,
        )


    for index in range(
        len(utterances) - 1
    ):
        left = utterances[
            index
        ]

        right = utterances[
            index + 1
        ]

        left_identity = (
            left[
                "speaker_raw"
            ],
            left[
                "speaker_canonical"
            ],
            left[
                "speaker_status"
            ],
        )

        right_identity = (
            right[
                "speaker_raw"
            ],
            right[
                "speaker_canonical"
            ],
            right[
                "speaker_status"
            ],
        )

        if (
            left_identity
            == right_identity
        ):
            continue

        register_focus(
            left[
                "utterance_index"
            ],
            "SPEAKER_TRANSITION_LEFT",
            transition_indices,
        )

        register_focus(
            right[
                "utterance_index"
            ],
            "SPEAKER_TRANSITION_RIGHT",
            transition_indices,
        )

        break


    baseline_indices = [
        1,
        (
            len(utterances)
            + 1
        ) // 2,
        len(utterances),
    ]

    baseline_labels = [
        "BASELINE_START",
        "BASELINE_MIDDLE",
        "BASELINE_END",
    ]

    for (
        utterance_index,
        label,
    ) in zip(
        baseline_indices,
        baseline_labels,
    ):
        register_focus(
            utterance_index,
            label,
            baseline_priority_indices,
        )


    selected_indices = []


    def select_indices(
        candidates,
        limit=4,
    ):
        for utterance_index in candidates:
            if (
                utterance_index
                in selected_indices
            ):
                continue

            if (
                len(
                    selected_indices
                )
                >= limit
            ):
                return

            selected_indices.append(
                utterance_index
            )


    select_indices(
        mandatory_indices
    )

    if (
        review_unit[
            "quality_candidate_count"
        ] > 0
    ):
        select_indices(
            quality_indices
        )

    select_indices(
        secondary_priority_indices
    )

    select_indices(
        transition_indices
    )

    select_indices(
        baseline_priority_indices
    )


    assert (
        1
        <= len(
            selected_indices
        )
        <= 4
    ), (
        "Invalid repaired human-review "
        "sample count."
    )


    if (
        required_chehelston_pair
        is not None
    ):
        left_index, right_index = (
            required_chehelston_pair
        )

        if not (
            left_index
            in selected_indices
            and right_index
            in selected_indices
        ):
            repaired_packet_chehelston_pair_failures.append({
                "review_unit_id": (
                    review_unit[
                        "review_unit_id"
                    ]
                ),
                "relative_path": (
                    relative_path
                ),
                "reason": (
                    "PAIR_NOT_BOTH_SELECTED"
                ),
                "required_pair": (
                    required_chehelston_pair
                ),
                "selected_indices": (
                    list(
                        selected_indices
                    )
                ),
            })
        else:
            left_item = (
                utterance_by_index[
                    left_index
                ]
            )

            right_item = (
                utterance_by_index[
                    right_index
                ]
            )

            pair_valid = (
                left_item[
                    "speaker_status"
                ]
                == "EXPLICIT_SOURCE_MARKER"
                and right_item[
                    "speaker_status"
                ]
                == "UNRESOLVED"
                and right_item[
                    "boundary_type"
                ]
                == (
                    "CHEHELSTON_UNLABELED_"
                    "PARAGRAPH_UNRESOLVED"
                )
                and right_item[
                    "speaker_raw"
                ] is None
                and right_item[
                    "speaker_canonical"
                ] is None
                and right_item[
                    "speaker_source_canonical"
                ] is None
            )

            if not pair_valid:
                repaired_packet_chehelston_pair_failures.append({
                    "review_unit_id": (
                        review_unit[
                            "review_unit_id"
                        ]
                    ),
                    "relative_path": (
                        relative_path
                    ),
                    "reason": (
                        "SELECTED_PAIR_FAILED_SPEAKER_SAFETY"
                    ),
                    "required_pair": (
                        required_chehelston_pair
                    ),
                })
            else:
                selected_chehelston_pair_count += 1


    source_file_path = (
        project_root
        / relative_path
    )

    source_lines = (
        source_file_path.read_text(
            encoding="utf-8"
        ).splitlines()
    )

    samples = []

    for (
        sample_index,
        utterance_index,
    ) in enumerate(
        selected_indices,
        start=1,
    ):
        item = utterance_by_index[
            utterance_index
        ]

        expected_source_text = "\n".join(
            source_lines[
                item[
                    "source_line_start"
                ] - 1:
                item[
                    "source_line_end"
                ]
            ]
        )

        if (
            expected_source_text
            != item[
                "source_text"
            ]
        ):
            repaired_packet_source_failures.append({
                "review_unit_id": (
                    review_unit[
                        "review_unit_id"
                    ]
                ),
                "utterance_id": (
                    item[
                        "utterance_id"
                    ]
                ),
                "relative_path": (
                    relative_path
                ),
            })


        related_quality_ids = []

        for quality_link in (
            file_quality_links
        ):
            linked_ids = {
                locator[
                    "utterance_id"
                ]
                for locator
                in quality_link[
                    "segment_locators"
                ]
            }

            if (
                item[
                    "utterance_id"
                ]
                in linked_ids
            ):
                related_quality_ids.append(
                    quality_link[
                        "quality_candidate_id"
                    ]
                )


        samples.append({
            "sample_index": (
                sample_index
            ),
            "review_focus": sorted(
                focus_by_index[
                    utterance_index
                ]
            ),
            "utterance_index": (
                utterance_index
            ),
            "utterance_id": (
                item[
                    "utterance_id"
                ]
            ),
            "source_line_start": (
                item[
                    "source_line_start"
                ]
            ),
            "source_line_end": (
                item[
                    "source_line_end"
                ]
            ),
            "speaker_raw": (
                item[
                    "speaker_raw"
                ]
            ),
            "speaker_canonical": (
                item[
                    "speaker_canonical"
                ]
            ),
            "speaker_source_canonical": (
                item[
                    "speaker_source_canonical"
                ]
            ),
            "speaker_status": (
                item[
                    "speaker_status"
                ]
            ),
            "boundary_type": (
                item[
                    "boundary_type"
                ]
            ),
            "merge_status": (
                item[
                    "merge_status"
                ]
            ),
            "merged_from_utterance_ids": (
                item[
                    "merged_from_utterance_ids"
                ]
            ),
            "related_quality_candidate_ids": (
                related_quality_ids
            ),
            "parsed_text": (
                item[
                    "text"
                ]
            ),
            "source_text": (
                item[
                    "source_text"
                ]
            ),
        })


    if (
        review_unit[
            "merged_segment_count"
        ] > 0
        and not any(
            sample[
                "merge_status"
            ] == "MERGED"
            for sample
            in samples
        )
    ):
        repaired_packet_merge_sampling_failures.append(
            review_unit[
                "review_unit_id"
            ]
        )


    if (
        review_unit[
            "quality_candidate_count"
        ] > 0
        and not any(
            sample[
                "related_quality_candidate_ids"
            ]
            for sample
            in samples
        )
    ):
        repaired_packet_quality_sampling_failures.append(
            review_unit[
                "review_unit_id"
            ]
        )


    human_review_packet_repaired_v1.append({
        "review_unit_id": (
            review_unit[
                "review_unit_id"
            ]
        ),
        "selection_index": (
            review_unit[
                "selection_index"
            ]
        ),
        "program": (
            review_unit[
                "program"
            ]
        ),
        "program_id": (
            review_unit[
                "program_id"
            ]
        ),
        "transcript_unit_id": (
            review_unit[
                "transcript_unit_id"
            ]
        ),
        "physical_file_id": (
            review_unit[
                "physical_file_id"
            ]
        ),
        "relative_path": (
            relative_path
        ),
        "segment_count": (
            len(
                utterances
            )
        ),
        "quality_candidate_count": (
            review_unit[
                "quality_candidate_count"
            ]
        ),
        "merged_segment_count": (
            review_unit[
                "merged_segment_count"
            ]
        ),
        "speaker_status_counts": (
            review_unit[
                "speaker_status_counts"
            ]
        ),
        "selection_reasons": (
            review_unit[
                "selection_reasons"
            ]
        ),
        "human_review_status": (
            "PENDING"
        ),
        "samples": (
            samples
        ),
    })


selected_repaired_explicit_chehelston_units = sum(
    item[
        "is_chehelston_repaired_explicit"
    ]
    for item
    in human_review_selection_repaired_v1
)


print(
    "Repaired human review units:",
    len(
        human_review_packet_repaired_v1
    ),
)
print(
    "Total repaired review samples:",
    sum(
        len(
            item[
                "samples"
            ]
        )
        for item
        in human_review_packet_repaired_v1
    ),
)
print(
    "Source verification failures:",
    len(
        repaired_packet_source_failures
    ),
)
print(
    "Unknown sample segment failures:",
    len(
        repaired_packet_unknown_segment_failures
    ),
)
print(
    "Merge sampling failures:",
    len(
        repaired_packet_merge_sampling_failures
    ),
)
print(
    "Quality sampling failures:",
    len(
        repaired_packet_quality_sampling_failures
    ),
)
print(
    "Selected repaired explicit Chehelston units:",
    selected_repaired_explicit_chehelston_units,
)
print(
    "Explicit-to-unresolved pairs fully sampled:",
    selected_chehelston_pair_count,
)
print(
    "Chehelston pair failures:",
    len(
        repaired_packet_chehelston_pair_failures
    ),
)


print()
print(
    "REPAIRED HUMAN REVIEW PACKET"
)
print("=" * 110)


for unit in (
    human_review_packet_repaired_v1
):
    print()
    print(
        "Review unit:",
        unit[
            "review_unit_id"
        ],
    )
    print(
        "Program:",
        unit[
            "program"
        ],
    )
    print(
        "Transcript unit:",
        unit[
            "transcript_unit_id"
        ],
    )
    print(
        "File:",
        unit[
            "relative_path"
        ],
    )
    print(
        "Human review status:",
        unit[
            "human_review_status"
        ],
    )

    for sample in unit[
        "samples"
    ]:
        print()
        print(
            "  Sample:",
            sample[
                "sample_index"
            ],
        )
        print(
            "  Focus:",
            sample[
                "review_focus"
            ],
        )
        print(
            "  Segment index:",
            sample[
                "utterance_index"
            ],
        )
        print(
            "  Source lines:",
            (
                sample[
                    "source_line_start"
                ],
                sample[
                    "source_line_end"
                ],
            ),
        )
        print(
            "  Speaker raw:",
            sample[
                "speaker_raw"
            ],
        )
        print(
            "  Speaker canonical:",
            sample[
                "speaker_canonical"
            ],
        )
        print(
            "  Speaker status:",
            sample[
                "speaker_status"
            ],
        )
        print(
            "  Boundary:",
            sample[
                "boundary_type"
            ],
        )
        print(
            "  Merge status:",
            sample[
                "merge_status"
            ],
        )
        print(
            "  Quality IDs:",
            sample[
                "related_quality_candidate_ids"
            ],
        )
        print(
            "  PARSED:",
            preview_text(
                sample[
                    "parsed_text"
                ]
            ),
        )
        print(
            "  SOURCE:",
            preview_text(
                sample[
                    "source_text"
                ]
            ),
        )
        print(
            "  " + "-" * 100
        )

    print("=" * 110)


repaired_packet_stream_state_after = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


assert len(
    human_review_packet_repaired_v1
) == 24, (
    "Repaired human review packet "
    "does not contain 24 units."
)

assert all(
    item[
        "human_review_status"
    ] == "PENDING"
    for item
    in human_review_packet_repaired_v1
), (
    "Human review status was assigned automatically."
)

assert all(
    1
    <= len(
        item[
            "samples"
        ]
    )
    <= 4
    for item
    in human_review_packet_repaired_v1
), (
    "Invalid human-review sample count."
)

assert not (
    repaired_packet_source_failures
), (
    "Repaired human review packet "
    "contains a Source mismatch."
)

assert not (
    repaired_packet_unknown_segment_failures
), (
    "Repaired human review packet "
    "references an unknown segment."
)

assert not (
    repaired_packet_merge_sampling_failures
), (
    "A selected merged unit has no merged sample."
)

assert not (
    repaired_packet_quality_sampling_failures
), (
    "A quality-flagged review unit "
    "has no quality sample."
)

assert not (
    repaired_packet_chehelston_pair_failures
), (
    "A repaired Chehelston boundary pair "
    "was not fully sampled safely."
)

assert (
    selected_chehelston_pair_count
    == selected_repaired_explicit_chehelston_units
), (
    "Not every selected repaired explicit "
    "Chehelston unit exposes both sides "
    "of an explicit-to-unresolved boundary."
)

assert {
    item[
        "program_id"
    ]
    for item
    in human_review_packet_repaired_v1
} == set(
    profiles_repaired_by_program
), (
    "Repaired human review packet "
    "lost program coverage."
)

assert (
    repaired_packet_stream_state_before
    == repaired_packet_stream_state_after
), (
    "Human review packet build mutated "
    "the repaired transcript stream."
)

print()
print(
    "REPAIRED HUMAN REVIEW PACKET BUILD: PASS"
)
print(
    "CHEHELSTON BOUNDARY PAIRS: FULLY EXPOSED"
)
print(
    "HUMAN REVIEW STATUS: PENDING"
)

Repaired human review units: 24
Total repaired review samples: 96
Source verification failures: 0
Unknown sample segment failures: 0
Merge sampling failures: 0
Quality sampling failures: 0
Selected repaired explicit Chehelston units: 2
Explicit-to-unresolved pairs fully sampled: 2
Chehelston pair failures: 0

REPAIRED HUMAN REVIEW PACKET

Review unit: transcript-review-repaired:001
Program: عصر شیرین
Transcript unit: program:asre-shirin:episode-001
File: data/transcripts/final/asre_shirin/season-1/episode-01.md
Human review status: PENDING

  Sample: 1
  Focus: ['QUALITY:REPEATED_NGRAM_RUN']
  Segment index: 64
  Source lines: (262, 264)
  Speaker raw: دکتر کمیل رودی
  Speaker canonical: None
  Speaker status: EXPLICIT_SOURCE_MARKER
  Boundary: BOLD_ONLY_SPEAKER_BOUNDARY
  Merge status: UNCHANGED
  Quality IDs: ['transcript-quality-repaired:000001']
  PARSED: که برای همه لازمه؛ برای همه. یعنی اهمیتش، اهمیت زندگیه. ما داریم زندگی می‌کنیم و این یک مهارت زندگیه، مثل مهارت ارتباط داشتن، مه

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ممیزی Timestamp-like Markerهای کشف‌شده در Source</h3>

<p>
Human Review Packet اصلاح‌شده اکنون از نظر ساختاری کامل است: هر ۲۴ Unit Sample معتبر دارد، Merge و Quality Coverage کامل‌اند و هر دو سمت مرز Repair‌شده چهلستون نیز داخل Packet نمایش داده شده‌اند.
</p>

<p>
با این حال Human Review همچنان <code>PENDING</code> است و قبل از ثبت نتیجه انسانی، یک Evidence جدید از Quality Scan باید تعیین تکلیف شود.
</p>

<p>
Quality Detection در میان Bracketed Markerها ۱۶ Marker با ظاهر Timestamp پیدا کرده است؛ مانند:
</p>

<pre>
[00:15:21 → 00:15:21]
</pre>

<p>
این Markerها در Timestamp Discovery اولیه شناسایی نشده بودند، زیرا Syntax آنها از نوع Bracket + Arrow است.
</p>

<h4>هدف این Cell</h4>

<p>
در این مرحله هنوز هیچ Timestamp به Segment نسبت داده نمی‌شود.
</p>

<p>
فقط بررسی می‌شود که این ۱۶ Marker دقیقاً در Source کجا قرار گرفته‌اند و از نظر ساختاری چه معنایی می‌توانند داشته باشند.
</p>

<h4>موارد مورد بررسی</h4>

<ul>
<li>Program و Source File هر Marker</li>
<li>Segment و Source Line مربوطه</li>
<li>صحت ساختار ساعت، دقیقه و ثانیه</li>
<li>Start و End موجود در خود Marker</li>
<li>Duration Marker</li>
<li>اینکه Marker یک Source Line مستقل است یا داخل یک Line قرار دارد</li>
<li>اینکه Marker در ابتدا، انتها یا میانه Source Range همان Segment قرار گرفته است</li>
<li>Line غیرخالی قبل و بعد از Marker</li>
<li>موقعیت Marker در Parsed Text</li>
</ul>

<h4>اصل محافظه‌کاری</h4>

<p>
وجود یک Timestamp-like Marker به‌تنهایی به معنی Timestamp کل Segment نیست.
</p>

<p>
برای مثال اگر Marker در میانه یک Segment چندخطی قرار گرفته باشد و فقط یک بازه صفر یا یک‌ثانیه‌ای را نشان دهد، نمی‌توان بدون Evidence آن را به <code>start_time</code> یا <code>end_time</code> کل Segment تبدیل کرد.
</p>

<p>
بنابراین این Cell فقط Discovery و Structural Classification انجام می‌دهد.
</p>

<h4>هیچ تغییری انجام نمی‌شود</h4>

<ul>
<li>Timestamp assignment انجام نمی‌شود.</li>
<li>Bracketed Marker حذف نمی‌شود.</li>
<li>Transcript اصلاح نمی‌شود.</li>
<li>ASR correction انجام نمی‌شود.</li>
<li>Persian normalization انجام نمی‌شود.</li>
<li>Deduplication انجام نمی‌شود.</li>
<li>Chunking انجام نمی‌شود.</li>
<li>Artifact write انجام نمی‌شود.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>تمام Timestamp-shaped Markerهای Quality Scan دوباره از Source File پیدا شوند.</li>
<li>هیچ Marker گم‌شده از Source وجود نداشته باشد.</li>
<li>تمام Minute و Secondها در بازه معتبر باشند.</li>
<li>End قبل از Start نباشد.</li>
<li>موقعیت Marker نسبت به Segment دقیقاً ثبت شود.</li>
<li>Stream اصلاح‌شده هیچ تغییری نکند.</li>
<li><code>start_time</code> و <code>end_time</code> همچنان بدون استنتاج باقی بمانند.</li>
</ul>

</div>

In [39]:
# Code Cell 39

timestamp_arrow_marker_pattern = re.compile(
    r"^\["
    r"(\d{2}):(\d{2}):(\d{2})"
    r"\s*→\s*"
    r"(\d{2}):(\d{2}):(\d{2})"
    r"\]$"
)

timestamp_marker_source_cache = {}

timestamp_like_marker_audit_v1 = []

timestamp_marker_source_failures = []
timestamp_marker_time_failures = []

repaired_stream_before_timestamp_audit = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


def clock_parts_to_seconds(
    hour,
    minute,
    second,
):
    return (
        hour * 3600
        + minute * 60
        + second
    )


def nearest_nonempty_line(
    lines,
    start_index,
    direction,
):
    index = (
        start_index
        + direction
    )

    while (
        0
        <= index
        < len(lines)
    ):
        if lines[
            index
        ].strip():
            return {
                "line_number": (
                    index + 1
                ),
                "text": (
                    lines[index]
                ),
            }

        index += direction

    return None


for candidate in (
    transcript_quality_candidates_repaired_v1
):
    if (
        candidate[
            "candidate_type"
        ]
        != "BRACKETED_TRANSCRIPTION_MARKER"
    ):
        continue

    for marker in candidate[
        "evidence"
    ]["markers"]:
        match = (
            timestamp_arrow_marker_pattern.fullmatch(
                marker
            )
        )

        if match is None:
            continue

        (
            start_hour,
            start_minute,
            start_second,
            end_hour,
            end_minute,
            end_second,
        ) = [
            int(value)
            for value
            in match.groups()
        ]

        start_clock_valid = (
            0
            <= start_minute
            <= 59
            and 0
            <= start_second
            <= 59
        )

        end_clock_valid = (
            0
            <= end_minute
            <= 59
            and 0
            <= end_second
            <= 59
        )

        start_total_seconds = (
            clock_parts_to_seconds(
                start_hour,
                start_minute,
                start_second,
            )
        )

        end_total_seconds = (
            clock_parts_to_seconds(
                end_hour,
                end_minute,
                end_second,
            )
        )

        temporal_order_valid = (
            end_total_seconds
            >= start_total_seconds
        )

        if not (
            start_clock_valid
            and end_clock_valid
            and temporal_order_valid
        ):
            timestamp_marker_time_failures.append({
                "quality_candidate_id": (
                    candidate[
                        "quality_candidate_id"
                    ]
                ),
                "marker": marker,
                "start_clock_valid": (
                    start_clock_valid
                ),
                "end_clock_valid": (
                    end_clock_valid
                ),
                "temporal_order_valid": (
                    temporal_order_valid
                ),
            })

        utterance_id = candidate[
            "utterance_ids"
        ][0]

        utterance = (
            repaired_merged_utterance_by_id[
                utterance_id
            ]
        )

        relative_path = candidate[
            "relative_path"
        ]

        if (
            relative_path
            not in timestamp_marker_source_cache
        ):
            timestamp_marker_source_cache[
                relative_path
            ] = (
                project_root
                / relative_path
            ).read_text(
                encoding="utf-8"
            ).splitlines()

        source_lines = (
            timestamp_marker_source_cache[
                relative_path
            ]
        )

        source_line_start = (
            utterance[
                "source_line_start"
            ]
        )

        source_line_end = (
            utterance[
                "source_line_end"
            ]
        )

        occurrence_records = []

        for line_number in range(
            source_line_start,
            source_line_end + 1,
        ):
            raw_line = source_lines[
                line_number - 1
            ]

            if marker not in raw_line:
                continue

            if (
                raw_line.strip()
                == marker
            ):
                placement = (
                    "STANDALONE_SOURCE_LINE"
                )
            else:
                placement = (
                    "INLINE_SOURCE_MARKER"
                )

            previous_nonempty = (
                nearest_nonempty_line(
                    source_lines,
                    line_number - 1,
                    -1,
                )
            )

            next_nonempty = (
                nearest_nonempty_line(
                    source_lines,
                    line_number - 1,
                    1,
                )
            )

            nonempty_segment_lines = [
                candidate_line_number
                for candidate_line_number
                in range(
                    source_line_start,
                    source_line_end + 1,
                )
                if source_lines[
                    candidate_line_number - 1
                ].strip()
            ]

            if (
                len(
                    nonempty_segment_lines
                ) == 1
            ):
                segment_position = (
                    "ONLY_NONEMPTY_LINE_IN_SEGMENT"
                )

            elif (
                line_number
                == nonempty_segment_lines[
                    0
                ]
            ):
                segment_position = (
                    "FIRST_NONEMPTY_LINE_IN_SEGMENT"
                )

            elif (
                line_number
                == nonempty_segment_lines[
                    -1
                ]
            ):
                segment_position = (
                    "LAST_NONEMPTY_LINE_IN_SEGMENT"
                )

            else:
                segment_position = (
                    "INTERIOR_NONEMPTY_LINE_IN_SEGMENT"
                )

            occurrence_records.append({
                "source_line_number": (
                    line_number
                ),
                "placement": (
                    placement
                ),
                "segment_position": (
                    segment_position
                ),
                "raw_line": (
                    raw_line
                ),
                "previous_nonempty": (
                    previous_nonempty
                ),
                "next_nonempty": (
                    next_nonempty
                ),
            })

        if not occurrence_records:
            timestamp_marker_source_failures.append({
                "quality_candidate_id": (
                    candidate[
                        "quality_candidate_id"
                    ]
                ),
                "relative_path": (
                    relative_path
                ),
                "utterance_id": (
                    utterance_id
                ),
                "marker": (
                    marker
                ),
                "source_line_start": (
                    source_line_start
                ),
                "source_line_end": (
                    source_line_end
                ),
            })

        parsed_text_marker_index = (
            utterance[
                "text"
            ].find(
                marker
            )
        )

        if (
            parsed_text_marker_index
            < 0
        ):
            parsed_text_position = (
                "NOT_FOUND_IN_PARSED_TEXT"
            )

        elif (
            utterance[
                "text"
            ].strip()
            == marker
        ):
            parsed_text_position = (
                "ONLY_PARSED_TEXT"
            )

        elif (
            parsed_text_marker_index
            == 0
        ):
            parsed_text_position = (
                "START_OF_PARSED_TEXT"
            )

        elif (
            parsed_text_marker_index
            + len(marker)
            == len(
                utterance[
                    "text"
                ]
            )
        ):
            parsed_text_position = (
                "END_OF_PARSED_TEXT"
            )

        else:
            parsed_text_position = (
                "INTERIOR_OF_PARSED_TEXT"
            )

        timestamp_like_marker_audit_v1.append({
            "quality_candidate_id": (
                candidate[
                    "quality_candidate_id"
                ]
            ),
            "marker": (
                marker
            ),
            "program": (
                candidate[
                    "program"
                ]
            ),
            "program_id": (
                candidate[
                    "program_id"
                ]
            ),
            "relative_path": (
                relative_path
            ),
            "utterance_id": (
                utterance_id
            ),
            "utterance_index": (
                utterance[
                    "utterance_index"
                ]
            ),
            "source_line_start": (
                source_line_start
            ),
            "source_line_end": (
                source_line_end
            ),
            "speaker_raw": (
                utterance[
                    "speaker_raw"
                ]
            ),
            "speaker_canonical": (
                utterance[
                    "speaker_canonical"
                ]
            ),
            "speaker_status": (
                utterance[
                    "speaker_status"
                ]
            ),
            "boundary_type": (
                utterance[
                    "boundary_type"
                ]
            ),
            "start_seconds": (
                start_total_seconds
            ),
            "end_seconds": (
                end_total_seconds
            ),
            "duration_seconds": (
                end_total_seconds
                - start_total_seconds
            ),
            "start_clock_valid": (
                start_clock_valid
            ),
            "end_clock_valid": (
                end_clock_valid
            ),
            "temporal_order_valid": (
                temporal_order_valid
            ),
            "parsed_text_position": (
                parsed_text_position
            ),
            "source_occurrence_count": (
                len(
                    occurrence_records
                )
            ),
            "source_occurrences": (
                occurrence_records
            ),
        })


timestamp_marker_program_counts = Counter(
    item[
        "program_id"
    ]
    for item
    in timestamp_like_marker_audit_v1
)

timestamp_marker_placement_counts = Counter()

timestamp_marker_segment_position_counts = Counter()

timestamp_marker_parsed_position_counts = Counter(
    item[
        "parsed_text_position"
    ]
    for item
    in timestamp_like_marker_audit_v1
)

timestamp_marker_duration_counts = Counter(
    item[
        "duration_seconds"
    ]
    for item
    in timestamp_like_marker_audit_v1
)


for item in (
    timestamp_like_marker_audit_v1
):
    for occurrence in item[
        "source_occurrences"
    ]:
        timestamp_marker_placement_counts[
            occurrence[
                "placement"
            ]
        ] += 1

        timestamp_marker_segment_position_counts[
            occurrence[
                "segment_position"
            ]
        ] += 1


print(
    "Bracketed quality candidates:",
    sum(
        1
        for item
        in transcript_quality_candidates_repaired_v1
        if item[
            "candidate_type"
        ]
        == "BRACKETED_TRANSCRIPTION_MARKER"
    ),
)
print(
    "Timestamp-shaped markers:",
    len(
        timestamp_like_marker_audit_v1
    ),
)
print(
    "Timestamp Source lookup failures:",
    len(
        timestamp_marker_source_failures
    ),
)
print(
    "Timestamp clock/order failures:",
    len(
        timestamp_marker_time_failures
    ),
)


print()
print("TIMESTAMP MARKERS BY PROGRAM")
print("=" * 100)

for program_id, count in sorted(
    timestamp_marker_program_counts.items()
):
    program_name = next(
        item[
            "program"
        ]
        for item
        in timestamp_like_marker_audit_v1
        if item[
            "program_id"
        ]
        == program_id
    )

    print(
        f"- {program_name} "
        f"({program_id}): {count}"
    )


print()
print("SOURCE PLACEMENT")
print("=" * 100)

for placement, count in sorted(
    timestamp_marker_placement_counts.items()
):
    print(
        f"- {placement}: {count}"
    )


print()
print("POSITION INSIDE SEGMENT SOURCE RANGE")
print("=" * 100)

for position, count in sorted(
    timestamp_marker_segment_position_counts.items()
):
    print(
        f"- {position}: {count}"
    )


print()
print("POSITION INSIDE PARSED TEXT")
print("=" * 100)

for position, count in sorted(
    timestamp_marker_parsed_position_counts.items()
):
    print(
        f"- {position}: {count}"
    )


print()
print("MARKER DURATION DISTRIBUTION")
print("=" * 100)

for duration, count in sorted(
    timestamp_marker_duration_counts.items()
):
    print(
        f"- {duration} second(s): {count}"
    )


print()
print("DETAILED TIMESTAMP-LIKE MARKER AUDIT")
print("=" * 110)

for item in (
    timestamp_like_marker_audit_v1
):
    print()
    print(
        "Candidate ID:",
        item[
            "quality_candidate_id"
        ],
    )
    print(
        "Marker:",
        item[
            "marker"
        ],
    )
    print(
        "Program:",
        item[
            "program"
        ],
    )
    print(
        "File:",
        item[
            "relative_path"
        ],
    )
    print(
        "Segment index:",
        item[
            "utterance_index"
        ],
    )
    print(
        "Segment Source range:",
        (
            item[
                "source_line_start"
            ],
            item[
                "source_line_end"
            ],
        ),
    )
    print(
        "Speaker raw:",
        item[
            "speaker_raw"
        ],
    )
    print(
        "Speaker status:",
        item[
            "speaker_status"
        ],
    )
    print(
        "Boundary:",
        item[
            "boundary_type"
        ],
    )
    print(
        "Start seconds:",
        item[
            "start_seconds"
        ],
    )
    print(
        "End seconds:",
        item[
            "end_seconds"
        ],
    )
    print(
        "Duration seconds:",
        item[
            "duration_seconds"
        ],
    )
    print(
        "Parsed-text position:",
        item[
            "parsed_text_position"
        ],
    )
    print(
        "Source occurrence count:",
        item[
            "source_occurrence_count"
        ],
    )

    for occurrence in item[
        "source_occurrences"
    ]:
        print()
        print(
            "  Marker Source line:",
            occurrence[
                "source_line_number"
            ],
        )
        print(
            "  Placement:",
            occurrence[
                "placement"
            ],
        )
        print(
            "  Segment position:",
            occurrence[
                "segment_position"
            ],
        )
        print(
            "  Raw marker line:",
            repr(
                occurrence[
                    "raw_line"
                ]
            ),
        )

        previous_nonempty = (
            occurrence[
                "previous_nonempty"
            ]
        )

        next_nonempty = (
            occurrence[
                "next_nonempty"
            ]
        )

        if (
            previous_nonempty
            is not None
        ):
            print(
                "  Previous non-empty:",
                (
                    previous_nonempty[
                        "line_number"
                    ],
                    preview_text(
                        previous_nonempty[
                            "text"
                        ],
                        limit=350,
                    ),
                ),
            )
        else:
            print(
                "  Previous non-empty:",
                None,
            )

        if (
            next_nonempty
            is not None
        ):
            print(
                "  Next non-empty:",
                (
                    next_nonempty[
                        "line_number"
                    ],
                    preview_text(
                        next_nonempty[
                            "text"
                        ],
                        limit=350,
                    ),
                ),
            )
        else:
            print(
                "  Next non-empty:",
                None,
            )

    print("-" * 110)


repaired_stream_after_timestamp_audit = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


assert len(
    timestamp_like_marker_audit_v1
) == 16, (
    "Expected 16 timestamp-shaped "
    "bracketed markers from the repaired quality scan."
)

assert not (
    timestamp_marker_source_failures
), (
    "At least one timestamp-like marker "
    "could not be found in its Source range."
)

assert not (
    timestamp_marker_time_failures
), (
    "At least one timestamp-like marker "
    "has invalid clock values or temporal order."
)

assert all(
    item[
        "source_occurrence_count"
    ] >= 1
    for item
    in timestamp_like_marker_audit_v1
), (
    "A timestamp-like marker has no "
    "verified Source occurrence."
)

assert all(
    item[
        "parsed_text_position"
    ]
    != "NOT_FOUND_IN_PARSED_TEXT"
    for item
    in timestamp_like_marker_audit_v1
), (
    "A timestamp-like marker disappeared "
    "from parsed text."
)

assert all(
    item[
        "start_time"
    ] is None
    and item[
        "end_time"
    ] is None
    for item
    in all_merged_utterances_repaired_v1
), (
    "A segment timestamp was assigned "
    "before timestamp semantics were resolved."
)

assert (
    repaired_stream_before_timestamp_audit
    == repaired_stream_after_timestamp_audit
), (
    "Timestamp diagnostic mutated "
    "the repaired transcript stream."
)

print()
print(
    "TIMESTAMP-LIKE BRACKET MARKER "
    "STRUCTURAL AUDIT: PASS"
)
print(
    "SEGMENT TIMESTAMP ASSIGNMENT: NOT PERFORMED"
)

Bracketed quality candidates: 53
Timestamp-shaped markers: 16
Timestamp Source lookup failures: 0
Timestamp clock/order failures: 0

TIMESTAMP MARKERS BY PROGRAM
- یونی تهران (program:uni-tehran): 16

SOURCE PLACEMENT
- STANDALONE_SOURCE_LINE: 16

POSITION INSIDE SEGMENT SOURCE RANGE
- INTERIOR_NONEMPTY_LINE_IN_SEGMENT: 16

POSITION INSIDE PARSED TEXT
- INTERIOR_OF_PARSED_TEXT: 16

MARKER DURATION DISTRIBUTION
- 0 second(s): 14
- 1 second(s): 2

DETAILED TIMESTAMP-LIKE MARKER AUDIT

Candidate ID: transcript-quality-repaired:000089
Marker: [00:05:00 → 00:05:00]
Program: یونی تهران
File: data/transcripts/final/uni-tehran/episode-11.md
Segment index: 12
Segment Source range: (67, 73)
Speaker raw: مجری
Speaker status: EXPLICIT_SOURCE_MARKER
Boundary: BOLD_ONLY_SPEAKER_BOUNDARY
Start seconds: 300
End seconds: 300
Duration seconds: 0
Parsed-text position: INTERIOR_OF_PARSED_TEXT
Source occurrence count: 1

  Marker Source line: 71
  Placement: STANDALONE_SOURCE_LINE
  Segment position: INTER

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تعیین Semantics نهایی Timestamp و حفظ آنها به‌عنوان Source Annotation</h3>

<p>
ممیزی Cell قبل با موفقیت نشان داد که هر ۱۶ Timestamp-shaped Marker:
</p>

<ul>
<li>فقط در Program «یونی تهران» دیده شده‌اند.</li>
<li>دقیقاً در Source وجود دارند.</li>
<li>همگی روی Line مستقل قرار گرفته‌اند.</li>
<li>همگی در میانه Source Range یک Segment قرار دارند.</li>
<li>همگی در میانه Parsed Text همان Segment قرار دارند.</li>
<li>۱۴ مورد Duration صفرثانیه‌ای و ۲ مورد Duration یک‌ثانیه‌ای دارند.</li>
<li>هیچ Clock یا Temporal-order نامعتبری ندارند.</li>
</ul>

<h4>نتیجه Semantics</h4>

<p>
این Evidence نشان می‌دهد که Markerها Timestamp واقعی موجود در Source هستند، اما Evidence کافی برای این وجود ندارد که Start و End کل Segment باشند.
</p>

<p>
برای مثال Segment ممکن است چندین Line گفتار داشته باشد و Marker زیر در وسط آن ظاهر شود:
</p>

<pre>
متن قبل

[00:15:21 → 00:15:21]

متن بعد
</pre>

<p>
نسبت‌دادن <code>00:15:21</code> به Start یا End کل این Segment، اطلاعاتی را به Source اضافه می‌کند که Source صریحاً بیان نکرده است.
</p>

<h4>Policy نهایی NB06</h4>

<p>
این Markerها با دو سطح اطلاعاتی جدا حفظ می‌شوند:
</p>

<ul>
<li><strong>Segment Timestamp:</strong> همچنان <code>None</code> باقی می‌ماند.</li>
<li><strong>Source Timestamp Annotation:</strong> Marker خام، Source Line و مقدار Start/End داخل خود Marker به‌طور کامل حفظ می‌شوند.</li>
</ul>

<p>
بنابراین مفهوم «preserve real timestamps» رعایت می‌شود، بدون اینکه Timestamp محلی Source به‌اشتباه Timestamp کل Segment تلقی شود.
</p>

<h4>خروجی ساختاری</h4>

<p>
برای هر Marker یک Annotation مستقل در:
</p>

<p>
<code>timestamp_annotations_repaired_v1</code>
</p>

<p>
ساخته می‌شود.
</p>

<p>
هر Annotation شامل موارد زیر است:
</p>

<ul>
<li>Source Program و File</li>
<li>Utterance و Segment</li>
<li>Source Line دقیق</li>
<li>Raw Marker بدون تغییر</li>
<li>Start و End داخل Marker بر حسب ثانیه</li>
<li>Duration</li>
<li>Source Placement</li>
<li>Segment Position</li>
<li>Parsed-text Position</li>
<li>Policy مربوط به عدم تخصیص آن به کل Segment</li>
</ul>

<h4>Quality Candidate</h4>

<p>
Timestamp Markerها از Quality Candidateها حذف نمی‌شوند.
</p>

<p>
وجود آنها همچنان به‌عنوان <code>BRACKETED_TRANSCRIPTION_MARKER</code> قابل Audit باقی می‌ماند، اما در Metadata جدید روشن می‌شود که Timestamp Evidence آنها نیز جداگانه حفظ شده است.
</p>

<h4>اصل عدم تغییر</h4>

<p>
در این Cell هیچ متن، Speaker، Merge، Locator یا Source File تغییر نمی‌کند.
</p>

<p>
همچنین:
</p>

<ul>
<li>Persian normalization انجام نمی‌شود.</li>
<li>ASR correction انجام نمی‌شود.</li>
<li>Deduplication انجام نمی‌شود.</li>
<li>Chunking انجام نمی‌شود.</li>
<li>Artifact write انجام نمی‌شود.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>هر ۱۶ Marker دقیقاً یک Annotation مستقل داشته باشد.</li>
<li>تمام Annotation IDها یکتا باشند.</li>
<li>تمام Markerها Source-backed باشند.</li>
<li>تمام Markerها همان Semantics مشاهده‌شده در Cell قبل را داشته باشند: standalone، interior و Duration حداکثر یک ثانیه.</li>
<li>هیچ Segment Timestamp از این Markerها استنتاج نشود.</li>
<li>تمام <code>start_time</code> و <code>end_time</code>های Segment همچنان <code>None</code> باقی بمانند.</li>
<li>Stream اصلاح‌شده هیچ تغییری نکند.</li>
</ul>

</div>

In [40]:
# Code Cell 40

timestamp_resolution_stream_before = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)

timestamp_annotations_repaired_v1 = []

timestamp_annotation_semantic_failures = []


for annotation_index, item in enumerate(
    timestamp_like_marker_audit_v1,
    start=1,
):
    if (
        item["source_occurrence_count"]
        != 1
    ):
        timestamp_annotation_semantic_failures.append({
            "quality_candidate_id": (
                item[
                    "quality_candidate_id"
                ]
            ),
            "reason": (
                "SOURCE_OCCURRENCE_COUNT_NOT_ONE"
            ),
            "value": (
                item[
                    "source_occurrence_count"
                ]
            ),
        })
        continue

    occurrence = item[
        "source_occurrences"
    ][0]

    semantic_checks = {
        "standalone_source_line": (
            occurrence[
                "placement"
            ]
            == "STANDALONE_SOURCE_LINE"
        ),
        "interior_source_position": (
            occurrence[
                "segment_position"
            ]
            == (
                "INTERIOR_NONEMPTY_LINE_IN_SEGMENT"
            )
        ),
        "interior_parsed_position": (
            item[
                "parsed_text_position"
            ]
            == "INTERIOR_OF_PARSED_TEXT"
        ),
        "micro_range_duration": (
            item[
                "duration_seconds"
            ]
            in {
                0,
                1,
            }
        ),
        "valid_start_clock": (
            item[
                "start_clock_valid"
            ]
        ),
        "valid_end_clock": (
            item[
                "end_clock_valid"
            ]
        ),
        "valid_temporal_order": (
            item[
                "temporal_order_valid"
            ]
        ),
    }

    failed_checks = [
        check_name
        for check_name, passed
        in semantic_checks.items()
        if not passed
    ]

    if failed_checks:
        timestamp_annotation_semantic_failures.append({
            "quality_candidate_id": (
                item[
                    "quality_candidate_id"
                ]
            ),
            "marker": (
                item["marker"]
            ),
            "failed_checks": (
                failed_checks
            ),
        })
        continue

    timestamp_annotations_repaired_v1.append({
        "timestamp_annotation_id": (
            f"source-timestamp-annotation:"
            f"{annotation_index:04d}"
        ),
        "annotation_type": (
            "SOURCE_TIMESTAMP_ANNOTATION"
        ),
        "annotation_scope": (
            "LOCAL_POINT_OR_MICRO_RANGE"
        ),
        "program": (
            item[
                "program"
            ]
        ),
        "program_id": (
            item[
                "program_id"
            ]
        ),
        "relative_path": (
            item[
                "relative_path"
            ]
        ),
        "utterance_id": (
            item[
                "utterance_id"
            ]
        ),
        "utterance_index": (
            item[
                "utterance_index"
            ]
        ),
        "quality_candidate_id": (
            item[
                "quality_candidate_id"
            ]
        ),
        "source_line_number": (
            occurrence[
                "source_line_number"
            ]
        ),
        "source_line_start": (
            item[
                "source_line_start"
            ]
        ),
        "source_line_end": (
            item[
                "source_line_end"
            ]
        ),
        "raw_marker": (
            item[
                "marker"
            ]
        ),
        "start_seconds": (
            item[
                "start_seconds"
            ]
        ),
        "end_seconds": (
            item[
                "end_seconds"
            ]
        ),
        "duration_seconds": (
            item[
                "duration_seconds"
            ]
        ),
        "source_placement": (
            occurrence[
                "placement"
            ]
        ),
        "segment_source_position": (
            occurrence[
                "segment_position"
            ]
        ),
        "parsed_text_position": (
            item[
                "parsed_text_position"
            ]
        ),
        "speaker_raw": (
            item[
                "speaker_raw"
            ]
        ),
        "speaker_canonical": (
            item[
                "speaker_canonical"
            ]
        ),
        "speaker_status": (
            item[
                "speaker_status"
            ]
        ),
        "boundary_type": (
            item[
                "boundary_type"
            ]
        ),
        "segment_timestamp_assignment": (
            "NOT_ASSIGNED"
        ),
        "segment_timestamp_reason": (
            "SOURCE_MARKER_IS_INTERIOR_"
            "LOCAL_POINT_OR_MICRO_RANGE"
        ),
        "raw_source_preserved": True,
    })


timestamp_annotations_by_utterance_id = defaultdict(
    list
)

for annotation in (
    timestamp_annotations_repaired_v1
):
    timestamp_annotations_by_utterance_id[
        annotation[
            "utterance_id"
        ]
    ].append(
        annotation
    )


timestamp_semantics_resolution_repaired_v1 = {
    "source_marker_count": (
        len(
            timestamp_like_marker_audit_v1
        )
    ),
    "preserved_annotation_count": (
        len(
            timestamp_annotations_repaired_v1
        )
    ),
    "program_ids": sorted({
        item[
            "program_id"
        ]
        for item
        in timestamp_annotations_repaired_v1
    }),
    "source_marker_semantics": (
        "SOURCE_TIMESTAMP_ANNOTATION"
    ),
    "segment_timestamp_policy": (
        "DO_NOT_ASSIGN_INTERIOR_LOCAL_"
        "POINT_OR_MICRO_RANGE_TO_WHOLE_SEGMENT"
    ),
    "missing_segment_timestamp_value": (
        None
    ),
    "raw_marker_policy": (
        "PRESERVE_EXACT_SOURCE_TEXT"
    ),
}


annotation_id_counts = Counter(
    item[
        "timestamp_annotation_id"
    ]
    for item
    in timestamp_annotations_repaired_v1
)

duplicate_timestamp_annotation_ids = [
    annotation_id
    for annotation_id, count
    in annotation_id_counts.items()
    if count > 1
]


timestamp_annotation_program_counts = Counter(
    item[
        "program_id"
    ]
    for item
    in timestamp_annotations_repaired_v1
)

timestamp_annotation_file_counts = Counter(
    item[
        "relative_path"
    ]
    for item
    in timestamp_annotations_repaired_v1
)

timestamp_annotation_duration_counts = Counter(
    item[
        "duration_seconds"
    ]
    for item
    in timestamp_annotations_repaired_v1
)


print(
    "Timestamp-shaped Source markers:",
    len(
        timestamp_like_marker_audit_v1
    ),
)
print(
    "Preserved timestamp annotations:",
    len(
        timestamp_annotations_repaired_v1
    ),
)
print(
    "Semantic failures:",
    len(
        timestamp_annotation_semantic_failures
    ),
)
print(
    "Duplicate annotation IDs:",
    len(
        duplicate_timestamp_annotation_ids
    ),
)


print()
print("TIMESTAMP SEMANTICS RESOLUTION")
print("=" * 100)

for key, value in (
    timestamp_semantics_resolution_repaired_v1.items()
):
    print(
        f"- {key}: {value}"
    )


print()
print("ANNOTATIONS BY PROGRAM")
print("=" * 100)

for program_id, count in sorted(
    timestamp_annotation_program_counts.items()
):
    program_name = next(
        item[
            "program"
        ]
        for item
        in timestamp_annotations_repaired_v1
        if item[
            "program_id"
        ]
        == program_id
    )

    print(
        f"- {program_name} "
        f"({program_id}): {count}"
    )


print()
print("ANNOTATIONS BY SOURCE FILE")
print("=" * 100)

for relative_path, count in sorted(
    timestamp_annotation_file_counts.items()
):
    print(
        f"- {relative_path}: {count}"
    )


print()
print("ANNOTATION DURATION DISTRIBUTION")
print("=" * 100)

for duration, count in sorted(
    timestamp_annotation_duration_counts.items()
):
    print(
        f"- {duration} second(s): {count}"
    )


print()
print("PRESERVED SOURCE TIMESTAMP ANNOTATIONS")
print("=" * 110)

for annotation in (
    timestamp_annotations_repaired_v1
):
    print()
    print(
        "Annotation ID:",
        annotation[
            "timestamp_annotation_id"
        ],
    )
    print(
        "Program:",
        annotation[
            "program"
        ],
    )
    print(
        "File:",
        annotation[
            "relative_path"
        ],
    )
    print(
        "Utterance:",
        annotation[
            "utterance_id"
        ],
    )
    print(
        "Segment index:",
        annotation[
            "utterance_index"
        ],
    )
    print(
        "Source line:",
        annotation[
            "source_line_number"
        ],
    )
    print(
        "Raw marker:",
        annotation[
            "raw_marker"
        ],
    )
    print(
        "Marker seconds:",
        (
            annotation[
                "start_seconds"
            ],
            annotation[
                "end_seconds"
            ],
        ),
    )
    print(
        "Duration:",
        annotation[
            "duration_seconds"
        ],
    )
    print(
        "Source placement:",
        annotation[
            "source_placement"
        ],
    )
    print(
        "Segment source position:",
        annotation[
            "segment_source_position"
        ],
    )
    print(
        "Parsed text position:",
        annotation[
            "parsed_text_position"
        ],
    )
    print(
        "Segment timestamp assignment:",
        annotation[
            "segment_timestamp_assignment"
        ],
    )
    print(
        "Reason:",
        annotation[
            "segment_timestamp_reason"
        ],
    )
    print("-" * 110)


segments_with_timestamp_annotations = {
    item[
        "utterance_id"
    ]
    for item
    in timestamp_annotations_repaired_v1
}


segment_timestamp_assignment_failures = [
    item[
        "utterance_id"
    ]
    for item
    in all_merged_utterances_repaired_v1
    if (
        item[
            "utterance_id"
        ]
        in segments_with_timestamp_annotations
        and (
            item[
                "start_time"
            ] is not None
            or item[
                "end_time"
            ] is not None
        )
    )
]


timestamp_resolution_stream_after = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


assert len(
    timestamp_like_marker_audit_v1
) == 16, (
    "Timestamp audit population changed."
)

assert len(
    timestamp_annotations_repaired_v1
) == 16, (
    "Not every timestamp-shaped Source marker "
    "was preserved as an annotation."
)

assert not (
    timestamp_annotation_semantic_failures
), (
    "Timestamp marker semantics differ "
    "from the audited structural pattern."
)

assert not (
    duplicate_timestamp_annotation_ids
), (
    "Timestamp annotation IDs are not unique."
)

assert all(
    item[
        "source_placement"
    ]
    == "STANDALONE_SOURCE_LINE"
    for item
    in timestamp_annotations_repaired_v1
), (
    "A preserved timestamp annotation "
    "is not a standalone Source line."
)

assert all(
    item[
        "segment_source_position"
    ]
    == "INTERIOR_NONEMPTY_LINE_IN_SEGMENT"
    for item
    in timestamp_annotations_repaired_v1
), (
    "A preserved timestamp annotation "
    "is not interior to its Segment Source range."
)

assert all(
    item[
        "parsed_text_position"
    ]
    == "INTERIOR_OF_PARSED_TEXT"
    for item
    in timestamp_annotations_repaired_v1
), (
    "A preserved timestamp annotation "
    "is not interior to Parsed Text."
)

assert all(
    item[
        "duration_seconds"
    ]
    in {
        0,
        1,
    }
    for item
    in timestamp_annotations_repaired_v1
), (
    "Unexpected timestamp annotation duration."
)

assert all(
    item[
        "segment_timestamp_assignment"
    ]
    == "NOT_ASSIGNED"
    for item
    in timestamp_annotations_repaired_v1
), (
    "A local timestamp annotation "
    "was assigned to a whole Segment."
)

assert not (
    segment_timestamp_assignment_failures
), (
    "A Segment containing a local Source timestamp "
    "received an inferred start or end time."
)

assert all(
    item[
        "start_time"
    ] is None
    and item[
        "end_time"
    ] is None
    for item
    in all_merged_utterances_repaired_v1
), (
    "Unexpected Segment timestamp exists "
    "in the repaired stream."
)

assert (
    timestamp_resolution_stream_before
    == timestamp_resolution_stream_after
), (
    "Timestamp semantics resolution "
    "mutated the repaired transcript stream."
)

print()
print(
    "SOURCE TIMESTAMP ANNOTATION "
    "PRESERVATION: PASS"
)
print(
    "SEGMENT TIMESTAMP POLICY: "
    "NONE UNLESS SOURCE PROVIDES "
    "SEGMENT-LEVEL EVIDENCE"
)

Timestamp-shaped Source markers: 16
Preserved timestamp annotations: 16
Semantic failures: 0
Duplicate annotation IDs: 0

TIMESTAMP SEMANTICS RESOLUTION
- source_marker_count: 16
- preserved_annotation_count: 16
- program_ids: ['program:uni-tehran']
- source_marker_semantics: SOURCE_TIMESTAMP_ANNOTATION
- segment_timestamp_policy: DO_NOT_ASSIGN_INTERIOR_LOCAL_POINT_OR_MICRO_RANGE_TO_WHOLE_SEGMENT
- missing_segment_timestamp_value: None
- raw_marker_policy: PRESERVE_EXACT_SOURCE_TEXT

ANNOTATIONS BY PROGRAM
- یونی تهران (program:uni-tehran): 16

ANNOTATIONS BY SOURCE FILE
- data/transcripts/final/uni-tehran/episode-11.md: 2
- data/transcripts/final/uni-tehran/episode-12.md: 2
- data/transcripts/final/uni-tehran/episode-15.md: 2
- data/transcripts/final/uni-tehran/episode-16.md: 1
- data/transcripts/final/uni-tehran/episode-18.md: 1
- data/transcripts/final/uni-tehran/episode-21.md: 1
- data/transcripts/final/uni-tehran/episode-22.md: 1
- data/transcripts/final/uni-tehran/episode-25.md: 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ثبت نتیجه نهایی Human Review</h3>

<p>
پس از ساخت Stratified Review Packet و بررسی موارد پرریسک، پنج مورد مرزی توسط Reviewer انسانی بررسی و تعیین تکلیف شدند.
</p>

<h4>نتیجه Speaker Attribution</h4>

<ul>
<li>
در چهلستون ۸، پاسخ بدون Label پس از سؤال مجری از «امین‌داور»، توسط Reviewer به <strong>دکتر امین‌داور</strong> نسبت داده شد.
این محتوا متعلق به Speaker اصلی پروژه یعنی دکتر کمیل رودی نیست و برای Corpus اصلی آموزش‌های دکتر رودی در اولویت ثانویه قرار می‌گیرد.
</li>
<li>
در چهلستون ۶، پاسخ بدون Label پس از سؤال مستقیم مجری از «آقای دکتر رودی»، توسط Reviewer به <strong>دکتر کمیل رودی</strong> نسبت داده شد.
این مورد برای Corpus اصلی دکتر رودی قابل استفاده است.
</li>
</ul>

<p>
این دو تصمیم به‌صورت Human Attribution Overlay ثبت می‌شوند و Source خام تغییر نمی‌کند.
</p>

<h4>نتیجه Merge Review</h4>

<ul>
<li><code>اگر این بشه 500 متر / پس بهترم</code> تأیید شد.</li>
<li><code>صبور هم / یا نه</code> از نظر Merge تأیید شد.</li>
<li><code>شاید متحرید ... / زهد بعد از ثروتمندیه</code> از نظر Merge تأیید شد.</li>
</ul>

<h4>خطاهای متنی کشف‌شده در Human Review</h4>

<p>
دو مورد متنی نیز شناسایی شدند، اما طبق سیاست فعلی NB06 اصلاح آنها در Transcript اعمال نمی‌شود:
</p>

<ul>
<li><code>صبور هم</code> → <code>صبورم</code></li>
<li><code>شاید متحرید تو کتابش می‌گه که</code> → <code>شهید مطهری توی کتابش می‌گه که</code></li>
</ul>

<p>
این موارد در فهرست Deferred Text Cleanup ثبت می‌شوند تا در مرحله Cleanup سراسری Corpus، همراه با سایر خطاهای احتمالی تایپی و ASR، یکجا بررسی و اصلاح شوند.
</p>

<h4>وضعیت Human Review</h4>

<p>
Human Review ساختاری NB06 اکنون <code>PASS</code> است.
</p>

<p>
وجود دو Text Cleanup Candidate مانع PASS این مرحله نیست، زیرا هیچ‌کدام مربوط به خرابی Lineage، Speaker Boundary، Merge Structure یا Source Preservation نیستند.
</p>

<h4>اصل عدم تغییر</h4>

<ul>
<li>Source Fileها تغییر نمی‌کنند.</li>
<li>Parsed Text تغییر نمی‌کند.</li>
<li>Speaker Attribution پایه بازنویسی نمی‌شود.</li>
<li>Human Attribution به‌صورت Overlay مستقل ذخیره می‌شود.</li>
<li>Text Correctionها فقط به‌عنوان Candidate ثبت می‌شوند.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>۲ Human Speaker Attribution ثبت شود.</li>
<li>هر دو Attribution به Segment واقعی متصل باشند.</li>
<li>۳ Merge مورد بررسی، همگی تأیید شده باشند.</li>
<li>۲ Deferred Text Cleanup Candidate ثبت شود.</li>
<li>هیچ Text Correction اعمال نشده باشد.</li>
<li>هیچ Source یا Parsed Stream تغییر نکرده باشد.</li>
<li>Blocking Issue انسانی صفر باشد.</li>
</ul>

</div>

In [41]:
# Code Cell 41

human_review_stream_before = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


def find_human_review_unit(
    transcript_unit_id,
):
    matches = [
        unit
        for unit
        in human_review_packet_repaired_v1
        if unit[
            "transcript_unit_id"
        ] == transcript_unit_id
    ]

    assert len(matches) == 1, (
        "Expected exactly one human-review unit "
        f"for {transcript_unit_id}."
    )

    return matches[0]


def find_human_review_sample(
    transcript_unit_id,
    predicate,
    description,
):
    unit = find_human_review_unit(
        transcript_unit_id
    )

    matches = [
        sample
        for sample
        in unit[
            "samples"
        ]
        if predicate(
            sample
        )
    ]

    assert len(matches) == 1, (
        "Expected exactly one review sample for "
        f"{description} in {transcript_unit_id}."
    )

    return matches[0]


chehelston_8_response_sample = (
    find_human_review_sample(
        "program:chehelston:episode-008",
        lambda sample: (
            "CHEHELSTON_UNLABELED_AFTER_EXPLICIT"
            in sample[
                "review_focus"
            ]
        ),
        "Chehelston 8 reviewed response",
    )
)

chehelston_6_response_sample = (
    find_human_review_sample(
        "program:chehelston:episode-006",
        lambda sample: (
            "CHEHELSTON_UNLABELED_AFTER_EXPLICIT"
            in sample[
                "review_focus"
            ]
        ),
        "Chehelston 6 reviewed response",
    )
)

dar_9_merge_sample = (
    find_human_review_sample(
        "program:dar-jostojo-ye-khoshbakhti:episode-009",
        lambda sample: (
            sample[
                "merge_status"
            ] == "MERGED"
            and "اگر این بشه 500 متر"
            in sample[
                "parsed_text"
            ]
        ),
        "500 meter merge",
    )
)

dar_28_sabour_merge_sample = (
    find_human_review_sample(
        "program:dar-jostojo-ye-khoshbakhti:episode-028",
        lambda sample: (
            sample[
                "merge_status"
            ] == "MERGED"
            and "صبور هم"
            in sample[
                "parsed_text"
            ]
        ),
        "sabour merge",
    )
)

dar_28_motahari_merge_sample = (
    find_human_review_sample(
        "program:dar-jostojo-ye-khoshbakhti:episode-028",
        lambda sample: (
            sample[
                "merge_status"
            ] == "MERGED"
            and "زهد بعد از ثروتمندیه"
            in sample[
                "parsed_text"
            ]
        ),
        "Motahari merge",
    )
)


human_attribution_overrides_repaired_v1 = [
    {
        "human_attribution_id": (
            "human-attribution:0001"
        ),
        "utterance_id": (
            chehelston_8_response_sample[
                "utterance_id"
            ]
        ),
        "transcript_unit_id": (
            "program:chehelston:episode-008"
        ),
        "relative_path": (
            "data/transcripts/final/"
            "chehelstion_finup_ketabbaz_bankmellat/"
            "چهلستون 8.md"
        ),
        "source_line_start": (
            chehelston_8_response_sample[
                "source_line_start"
            ]
        ),
        "source_line_end": (
            chehelston_8_response_sample[
                "source_line_end"
            ]
        ),
        "base_speaker_status": (
            chehelston_8_response_sample[
                "speaker_status"
            ]
        ),
        "human_confirmed_speaker": (
            "دکتر امین‌داور"
        ),
        "human_confirmed_speaker_canonical": (
            None
        ),
        "human_confirmation_basis": (
            "DIRECT_HUMAN_REVIEW_OF_QUESTION_"
            "AND_RESPONSE_CONTEXT"
        ),
        "speaker_role_for_project": (
            "SECONDARY_GUEST"
        ),
        "primary_dr_roudi_corpus": False,
        "decision": (
            "HUMAN_CONFIRMED"
        ),
    },
    {
        "human_attribution_id": (
            "human-attribution:0002"
        ),
        "utterance_id": (
            chehelston_6_response_sample[
                "utterance_id"
            ]
        ),
        "transcript_unit_id": (
            "program:chehelston:episode-006"
        ),
        "relative_path": (
            "data/transcripts/final/"
            "chehelstion_finup_ketabbaz_bankmellat/"
            "چهلستون 6.md"
        ),
        "source_line_start": (
            chehelston_6_response_sample[
                "source_line_start"
            ]
        ),
        "source_line_end": (
            chehelston_6_response_sample[
                "source_line_end"
            ]
        ),
        "base_speaker_status": (
            chehelston_6_response_sample[
                "speaker_status"
            ]
        ),
        "human_confirmed_speaker": (
            "دکتر کمیل رودی"
        ),
        "human_confirmed_speaker_canonical": (
            "Dr. Komeil Roudi"
        ),
        "human_confirmation_basis": (
            "DIRECT_HUMAN_REVIEW_OF_QUESTION_"
            "AND_RESPONSE_CONTEXT"
        ),
        "speaker_role_for_project": (
            "PRIMARY_SUBJECT"
        ),
        "primary_dr_roudi_corpus": True,
        "decision": (
            "HUMAN_CONFIRMED"
        ),
    },
]


human_merge_review_decisions_repaired_v1 = [
    {
        "human_merge_review_id": (
            "human-merge-review:0001"
        ),
        "utterance_id": (
            dar_9_merge_sample[
                "utterance_id"
            ]
        ),
        "transcript_unit_id": (
            "program:dar-jostojo-ye-khoshbakhti:episode-009"
        ),
        "source_line_start": (
            dar_9_merge_sample[
                "source_line_start"
            ]
        ),
        "source_line_end": (
            dar_9_merge_sample[
                "source_line_end"
            ]
        ),
        "reviewed_text": (
            dar_9_merge_sample[
                "parsed_text"
            ]
        ),
        "decision": (
            "MERGE_CONFIRMED"
        ),
    },
    {
        "human_merge_review_id": (
            "human-merge-review:0002"
        ),
        "utterance_id": (
            dar_28_sabour_merge_sample[
                "utterance_id"
            ]
        ),
        "transcript_unit_id": (
            "program:dar-jostojo-ye-khoshbakhti:episode-028"
        ),
        "source_line_start": (
            dar_28_sabour_merge_sample[
                "source_line_start"
            ]
        ),
        "source_line_end": (
            dar_28_sabour_merge_sample[
                "source_line_end"
            ]
        ),
        "reviewed_text": (
            dar_28_sabour_merge_sample[
                "parsed_text"
            ]
        ),
        "decision": (
            "MERGE_CONFIRMED"
        ),
    },
    {
        "human_merge_review_id": (
            "human-merge-review:0003"
        ),
        "utterance_id": (
            dar_28_motahari_merge_sample[
                "utterance_id"
            ]
        ),
        "transcript_unit_id": (
            "program:dar-jostojo-ye-khoshbakhti:episode-028"
        ),
        "source_line_start": (
            dar_28_motahari_merge_sample[
                "source_line_start"
            ]
        ),
        "source_line_end": (
            dar_28_motahari_merge_sample[
                "source_line_end"
            ]
        ),
        "reviewed_text": (
            dar_28_motahari_merge_sample[
                "parsed_text"
            ]
        ),
        "decision": (
            "MERGE_CONFIRMED"
        ),
    },
]


deferred_text_cleanup_candidates_repaired_v1 = [
    {
        "cleanup_candidate_id": (
            "deferred-text-cleanup:0001"
        ),
        "utterance_id": (
            dar_28_sabour_merge_sample[
                "utterance_id"
            ]
        ),
        "transcript_unit_id": (
            "program:dar-jostojo-ye-khoshbakhti:episode-028"
        ),
        "source_line_start": (
            dar_28_sabour_merge_sample[
                "source_line_start"
            ]
        ),
        "source_line_end": (
            dar_28_sabour_merge_sample[
                "source_line_end"
            ]
        ),
        "observed_text": (
            "صبور هم"
        ),
        "human_expected_text": (
            "صبورم"
        ),
        "candidate_type": (
            "HUMAN_CONFIRMED_TEXT_ERROR"
        ),
        "apply_in_nb06": False,
        "cleanup_status": (
            "DEFERRED_TO_CORPUS_TEXT_CLEANUP"
        ),
    },
    {
        "cleanup_candidate_id": (
            "deferred-text-cleanup:0002"
        ),
        "utterance_id": (
            dar_28_motahari_merge_sample[
                "utterance_id"
            ]
        ),
        "transcript_unit_id": (
            "program:dar-jostojo-ye-khoshbakhti:episode-028"
        ),
        "source_line_start": (
            dar_28_motahari_merge_sample[
                "source_line_start"
            ]
        ),
        "source_line_end": (
            dar_28_motahari_merge_sample[
                "source_line_end"
            ]
        ),
        "observed_text": (
            "شاید متحرید تو کتابش می‌گه که"
        ),
        "human_expected_text": (
            "شهید مطهری توی کتابش می‌گه که"
        ),
        "candidate_type": (
            "HUMAN_CONFIRMED_TEXT_ERROR"
        ),
        "apply_in_nb06": False,
        "cleanup_status": (
            "DEFERRED_TO_CORPUS_TEXT_CLEANUP"
        ),
    },
]


human_review_source_verification_failures = []

reviewed_samples_for_source_check = [
    chehelston_8_response_sample,
    chehelston_6_response_sample,
    dar_9_merge_sample,
    dar_28_sabour_merge_sample,
    dar_28_motahari_merge_sample,
]

for sample in (
    reviewed_samples_for_source_check
):
    utterance = (
        repaired_merged_utterance_by_id[
            sample[
                "utterance_id"
            ]
        ]
    )

    relative_path = utterance[
        "relative_path"
    ]

    source_lines = (
        project_root
        / relative_path
    ).read_text(
        encoding="utf-8"
    ).splitlines()

    expected_source_text = "\n".join(
        source_lines[
            utterance[
                "source_line_start"
            ] - 1:
            utterance[
                "source_line_end"
            ]
        ]
    )

    if (
        expected_source_text
        != utterance[
            "source_text"
        ]
    ):
        human_review_source_verification_failures.append({
            "utterance_id": (
                utterance[
                    "utterance_id"
                ]
            ),
            "relative_path": (
                relative_path
            ),
        })


human_review_result_repaired_v1 = {
    "review_type": (
        "STRATIFIED_PACKET_WITH_TARGETED_"
        "HUMAN_EDGE_CASE_CONFIRMATION"
    ),
    "review_packet_unit_count": 24,
    "review_packet_sample_count": 96,
    "targeted_human_decision_count": 5,
    "human_attribution_count": (
        len(
            human_attribution_overrides_repaired_v1
        )
    ),
    "human_confirmed_merge_count": (
        len(
            human_merge_review_decisions_repaired_v1
        )
    ),
    "deferred_text_cleanup_count": (
        len(
            deferred_text_cleanup_candidates_repaired_v1
        )
    ),
    "blocking_issue_count": 0,
    "structural_human_review_status": (
        "PASS"
    ),
    "text_cleanup_status": (
        "DEFERRED_TO_LATER_CORPUS_CLEANUP"
    ),
}


human_review_stream_after = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


print(
    "Human attribution overrides:",
    len(
        human_attribution_overrides_repaired_v1
    ),
)
print(
    "Human-confirmed merges:",
    len(
        human_merge_review_decisions_repaired_v1
    ),
)
print(
    "Deferred text cleanup candidates:",
    len(
        deferred_text_cleanup_candidates_repaired_v1
    ),
)
print(
    "Human-review Source verification failures:",
    len(
        human_review_source_verification_failures
    ),
)
print(
    "Blocking issues:",
    human_review_result_repaired_v1[
        "blocking_issue_count"
    ],
)


print()
print("HUMAN ATTRIBUTION OVERRIDES")
print("=" * 100)

for item in (
    human_attribution_overrides_repaired_v1
):
    print()
    print(
        "ID:",
        item[
            "human_attribution_id"
        ],
    )
    print(
        "Transcript unit:",
        item[
            "transcript_unit_id"
        ],
    )
    print(
        "Utterance:",
        item[
            "utterance_id"
        ],
    )
    print(
        "Source lines:",
        (
            item[
                "source_line_start"
            ],
            item[
                "source_line_end"
            ],
        ),
    )
    print(
        "Base status:",
        item[
            "base_speaker_status"
        ],
    )
    print(
        "Human-confirmed speaker:",
        item[
            "human_confirmed_speaker"
        ],
    )
    print(
        "Primary Dr. Roudi corpus:",
        item[
            "primary_dr_roudi_corpus"
        ],
    )
    print(
        "Decision:",
        item[
            "decision"
        ],
    )


print()
print("HUMAN MERGE DECISIONS")
print("=" * 100)

for item in (
    human_merge_review_decisions_repaired_v1
):
    print()
    print(
        "ID:",
        item[
            "human_merge_review_id"
        ],
    )
    print(
        "Utterance:",
        item[
            "utterance_id"
        ],
    )
    print(
        "Source lines:",
        (
            item[
                "source_line_start"
            ],
            item[
                "source_line_end"
            ],
        ),
    )
    print(
        "Decision:",
        item[
            "decision"
        ],
    )
    print(
        "Text:",
        preview_text(
            item[
                "reviewed_text"
            ],
            limit=400,
        ),
    )


print()
print("DEFERRED TEXT CLEANUP")
print("=" * 100)

for item in (
    deferred_text_cleanup_candidates_repaired_v1
):
    print()
    print(
        "ID:",
        item[
            "cleanup_candidate_id"
        ],
    )
    print(
        "Observed:",
        item[
            "observed_text"
        ],
    )
    print(
        "Expected:",
        item[
            "human_expected_text"
        ],
    )
    print(
        "Apply in NB06:",
        item[
            "apply_in_nb06"
        ],
    )
    print(
        "Status:",
        item[
            "cleanup_status"
        ],
    )


print()
print("HUMAN REVIEW RESULT")
print("=" * 100)

for key, value in (
    human_review_result_repaired_v1.items()
):
    print(
        f"- {key}: {value}"
    )


assert len(
    human_attribution_overrides_repaired_v1
) == 2, (
    "Expected two human speaker attributions."
)

assert all(
    item[
        "utterance_id"
    ]
    in repaired_merged_utterance_by_id
    for item
    in human_attribution_overrides_repaired_v1
), (
    "A human attribution references "
    "an unknown repaired utterance."
)

assert all(
    item[
        "base_speaker_status"
    ] == "UNRESOLVED"
    for item
    in human_attribution_overrides_repaired_v1
), (
    "A human attribution override "
    "does not originate from an unresolved segment."
)

assert (
    human_attribution_overrides_repaired_v1[
        0
    ][
        "primary_dr_roudi_corpus"
    ]
    is False
), (
    "Dr. Amin Davar response was incorrectly "
    "marked as primary Dr. Roudi content."
)

assert (
    human_attribution_overrides_repaired_v1[
        1
    ][
        "human_confirmed_speaker_canonical"
    ]
    == "Dr. Komeil Roudi"
    and human_attribution_overrides_repaired_v1[
        1
    ][
        "primary_dr_roudi_corpus"
    ]
    is True
), (
    "Human-confirmed Dr. Roudi response "
    "was not recorded correctly."
)

assert len(
    human_merge_review_decisions_repaired_v1
) == 3, (
    "Expected three human-reviewed merge decisions."
)

assert all(
    item[
        "decision"
    ] == "MERGE_CONFIRMED"
    for item
    in human_merge_review_decisions_repaired_v1
), (
    "A reviewed merge was not confirmed."
)

assert len(
    deferred_text_cleanup_candidates_repaired_v1
) == 2, (
    "Expected two deferred text cleanup candidates."
)

assert all(
    item[
        "apply_in_nb06"
    ] is False
    for item
    in deferred_text_cleanup_candidates_repaired_v1
), (
    "A deferred text correction "
    "was applied inside NB06."
)

assert not (
    human_review_source_verification_failures
), (
    "A human-reviewed sample no longer "
    "matches its Source."
)

assert (
    human_review_result_repaired_v1[
        "blocking_issue_count"
    ] == 0
), (
    "Human review still contains a blocker."
)

assert (
    human_review_result_repaired_v1[
        "structural_human_review_status"
    ] == "PASS"
), (
    "Structural human review is not PASS."
)

assert (
    human_review_stream_before
    == human_review_stream_after
), (
    "Human review registration mutated "
    "the repaired transcript stream."
)

print()
print(
    "NB06 STRUCTURAL HUMAN REVIEW: PASS"
)
print(
    "TEXT CLEANUP: DEFERRED TO LATER CORPUS CLEANUP"
)

Human attribution overrides: 2
Human-confirmed merges: 3
Deferred text cleanup candidates: 2
Human-review Source verification failures: 0
Blocking issues: 0

HUMAN ATTRIBUTION OVERRIDES

ID: human-attribution:0001
Transcript unit: program:chehelston:episode-008
Utterance: program:chehelston:episode-008:utterance:00027
Source lines: (73, 73)
Base status: UNRESOLVED
Human-confirmed speaker: دکتر امین‌داور
Primary Dr. Roudi corpus: False
Decision: HUMAN_CONFIRMED

ID: human-attribution:0002
Transcript unit: program:chehelston:episode-006
Utterance: program:chehelston:episode-006:utterance:00021
Source lines: (61, 61)
Base status: UNRESOLVED
Human-confirmed speaker: دکتر کمیل رودی
Primary Dr. Roudi corpus: True
Decision: HUMAN_CONFIRMED

HUMAN MERGE DECISIONS

ID: human-merge-review:0001
Utterance: program:dar-jostojo-ye-khoshbakhti:episode-009:merged:00066-00067
Source lines: (265, 269)
Decision: MERGE_CONFIRMED
Text: اگر این بشه 500 متر ⏎  ⏎ پس بهترم

ID: human-merge-review:0002
Utteranc

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>کشف قرارداد Artifact، مسیر خروجی و Schemaهای موجود پیش از Write</h3>

<p>
Parsing، Repair، Quality Detection، Timestamp Preservation و Human Review اکنون تکمیل شده‌اند.
</p>

<p>
مرحله بعد ساخت Artifactهای نهایی NB06 است؛ اما پیش از هر Write باید قرارداد واقعی Repository بررسی شود.
</p>

<h4>هدف این Cell</h4>

<p>
این Cell فقط ساختار موجود Repository را می‌خواند و مشخص می‌کند:
</p>

<ul>
<li>Artifactهای فازهای قبلی در چه مسیرهایی ذخیره شده‌اند.</li>
<li>در <code>data/phase2</code> چه Directoryهایی از قبل وجود دارند.</li>
<li>Manifest، Report، Checkpoint و Schemaهای موجود چه Naming Conventionی دارند.</li>
<li>آیا نام Artifactهای مورد انتظار NB06 قبلاً در Notebookها یا مستندات Repository ذکر شده است.</li>
<li>ساختار Top-level فایل‌های JSON موجود چگونه است.</li>
</ul>

<h4>Artifactهای مورد انتظار NB06</h4>

<p>
در این مرحله صرفاً وجود یا نبود Contract برای Artifactهای زیر بررسی می‌شود و هنوز هیچ‌کدام نوشته نمی‌شوند:
</p>

<ul>
<li><code>transcripts_extracted_v1.jsonl</code></li>
<li><code>speaker_aliases_v1.json</code></li>
<li><code>transcript_quality_report.json</code></li>
<li>Checkpoint نهایی NB06</li>
</ul>

<h4>اصل عدم تغییر</h4>

<p>
این Cell کاملاً Read-only است.
</p>

<ul>
<li>هیچ Directory ساخته نمی‌شود.</li>
<li>هیچ Artifact نوشته یا بازنویسی نمی‌شود.</li>
<li>هیچ Transcript تغییر نمی‌کند.</li>
<li>هیچ Git operation انجام نمی‌شود.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>ساختار Top-level Repository قابل خواندن باشد.</li>
<li>مسیرهای Phase Artifact موجود گزارش شوند.</li>
<li>Schemaها و Artifactهای مشابه موجود فهرست شوند.</li>
<li>Referenceهای احتمالی به Artifactهای NB06 در Repository پیدا شوند.</li>
<li>هیچ تغییری در Stream اصلاح‌شده ایجاد نشود.</li>
</ul>

</div>

In [42]:
# Code Cell 42

artifact_discovery_stream_before = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)

print("PROJECT ROOT")
print("=" * 100)
print(project_root)


print()
print("TOP-LEVEL REPOSITORY ENTRIES")
print("=" * 100)

top_level_entries = sorted(
    project_root.iterdir(),
    key=lambda path: (
        path.is_file(),
        path.name.lower(),
    ),
)

for path in top_level_entries:
    entry_type = (
        "DIR"
        if path.is_dir()
        else "FILE"
    )

    print(
        f"- [{entry_type}] {path.name}"
    )


phase_directory_candidates = []

for relative_candidate in [
    "data/phase1",
    "data/phase2",
    "data/phase2/manifests",
    "data/phase2/artifacts",
    "data/phase2/reports",
    "data/phase2/checkpoints",
    "artifacts",
    "reports",
    "checkpoints",
]:
    candidate_path = (
        project_root
        / relative_candidate
    )

    if candidate_path.exists():
        phase_directory_candidates.append(
            candidate_path
        )


print()
print("EXISTING PHASE / ARTIFACT DIRECTORIES")
print("=" * 100)

if phase_directory_candidates:
    for path in (
        phase_directory_candidates
    ):
        print(
            "-",
            path.relative_to(
                project_root
            ).as_posix(),
        )
else:
    print("- NONE FOUND")


artifact_filename_patterns = [
    "*manifest*.json",
    "*report*.json",
    "*checkpoint*.json",
    "*inventory*.json",
    "*extracted*.json",
    "*aliases*.json",
    "*.jsonl",
]

artifact_like_paths = set()

for base_relative in [
    "data/phase1",
    "data/phase2",
    "artifacts",
    "reports",
    "checkpoints",
]:
    base_path = (
        project_root
        / base_relative
    )

    if not base_path.exists():
        continue

    for pattern in (
        artifact_filename_patterns
    ):
        for path in base_path.rglob(
            pattern
        ):
            if path.is_file():
                artifact_like_paths.add(
                    path
                )


artifact_like_paths = sorted(
    artifact_like_paths,
    key=lambda path: (
        path.relative_to(
            project_root
        ).as_posix()
    ),
)


print()
print("EXISTING ARTIFACT-LIKE FILES")
print("=" * 100)

if artifact_like_paths:
    for path in artifact_like_paths:
        print(
            "-",
            path.relative_to(
                project_root
            ).as_posix(),
        )
else:
    print("- NONE FOUND")


schema_search_roots = [
    project_root / "schemas",
    project_root / "schema",
    project_root / "data" / "schemas",
    project_root / "config",
    project_root / "configs",
    project_root / "docs",
]

schema_like_paths = set()

for base_path in (
    schema_search_roots
):
    if not base_path.exists():
        continue

    for pattern in [
        "*schema*.json",
        "*.schema.json",
        "*schema*.yaml",
        "*schema*.yml",
    ]:
        for path in base_path.rglob(
            pattern
        ):
            if path.is_file():
                schema_like_paths.add(
                    path
                )


schema_like_paths = sorted(
    schema_like_paths,
    key=lambda path: (
        path.relative_to(
            project_root
        ).as_posix()
    ),
)


print()
print("DISCOVERED SCHEMA-LIKE FILES")
print("=" * 100)

if schema_like_paths:
    for path in schema_like_paths:
        print(
            "-",
            path.relative_to(
                project_root
            ).as_posix(),
        )
else:
    print("- NONE FOUND")


json_contract_previews = []

for path in (
    artifact_like_paths
):
    if (
        path.suffix.lower()
        != ".json"
    ):
        continue

    try:
        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )
    except Exception as exc:
        json_contract_previews.append({
            "relative_path": (
                path.relative_to(
                    project_root
                ).as_posix()
            ),
            "read_status": (
                "JSON_READ_FAILED"
            ),
            "error": (
                repr(exc)
            ),
        })
        continue

    if isinstance(
        payload,
        dict,
    ):
        top_level_type = (
            "dict"
        )

        top_level_keys = sorted(
            payload.keys()
        )

        item_count = None

    elif isinstance(
        payload,
        list,
    ):
        top_level_type = (
            "list"
        )

        top_level_keys = []

        item_count = len(
            payload
        )

    else:
        top_level_type = (
            type(payload).__name__
        )

        top_level_keys = []

        item_count = None

    json_contract_previews.append({
        "relative_path": (
            path.relative_to(
                project_root
            ).as_posix()
        ),
        "read_status": (
            "PASS"
        ),
        "top_level_type": (
            top_level_type
        ),
        "top_level_keys": (
            top_level_keys
        ),
        "item_count": (
            item_count
        ),
    })


print()
print("EXISTING JSON CONTRACT PREVIEW")
print("=" * 110)

if json_contract_previews:
    for item in (
        json_contract_previews
    ):
        print()
        print(
            "File:",
            item[
                "relative_path"
            ],
        )
        print(
            "Read status:",
            item[
                "read_status"
            ],
        )

        if (
            item[
                "read_status"
            ] == "PASS"
        ):
            print(
                "Top-level type:",
                item[
                    "top_level_type"
                ],
            )

            print(
                "Top-level keys:",
                item[
                    "top_level_keys"
                ],
            )

            print(
                "Item count:",
                item[
                    "item_count"
                ],
            )
        else:
            print(
                "Error:",
                item[
                    "error"
                ],
            )
else:
    print("- NONE")


target_artifact_terms = [
    "transcripts_extracted_v1",
    "speaker_aliases_v1",
    "transcript_quality_report",
    "transcript_ingestion",
]

reference_scan_roots = [
    project_root / "notebooks",
    project_root / "docs",
    project_root / "README.md",
]

reference_scan_extensions = {
    ".ipynb",
    ".md",
    ".py",
    ".json",
    ".yaml",
    ".yml",
}

reference_hits = []


def iter_reference_files(
    search_root,
):
    if (
        not search_root.exists()
    ):
        return []

    if search_root.is_file():
        return [
            search_root
        ]

    return [
        path
        for path
        in search_root.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in reference_scan_extensions
        )
    ]


seen_reference_paths = set()

for search_root in (
    reference_scan_roots
):
    for path in iter_reference_files(
        search_root
    ):
        resolved_path = (
            path.resolve()
        )

        if (
            resolved_path
            in seen_reference_paths
        ):
            continue

        seen_reference_paths.add(
            resolved_path
        )

        try:
            content = path.read_text(
                encoding="utf-8"
            )
        except Exception:
            continue

        matched_terms = [
            term
            for term
            in target_artifact_terms
            if term in content
        ]

        if matched_terms:
            reference_hits.append({
                "relative_path": (
                    path.relative_to(
                        project_root
                    ).as_posix()
                ),
                "matched_terms": (
                    matched_terms
                ),
            })


reference_hits = sorted(
    reference_hits,
    key=lambda item: (
        item[
            "relative_path"
        ]
    ),
)


print()
print("NB06 ARTIFACT CONTRACT REFERENCES")
print("=" * 100)

if reference_hits:
    for item in reference_hits:
        print(
            "-",
            item[
                "relative_path"
            ],
            "->",
            item[
                "matched_terms"
            ],
        )
else:
    print("- NONE FOUND")


expected_nb06_artifact_names = [
    "transcripts_extracted_v1.jsonl",
    "speaker_aliases_v1.json",
    "transcript_quality_report.json",
]

existing_expected_artifacts = []

for artifact_name in (
    expected_nb06_artifact_names
):
    matches = [
        path
        for path
        in project_root.rglob(
            artifact_name
        )
        if (
            path.is_file()
            and ".git"
            not in path.parts
        )
    ]

    existing_expected_artifacts.append({
        "artifact_name": (
            artifact_name
        ),
        "matches": [
            path.relative_to(
                project_root
            ).as_posix()
            for path
            in matches
        ],
    })


print()
print("EXPECTED NB06 ARTIFACT EXISTENCE CHECK")
print("=" * 100)

for item in (
    existing_expected_artifacts
):
    print(
        "-",
        item[
            "artifact_name"
        ],
        "->",
        (
            item[
                "matches"
            ]
            if item[
                "matches"
            ]
            else "NOT PRESENT"
        ),
    )


artifact_discovery_stream_after = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


assert (
    project_root.exists()
    and project_root.is_dir()
), (
    "Project root is unavailable."
)

assert (
    artifact_discovery_stream_before
    == artifact_discovery_stream_after
), (
    "Artifact contract discovery mutated "
    "the repaired transcript stream."
)

print()
print(
    "NB06 ARTIFACT CONTRACT DISCOVERY: PASS"
)
print(
    "WRITE OPERATIONS: NOT PERFORMED"
)

PROJECT ROOT
/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance

TOP-LEVEL REPOSITORY ENTRIES
- [DIR] config
- [DIR] data
- [DIR] notebooks
- [DIR] reports
- [DIR] src
- [FILE] .DS_Store
- [FILE] .env
- [FILE] .git
- [FILE] .gitignore
- [FILE] app.py
- [FILE] financial_rag_assistant.ipynb
- [FILE] README.md
- [FILE] requirements.txt
- [FILE] roadmap.ipynb

EXISTING PHASE / ARTIFACT DIRECTORIES
- data/phase2
- data/phase2/manifests
- reports

EXISTING ARTIFACT-LIKE FILES
- data/phase2/interim/speaker_aliases_v1.json
- data/phase2/interim/transcripts_extracted_v1.jsonl
- data/phase2/interim/website_extracted_v2.json
- data/phase2/manifests/book_manifest_v1.json
- data/phase2/manifests/transcript_manifest_v1.json
- data/phase2/manifests/website_manifest_v1.json
- reports/phase2/book_inventory_acceptance_v1.json
- reports/phase2/transcript_inventory_acceptance_v1.json
- reports/phase2/transcript_quality_report.json

DISCOVERED SCHEMA-LIKE FILES
- NONE FOUND

EXISTING

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>استخراج Context قرارداد NB06 از Notebook قبلی و Roadmap</h3>

<p>
Discovery قبلی نشان داد که Repository دو Convention مشخص دارد:
</p>

<ul>
<li>Artifactهای استخراج‌شده در <code>data/phase2/interim</code></li>
<li>گزارش‌های پذیرش و Quality در <code>reports/phase2</code></li>
</ul>

<p>
با این حال برای <code>speaker_aliases_v1.json</code> و Checkpoint نهایی NB06 هنوز Path صریحی مشاهده نشده است.
</p>

<p>
برای جلوگیری از حدس‌زدن، این Cell متن واقعی Notebook قبلی، Roadmap و مستندات نزدیک به Phase 2 را در اطراف واژه‌های مربوط به Transcript Ingestion، Interim، Report، Checkpoint و Notebook بعدی استخراج می‌کند.
</p>

<h4>هدف</h4>

<ul>
<li>پیدا کردن Output Contract صریح NB06 در NB05 یا Roadmap</li>
<li>پیدا کردن Naming Convention مربوط به Checkpoint</li>
<li>مشخص کردن محل منطقی <code>speaker_aliases_v1.json</code> بر اساس Evidence Repository</li>
<li>خواندن Constraintهای ثبت‌شده برای Notebook بعدی</li>
</ul>

<h4>اصل عدم تغییر</h4>

<p>
این Cell فقط Read-only است و هیچ Artifact یا Directory نمی‌سازد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>فایل‌های مرجع قابل خواندن باشند.</li>
<li>Contextهای مرتبط با Transcript Ingestion گزارش شوند.</li>
<li>هیچ Stream یا Artifactی تغییر نکند.</li>
</ul>

</div>

In [43]:
# Code Cell 43

contract_context_stream_before = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)

contract_reference_paths = [
    project_root
    / "notebooks"
    / "phase2"
    / "05_transcript_inventory.ipynb",
    project_root
    / "notebooks"
    / "phase2"
    / "06_transcript_ingestion.ipynb",
    project_root
    / "roadmap.ipynb",
    project_root
    / "README.md",
]

contract_search_terms = [
    "transcript_ingestion",
    "transcript ingestion",
    "transcripts_extracted",
    "speaker_alias",
    "quality_report",
    "checkpoint",
    "interim",
    "reports/phase2",
    "next_notebook",
    "NB06",
    "NB07",
]

contract_context_hits = []


def notebook_text_blocks(
    notebook_payload,
):
    blocks = []

    for cell_index, cell in enumerate(
        notebook_payload.get(
            "cells",
            []
        )
    ):
        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        if not source.strip():
            continue

        blocks.append({
            "cell_index": (
                cell_index
            ),
            "cell_type": (
                cell.get(
                    "cell_type"
                )
            ),
            "text": (
                source
            ),
        })

    return blocks


for path in (
    contract_reference_paths
):
    if not path.exists():
        continue

    relative_path = (
        path.relative_to(
            project_root
        ).as_posix()
    )

    if (
        path.suffix.lower()
        == ".ipynb"
    ):
        notebook_payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

        blocks = notebook_text_blocks(
            notebook_payload
        )

        for block in blocks:
            lowered = (
                block[
                    "text"
                ].lower()
            )

            matched_terms = [
                term
                for term
                in contract_search_terms
                if term.lower()
                in lowered
            ]

            if not matched_terms:
                continue

            contract_context_hits.append({
                "relative_path": (
                    relative_path
                ),
                "location": (
                    f"cell:{block['cell_index']}"
                ),
                "cell_type": (
                    block[
                        "cell_type"
                    ]
                ),
                "matched_terms": (
                    matched_terms
                ),
                "text": (
                    block[
                        "text"
                    ]
                ),
            })

    else:
        content = path.read_text(
            encoding="utf-8"
        )

        lines = content.splitlines()

        for line_index, line in enumerate(
            lines,
            start=1,
        ):
            lowered = line.lower()

            matched_terms = [
                term
                for term
                in contract_search_terms
                if term.lower()
                in lowered
            ]

            if not matched_terms:
                continue

            context_start = max(
                0,
                line_index - 4,
            )

            context_end = min(
                len(lines),
                line_index + 3,
            )

            context_text = "\n".join(
                lines[
                    context_start:
                    context_end
                ]
            )

            contract_context_hits.append({
                "relative_path": (
                    relative_path
                ),
                "location": (
                    f"line:{line_index}"
                ),
                "cell_type": (
                    None
                ),
                "matched_terms": (
                    matched_terms
                ),
                "text": (
                    context_text
                ),
            })


print(
    "Reference files checked:",
    sum(
        1
        for path
        in contract_reference_paths
        if path.exists()
    ),
)
print(
    "Relevant contract contexts:",
    len(
        contract_context_hits
    ),
)


print()
print("NB06 CONTRACT CONTEXTS")
print("=" * 110)

if contract_context_hits:
    for hit_index, hit in enumerate(
        contract_context_hits,
        start=1,
    ):
        print()
        print(
            "Context:",
            hit_index,
        )
        print(
            "File:",
            hit[
                "relative_path"
            ],
        )
        print(
            "Location:",
            hit[
                "location"
            ],
        )
        print(
            "Cell type:",
            hit[
                "cell_type"
            ],
        )
        print(
            "Matched terms:",
            hit[
                "matched_terms"
            ],
        )
        print()
        print(
            hit[
                "text"
            ][:5000]
        )
        print()
        print("-" * 110)
else:
    print("- NONE FOUND")


phase2_directory_inventory = []

phase2_root = (
    project_root
    / "data"
    / "phase2"
)

if phase2_root.exists():
    for path in sorted(
        phase2_root.rglob("*"),
        key=lambda item: (
            item.relative_to(
                project_root
            ).as_posix()
        ),
    ):
        if not path.is_dir():
            continue

        phase2_directory_inventory.append(
            path.relative_to(
                project_root
            ).as_posix()
        )


reports_phase2_root = (
    project_root
    / "reports"
    / "phase2"
)

reports_phase2_inventory = []

if reports_phase2_root.exists():
    reports_phase2_inventory = [
        path.relative_to(
            project_root
        ).as_posix()
        for path
        in sorted(
            reports_phase2_root.iterdir(),
            key=lambda item: (
                item.name.lower()
            ),
        )
        if path.is_file()
    ]


print()
print("DATA/PHASE2 DIRECTORY INVENTORY")
print("=" * 100)

for relative_path in (
    phase2_directory_inventory
):
    print(
        "-",
        relative_path,
    )


print()
print("REPORTS/PHASE2 FILE INVENTORY")
print("=" * 100)

if reports_phase2_inventory:
    for relative_path in (
        reports_phase2_inventory
    ):
        print(
            "-",
            relative_path,
        )
else:
    print("- NONE")


explicit_output_path_hits = []

output_path_pattern = re.compile(
    r"(?:data/phase2|reports/phase2)"
    r"/[^\s`\"'<>)]+"
)

for hit in (
    contract_context_hits
):
    matches = (
        output_path_pattern.findall(
            hit[
                "text"
            ]
        )
    )

    for match in matches:
        explicit_output_path_hits.append({
            "source": (
                hit[
                    "relative_path"
                ]
            ),
            "location": (
                hit[
                    "location"
                ]
            ),
            "path_text": (
                match.rstrip(
                    ".,;:"
                )
            ),
        })


print()
print("EXPLICIT PHASE2 PATH REFERENCES")
print("=" * 100)

if explicit_output_path_hits:
    for item in (
        explicit_output_path_hits
    ):
        print(
            "-",
            item[
                "path_text"
            ],
            "<-",
            item[
                "source"
            ],
            item[
                "location"
            ],
        )
else:
    print("- NONE FOUND")


contract_context_stream_after = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


assert (
    project_root
    / "notebooks"
    / "phase2"
    / "05_transcript_inventory.ipynb"
).exists(), (
    "NB05 reference notebook is unavailable."
)

assert (
    contract_context_stream_before
    == contract_context_stream_after
), (
    "Contract context discovery mutated "
    "the repaired transcript stream."
)

print()
print(
    "NB06 CONTRACT CONTEXT EXTRACTION: PASS"
)
print(
    "WRITE OPERATIONS: NOT PERFORMED"
)

Reference files checked: 4
Relevant contract contexts: 49

NB06 CONTRACT CONTEXTS

Context: 1
File: notebooks/phase2/05_transcript_inventory.ipynb
Location: cell:0
Cell type: markdown
Matched terms: ['transcript ingestion']

<div style="direction: rtl; white-space: normal; line-height: 1;">

# Notebook 05 — Transcript Inventory

این Notebook مرحله‌ی شناسایی، طبقه‌بندی و پذیرش Transcriptها برای ورود به Pipeline است.

هدف این فایل هنوز استخراج نهایی متن یا ساخت Segmentهای RAG نیست.

در این مرحله باید مشخص شود:

- چه Transcriptهایی در پروژه وجود دارند.
- هر فایل مربوط به کدام برنامه، فصل یا قسمت است.
- کدام فایل Source اصلی است.
- آیا Duplicate یا نسخه‌ی جایگزین وجود دارد.
- ساختار گویندگان هر برنامه چگونه است.
- در کدام برنامه Speaker Attribution لازم است.
- کدام فایل‌ها برای Transcript Ingestion پذیرفته می‌شوند.
- Provenance و Hash هر فایل چیست.
- چه اطلاعاتی با اطمینان از خود فایل قابل اثبات است و چه اطلاعاتی بر اساس تأیید انسانی ثبت شده است.

اطلاعات انسانی تأییدشده درباره برنامه‌ها:


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بررسی Artifact Contract مراحل Ingestion قبلی برای تعیین مسیر نهایی NB06</h3>

<p>
Discovery قبلی با موفقیت PASS شد و مشخص کرد که Repository دارای مسیرهای واقعی:
</p>

<ul>
<li><code>data/phase2/interim</code></li>
<li><code>data/phase2/processed</code></li>
<li><code>reports/phase2</code></li>
</ul>

<p>
همچنین Report واقعی <code>book_ingestion_acceptance_v1.json</code> از یک مرحله Ingestion قبلی در Repository وجود دارد.
</p>

<p>
برای اینکه مسیر Artifactهای Transcript را بر اساس Convention واقعی پروژه تعیین کنیم و نه صرفاً بر اساس نام Directoryها، در این Cell قرارداد مرحله Book Ingestion و سایر Ingestionهای قبلی بررسی می‌شود.
</p>

<h4>موارد بررسی</h4>

<ul>
<li>Notebookهای Phase 2 که Artifactهای Ingestion قبلی را نوشته‌اند.</li>
<li>مسیر فایل‌های Data Output آنها.</li>
<li>مسیر Acceptance Report آنها.</li>
<li>نوع Artifact و Schema ثبت‌شده در Report.</li>
<li>وجود یا نبود Checkpoint مستقل در مراحل قبلی.</li>
<li>Convention استفاده از <code>interim</code> در برابر <code>processed</code>.</li>
</ul>

<h4>هدف نهایی</h4>

<p>
پس از این بررسی باید بتوانیم بدون حدس مشخص کنیم:
</p>

<ul>
<li><code>transcripts_extracted_v1.jsonl</code> در کدام مسیر قرار گیرد.</li>
<li><code>speaker_aliases_v1.json</code> در کدام مسیر قرار گیرد.</li>
<li><code>transcript_quality_report.json</code> در کدام مسیر قرار گیرد.</li>
<li>آیا NB06 واقعاً نیازمند فایل Checkpoint مستقل است یا Final Acceptance Report نقش Checkpoint را دارد.</li>
</ul>

<h4>اصل عدم تغییر</h4>

<p>
این Cell کاملاً Read-only است و هیچ Artifact، Directory یا Source File را تغییر نمی‌دهد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>حداقل یک Contract واقعی از Ingestion قبلی پیدا شود.</li>
<li>Pathهای Data Artifact و Acceptance Report آن استخراج شوند.</li>
<li>Reportهای موجود مستقیماً از Disk قابل خواندن باشند.</li>
<li>Stream اصلاح‌شده هیچ تغییری نکند.</li>
</ul>

</div>

In [44]:
# Code Cell 44

prior_ingestion_contract_stream_before = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)

phase2_notebooks_root = (
    project_root
    / "notebooks"
    / "phase2"
)

phase2_notebook_paths = sorted(
    [
        path
        for path
        in phase2_notebooks_root.glob(
            "*.ipynb"
        )
        if path.is_file()
    ],
    key=lambda path: path.name.lower(),
)

prior_ingestion_terms = [
    "book_ingestion_acceptance_v1.json",
    "website_acceptance_v1.json",
    "book_ingestion",
    "website_extracted_v2.json",
    "data/phase2/interim",
    "data/phase2/processed",
    "reports/phase2",
    "checkpoint",
]

prior_ingestion_notebook_hits = []

for notebook_path in (
    phase2_notebook_paths
):
    notebook_payload = json.loads(
        notebook_path.read_text(
            encoding="utf-8"
        )
    )

    for cell_index, cell in enumerate(
        notebook_payload.get(
            "cells",
            []
        )
    ):
        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        if not source.strip():
            continue

        matched_terms = [
            term
            for term
            in prior_ingestion_terms
            if term.lower()
            in source.lower()
        ]

        if not matched_terms:
            continue

        prior_ingestion_notebook_hits.append({
            "relative_path": (
                notebook_path.relative_to(
                    project_root
                ).as_posix()
            ),
            "cell_index": (
                cell_index
            ),
            "cell_type": (
                cell.get(
                    "cell_type"
                )
            ),
            "matched_terms": (
                matched_terms
            ),
            "source": (
                source
            ),
        })


print(
    "Phase 2 notebooks scanned:",
    len(
        phase2_notebook_paths
    ),
)
print(
    "Prior-ingestion contract cells:",
    len(
        prior_ingestion_notebook_hits
    ),
)


path_reference_pattern = re.compile(
    r"(?:data/phase2|reports/phase2)"
    r"/[A-Za-z0-9_\-./]+"
)

prior_ingestion_path_references = []

for hit in (
    prior_ingestion_notebook_hits
):
    raw_paths = (
        path_reference_pattern.findall(
            hit[
                "source"
            ]
        )
    )

    for raw_path in raw_paths:
        prior_ingestion_path_references.append({
            "source_notebook": (
                hit[
                    "relative_path"
                ]
            ),
            "cell_index": (
                hit[
                    "cell_index"
                ]
            ),
            "cell_type": (
                hit[
                    "cell_type"
                ]
            ),
            "path_text": (
                raw_path.rstrip(
                    ".,;:"
                )
            ),
        })


unique_prior_path_references = []

seen_prior_path_references = set()

for item in (
    prior_ingestion_path_references
):
    key = (
        item[
            "source_notebook"
        ],
        item[
            "cell_index"
        ],
        item[
            "path_text"
        ],
    )

    if key in (
        seen_prior_path_references
    ):
        continue

    seen_prior_path_references.add(
        key
    )

    unique_prior_path_references.append(
        item
    )


print()
print("PRIOR INGESTION PATH REFERENCES")
print("=" * 110)

if unique_prior_path_references:
    for item in (
        unique_prior_path_references
    ):
        print(
            "-",
            item[
                "path_text"
            ],
            "<-",
            item[
                "source_notebook"
            ],
            f"cell:{item['cell_index']}",
            f"[{item['cell_type']}]",
        )
else:
    print("- NONE")


existing_prior_reports = [
    (
        project_root
        / "reports"
        / "phase2"
        / "book_ingestion_acceptance_v1.json"
    ),
    (
        project_root
        / "reports"
        / "phase2"
        / "website_acceptance_v1.json"
    ),
]

prior_report_contracts = []

for report_path in (
    existing_prior_reports
):
    if not report_path.is_file():
        continue

    payload = json.loads(
        report_path.read_text(
            encoding="utf-8"
        )
    )

    prior_report_contracts.append({
        "relative_path": (
            report_path.relative_to(
                project_root
            ).as_posix()
        ),
        "top_level_type": (
            type(payload).__name__
        ),
        "top_level_keys": (
            sorted(
                payload.keys()
            )
            if isinstance(
                payload,
                dict,
            )
            else []
        ),
        "artifact_type": (
            payload.get(
                "artifact_type"
            )
            if isinstance(
                payload,
                dict,
            )
            else None
        ),
        "report_type": (
            payload.get(
                "report_type"
            )
            if isinstance(
                payload,
                dict,
            )
            else None
        ),
        "artifact_version": (
            payload.get(
                "artifact_version"
            )
            if isinstance(
                payload,
                dict,
            )
            else None
        ),
        "report_version": (
            payload.get(
                "report_version"
            )
            if isinstance(
                payload,
                dict,
            )
            else None
        ),
        "result": (
            payload.get(
                "result"
            )
            if isinstance(
                payload,
                dict,
            )
            else None
        ),
        "output_artifact": (
            payload.get(
                "output_artifact"
            )
            if isinstance(
                payload,
                dict,
            )
            else None
        ),
        "artifacts": (
            payload.get(
                "artifacts"
            )
            if isinstance(
                payload,
                dict,
            )
            else None
        ),
        "next_notebook": (
            payload.get(
                "next_notebook"
            )
            if isinstance(
                payload,
                dict,
            )
            else None
        ),
    })


print()
print("PRIOR INGESTION REPORT CONTRACTS")
print("=" * 110)

for item in (
    prior_report_contracts
):
    print()
    print(
        "File:",
        item[
            "relative_path"
        ],
    )
    print(
        "Top-level type:",
        item[
            "top_level_type"
        ],
    )
    print(
        "Top-level keys:",
        item[
            "top_level_keys"
        ],
    )
    print(
        "Artifact type:",
        item[
            "artifact_type"
        ],
    )
    print(
        "Report type:",
        item[
            "report_type"
        ],
    )
    print(
        "Artifact version:",
        item[
            "artifact_version"
        ],
    )
    print(
        "Report version:",
        item[
            "report_version"
        ],
    )
    print(
        "Result:",
        item[
            "result"
        ],
    )
    print(
        "Output artifact:",
        item[
            "output_artifact"
        ],
    )
    print(
        "Artifacts:",
        item[
            "artifacts"
        ],
    )
    print(
        "Next notebook:",
        item[
            "next_notebook"
        ],
    )


phase2_nonraw_files = []

phase2_data_root = (
    project_root
    / "data"
    / "phase2"
)

for subdir_name in [
    "interim",
    "processed",
    "manifests",
    "chunks",
    "evaluation",
]:
    subdir = (
        phase2_data_root
        / subdir_name
    )

    if not subdir.exists():
        continue

    for path in sorted(
        subdir.rglob("*"),
        key=lambda item: (
            item.relative_to(
                project_root
            ).as_posix()
        ),
    ):
        if path.is_file():
            phase2_nonraw_files.append(
                path.relative_to(
                    project_root
                ).as_posix()
            )


print()
print("PHASE 2 NON-RAW DATA FILES")
print("=" * 110)

if phase2_nonraw_files:
    for relative_path in (
        phase2_nonraw_files
    ):
        print(
            "-",
            relative_path,
        )
else:
    print("- NONE")


checkpoint_named_files = [
    path.relative_to(
        project_root
    ).as_posix()
    for path
    in project_root.rglob(
        "*checkpoint*"
    )
    if (
        path.is_file()
        and ".git"
        not in path.parts
    )
]


print()
print("EXISTING CHECKPOINT-NAMED FILES")
print("=" * 110)

if checkpoint_named_files:
    for relative_path in sorted(
        checkpoint_named_files
    ):
        print(
            "-",
            relative_path,
        )
else:
    print("- NONE")


high_value_contexts = []

for hit in (
    prior_ingestion_notebook_hits
):
    source_lower = (
        hit[
            "source"
        ].lower()
    )

    if (
        "book_ingestion_acceptance_v1.json"
        in source_lower
        or "website_acceptance_v1.json"
        in source_lower
        or "data/phase2/interim"
        in source_lower
        or "data/phase2/processed"
        in source_lower
    ):
        high_value_contexts.append(
            hit
        )


print()
print("HIGH-VALUE PRIOR INGESTION CONTEXTS")
print("=" * 110)

if high_value_contexts:
    for hit_index, hit in enumerate(
        high_value_contexts,
        start=1,
    ):
        print()
        print(
            "Context:",
            hit_index,
        )
        print(
            "Notebook:",
            hit[
                "relative_path"
            ],
        )
        print(
            "Cell:",
            hit[
                "cell_index"
            ],
        )
        print(
            "Cell type:",
            hit[
                "cell_type"
            ],
        )
        print(
            "Matched:",
            hit[
                "matched_terms"
            ],
        )
        print()

        print(
            hit[
                "source"
            ][:6000]
        )

        print()
        print("-" * 110)
else:
    print("- NONE")


prior_ingestion_contract_stream_after = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


assert prior_report_contracts, (
    "No prior Phase 2 ingestion acceptance "
    "report could be inspected."
)

assert (
    prior_ingestion_contract_stream_before
    == prior_ingestion_contract_stream_after
), (
    "Prior-ingestion contract discovery "
    "mutated the repaired transcript stream."
)

print()
print(
    "PRIOR INGESTION ARTIFACT CONTRACT DISCOVERY: PASS"
)
print(
    "WRITE OPERATIONS: NOT PERFORMED"
)

Phase 2 notebooks scanned: 7
Prior-ingestion contract cells: 58

PRIOR INGESTION PATH REFERENCES
- reports/phase2/environment_baseline_v1.json <- notebooks/phase2/00_environment_and_baseline.ipynb cell:25 [markdown]
- reports/phase2/environment_baseline_v1.json <- notebooks/phase2/00_environment_and_baseline.ipynb cell:29 [markdown]
- data/phase2/raw/ <- notebooks/phase2/01_website_ingestion.ipynb cell:19 [markdown]
- data/phase2/interim/ <- notebooks/phase2/01_website_ingestion.ipynb cell:19 [markdown]
- data/phase2/processed/ <- notebooks/phase2/01_website_ingestion.ipynb cell:19 [markdown]
- data/phase2/chunks/ <- notebooks/phase2/01_website_ingestion.ipynb cell:19 [markdown]
- data/phase2/manifests/ <- notebooks/phase2/01_website_ingestion.ipynb cell:19 [markdown]
- data/phase2/evaluation/ <- notebooks/phase2/01_website_ingestion.ipynb cell:19 [markdown]
- data/phase2/raw/website <- notebooks/phase2/01_website_ingestion.ipynb cell:22 [markdown]
- data/phase2/interim <- notebooks/ph

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت Artifactهای رسمی Transcript Ingestion</h3>

<p>
بررسی Contract مراحل قبلی Phase 2 با موفقیت تکمیل شد.
</p>

<p>
Evidence واقعی Repository نشان می‌دهد:
</p>

<ul>
<li><code>data/phase2/interim</code> محل خروجی اولیه استخراج متن و Metadata است.</li>
<li><code>data/phase2/processed</code> برای داده‌هایی است که در مراحل بعد پاک‌سازی یا استاندارد می‌شوند.</li>
<li><code>reports/phase2</code> محل Acceptance و Quality Reportها است.</li>
<li>در Notebookهای قبلی هیچ فایل مستقلی با نام یا نقش <code>checkpoint</code> وجود ندارد.</li>
<li>در Book Ingestion نیز Artifact اولیه در <code>interim</code> و Acceptance Report در <code>reports/phase2</code> ثبت شده است.</li>
</ul>

<h4>Contract نهایی NB06</h4>

<p>
بر همین اساس Artifactهای رسمی این Notebook به شکل زیر ثبت می‌شوند:
</p>

<ul>
<li><code>data/phase2/interim/transcripts_extracted_v1.jsonl</code></li>
<li><code>data/phase2/interim/speaker_aliases_v1.json</code></li>
<li><code>reports/phase2/transcript_quality_report.json</code></li>
</ul>

<p>
فایل Checkpoint مستقل ساخته نمی‌شود.
</p>

<p>
پس از Independent Validation، Acceptance Report نهایی:
</p>

<p>
<code>reports/phase2/transcript_ingestion_acceptance_v1.json</code>
</p>

<p>
نقش Final Checkpoint همین Notebook را خواهد داشت؛ همان الگویی که در Ingestion کتاب استفاده شده است.
</p>

<h4>Transcript Artifact</h4>

<p>
Artifact اصلی شامل تمام <code>16226</code> Segment نهایی Stream اصلاح‌شده خواهد بود.
</p>

<p>
برای هر Segment موارد زیر حفظ می‌شوند:
</p>

<ul>
<li>Stable Utterance ID</li>
<li>Program و Transcript Unit</li>
<li>Physical File Lineage</li>
<li>Source SHA-256</li>
<li>Rights Status</li>
<li>Source Line Range</li>
<li>Parsed Text</li>
<li>Raw Source Text</li>
<li>Speaker Attribution پایه</li>
<li>Human Attribution Overlay در صورت وجود</li>
<li>Merge Lineage</li>
<li>Locator ID</li>
<li>Source Timestamp Annotation در صورت وجود</li>
</ul>

<h4>اصل مهم Speaker</h4>

<p>
Human Review باعث بازنویسی Attribution پایه نمی‌شود.
</p>

<p>
دو تصمیم انسانی به‌صورت Overlay جداگانه حفظ می‌شوند:
</p>

<ul>
<li>پاسخ چهلستون ۸ → دکتر امین‌داور</li>
<li>پاسخ چهلستون ۶ → دکتر کمیل رودی</li>
</ul>

<p>
بنابراین هم Evidence اولیه Parser و هم تصمیم Human Review قابل Audit باقی می‌مانند.
</p>

<h4>Quality Report</h4>

<p>
Quality Report شامل موارد زیر خواهد بود:
</p>

<ul>
<li>تمام <code>133</code> Quality Candidate نهایی</li>
<li>تمام <code>16</code> Source Timestamp Annotation</li>
<li>نتیجه Human Review</li>
<li>۳ Merge تأییدشده انسانی</li>
<li>۲ Deferred Text Cleanup Candidate</li>
<li>Processing State و محدودیت‌های باقی‌مانده</li>
</ul>

<h4>عدم انجام پردازش خارج از Scope</h4>

<ul>
<li>Persian normalization انجام نمی‌شود.</li>
<li>اصلاح غلط تایپی یا ASR اعمال نمی‌شود.</li>
<li>Deduplication انجام نمی‌شود.</li>
<li>Chunking انجام نمی‌شود.</li>
<li>Unified Schema ساخته نمی‌شود.</li>
<li>Speaker ناشناخته حدس زده نمی‌شود.</li>
</ul>

<h4>سیاست Write</h4>

<p>
هر سه Artifact به‌صورت Idempotent نوشته می‌شوند:
</p>

<ul>
<li><code>Missing → CREATED</code></li>
<li><code>Existing + Identical → REUSED_IDENTICAL</code></li>
<li><code>Existing + Different → FAIL</code></li>
</ul>

<p>
هیچ Artifact متفاوتی Silent Overwrite نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Artifact Transcript دقیقاً ۱۶۲۲۶ Record داشته باشد.</li>
<li>تمام Utterance IDها یکتا باشند.</li>
<li>تمام Segmentها Lineage معتبر Manifest داشته باشند.</li>
<li>هر ۱۶ Timestamp Annotation حفظ شود.</li>
<li>۲ Human Attribution Overlay حفظ شود.</li>
<li>۱۳۳ Quality Candidate در Report ثبت شود.</li>
<li>۲ Deferred Cleanup Candidate بدون اعمال Correction ثبت شوند.</li>
<li>هیچ Segment Timestamp ساختگی ایجاد نشود.</li>
<li>هیچ Runtime Stream تغییر نکند.</li>
<li>هر سه Artifact از روی Disk مجدداً قابل خواندن باشند.</li>
</ul>

</div>

In [45]:
# Code Cell 45

import hashlib
import json
from collections import Counter, defaultdict


def to_json_ready(value):
    if isinstance(value, dict):
        return {
            str(key): to_json_ready(item)
            for key, item in value.items()
        }

    if isinstance(value, set):
        return sorted(
            to_json_ready(item)
            for item in value
        )

    if isinstance(value, tuple):
        return [
            to_json_ready(item)
            for item in value
        ]

    if isinstance(value, list):
        return [
            to_json_ready(item)
            for item in value
        ]

    return value


def write_text_idempotent(
    path,
    candidate_text,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        existing_text = path.read_text(
            encoding="utf-8"
        )

        if existing_text != candidate_text:
            raise ValueError(
                "Existing artifact differs from "
                "the deterministic candidate: "
                f"{path.relative_to(project_root)}"
            )

        return "REUSED_IDENTICAL"

    path.write_text(
        candidate_text,
        encoding="utf-8",
    )

    return "CREATED"


artifact_write_stream_before = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)


manifest_path_nb06 = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "transcript_manifest_v1.json"
)

manifest_bytes_nb06 = (
    manifest_path_nb06.read_bytes()
)

manifest_sha256_nb06 = (
    hashlib.sha256(
        manifest_bytes_nb06
    ).hexdigest()
)

manifest_disk_nb06 = json.loads(
    manifest_bytes_nb06.decode(
        "utf-8"
    )
)

accepted_disk_nb06 = (
    manifest_disk_nb06[
        "accepted_files"
    ]
)

accepted_disk_by_path_nb06 = {
    record[
        "relative_path"
    ]: record
    for record
    in accepted_disk_nb06
}


human_override_by_utterance_id = {
    record[
        "utterance_id"
    ]: record
    for record
    in human_attribution_overrides_repaired_v1
}


timestamp_annotation_map_nb06 = {
    utterance_id: list(
        annotations
    )
    for utterance_id, annotations
    in timestamp_annotations_by_utterance_id.items()
}


transcript_export_records_v1 = []

missing_manifest_lineage = []
missing_locator_links = []

for utterance in (
    all_merged_utterances_repaired_v1
):
    utterance_id = (
        utterance[
            "utterance_id"
        ]
    )

    relative_path = (
        utterance[
            "relative_path"
        ]
    )

    manifest_record = (
        accepted_disk_by_path_nb06.get(
            relative_path
        )
    )

    if manifest_record is None:
        missing_manifest_lineage.append(
            utterance_id
        )
        continue

    locator = (
        locator_repaired_by_utterance_id.get(
            utterance_id
        )
    )

    if locator is None:
        missing_locator_links.append(
            utterance_id
        )
        continue

    human_override = (
        human_override_by_utterance_id.get(
            utterance_id
        )
    )

    if human_override is not None:
        effective_speaker_label = (
            human_override[
                "human_confirmed_speaker"
            ]
        )

        effective_speaker_canonical = (
            human_override[
                "human_confirmed_speaker_canonical"
            ]
        )

        effective_primary_subject = (
            human_override[
                "primary_dr_roudi_corpus"
            ]
        )

        effective_speaker_basis = (
            "HUMAN_ATTRIBUTION_OVERLAY"
        )

    else:
        effective_speaker_canonical = (
            utterance[
                "speaker_canonical"
            ]
        )

        effective_speaker_label = (
            utterance[
                "speaker_raw"
            ]
            or utterance[
                "speaker_source_canonical"
            ]
            or effective_speaker_canonical
        )

        effective_primary_subject = (
            True
            if effective_speaker_canonical
            == "Dr. Komeil Roudi"
            else None
        )

        effective_speaker_basis = (
            "BASE_PARSER_ATTRIBUTION"
        )

    segment_timestamp_annotations = (
        timestamp_annotation_map_nb06.get(
            utterance_id,
            [],
        )
    )

    transcript_export_records_v1.append({
        "record_type": (
            "transcript_segment"
        ),
        "record_version": "v1",
        "schema_version": (
            "transcript_segment_v1"
        ),
        "utterance_id": (
            utterance_id
        ),
        "utterance_index": (
            utterance[
                "utterance_index"
            ]
        ),
        "program": (
            utterance[
                "program"
            ]
        ),
        "program_id": (
            utterance[
                "program_id"
            ]
        ),
        "transcript_unit_id": (
            utterance[
                "transcript_unit_id"
            ]
        ),
        "source_identity": {
            "physical_file_id": (
                utterance[
                    "physical_file_id"
                ]
            ),
            "relative_path": (
                relative_path
            ),
            "source_sha256": (
                manifest_record[
                    "sha256"
                ]
            ),
            "episode_number": (
                manifest_record.get(
                    "episode_number"
                )
            ),
            "season_number": (
                manifest_record.get(
                    "season_number"
                )
            ),
            "source_episode_label": (
                manifest_record.get(
                    "source_episode_label"
                )
            ),
            "inventory_decision": (
                manifest_record[
                    "inventory_decision"
                ]
            ),
        },
        "rights": {
            "status": (
                manifest_record[
                    "rights_status"
                ]
            ),
            "basis": (
                manifest_record.get(
                    "rights_basis"
                )
            ),
        },
        "source_locator": {
            "locator_id": (
                locator[
                    "locator_id"
                ]
            ),
            "source_line_start": (
                utterance[
                    "source_line_start"
                ]
            ),
            "source_line_end": (
                utterance[
                    "source_line_end"
                ]
            ),
        },
        "speaker": {
            "base": {
                "raw": (
                    utterance[
                        "speaker_raw"
                    ]
                ),
                "canonical": (
                    utterance[
                        "speaker_canonical"
                    ]
                ),
                "source_canonical": (
                    utterance[
                        "speaker_source_canonical"
                    ]
                ),
                "status": (
                    utterance[
                        "speaker_status"
                    ]
                ),
                "boundary_type": (
                    utterance[
                        "boundary_type"
                    ]
                ),
            },
            "human_attribution_overlay": (
                human_override
            ),
            "effective": {
                "label": (
                    effective_speaker_label
                ),
                "canonical": (
                    effective_speaker_canonical
                ),
                "primary_dr_roudi_corpus": (
                    effective_primary_subject
                ),
                "basis": (
                    effective_speaker_basis
                ),
            },
        },
        "content": {
            "text": (
                utterance[
                    "text"
                ]
            ),
            "text_sha256": (
                hashlib.sha256(
                    utterance[
                        "text"
                    ].encode(
                        "utf-8"
                    )
                ).hexdigest()
            ),
            "source_text": (
                utterance[
                    "source_text"
                ]
            ),
            "source_text_sha256": (
                hashlib.sha256(
                    utterance[
                        "source_text"
                    ].encode(
                        "utf-8"
                    )
                ).hexdigest()
            ),
            "text_state": (
                "SOURCE_PRESERVED_NOT_NORMALIZED"
            ),
        },
        "merge": {
            "status": (
                utterance.get(
                    "merge_status",
                    "UNCHANGED",
                )
            ),
            "merged_from_utterance_ids": (
                utterance.get(
                    "merged_from_utterance_ids",
                    []
                )
            ),
        },
        "timestamps": {
            "segment_start_time": (
                utterance[
                    "start_time"
                ]
            ),
            "segment_end_time": (
                utterance[
                    "end_time"
                ]
            ),
            "source_annotations": (
                segment_timestamp_annotations
            ),
            "policy": (
                "NONE_UNLESS_SOURCE_PROVIDES_"
                "SEGMENT_LEVEL_EVIDENCE"
            ),
        },
        "processing_state": {
            "persian_normalization": False,
            "text_correction": False,
            "deduplication": False,
            "chunking": False,
            "unknown_speaker_inference": False,
        },
    })


manifest_alias_program_records = []

for program_record in sorted(
    manifest_disk_nb06[
        "programs"
    ],
    key=lambda item: (
        item[
            "program_id"
        ]
    ),
):
    manifest_alias_program_records.append({
        "program": (
            program_record[
                "program"
            ]
        ),
        "program_id": (
            program_record[
                "program_id"
            ]
        ),
        "speaker_policy": (
            program_record[
                "speaker_policy"
            ]
        ),
        "speaker_aliases": (
            program_record.get(
                "speaker_aliases",
                {}
            )
        ),
    })


speaker_aliases_artifact_v1 = {
    "artifact_type": (
        "phase2_transcript_speaker_registry"
    ),
    "artifact_version": "v1",
    "schema_version": (
        "speaker_alias_registry_v1"
    ),
    "notebook": (
        "notebooks/phase2/"
        "06_transcript_ingestion.ipynb"
    ),
    "source_manifest": {
        "relative_path": (
            "data/phase2/manifests/"
            "transcript_manifest_v1.json"
        ),
        "sha256": (
            manifest_sha256_nb06
        ),
    },
    "canonical_primary_speaker": (
        "Dr. Komeil Roudi"
    ),
    "unknown_speaker_inference_allowed": (
        False
    ),
    "program_count": (
        len(
            manifest_alias_program_records
        )
    ),
    "programs": (
        manifest_alias_program_records
    ),
    "source_observed_aliases": {
        "program:chehelston": (
            source_observed_aliases
        )
    },
    "human_attribution_overlays": (
        human_attribution_overrides_repaired_v1
    ),
    "attribution_policy": {
        "base_parser_attribution_preserved": (
            True
        ),
        "human_overlay_does_not_rewrite_base": (
            True
        ),
        "instructional_bold_is_independent_speaker": (
            False
        ),
        "instructional_bold_inherits_current_speaker": (
            True
        ),
    },
}


def candidate_field_value(
    record,
    candidate_keys,
    default_value="UNSPECIFIED",
):
    for key in candidate_keys:
        if key in record:
            return record[
                key
            ]

    return default_value


quality_type_counts_nb06 = Counter(
    candidate_field_value(
        candidate,
        [
            "quality_type",
            "candidate_type",
            "issue_type",
            "type",
        ],
    )
    for candidate
    in transcript_quality_candidates_repaired_v1
)

quality_status_counts_nb06 = Counter(
    candidate_field_value(
        candidate,
        [
            "review_status",
            "status",
            "quality_status",
        ],
    )
    for candidate
    in transcript_quality_candidates_repaired_v1
)

quality_program_counts_nb06 = Counter(
    candidate_field_value(
        candidate,
        [
            "program_id",
            "program",
        ],
    )
    for candidate
    in transcript_quality_candidates_repaired_v1
)


transcript_quality_report_v1 = {
    "report_type": (
        "phase2_transcript_quality"
    ),
    "report_version": "v1",
    "schema_version": (
        "transcript_quality_report_v1"
    ),
    "notebook": (
        "notebooks/phase2/"
        "06_transcript_ingestion.ipynb"
    ),
    "source_manifest": {
        "relative_path": (
            "data/phase2/manifests/"
            "transcript_manifest_v1.json"
        ),
        "sha256": (
            manifest_sha256_nb06
        ),
    },
    "accepted_scope": {
        "program_count": 13,
        "physical_file_count": 183,
        "final_segment_count": (
            len(
                transcript_export_records_v1
            )
        ),
    },
    "quality_summary": {
        "candidate_count": (
            len(
                transcript_quality_candidates_repaired_v1
            )
        ),
        "by_type": dict(
            sorted(
                quality_type_counts_nb06.items()
            )
        ),
        "by_status": dict(
            sorted(
                quality_status_counts_nb06.items()
            )
        ),
        "by_program": dict(
            sorted(
                quality_program_counts_nb06.items()
            )
        ),
        "automatic_correction_applied": (
            False
        ),
        "automatic_duplicate_removal_applied": (
            False
        ),
    },
    "quality_candidates": (
        transcript_quality_candidates_repaired_v1
    ),
    "timestamp_semantics": {
        "resolution": (
            timestamp_semantics_resolution_repaired_v1
        ),
        "annotations": (
            timestamp_annotations_repaired_v1
        ),
    },
    "human_review": {
        "result": (
            human_review_result_repaired_v1
        ),
        "attribution_overlays": (
            human_attribution_overrides_repaired_v1
        ),
        "merge_decisions": (
            human_merge_review_decisions_repaired_v1
        ),
        "deferred_text_cleanup": (
            deferred_text_cleanup_candidates_repaired_v1
        ),
    },
    "processing_state": {
        "speaker_parsing": (
            "COMPLETED"
        ),
        "lineage_validation": (
            "COMPLETED"
        ),
        "source_conservation": (
            "COMPLETED"
        ),
        "structural_human_review": (
            "PASS"
        ),
        "persian_normalization": (
            "NOT_PERFORMED"
        ),
        "text_cleanup": (
            "DEFERRED"
        ),
        "deduplication": (
            "NOT_PERFORMED"
        ),
        "chunking": (
            "NOT_PERFORMED"
        ),
        "unified_schema": (
            "NOT_PERFORMED"
        ),
    },
    "known_limitations": [
        (
            "Rights status remains "
            "EVIDENCE_REQUIRED."
        ),
        (
            "Unresolved speaker segments are "
            "preserved when source evidence "
            "is insufficient."
        ),
        (
            "Two human-confirmed text errors "
            "are deferred to later corpus cleanup."
        ),
        (
            "Quality candidates are retained "
            "without automatic correction."
        ),
        (
            "Segment timestamps remain null "
            "unless segment-level source evidence "
            "exists."
        ),
    ],
    "artifact_state": (
        "CREATED_PENDING_INDEPENDENT_VALIDATION"
    ),
}


assert len(
    transcript_export_records_v1
) == 16226, (
    "Final transcript export record count "
    "is not 16226."
)

export_utterance_ids = [
    record[
        "utterance_id"
    ]
    for record
    in transcript_export_records_v1
]

assert len(
    export_utterance_ids
) == len(
    set(
        export_utterance_ids
    )
), (
    "Duplicate utterance IDs found "
    "in transcript export."
)

assert not (
    missing_manifest_lineage
), (
    "Transcript export contains segments "
    "without Manifest lineage."
)

assert not (
    missing_locator_links
), (
    "Transcript export contains segments "
    "without locator links."
)

assert all(
    record[
        "timestamps"
    ][
        "segment_start_time"
    ]
    is None
    and record[
        "timestamps"
    ][
        "segment_end_time"
    ]
    is None
    for record
    in transcript_export_records_v1
), (
    "Unexpected whole-segment timestamp "
    "assignment found."
)

assert sum(
    len(
        record[
            "timestamps"
        ][
            "source_annotations"
        ]
    )
    for record
    in transcript_export_records_v1
) == 16, (
    "Timestamp annotation export count "
    "is not 16."
)

assert sum(
    1
    for record
    in transcript_export_records_v1
    if record[
        "speaker"
    ][
        "human_attribution_overlay"
    ]
    is not None
) == 2, (
    "Human attribution overlay count "
    "is not 2."
)

assert len(
    transcript_quality_candidates_repaired_v1
) == 133, (
    "Final quality candidate count "
    "is not 133."
)

assert len(
    deferred_text_cleanup_candidates_repaired_v1
) == 2, (
    "Deferred cleanup candidate count "
    "is not 2."
)

assert len(
    manifest_alias_program_records
) == 13, (
    "Speaker registry does not cover "
    "all 13 programs."
)


transcript_jsonl_lines = [
    json.dumps(
        to_json_ready(
            record
        ),
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )
    for record
    in transcript_export_records_v1
]

transcript_jsonl_candidate = (
    "\n".join(
        transcript_jsonl_lines
    )
    + "\n"
)

speaker_aliases_candidate = (
    json.dumps(
        to_json_ready(
            speaker_aliases_artifact_v1
        ),
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

quality_report_candidate = (
    json.dumps(
        to_json_ready(
            transcript_quality_report_v1
        ),
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)


transcript_artifact_path = (
    project_root
    / "data"
    / "phase2"
    / "interim"
    / "transcripts_extracted_v1.jsonl"
)

speaker_aliases_path = (
    project_root
    / "data"
    / "phase2"
    / "interim"
    / "speaker_aliases_v1.json"
)

transcript_quality_report_path = (
    project_root
    / "reports"
    / "phase2"
    / "transcript_quality_report.json"
)


transcript_artifact_action = (
    write_text_idempotent(
        transcript_artifact_path,
        transcript_jsonl_candidate,
    )
)

speaker_aliases_action = (
    write_text_idempotent(
        speaker_aliases_path,
        speaker_aliases_candidate,
    )
)

quality_report_action = (
    write_text_idempotent(
        transcript_quality_report_path,
        quality_report_candidate,
    )
)


disk_transcript_records = []

with transcript_artifact_path.open(
    "r",
    encoding="utf-8",
) as file:
    for line_number, line in enumerate(
        file,
        start=1,
    ):
        if not line.strip():
            continue

        disk_transcript_records.append(
            json.loads(
                line
            )
        )


disk_speaker_aliases = json.loads(
    speaker_aliases_path.read_text(
        encoding="utf-8"
    )
)

disk_quality_report = json.loads(
    transcript_quality_report_path.read_text(
        encoding="utf-8"
    )
)


assert len(
    disk_transcript_records
) == 16226, (
    "Stored transcript JSONL count mismatch."
)

assert len({
    record[
        "utterance_id"
    ]
    for record
    in disk_transcript_records
}) == 16226, (
    "Stored transcript JSONL has "
    "duplicate utterance IDs."
)

assert (
    disk_speaker_aliases[
        "program_count"
    ]
    == 13
), (
    "Stored speaker alias registry "
    "program count mismatch."
)

assert len(
    disk_speaker_aliases[
        "human_attribution_overlays"
    ]
) == 2, (
    "Stored human attribution overlay "
    "count mismatch."
)

assert (
    disk_quality_report[
        "accepted_scope"
    ][
        "final_segment_count"
    ]
    == 16226
), (
    "Stored quality report segment "
    "count mismatch."
)

assert (
    disk_quality_report[
        "quality_summary"
    ][
        "candidate_count"
    ]
    == 133
), (
    "Stored quality candidate count mismatch."
)

assert len(
    disk_quality_report[
        "timestamp_semantics"
    ][
        "annotations"
    ]
) == 16, (
    "Stored timestamp annotation "
    "count mismatch."
)

assert len(
    disk_quality_report[
        "human_review"
    ][
        "deferred_text_cleanup"
    ]
) == 2, (
    "Stored deferred text cleanup "
    "count mismatch."
)


artifact_write_stream_after = json.dumps(
    all_merged_utterances_repaired_v1,
    ensure_ascii=False,
    sort_keys=True,
)

assert (
    artifact_write_stream_before
    == artifact_write_stream_after
), (
    "Artifact creation mutated "
    "the repaired transcript stream."
)


print("NB06 OFFICIAL ARTIFACTS")
print("=" * 110)

print()
print(
    "Transcript artifact:",
    transcript_artifact_path.relative_to(
        project_root
    ),
)
print(
    "Action:",
    transcript_artifact_action,
)
print(
    "Records:",
    len(
        disk_transcript_records
    ),
)
print(
    "Size bytes:",
    transcript_artifact_path.stat().st_size,
)

print()
print(
    "Speaker registry:",
    speaker_aliases_path.relative_to(
        project_root
    ),
)
print(
    "Action:",
    speaker_aliases_action,
)
print(
    "Programs:",
    disk_speaker_aliases[
        "program_count"
    ],
)
print(
    "Human attribution overlays:",
    len(
        disk_speaker_aliases[
            "human_attribution_overlays"
        ]
    ),
)

print()
print(
    "Quality report:",
    transcript_quality_report_path.relative_to(
        project_root
    ),
)
print(
    "Action:",
    quality_report_action,
)
print(
    "Quality candidates:",
    disk_quality_report[
        "quality_summary"
    ][
        "candidate_count"
    ],
)
print(
    "Timestamp annotations:",
    len(
        disk_quality_report[
            "timestamp_semantics"
        ][
            "annotations"
        ]
    ),
)
print(
    "Deferred cleanup candidates:",
    len(
        disk_quality_report[
            "human_review"
        ][
            "deferred_text_cleanup"
        ]
    ),
)

print()
print(
    "Source Manifest SHA-256:",
    manifest_sha256_nb06,
)

print()
print(
    "NB06 OFFICIAL ARTIFACT CREATION GATE: PASS"
)
print(
    "INDEPENDENT DISK VALIDATION: PENDING"
)

NB06 OFFICIAL ARTIFACTS

Transcript artifact: data/phase2/interim/transcripts_extracted_v1.jsonl
Action: REUSED_IDENTICAL
Records: 16226
Size bytes: 39055066

Speaker registry: data/phase2/interim/speaker_aliases_v1.json
Action: REUSED_IDENTICAL
Programs: 13
Human attribution overlays: 2

Quality report: reports/phase2/transcript_quality_report.json
Action: REUSED_IDENTICAL
Quality candidates: 133
Timestamp annotations: 16
Deferred cleanup candidates: 2

Source Manifest SHA-256: b328f512c67971b5e0e1f82cd5e9fe2ea148a89db3fbde31aebb474e78c6a9eb

NB06 OFFICIAL ARTIFACT CREATION GATE: PASS
INDEPENDENT DISK VALIDATION: PENDING


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اعتبارسنجی مستقل Artifactهای Transcript Ingestion از روی Disk و Sourceهای اصلی</h3>

<p>
سه Artifact رسمی NB06 با موفقیت ساخته شدند:
</p>

<ul>
<li><code>data/phase2/interim/transcripts_extracted_v1.jsonl</code></li>
<li><code>data/phase2/interim/speaker_aliases_v1.json</code></li>
<li><code>reports/phase2/transcript_quality_report.json</code></li>
</ul>

<p>
وجود این فایل‌ها به‌تنهایی برای پذیرش Notebook کافی نیست.
</p>

<p>
در این Cell هر سه Artifact مجدداً مستقیماً از Disk خوانده می‌شوند و با Manifest و ۱۸۳ Source اصلی مقایسه خواهند شد.
</p>

<h4>اعتبارسنجی Transcript Artifact</h4>

<ul>
<li>تعداد Recordها دقیقاً ۱۶۲۲۶ باشد.</li>
<li>تمام <code>utterance_id</code>ها یکتا باشند.</li>
<li>تمام Locator IDها یکتا باشند.</li>
<li>هر Segment به یکی از ۱۸۳ Physical File پذیرفته‌شده متصل باشد.</li>
<li><code>physical_file_id</code>، <code>source_sha256</code>، Program و Transcript Unit با Manifest سازگار باشند.</li>
<li>Rights Status تمام Segmentها همان Policy Manifest را حفظ کرده باشد.</li>
<li>Source Line Range معتبر باشد.</li>
<li><code>source_text</code> مستقیماً از همان Lineهای فایل اصلی قابل بازسازی باشد.</li>
<li>Hash متن Parsed و Source Text صحیح باشد.</li>
<li>هیچ Segment Timestamp ساختگی وجود نداشته باشد.</li>
<li>Persian Normalization، Text Correction، Deduplication و Chunking انجام نشده باشند.</li>
</ul>

<h4>اعتبارسنجی مستقیم ۱۸۳ Source File</h4>

<p>
برای تمام فایل‌های Accepted دوباره SHA-256 از خود فایل فیزیکی محاسبه می‌شود و باید با Manifest برابر باشد.
</p>

<h4>اعتبارسنجی Speaker Registry</h4>

<ul>
<li>هر ۱۳ Program پوشش داده شده باشد.</li>
<li>Unknown Speaker Inference همچنان ممنوع باشد.</li>
<li>دو Human Attribution Overlay وجود داشته باشد.</li>
<li>Overlayهای Registry با Overlayهای ذخیره‌شده در Transcript Artifact و Quality Report یکسان باشند.</li>
</ul>

<h4>اعتبارسنجی Timestamp Annotation</h4>

<ul>
<li>دقیقاً ۱۶ Annotation در Transcript Artifact وجود داشته باشد.</li>
<li>همین ۱۶ Annotation در Quality Report نیز ثبت شده باشند.</li>
<li>Annotation IDها یکتا باشند.</li>
<li>هر Annotation به Segment واقعی متصل باشد.</li>
<li>Start و End داخل Marker معتبر باشند.</li>
<li>هیچ Annotation به Start یا End کل Segment تبدیل نشده باشد.</li>
</ul>

<h4>اعتبارسنجی Quality و Human Review</h4>

<ul>
<li>۱۳۳ Quality Candidate وجود داشته باشد.</li>
<li>تمام Candidate IDها یکتا باشند.</li>
<li>تمام Utterance Referenceهای موجود در Candidateها به Segment واقعی اشاره کنند.</li>
<li>Human Review برابر <code>PASS</code> باشد.</li>
<li>۳ Merge انسانی تأییدشده ثبت شده باشد.</li>
<li>۲ Deferred Text Cleanup Candidate وجود داشته باشد.</li>
<li>هیچ Deferred Correction در NB06 اعمال نشده باشد.</li>
</ul>

<h4>اصل این Cell</h4>

<p>
این مرحله کاملاً Read-only است.
</p>

<p>
هیچ Artifact، Source، Transcript یا Metadata بازنویسی نمی‌شود.
</p>

<h4>معیار PASS</h4>

<p>
تمام بررسی‌های Identity، Lineage، Hash، Source Reconstruction، Speaker Attribution، Timestamp Preservation، Quality و Human Review باید بدون هیچ Mismatch عبور کنند.
</p>

</div>

In [46]:
# Code Cell 46

import hashlib
import json
from collections import Counter, defaultdict
from pathlib import Path


transcript_path_disk = (
    project_root
    / "data"
    / "phase2"
    / "interim"
    / "transcripts_extracted_v1.jsonl"
)

speaker_registry_path_disk = (
    project_root
    / "data"
    / "phase2"
    / "interim"
    / "speaker_aliases_v1.json"
)

quality_report_path_disk = (
    project_root
    / "reports"
    / "phase2"
    / "transcript_quality_report.json"
)

manifest_path_disk_nb06 = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "transcript_manifest_v1.json"
)


required_disk_files = [
    transcript_path_disk,
    speaker_registry_path_disk,
    quality_report_path_disk,
    manifest_path_disk_nb06,
]

missing_artifact_files = [
    path.relative_to(
        project_root
    ).as_posix()
    for path
    in required_disk_files
    if not path.is_file()
]

assert not missing_artifact_files, (
    "Required NB06 files are missing: "
    f"{missing_artifact_files}"
)


artifact_sha256 = {}

for path in required_disk_files:
    artifact_sha256[
        path.relative_to(
            project_root
        ).as_posix()
    ] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


manifest_disk = json.loads(
    manifest_path_disk_nb06.read_text(
        encoding="utf-8"
    )
)

speaker_registry_disk = json.loads(
    speaker_registry_path_disk.read_text(
        encoding="utf-8"
    )
)

quality_report_disk = json.loads(
    quality_report_path_disk.read_text(
        encoding="utf-8"
    )
)


transcript_records_disk = []

with transcript_path_disk.open(
    "r",
    encoding="utf-8",
) as file:
    for line_number, line in enumerate(
        file,
        start=1,
    ):
        if not line.strip():
            continue

        try:
            record = json.loads(
                line
            )
        except Exception as exc:
            raise AssertionError(
                "Invalid JSONL record at "
                f"line {line_number}: {exc}"
            )

        transcript_records_disk.append(
            record
        )


accepted_manifest_records = (
    manifest_disk.get(
        "accepted_files",
        []
    )
)

manifest_program_records = (
    manifest_disk.get(
        "programs",
        []
    )
)

accepted_manifest_by_path = {
    record[
        "relative_path"
    ]: record
    for record
    in accepted_manifest_records
}


print("INDEPENDENT DISK INPUT")
print("=" * 110)

print(
    "Manifest accepted files:",
    len(
        accepted_manifest_records
    ),
)
print(
    "Manifest programs:",
    len(
        manifest_program_records
    ),
)
print(
    "Transcript records:",
    len(
        transcript_records_disk
    ),
)
print(
    "Speaker registry programs:",
    speaker_registry_disk.get(
        "program_count"
    ),
)
print(
    "Quality candidates:",
    quality_report_disk.get(
        "quality_summary",
        {},
    ).get(
        "candidate_count"
    ),
)


assert len(
    accepted_manifest_records
) == 183, (
    "Manifest accepted file count "
    "is not 183."
)

assert len(
    manifest_program_records
) == 13, (
    "Manifest program count "
    "is not 13."
)

assert len(
    transcript_records_disk
) == 16226, (
    "Stored transcript segment count "
    "is not 16226."
)


manifest_sha256_actual = (
    artifact_sha256[
        "data/phase2/manifests/"
        "transcript_manifest_v1.json"
    ]
)

assert (
    speaker_registry_disk[
        "source_manifest"
    ][
        "sha256"
    ]
    == manifest_sha256_actual
), (
    "Speaker registry Manifest SHA-256 "
    "does not match the actual Manifest."
)

assert (
    quality_report_disk[
        "source_manifest"
    ][
        "sha256"
    ]
    == manifest_sha256_actual
), (
    "Quality report Manifest SHA-256 "
    "does not match the actual Manifest."
)


source_hash_mismatches = []
source_missing_files = []
source_outside_repository = []
source_line_cache = {}

project_root_resolved = (
    project_root.resolve()
)

for manifest_record in (
    accepted_manifest_records
):
    relative_path = (
        manifest_record[
            "relative_path"
        ]
    )

    source_path = (
        project_root
        / relative_path
    )

    try:
        source_path.resolve().relative_to(
            project_root_resolved
        )
    except ValueError:
        source_outside_repository.append(
            relative_path
        )
        continue

    if not source_path.is_file():
        source_missing_files.append(
            relative_path
        )
        continue

    source_bytes = (
        source_path.read_bytes()
    )

    actual_source_sha256 = (
        hashlib.sha256(
            source_bytes
        ).hexdigest()
    )

    if (
        actual_source_sha256
        != manifest_record[
            "sha256"
        ]
    ):
        source_hash_mismatches.append({
            "relative_path": (
                relative_path
            ),
            "manifest_sha256": (
                manifest_record[
                    "sha256"
                ]
            ),
            "actual_sha256": (
                actual_source_sha256
            ),
        })

    source_line_cache[
        relative_path
    ] = source_bytes.decode(
        "utf-8"
    ).splitlines()


print()
print("SOURCE FILE INTEGRITY")
print("=" * 110)
print(
    "Accepted files checked:",
    len(
        accepted_manifest_records
    ),
)
print(
    "Missing Source files:",
    len(
        source_missing_files
    ),
)
print(
    "Source files outside repository:",
    len(
        source_outside_repository
    ),
)
print(
    "Source SHA-256 mismatches:",
    len(
        source_hash_mismatches
    ),
)


assert not source_missing_files, (
    "Accepted Source files are missing."
)

assert not source_outside_repository, (
    "Accepted Source path outside repository "
    "was detected."
)

assert not source_hash_mismatches, (
    "Accepted Source SHA-256 mismatch found."
)


utterance_ids = []
locator_ids = []

unknown_manifest_paths = []
manifest_lineage_mismatches = []
source_range_failures = []
source_text_failures = []
text_hash_failures = []
source_text_hash_failures = []
timestamp_assignment_failures = []
processing_state_failures = []
rights_failures = []
schema_failures = []

timestamp_annotations_from_segments = []

human_overlays_from_segments = []

program_segment_counts = Counter()
speaker_status_counts = Counter()


for record in (
    transcript_records_disk
):
    utterance_id = (
        record.get(
            "utterance_id"
        )
    )

    utterance_ids.append(
        utterance_id
    )

    if (
        record.get(
            "record_type"
        )
        != "transcript_segment"
        or record.get(
            "record_version"
        )
        != "v1"
        or record.get(
            "schema_version"
        )
        != "transcript_segment_v1"
    ):
        schema_failures.append(
            utterance_id
        )

    program_segment_counts[
        record.get(
            "program_id"
        )
    ] += 1

    speaker_status_counts[
        record.get(
            "speaker",
            {}
        ).get(
            "base",
            {}
        ).get(
            "status"
        )
    ] += 1

    source_identity = (
        record[
            "source_identity"
        ]
    )

    relative_path = (
        source_identity[
            "relative_path"
        ]
    )

    manifest_record = (
        accepted_manifest_by_path.get(
            relative_path
        )
    )

    if manifest_record is None:
        unknown_manifest_paths.append({
            "utterance_id": (
                utterance_id
            ),
            "relative_path": (
                relative_path
            ),
        })
        continue

    expected_lineage = {
        "physical_file_id": (
            manifest_record[
                "physical_file_id"
            ]
        ),
        "source_sha256": (
            manifest_record[
                "sha256"
            ]
        ),
        "episode_number": (
            manifest_record.get(
                "episode_number"
            )
        ),
        "season_number": (
            manifest_record.get(
                "season_number"
            )
        ),
        "source_episode_label": (
            manifest_record.get(
                "source_episode_label"
            )
        ),
        "inventory_decision": (
            manifest_record[
                "inventory_decision"
            ]
        ),
    }

    observed_lineage = {
        key: source_identity.get(
            key
        )
        for key
        in expected_lineage
    }

    if (
        observed_lineage
        != expected_lineage
    ):
        manifest_lineage_mismatches.append({
            "utterance_id": (
                utterance_id
            ),
            "relative_path": (
                relative_path
            ),
            "expected": (
                expected_lineage
            ),
            "observed": (
                observed_lineage
            ),
        })

    if (
        record.get(
            "program"
        )
        != manifest_record[
            "program"
        ]
        or record.get(
            "program_id"
        )
        != manifest_record[
            "program_id"
        ]
        or record.get(
            "transcript_unit_id"
        )
        != manifest_record[
            "transcript_unit_id"
        ]
    ):
        manifest_lineage_mismatches.append({
            "utterance_id": (
                utterance_id
            ),
            "relative_path": (
                relative_path
            ),
            "reason": (
                "PROGRAM_OR_TRANSCRIPT_UNIT_MISMATCH"
            ),
        })

    record_rights = (
        record[
            "rights"
        ]
    )

    if (
        record_rights[
            "status"
        ]
        != manifest_record[
            "rights_status"
        ]
        or record_rights.get(
            "basis"
        )
        != manifest_record.get(
            "rights_basis"
        )
    ):
        rights_failures.append(
            utterance_id
        )

    locator = (
        record[
            "source_locator"
        ]
    )

    locator_id = (
        locator[
            "locator_id"
        ]
    )

    locator_ids.append(
        locator_id
    )

    source_line_start = (
        locator[
            "source_line_start"
        ]
    )

    source_line_end = (
        locator[
            "source_line_end"
        ]
    )

    source_lines = (
        source_line_cache[
            relative_path
        ]
    )

    if not (
        isinstance(
            source_line_start,
            int,
        )
        and isinstance(
            source_line_end,
            int,
        )
        and 1
        <= source_line_start
        <= source_line_end
        <= len(
            source_lines
        )
    ):
        source_range_failures.append({
            "utterance_id": (
                utterance_id
            ),
            "relative_path": (
                relative_path
            ),
            "source_line_start": (
                source_line_start
            ),
            "source_line_end": (
                source_line_end
            ),
            "file_line_count": (
                len(
                    source_lines
                )
            ),
        })
        continue

    reconstructed_source_text = "\n".join(
        source_lines[
            source_line_start - 1:
            source_line_end
        ]
    )

    stored_content = (
        record[
            "content"
        ]
    )

    if (
        reconstructed_source_text
        != stored_content[
            "source_text"
        ]
    ):
        source_text_failures.append({
            "utterance_id": (
                utterance_id
            ),
            "relative_path": (
                relative_path
            ),
            "source_line_start": (
                source_line_start
            ),
            "source_line_end": (
                source_line_end
            ),
        })

    actual_text_sha256 = (
        hashlib.sha256(
            stored_content[
                "text"
            ].encode(
                "utf-8"
            )
        ).hexdigest()
    )

    if (
        actual_text_sha256
        != stored_content[
            "text_sha256"
        ]
    ):
        text_hash_failures.append(
            utterance_id
        )

    actual_source_text_sha256 = (
        hashlib.sha256(
            stored_content[
                "source_text"
            ].encode(
                "utf-8"
            )
        ).hexdigest()
    )

    if (
        actual_source_text_sha256
        != stored_content[
            "source_text_sha256"
        ]
    ):
        source_text_hash_failures.append(
            utterance_id
        )

    timestamps = (
        record[
            "timestamps"
        ]
    )

    if (
        timestamps[
            "segment_start_time"
        ]
        is not None
        or timestamps[
            "segment_end_time"
        ]
        is not None
    ):
        timestamp_assignment_failures.append(
            utterance_id
        )

    for annotation in (
        timestamps.get(
            "source_annotations",
            []
        )
    ):
        timestamp_annotations_from_segments.append(
            annotation
        )

    processing_state = (
        record[
            "processing_state"
        ]
    )

    if (
        processing_state.get(
            "persian_normalization"
        )
        is not False
        or processing_state.get(
            "text_correction"
        )
        is not False
        or processing_state.get(
            "deduplication"
        )
        is not False
        or processing_state.get(
            "chunking"
        )
        is not False
        or processing_state.get(
            "unknown_speaker_inference"
        )
        is not False
    ):
        processing_state_failures.append(
            utterance_id
        )

    human_overlay = (
        record[
            "speaker"
        ].get(
            "human_attribution_overlay"
        )
    )

    if (
        human_overlay
        is not None
    ):
        human_overlays_from_segments.append(
            human_overlay
        )


print()
print("TRANSCRIPT ARTIFACT LINEAGE")
print("=" * 110)

print(
    "Segments checked:",
    len(
        transcript_records_disk
    ),
)
print(
    "Unique utterance IDs:",
    len(
        set(
            utterance_ids
        )
    ),
)
print(
    "Unique locator IDs:",
    len(
        set(
            locator_ids
        )
    ),
)
print(
    "Unknown Manifest paths:",
    len(
        unknown_manifest_paths
    ),
)
print(
    "Manifest lineage mismatches:",
    len(
        manifest_lineage_mismatches
    ),
)
print(
    "Rights mismatches:",
    len(
        rights_failures
    ),
)
print(
    "Invalid Source ranges:",
    len(
        source_range_failures
    ),
)
print(
    "Source text mismatches:",
    len(
        source_text_failures
    ),
)
print(
    "Text hash failures:",
    len(
        text_hash_failures
    ),
)
print(
    "Source text hash failures:",
    len(
        source_text_hash_failures
    ),
)
print(
    "Unexpected segment timestamps:",
    len(
        timestamp_assignment_failures
    ),
)
print(
    "Processing-state failures:",
    len(
        processing_state_failures
    ),
)
print(
    "Schema failures:",
    len(
        schema_failures
    ),
)


assert len(
    utterance_ids
) == 16226

assert len(
    set(
        utterance_ids
    )
) == 16226, (
    "Duplicate utterance IDs found."
)

assert None not in (
    utterance_ids
), (
    "Missing utterance ID found."
)

assert len(
    locator_ids
) == 16226

assert len(
    set(
        locator_ids
    )
) == 16226, (
    "Duplicate locator IDs found."
)

assert not unknown_manifest_paths
assert not manifest_lineage_mismatches
assert not rights_failures
assert not source_range_failures
assert not source_text_failures
assert not text_hash_failures
assert not source_text_hash_failures
assert not timestamp_assignment_failures
assert not processing_state_failures
assert not schema_failures


assert set(
    program_segment_counts
) == {
    record[
        "program_id"
    ]
    for record
    in manifest_program_records
}, (
    "Transcript Artifact does not cover "
    "exactly the 13 Manifest programs."
)


print()
print("SEGMENTS BY PROGRAM")
print("=" * 110)

for program_id, count in sorted(
    program_segment_counts.items()
):
    print(
        f"- {program_id}: {count}"
    )


speaker_program_ids_disk = {
    record[
        "program_id"
    ]
    for record
    in speaker_registry_disk[
        "programs"
    ]
}

manifest_program_ids_disk = {
    record[
        "program_id"
    ]
    for record
    in manifest_program_records
}


assert (
    speaker_registry_disk[
        "artifact_type"
    ]
    == "phase2_transcript_speaker_registry"
)

assert (
    speaker_registry_disk[
        "artifact_version"
    ]
    == "v1"
)

assert (
    speaker_registry_disk[
        "schema_version"
    ]
    == "speaker_alias_registry_v1"
)

assert (
    speaker_registry_disk[
        "program_count"
    ]
    == 13
)

assert (
    speaker_program_ids_disk
    == manifest_program_ids_disk
), (
    "Speaker registry Program coverage "
    "does not match Manifest."
)

assert (
    speaker_registry_disk[
        "unknown_speaker_inference_allowed"
    ]
    is False
), (
    "Unknown speaker inference became enabled."
)


speaker_registry_overlays = (
    speaker_registry_disk[
        "human_attribution_overlays"
    ]
)

quality_report_overlays = (
    quality_report_disk[
        "human_review"
    ][
        "attribution_overlays"
    ]
)


def canonical_json_value(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


assert len(
    human_overlays_from_segments
) == 2

assert len(
    speaker_registry_overlays
) == 2

assert len(
    quality_report_overlays
) == 2

assert {
    canonical_json_value(
        value
    )
    for value
    in human_overlays_from_segments
} == {
    canonical_json_value(
        value
    )
    for value
    in speaker_registry_overlays
} == {
    canonical_json_value(
        value
    )
    for value
    in quality_report_overlays
}, (
    "Human attribution overlays differ "
    "between NB06 artifacts."
)


print()
print("SPEAKER REGISTRY")
print("=" * 110)
print(
    "Programs:",
    speaker_registry_disk[
        "program_count"
    ],
)
print(
    "Human attribution overlays:",
    len(
        speaker_registry_overlays
    ),
)
print(
    "Unknown speaker inference allowed:",
    speaker_registry_disk[
        "unknown_speaker_inference_allowed"
    ],
)


quality_timestamp_annotations = (
    quality_report_disk[
        "timestamp_semantics"
    ][
        "annotations"
    ]
)


segment_annotation_ids = [
    annotation[
        "timestamp_annotation_id"
    ]
    for annotation
    in timestamp_annotations_from_segments
]

quality_annotation_ids = [
    annotation[
        "timestamp_annotation_id"
    ]
    for annotation
    in quality_timestamp_annotations
]


assert len(
    timestamp_annotations_from_segments
) == 16

assert len(
    quality_timestamp_annotations
) == 16

assert len(
    set(
        segment_annotation_ids
    )
) == 16

assert len(
    set(
        quality_annotation_ids
    )
) == 16

assert set(
    segment_annotation_ids
) == set(
    quality_annotation_ids
), (
    "Timestamp annotation IDs differ "
    "between Transcript Artifact "
    "and Quality Report."
)

assert {
    canonical_json_value(
        annotation
    )
    for annotation
    in timestamp_annotations_from_segments
} == {
    canonical_json_value(
        annotation
    )
    for annotation
    in quality_timestamp_annotations
}, (
    "Timestamp annotation payloads differ "
    "between NB06 artifacts."
)


utterance_id_set = set(
    utterance_ids
)

timestamp_semantic_failures = []

for annotation in (
    quality_timestamp_annotations
):
    if (
        annotation[
            "utterance_id"
        ]
        not in utterance_id_set
    ):
        timestamp_semantic_failures.append({
            "annotation_id": (
                annotation[
                    "timestamp_annotation_id"
                ]
            ),
            "reason": (
                "UNKNOWN_UTTERANCE"
            ),
        })
        continue

    start_seconds = (
        annotation[
            "start_seconds"
        ]
    )

    end_seconds = (
        annotation[
            "end_seconds"
        ]
    )

    duration_seconds = (
        annotation[
            "duration_seconds"
        ]
    )

    if (
        end_seconds
        < start_seconds
        or duration_seconds
        != (
            end_seconds
            - start_seconds
        )
        or annotation[
            "segment_timestamp_assignment"
        ]
        != "NOT_ASSIGNED"
    ):
        timestamp_semantic_failures.append({
            "annotation_id": (
                annotation[
                    "timestamp_annotation_id"
                ]
            ),
            "reason": (
                "INVALID_TIMESTAMP_SEMANTICS"
            ),
        })


assert not (
    timestamp_semantic_failures
), (
    "Timestamp semantic validation failed."
)


print()
print("TIMESTAMP PRESERVATION")
print("=" * 110)
print(
    "Transcript annotations:",
    len(
        timestamp_annotations_from_segments
    ),
)
print(
    "Quality-report annotations:",
    len(
        quality_timestamp_annotations
    ),
)
print(
    "Semantic failures:",
    len(
        timestamp_semantic_failures
    ),
)
print(
    "Whole-segment timestamp assignments:",
    len(
        timestamp_assignment_failures
    ),
)


quality_candidates_disk = (
    quality_report_disk[
        "quality_candidates"
    ]
)

assert len(
    quality_candidates_disk
) == 133


quality_candidate_id_fields = [
    (
        "quality_candidate_id"
        if "quality_candidate_id"
        in candidate
        else (
            "candidate_id"
            if "candidate_id"
            in candidate
            else None
        )
    )
    for candidate
    in quality_candidates_disk
]

assert all(
    field is not None
    for field
    in quality_candidate_id_fields
), (
    "A Quality Candidate has no stable ID field."
)

quality_candidate_ids = [
    candidate[
        id_field
    ]
    for candidate, id_field
    in zip(
        quality_candidates_disk,
        quality_candidate_id_fields,
    )
]

assert len(
    set(
        quality_candidate_ids
    )
) == 133, (
    "Quality Candidate IDs are not unique."
)


def collect_direct_utterance_references(
    value,
):
    references = []

    if not isinstance(
        value,
        dict,
    ):
        return references

    for key, item in value.items():
        if (
            key.endswith(
                "utterance_id"
            )
            and isinstance(
                item,
                str,
            )
        ):
            references.append(
                item
            )

        elif (
            key.endswith(
                "utterance_ids"
            )
            and isinstance(
                item,
                list,
            )
        ):
            references.extend(
                candidate
                for candidate
                in item
                if isinstance(
                    candidate,
                    str,
                )
            )

    return references


unknown_quality_utterance_references = []

quality_reference_counts = Counter()

for candidate in (
    quality_candidates_disk
):
    references = (
        collect_direct_utterance_references(
            candidate
        )
    )

    quality_reference_counts[
        len(
            set(
                references
            )
        )
    ] += 1

    for reference in references:
        if (
            reference
            not in utterance_id_set
        ):
            unknown_quality_utterance_references.append({
                "quality_candidate_id": (
                    candidate.get(
                        "quality_candidate_id",
                        candidate.get(
                            "candidate_id"
                        ),
                    )
                ),
                "utterance_id": (
                    reference
                ),
            })


assert not (
    unknown_quality_utterance_references
), (
    "Quality Candidate references "
    "unknown utterance IDs."
)


human_review_disk = (
    quality_report_disk[
        "human_review"
    ]
)

human_review_result_disk = (
    human_review_disk[
        "result"
    ]
)

merge_decisions_disk = (
    human_review_disk[
        "merge_decisions"
    ]
)

deferred_cleanup_disk = (
    human_review_disk[
        "deferred_text_cleanup"
    ]
)


assert (
    human_review_result_disk[
        "structural_human_review_status"
    ]
    == "PASS"
)

assert (
    human_review_result_disk[
        "blocking_issue_count"
    ]
    == 0
)

assert len(
    merge_decisions_disk
) == 3

assert all(
    record[
        "decision"
    ]
    == "MERGE_CONFIRMED"
    for record
    in merge_decisions_disk
)

assert len(
    deferred_cleanup_disk
) == 2

assert all(
    record[
        "apply_in_nb06"
    ]
    is False
    for record
    in deferred_cleanup_disk
)

assert (
    quality_report_disk[
        "processing_state"
    ][
        "persian_normalization"
    ]
    == "NOT_PERFORMED"
)

assert (
    quality_report_disk[
        "processing_state"
    ][
        "deduplication"
    ]
    == "NOT_PERFORMED"
)

assert (
    quality_report_disk[
        "processing_state"
    ][
        "chunking"
    ]
    == "NOT_PERFORMED"
)


print()
print("QUALITY AND HUMAN REVIEW")
print("=" * 110)

print(
    "Quality candidates:",
    len(
        quality_candidates_disk
    ),
)
print(
    "Unique Quality Candidate IDs:",
    len(
        set(
            quality_candidate_ids
        )
    ),
)
print(
    "Unknown Quality utterance references:",
    len(
        unknown_quality_utterance_references
    ),
)
print(
    "Quality candidates by linked-utterance count:",
    dict(
        sorted(
            quality_reference_counts.items()
        )
    ),
)
print(
    "Human review status:",
    human_review_result_disk[
        "structural_human_review_status"
    ],
)
print(
    "Blocking human-review issues:",
    human_review_result_disk[
        "blocking_issue_count"
    ],
)
print(
    "Human-confirmed merges:",
    len(
        merge_decisions_disk
    ),
)
print(
    "Deferred text cleanup candidates:",
    len(
        deferred_cleanup_disk
    ),
)


assert (
    quality_report_disk[
        "accepted_scope"
    ][
        "program_count"
    ]
    == 13
)

assert (
    quality_report_disk[
        "accepted_scope"
    ][
        "physical_file_count"
    ]
    == 183
)

assert (
    quality_report_disk[
        "accepted_scope"
    ][
        "final_segment_count"
    ]
    == 16226
)

assert (
    quality_report_disk[
        "quality_summary"
    ][
        "candidate_count"
    ]
    == 133
)


print()
print("ARTIFACT SHA-256")
print("=" * 110)

for relative_path, sha256_value in sorted(
    artifact_sha256.items()
):
    print(
        "-",
        relative_path,
        "->",
        sha256_value,
    )


print()
print("FINAL INDEPENDENT VALIDATION SUMMARY")
print("=" * 110)

print(
    "Physical Source files validated:",
    len(
        accepted_manifest_records
    ),
)
print(
    "Final transcript segments validated:",
    len(
        transcript_records_disk
    ),
)
print(
    "Programs covered:",
    len(
        program_segment_counts
    ),
)
print(
    "Speaker overlays validated:",
    len(
        human_overlays_from_segments
    ),
)
print(
    "Timestamp annotations validated:",
    len(
        timestamp_annotations_from_segments
    ),
)
print(
    "Quality candidates validated:",
    len(
        quality_candidates_disk
    ),
)
print(
    "Deferred cleanup candidates:",
    len(
        deferred_cleanup_disk
    ),
)

print()
print(
    "NB06 INDEPENDENT ARTIFACT VALIDATION: PASS"
)
print(
    "FINAL ACCEPTANCE REPORT: PENDING"
)

INDEPENDENT DISK INPUT
Manifest accepted files: 183
Manifest programs: 13
Transcript records: 16226
Speaker registry programs: 13
Quality candidates: 133

SOURCE FILE INTEGRITY
Accepted files checked: 183
Missing Source files: 0
Source files outside repository: 0
Source SHA-256 mismatches: 0

TRANSCRIPT ARTIFACT LINEAGE
Segments checked: 16226
Unique utterance IDs: 16226
Unique locator IDs: 16226
Unknown Manifest paths: 0
Manifest lineage mismatches: 0
Rights mismatches: 0
Invalid Source ranges: 0
Source text mismatches: 0
Text hash failures: 0
Source text hash failures: 0
Unexpected segment timestamps: 0
Processing-state failures: 0
Schema failures: 0

SEGMENTS BY PROGRAM
- program:asre-shirin: 902
- program:bank-mellat: 453
- program:chehelston: 2455
- program:dar-jostojo-ye-khoshbakhti: 7236
- program:fanap: 82
- program:ketab-baz: 149
- program:khane-to: 560
- program:mizan: 217
- program:mq: 136
- program:oghyanous-aram: 31
- program:sherakat: 25
- program:tarbiat-morabi: 2597
- p

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت Final Acceptance Report و Checkpoint رسمی Notebook 06</h3>

<p>
Artifactهای رسمی Transcript Ingestion مستقیماً از روی Disk و در برابر Manifest و ۱۸۳ Source اصلی اعتبارسنجی شدند.
</p>

<p>
نتیجه Independent Validation:
</p>

<ul>
<li>۱۸۳ Source File معتبر</li>
<li>۱۶۲۲۶ Segment نهایی</li>
<li>۱۳ Program</li>
<li>۰ Source Hash Mismatch</li>
<li>۰ Manifest Lineage Mismatch</li>
<li>۰ Source Text Mismatch</li>
<li>۰ Rights Mismatch</li>
<li>۰ Segment Timestamp ساختگی</li>
<li>۲ Human Attribution Overlay معتبر</li>
<li>۱۶ Source Timestamp Annotation معتبر</li>
<li>۱۳۳ Quality Candidate معتبر</li>
<li>۳ Merge تأییدشده انسانی</li>
<li>۲ Deferred Text Cleanup Candidate</li>
<li>۰ Human Review Blocker</li>
</ul>

<h4>Checkpoint رسمی NB06</h4>

<p>
مطابق Contract واقعی Phase 2، فایل Checkpoint مستقل ساخته نمی‌شود.
</p>

<p>
Final Acceptance Report زیر نقش Checkpoint رسمی این Notebook را دارد:
</p>

<p>
<code>reports/phase2/transcript_ingestion_acceptance_v1.json</code>
</p>

<h4>Artifactهای پذیرفته‌شده</h4>

<ul>
<li><code>data/phase2/manifests/transcript_manifest_v1.json</code></li>
<li><code>data/phase2/interim/transcripts_extracted_v1.jsonl</code></li>
<li><code>data/phase2/interim/speaker_aliases_v1.json</code></li>
<li><code>reports/phase2/transcript_quality_report.json</code></li>
</ul>

<p>
برای هر Artifact، Path، SHA-256 و وضعیت Validation در Acceptance Report ثبت می‌شود.
</p>

<h4>وضعیت Speaker</h4>

<p>
Unknown Speaker Inference همچنان ممنوع است.
</p>

<p>
Segmentهایی که Source Evidence کافی ندارند به‌صورت <code>UNRESOLVED</code> حفظ شده‌اند.
</p>

<p>
دو Attribution انسانی نیز به‌صورت Overlay مستقل نگهداری شده‌اند و Attribution پایه Parser را بازنویسی نکرده‌اند.
</p>

<h4>وضعیت Timestamp</h4>

<p>
هیچ Timestamp برای کل Segment حدس زده نشده است.
</p>

<p>
۱۶ Timestamp واقعی Source به‌عنوان Annotation مستقل حفظ شده‌اند.
</p>

<h4>وضعیت Text Cleanup</h4>

<p>
دو خطای متنی که در Human Review تأیید شدند در این Notebook اصلاح نشده‌اند و برای Cleanup سراسری Corpus نگهداری می‌شوند:
</p>

<ul>
<li><code>صبور هم → صبورم</code></li>
<li><code>شاید متحرید ... → شهید مطهری ...</code></li>
</ul>

<h4>پردازش‌های Deferred</h4>

<ul>
<li>Persian Normalization</li>
<li>Corpus-wide Text Cleanup / ASR Correction</li>
<li>Deduplication</li>
<li>Unified Schema</li>
<li>Chunking</li>
<li>Rights Resolution</li>
</ul>

<h4>سیاست Write</h4>

<p>
Acceptance Report به‌صورت Idempotent ساخته می‌شود:
</p>

<ul>
<li><code>Missing → CREATED</code></li>
<li><code>Existing + Identical → REUSED_IDENTICAL</code></li>
<li><code>Existing + Different → FAIL</code></li>
</ul>

<p>
هیچ Report متفاوتی Silent Overwrite نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام Artifactهای پذیرفته‌شده واقعاً روی Disk وجود داشته باشند.</li>
<li>SHA-256 آنها مستقیماً محاسبه و ثبت شود.</li>
<li>Acceptance Scope برابر ۱۳ Program، ۱۸۳ Source و ۱۶۲۲۶ Segment باشد.</li>
<li>Independent Validation برابر PASS ثبت شود.</li>
<li>Human Review برابر PASS و Blocker برابر صفر باشد.</li>
<li>تمام Processingهای خارج از Scope به‌صورت Deferred ثبت شوند.</li>
<li>Acceptance Decision نهایی برابر <code>PASS</code> باشد.</li>
</ul>

</div>

In [47]:
# Code Cell 47

import hashlib
import json
from collections import Counter


acceptance_manifest_path = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "transcript_manifest_v1.json"
)

acceptance_transcript_path = (
    project_root
    / "data"
    / "phase2"
    / "interim"
    / "transcripts_extracted_v1.jsonl"
)

acceptance_speaker_registry_path = (
    project_root
    / "data"
    / "phase2"
    / "interim"
    / "speaker_aliases_v1.json"
)

acceptance_quality_report_path = (
    project_root
    / "reports"
    / "phase2"
    / "transcript_quality_report.json"
)

transcript_acceptance_report_path = (
    project_root
    / "reports"
    / "phase2"
    / "transcript_ingestion_acceptance_v1.json"
)


acceptance_required_artifacts = {
    "source_manifest": (
        acceptance_manifest_path
    ),
    "transcript_segments": (
        acceptance_transcript_path
    ),
    "speaker_registry": (
        acceptance_speaker_registry_path
    ),
    "quality_report": (
        acceptance_quality_report_path
    ),
}


missing_acceptance_artifacts = [
    name
    for name, path
    in acceptance_required_artifacts.items()
    if not path.is_file()
]

assert not (
    missing_acceptance_artifacts
), (
    "Required accepted Artifact is missing: "
    f"{missing_acceptance_artifacts}"
)


def file_sha256(
    path,
):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


acceptance_artifact_hashes = {
    name: file_sha256(
        path
    )
    for name, path
    in acceptance_required_artifacts.items()
}


acceptance_manifest_disk = json.loads(
    acceptance_manifest_path.read_text(
        encoding="utf-8"
    )
)

acceptance_speaker_registry_disk = json.loads(
    acceptance_speaker_registry_path.read_text(
        encoding="utf-8"
    )
)

acceptance_quality_report_disk = json.loads(
    acceptance_quality_report_path.read_text(
        encoding="utf-8"
    )
)


acceptance_segment_count = 0
acceptance_utterance_ids = set()
acceptance_locator_ids = set()
acceptance_program_counts = Counter()

acceptance_human_overlay_count = 0
acceptance_timestamp_annotation_count = 0
acceptance_nonnull_segment_timestamp_count = 0

acceptance_processing_violations = []

with acceptance_transcript_path.open(
    "r",
    encoding="utf-8",
) as file:
    for line_number, line in enumerate(
        file,
        start=1,
    ):
        if not line.strip():
            continue

        record = json.loads(
            line
        )

        acceptance_segment_count += 1

        acceptance_utterance_ids.add(
            record[
                "utterance_id"
            ]
        )

        acceptance_locator_ids.add(
            record[
                "source_locator"
            ][
                "locator_id"
            ]
        )

        acceptance_program_counts[
            record[
                "program_id"
            ]
        ] += 1

        if (
            record[
                "speaker"
            ][
                "human_attribution_overlay"
            ]
            is not None
        ):
            acceptance_human_overlay_count += 1

        acceptance_timestamp_annotation_count += (
            len(
                record[
                    "timestamps"
                ][
                    "source_annotations"
                ]
            )
        )

        if (
            record[
                "timestamps"
            ][
                "segment_start_time"
            ]
            is not None
            or record[
                "timestamps"
            ][
                "segment_end_time"
            ]
            is not None
        ):
            acceptance_nonnull_segment_timestamp_count += 1

        processing_state = (
            record[
                "processing_state"
            ]
        )

        if not (
            processing_state[
                "persian_normalization"
            ]
            is False
            and processing_state[
                "text_correction"
            ]
            is False
            and processing_state[
                "deduplication"
            ]
            is False
            and processing_state[
                "chunking"
            ]
            is False
            and processing_state[
                "unknown_speaker_inference"
            ]
            is False
        ):
            acceptance_processing_violations.append(
                record[
                    "utterance_id"
                ]
            )


acceptance_human_review = (
    acceptance_quality_report_disk[
        "human_review"
    ]
)

acceptance_human_result = (
    acceptance_human_review[
        "result"
    ]
)

acceptance_quality_candidates = (
    acceptance_quality_report_disk[
        "quality_candidates"
    ]
)

acceptance_timestamp_annotations = (
    acceptance_quality_report_disk[
        "timestamp_semantics"
    ][
        "annotations"
    ]
)

acceptance_human_overlays = (
    acceptance_speaker_registry_disk[
        "human_attribution_overlays"
    ]
)

acceptance_merge_decisions = (
    acceptance_human_review[
        "merge_decisions"
    ]
)

acceptance_deferred_cleanup = (
    acceptance_human_review[
        "deferred_text_cleanup"
    ]
)


acceptance_checks = {
    "source_manifest_exists": (
        acceptance_manifest_path.is_file()
    ),
    "transcript_artifact_exists": (
        acceptance_transcript_path.is_file()
    ),
    "speaker_registry_exists": (
        acceptance_speaker_registry_path.is_file()
    ),
    "quality_report_exists": (
        acceptance_quality_report_path.is_file()
    ),
    "manifest_program_count_13": (
        len(
            acceptance_manifest_disk[
                "programs"
            ]
        )
        == 13
    ),
    "manifest_accepted_files_183": (
        len(
            acceptance_manifest_disk[
                "accepted_files"
            ]
        )
        == 183
    ),
    "segment_count_16226": (
        acceptance_segment_count
        == 16226
    ),
    "utterance_ids_unique": (
        len(
            acceptance_utterance_ids
        )
        == 16226
    ),
    "locator_ids_unique": (
        len(
            acceptance_locator_ids
        )
        == 16226
    ),
    "all_13_programs_covered": (
        len(
            acceptance_program_counts
        )
        == 13
    ),
    "speaker_registry_program_count_13": (
        acceptance_speaker_registry_disk[
            "program_count"
        ]
        == 13
    ),
    "unknown_speaker_inference_forbidden": (
        acceptance_speaker_registry_disk[
            "unknown_speaker_inference_allowed"
        ]
        is False
    ),
    "human_attribution_overlay_count_2": (
        acceptance_human_overlay_count
        == 2
        and len(
            acceptance_human_overlays
        )
        == 2
    ),
    "timestamp_annotations_16": (
        acceptance_timestamp_annotation_count
        == 16
        and len(
            acceptance_timestamp_annotations
        )
        == 16
    ),
    "whole_segment_timestamps_absent": (
        acceptance_nonnull_segment_timestamp_count
        == 0
    ),
    "quality_candidates_133": (
        len(
            acceptance_quality_candidates
        )
        == 133
    ),
    "human_review_pass": (
        acceptance_human_result[
            "structural_human_review_status"
        ]
        == "PASS"
    ),
    "human_review_blockers_zero": (
        acceptance_human_result[
            "blocking_issue_count"
        ]
        == 0
    ),
    "human_confirmed_merges_3": (
        len(
            acceptance_merge_decisions
        )
        == 3
        and all(
            record[
                "decision"
            ]
            == "MERGE_CONFIRMED"
            for record
            in acceptance_merge_decisions
        )
    ),
    "deferred_text_cleanup_2": (
        len(
            acceptance_deferred_cleanup
        )
        == 2
        and all(
            record[
                "apply_in_nb06"
            ]
            is False
            for record
            in acceptance_deferred_cleanup
        )
    ),
    "segment_processing_scope_preserved": (
        not acceptance_processing_violations
    ),
    "persian_normalization_not_performed": (
        acceptance_quality_report_disk[
            "processing_state"
        ][
            "persian_normalization"
        ]
        == "NOT_PERFORMED"
    ),
    "deduplication_not_performed": (
        acceptance_quality_report_disk[
            "processing_state"
        ][
            "deduplication"
        ]
        == "NOT_PERFORMED"
    ),
    "chunking_not_performed": (
        acceptance_quality_report_disk[
            "processing_state"
        ][
            "chunking"
        ]
        == "NOT_PERFORMED"
    ),
}


all_acceptance_checks_pass = all(
    acceptance_checks.values()
)


transcript_ingestion_acceptance_v1 = {
    "report_type": (
        "phase2_transcript_ingestion_acceptance"
    ),
    "report_version": "v1",
    "schema_version": (
        "transcript_ingestion_acceptance_v1"
    ),
    "notebook": (
        "notebooks/phase2/"
        "06_transcript_ingestion.ipynb"
    ),
    "checkpoint_role": (
        "FINAL_NOTEBOOK_CHECKPOINT"
    ),
    "source_manifest": {
        "relative_path": (
            acceptance_manifest_path.relative_to(
                project_root
            ).as_posix()
        ),
        "sha256": (
            acceptance_artifact_hashes[
                "source_manifest"
            ]
        ),
        "program_count": 13,
        "accepted_physical_file_count": 183,
        "status": (
            "ACCEPTED_INPUT"
        ),
    },
    "accepted_artifacts": {
        "transcript_segments": {
            "relative_path": (
                acceptance_transcript_path.relative_to(
                    project_root
                ).as_posix()
            ),
            "sha256": (
                acceptance_artifact_hashes[
                    "transcript_segments"
                ]
            ),
            "record_count": (
                acceptance_segment_count
            ),
            "schema_version": (
                "transcript_segment_v1"
            ),
            "status": (
                "ACCEPTED_AFTER_INDEPENDENT_VALIDATION"
            ),
        },
        "speaker_registry": {
            "relative_path": (
                acceptance_speaker_registry_path.relative_to(
                    project_root
                ).as_posix()
            ),
            "sha256": (
                acceptance_artifact_hashes[
                    "speaker_registry"
                ]
            ),
            "program_count": (
                acceptance_speaker_registry_disk[
                    "program_count"
                ]
            ),
            "schema_version": (
                acceptance_speaker_registry_disk[
                    "schema_version"
                ]
            ),
            "status": (
                "ACCEPTED_AFTER_INDEPENDENT_VALIDATION"
            ),
        },
        "quality_report": {
            "relative_path": (
                acceptance_quality_report_path.relative_to(
                    project_root
                ).as_posix()
            ),
            "sha256": (
                acceptance_artifact_hashes[
                    "quality_report"
                ]
            ),
            "quality_candidate_count": (
                len(
                    acceptance_quality_candidates
                )
            ),
            "timestamp_annotation_count": (
                len(
                    acceptance_timestamp_annotations
                )
            ),
            "status": (
                "ACCEPTED_AFTER_INDEPENDENT_VALIDATION"
            ),
        },
    },
    "accepted_scope": {
        "program_count": (
            len(
                acceptance_program_counts
            )
        ),
        "physical_file_count": 183,
        "segment_count": (
            acceptance_segment_count
        ),
        "unique_utterance_id_count": (
            len(
                acceptance_utterance_ids
            )
        ),
        "unique_locator_id_count": (
            len(
                acceptance_locator_ids
            )
        ),
        "segments_by_program": dict(
            sorted(
                acceptance_program_counts.items()
            )
        ),
    },
    "speaker_state": {
        "canonical_primary_speaker": (
            "Dr. Komeil Roudi"
        ),
        "unknown_speaker_inference_allowed": (
            False
        ),
        "human_attribution_overlay_count": (
            acceptance_human_overlay_count
        ),
        "base_parser_attribution_preserved": (
            True
        ),
        "human_overlay_does_not_rewrite_base": (
            True
        ),
    },
    "timestamp_state": {
        "source_timestamp_annotation_count": (
            acceptance_timestamp_annotation_count
        ),
        "whole_segment_timestamp_assignment_count": (
            acceptance_nonnull_segment_timestamp_count
        ),
        "policy": (
            "NONE_UNLESS_SOURCE_PROVIDES_"
            "SEGMENT_LEVEL_EVIDENCE"
        ),
    },
    "quality_state": {
        "candidate_count": (
            len(
                acceptance_quality_candidates
            )
        ),
        "automatic_correction_applied": (
            False
        ),
        "automatic_duplicate_removal_applied": (
            False
        ),
    },
    "human_review": {
        "status": (
            acceptance_human_result[
                "structural_human_review_status"
            ]
        ),
        "blocking_issue_count": (
            acceptance_human_result[
                "blocking_issue_count"
            ]
        ),
        "human_attribution_count": (
            acceptance_human_result[
                "human_attribution_count"
            ]
        ),
        "human_confirmed_merge_count": (
            len(
                acceptance_merge_decisions
            )
        ),
        "deferred_text_cleanup_count": (
            len(
                acceptance_deferred_cleanup
            )
        ),
    },
    "processing_state": {
        "transcript_ingestion": (
            "COMPLETED"
        ),
        "speaker_parsing": (
            "COMPLETED"
        ),
        "lineage_validation": (
            "COMPLETED"
        ),
        "source_conservation": (
            "COMPLETED"
        ),
        "timestamp_preservation": (
            "COMPLETED"
        ),
        "human_review": (
            "PASS"
        ),
        "persian_normalization": (
            "NOT_PERFORMED"
        ),
        "text_cleanup": (
            "DEFERRED"
        ),
        "deduplication": (
            "NOT_PERFORMED"
        ),
        "unified_schema": (
            "NOT_PERFORMED"
        ),
        "chunking": (
            "NOT_PERFORMED"
        ),
    },
    "deferred_processing": [
        "PERSIAN_NORMALIZATION",
        "CORPUS_WIDE_TEXT_CLEANUP_AND_ASR_CORRECTION",
        "DEDUPLICATION",
        "UNIFIED_SCHEMA",
        "CHUNKING",
        "RIGHTS_RESOLUTION",
    ],
    "deferred_text_cleanup": (
        acceptance_deferred_cleanup
    ),
    "known_limitations": [
        (
            "Rights status for all accepted "
            "Transcript files remains EVIDENCE_REQUIRED."
        ),
        (
            "Speaker identity remains UNRESOLVED "
            "where Source evidence and Human Review "
            "do not establish an identity."
        ),
        (
            "Two Human-confirmed text errors are "
            "preserved as deferred cleanup candidates "
            "and are not corrected in Notebook 06."
        ),
        (
            "Quality candidates are retained without "
            "automatic correction or deletion."
        ),
        (
            "Sixteen real Source timestamp annotations "
            "are preserved, but no whole-segment "
            "timestamp is inferred from them."
        ),
    ],
    "validation_checks": (
        acceptance_checks
    ),
    "independent_artifact_validation": {
        "status": (
            "PASS"
            if all_acceptance_checks_pass
            else "FAIL"
        ),
        "source_file_count_validated": 183,
        "segment_count_validated": (
            acceptance_segment_count
        ),
        "source_hash_mismatch_count": 0,
        "manifest_lineage_mismatch_count": 0,
        "source_text_mismatch_count": 0,
        "rights_mismatch_count": 0,
        "timestamp_semantic_failure_count": 0,
        "unknown_quality_utterance_reference_count": 0,
    },
    "acceptance_decision": (
        "PASS"
        if all_acceptance_checks_pass
        else "FAIL"
    ),
}


transcript_acceptance_report_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

acceptance_report_candidate_text = (
    json.dumps(
        transcript_ingestion_acceptance_v1,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)


if transcript_acceptance_report_path.exists():
    existing_acceptance_text = (
        transcript_acceptance_report_path.read_text(
            encoding="utf-8"
        )
    )

    if (
        existing_acceptance_text
        != acceptance_report_candidate_text
    ):
        raise ValueError(
            "Existing transcript ingestion "
            "acceptance report differs from "
            "the deterministic candidate. "
            "Refusing to overwrite."
        )

    acceptance_report_action = (
        "REUSED_IDENTICAL"
    )

else:
    transcript_acceptance_report_path.write_text(
        acceptance_report_candidate_text,
        encoding="utf-8",
    )

    acceptance_report_action = (
        "CREATED"
    )


stored_acceptance_report = json.loads(
    transcript_acceptance_report_path.read_text(
        encoding="utf-8"
    )
)

stored_acceptance_sha256 = (
    file_sha256(
        transcript_acceptance_report_path
    )
)


assert (
    stored_acceptance_report[
        "acceptance_decision"
    ]
    == "PASS"
), (
    "Stored NB06 Acceptance Decision "
    "is not PASS."
)

assert (
    stored_acceptance_report[
        "independent_artifact_validation"
    ][
        "status"
    ]
    == "PASS"
), (
    "Stored Independent Artifact Validation "
    "is not PASS."
)

assert all(
    stored_acceptance_report[
        "validation_checks"
    ].values()
), (
    "At least one stored Acceptance Check "
    "is not True."
)

assert (
    stored_acceptance_report[
        "checkpoint_role"
    ]
    == "FINAL_NOTEBOOK_CHECKPOINT"
)

assert (
    stored_acceptance_report[
        "accepted_scope"
    ][
        "segment_count"
    ]
    == 16226
)

assert (
    stored_acceptance_report[
        "human_review"
    ][
        "status"
    ]
    == "PASS"
)

assert (
    stored_acceptance_report[
        "human_review"
    ][
        "blocking_issue_count"
    ]
    == 0
)

assert (
    stored_acceptance_report[
        "timestamp_state"
    ][
        "source_timestamp_annotation_count"
    ]
    == 16
)

assert (
    stored_acceptance_report[
        "timestamp_state"
    ][
        "whole_segment_timestamp_assignment_count"
    ]
    == 0
)


print("NB06 FINAL ACCEPTANCE REPORT")
print("=" * 110)

print(
    "Path:",
    transcript_acceptance_report_path.relative_to(
        project_root
    ),
)
print(
    "Action:",
    acceptance_report_action,
)
print(
    "SHA-256:",
    stored_acceptance_sha256,
)

print()
print(
    "Accepted programs:",
    stored_acceptance_report[
        "accepted_scope"
    ][
        "program_count"
    ],
)
print(
    "Accepted physical files:",
    stored_acceptance_report[
        "accepted_scope"
    ][
        "physical_file_count"
    ],
)
print(
    "Accepted segments:",
    stored_acceptance_report[
        "accepted_scope"
    ][
        "segment_count"
    ],
)
print(
    "Quality candidates:",
    stored_acceptance_report[
        "quality_state"
    ][
        "candidate_count"
    ],
)
print(
    "Timestamp annotations:",
    stored_acceptance_report[
        "timestamp_state"
    ][
        "source_timestamp_annotation_count"
    ],
)
print(
    "Human attribution overlays:",
    stored_acceptance_report[
        "speaker_state"
    ][
        "human_attribution_overlay_count"
    ],
)
print(
    "Human-confirmed merges:",
    stored_acceptance_report[
        "human_review"
    ][
        "human_confirmed_merge_count"
    ],
)
print(
    "Deferred text cleanup:",
    stored_acceptance_report[
        "human_review"
    ][
        "deferred_text_cleanup_count"
    ],
)
print(
    "Validation checks:",
    sum(
        stored_acceptance_report[
            "validation_checks"
        ].values()
    ),
    "/",
    len(
        stored_acceptance_report[
            "validation_checks"
        ]
    ),
)

print()
print(
    "Independent validation:",
    stored_acceptance_report[
        "independent_artifact_validation"
    ][
        "status"
    ],
)
print(
    "Acceptance decision:",
    stored_acceptance_report[
        "acceptance_decision"
    ],
)

print()
print(
    "NB06 FINAL ACCEPTANCE REPORT CREATION GATE: PASS"
)
print(
    "FINAL REPORT INDEPENDENT VALIDATION: PENDING"
)

NB06 FINAL ACCEPTANCE REPORT
Path: reports/phase2/transcript_ingestion_acceptance_v1.json
Action: REUSED_IDENTICAL
SHA-256: b9d4a5a29f52e9cc3031f21925d56f36f52d04d51e976a66843c491dc15a16ad

Accepted programs: 13
Accepted physical files: 183
Accepted segments: 16226
Quality candidates: 133
Timestamp annotations: 16
Human attribution overlays: 2
Human-confirmed merges: 3
Deferred text cleanup: 2
Validation checks: 24 / 24

Independent validation: PASS
Acceptance decision: PASS

NB06 FINAL ACCEPTANCE REPORT CREATION GATE: PASS
FINAL REPORT INDEPENDENT VALIDATION: PENDING


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اعتبارسنجی مستقل Final Acceptance Report از روی Disk</h3>

<p>
Final Acceptance Report رسمی Notebook 06 با وضعیت <code>PASS</code> ساخته شده است.
</p>

<p>
اما برای بسته‌شدن Notebook، خود این Report نیز نباید صرفاً بر اساس Variableهای Runtime Cell قبل پذیرفته شود.
</p>

<p>
در این Cell فایل:
</p>

<p>
<code>reports/phase2/transcript_ingestion_acceptance_v1.json</code>
</p>

<p>
مستقیماً از Disk دوباره خوانده می‌شود و تمام ادعاهای اصلی آن با Artifactها و Sourceهای واقعی مقایسه می‌شوند.
</p>

<h4>موارد Validation</h4>

<ul>
<li>نوع، Version، Schema و نقش Checkpoint گزارش</li>
<li>وجود تمام Artifactهای پذیرفته‌شده</li>
<li>تطبیق SHA-256 ثبت‌شده هر Artifact با فایل واقعی</li>
<li>تعداد ۱۳ Program و ۱۸۳ Physical Source</li>
<li>اعتبار Hash هر ۱۸۳ Source در برابر Manifest</li>
<li>تعداد دقیق ۱۶۲۲۶ Segment در JSONL</li>
<li>یکتایی تمام Utterance IDها و Locator IDها</li>
<li>پوشش دقیق ۱۳ Program</li>
<li>۲ Human Attribution Overlay</li>
<li>۱۶ Source Timestamp Annotation</li>
<li>صفر Whole-segment Timestamp Assignment</li>
<li>۱۳۳ Quality Candidate</li>
<li>Human Review برابر PASS و Blocker برابر صفر</li>
<li>۳ Merge تأییدشده انسانی</li>
<li>۲ Deferred Text Cleanup Candidate</li>
<li>عدم انجام Persian Normalization، Deduplication و Chunking</li>
<li>True بودن تمام Validation Checkهای ذخیره‌شده</li>
<li>Acceptance Decision نهایی برابر PASS</li>
</ul>

<h4>اصل استقلال</h4>

<p>
Countها و Hashها در این Cell دوباره از فایل‌های واقعی محاسبه می‌شوند و به Object ساخته‌شده در Cell 47 اتکا نمی‌شود.
</p>

<h4>اصل عدم تغییر</h4>

<p>
این Cell کاملاً Read-only است و هیچ Artifact، Source یا Notebook Cell را بازنویسی نمی‌کند.
</p>

<h4>معیار PASS</h4>

<p>
اگر تمام ادعاهای Final Acceptance Report با وضعیت واقعی Disk منطبق باشند، Report به‌عنوان Checkpoint معتبر NB06 پذیرفته می‌شود.
</p>

</div>

In [48]:
# Code Cell 48

import hashlib
import json
from collections import Counter


final_report_path_disk = (
    project_root
    / "reports"
    / "phase2"
    / "transcript_ingestion_acceptance_v1.json"
)

assert final_report_path_disk.is_file(), (
    "Final NB06 Acceptance Report is missing."
)


final_report_disk = json.loads(
    final_report_path_disk.read_text(
        encoding="utf-8"
    )
)


def independent_file_sha256(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


def resolve_repo_relative_path(
    relative_path,
):
    path = (
        project_root
        / relative_path
    )

    resolved = path.resolve()

    resolved.relative_to(
        project_root.resolve()
    )

    return path


print("FINAL ACCEPTANCE REPORT CONTRACT")
print("=" * 110)

print(
    "Report:",
    final_report_path_disk.relative_to(
        project_root
    ),
)
print(
    "Report SHA-256:",
    independent_file_sha256(
        final_report_path_disk
    ),
)
print(
    "Report type:",
    final_report_disk.get(
        "report_type"
    ),
)
print(
    "Report version:",
    final_report_disk.get(
        "report_version"
    ),
)
print(
    "Schema version:",
    final_report_disk.get(
        "schema_version"
    ),
)
print(
    "Checkpoint role:",
    final_report_disk.get(
        "checkpoint_role"
    ),
)
print(
    "Acceptance decision:",
    final_report_disk.get(
        "acceptance_decision"
    ),
)


assert (
    final_report_disk.get(
        "report_type"
    )
    == "phase2_transcript_ingestion_acceptance"
)

assert (
    final_report_disk.get(
        "report_version"
    )
    == "v1"
)

assert (
    final_report_disk.get(
        "schema_version"
    )
    == "transcript_ingestion_acceptance_v1"
)

assert (
    final_report_disk.get(
        "notebook"
    )
    == "notebooks/phase2/06_transcript_ingestion.ipynb"
)

assert (
    final_report_disk.get(
        "checkpoint_role"
    )
    == "FINAL_NOTEBOOK_CHECKPOINT"
)


source_manifest_contract = (
    final_report_disk[
        "source_manifest"
    ]
)

accepted_artifact_contracts = (
    final_report_disk[
        "accepted_artifacts"
    ]
)


artifact_contract_paths = {
    "source_manifest": (
        source_manifest_contract[
            "relative_path"
        ]
    ),
    "transcript_segments": (
        accepted_artifact_contracts[
            "transcript_segments"
        ][
            "relative_path"
        ]
    ),
    "speaker_registry": (
        accepted_artifact_contracts[
            "speaker_registry"
        ][
            "relative_path"
        ]
    ),
    "quality_report": (
        accepted_artifact_contracts[
            "quality_report"
        ][
            "relative_path"
        ]
    ),
}


artifact_contract_expected_hashes = {
    "source_manifest": (
        source_manifest_contract[
            "sha256"
        ]
    ),
    "transcript_segments": (
        accepted_artifact_contracts[
            "transcript_segments"
        ][
            "sha256"
        ]
    ),
    "speaker_registry": (
        accepted_artifact_contracts[
            "speaker_registry"
        ][
            "sha256"
        ]
    ),
    "quality_report": (
        accepted_artifact_contracts[
            "quality_report"
        ][
            "sha256"
        ]
    ),
}


artifact_contract_actual_hashes = {}
artifact_contract_missing = []
artifact_contract_hash_mismatches = []


for artifact_name, relative_path in (
    artifact_contract_paths.items()
):
    path = resolve_repo_relative_path(
        relative_path
    )

    if not path.is_file():
        artifact_contract_missing.append(
            artifact_name
        )
        continue

    actual_sha256 = (
        independent_file_sha256(
            path
        )
    )

    artifact_contract_actual_hashes[
        artifact_name
    ] = actual_sha256

    if (
        actual_sha256
        != artifact_contract_expected_hashes[
            artifact_name
        ]
    ):
        artifact_contract_hash_mismatches.append({
            "artifact": (
                artifact_name
            ),
            "relative_path": (
                relative_path
            ),
            "expected_sha256": (
                artifact_contract_expected_hashes[
                    artifact_name
                ]
            ),
            "actual_sha256": (
                actual_sha256
            ),
        })


print()
print("ACCEPTED ARTIFACT HASH VALIDATION")
print("=" * 110)

for artifact_name in (
    artifact_contract_paths
):
    print()
    print(
        "Artifact:",
        artifact_name,
    )
    print(
        "Path:",
        artifact_contract_paths[
            artifact_name
        ],
    )
    print(
        "Expected SHA-256:",
        artifact_contract_expected_hashes[
            artifact_name
        ],
    )
    print(
        "Actual SHA-256:",
        artifact_contract_actual_hashes.get(
            artifact_name
        ),
    )

print()
print(
    "Missing accepted artifacts:",
    len(
        artifact_contract_missing
    ),
)
print(
    "Artifact hash mismatches:",
    len(
        artifact_contract_hash_mismatches
    ),
)


assert not artifact_contract_missing
assert not artifact_contract_hash_mismatches


manifest_path_final = resolve_repo_relative_path(
    artifact_contract_paths[
        "source_manifest"
    ]
)

transcript_path_final = resolve_repo_relative_path(
    artifact_contract_paths[
        "transcript_segments"
    ]
)

speaker_registry_path_final = resolve_repo_relative_path(
    artifact_contract_paths[
        "speaker_registry"
    ]
)

quality_report_path_final = resolve_repo_relative_path(
    artifact_contract_paths[
        "quality_report"
    ]
)


manifest_final_disk = json.loads(
    manifest_path_final.read_text(
        encoding="utf-8"
    )
)

speaker_registry_final_disk = json.loads(
    speaker_registry_path_final.read_text(
        encoding="utf-8"
    )
)

quality_report_final_disk = json.loads(
    quality_report_path_final.read_text(
        encoding="utf-8"
    )
)


accepted_sources_final = (
    manifest_final_disk[
        "accepted_files"
    ]
)

manifest_programs_final = (
    manifest_final_disk[
        "programs"
    ]
)


source_integrity_failures_final = []

for source_record in (
    accepted_sources_final
):
    source_path = resolve_repo_relative_path(
        source_record[
            "relative_path"
        ]
    )

    if not source_path.is_file():
        source_integrity_failures_final.append({
            "relative_path": (
                source_record[
                    "relative_path"
                ]
            ),
            "reason": (
                "MISSING_SOURCE_FILE"
            ),
        })
        continue

    actual_sha256 = (
        independent_file_sha256(
            source_path
        )
    )

    if (
        actual_sha256
        != source_record[
            "sha256"
        ]
    ):
        source_integrity_failures_final.append({
            "relative_path": (
                source_record[
                    "relative_path"
                ]
            ),
            "reason": (
                "SOURCE_SHA256_MISMATCH"
            ),
            "expected_sha256": (
                source_record[
                    "sha256"
                ]
            ),
            "actual_sha256": (
                actual_sha256
            ),
        })


print()
print("SOURCE MANIFEST REVALIDATION")
print("=" * 110)

print(
    "Programs:",
    len(
        manifest_programs_final
    ),
)
print(
    "Accepted physical files:",
    len(
        accepted_sources_final
    ),
)
print(
    "Source integrity failures:",
    len(
        source_integrity_failures_final
    ),
)


assert len(
    manifest_programs_final
) == 13

assert len(
    accepted_sources_final
) == 183

assert not (
    source_integrity_failures_final
)


final_segment_count = 0
final_utterance_ids = set()
final_locator_ids = set()
final_program_counts = Counter()

final_human_overlay_count = 0
final_timestamp_annotation_count = 0
final_whole_segment_timestamp_count = 0

final_processing_scope_failures = []


with transcript_path_final.open(
    "r",
    encoding="utf-8",
) as file:
    for line_number, line in enumerate(
        file,
        start=1,
    ):
        if not line.strip():
            continue

        record = json.loads(
            line
        )

        final_segment_count += 1

        final_utterance_ids.add(
            record[
                "utterance_id"
            ]
        )

        final_locator_ids.add(
            record[
                "source_locator"
            ][
                "locator_id"
            ]
        )

        final_program_counts[
            record[
                "program_id"
            ]
        ] += 1

        if (
            record[
                "speaker"
            ][
                "human_attribution_overlay"
            ]
            is not None
        ):
            final_human_overlay_count += 1

        final_timestamp_annotation_count += (
            len(
                record[
                    "timestamps"
                ][
                    "source_annotations"
                ]
            )
        )

        if (
            record[
                "timestamps"
            ][
                "segment_start_time"
            ]
            is not None
            or record[
                "timestamps"
            ][
                "segment_end_time"
            ]
            is not None
        ):
            final_whole_segment_timestamp_count += 1

        processing_state = (
            record[
                "processing_state"
            ]
        )

        if not (
            processing_state[
                "persian_normalization"
            ]
            is False
            and processing_state[
                "text_correction"
            ]
            is False
            and processing_state[
                "deduplication"
            ]
            is False
            and processing_state[
                "chunking"
            ]
            is False
            and processing_state[
                "unknown_speaker_inference"
            ]
            is False
        ):
            final_processing_scope_failures.append(
                record[
                    "utterance_id"
                ]
            )


print()
print("TRANSCRIPT ARTIFACT REVALIDATION")
print("=" * 110)

print(
    "Segments:",
    final_segment_count,
)
print(
    "Unique utterance IDs:",
    len(
        final_utterance_ids
    ),
)
print(
    "Unique locator IDs:",
    len(
        final_locator_ids
    ),
)
print(
    "Programs covered:",
    len(
        final_program_counts
    ),
)
print(
    "Human attribution overlays:",
    final_human_overlay_count,
)
print(
    "Source timestamp annotations:",
    final_timestamp_annotation_count,
)
print(
    "Whole-segment timestamp assignments:",
    final_whole_segment_timestamp_count,
)
print(
    "Processing-scope failures:",
    len(
        final_processing_scope_failures
    ),
)


assert final_segment_count == 16226
assert len(final_utterance_ids) == 16226
assert len(final_locator_ids) == 16226
assert len(final_program_counts) == 13
assert final_human_overlay_count == 2
assert final_timestamp_annotation_count == 16
assert final_whole_segment_timestamp_count == 0

assert not (
    final_processing_scope_failures
)


report_scope = (
    final_report_disk[
        "accepted_scope"
    ]
)

assert (
    report_scope[
        "program_count"
    ]
    == len(
        final_program_counts
    )
    == 13
)

assert (
    report_scope[
        "physical_file_count"
    ]
    == len(
        accepted_sources_final
    )
    == 183
)

assert (
    report_scope[
        "segment_count"
    ]
    == final_segment_count
    == 16226
)

assert (
    report_scope[
        "unique_utterance_id_count"
    ]
    == len(
        final_utterance_ids
    )
)

assert (
    report_scope[
        "unique_locator_id_count"
    ]
    == len(
        final_locator_ids
    )
)

assert (
    report_scope[
        "segments_by_program"
    ]
    == dict(
        sorted(
            final_program_counts.items()
        )
    )
)


quality_candidates_final = (
    quality_report_final_disk[
        "quality_candidates"
    ]
)

timestamp_annotations_final = (
    quality_report_final_disk[
        "timestamp_semantics"
    ][
        "annotations"
    ]
)

human_review_final = (
    quality_report_final_disk[
        "human_review"
    ]
)

human_review_result_final = (
    human_review_final[
        "result"
    ]
)

merge_decisions_final = (
    human_review_final[
        "merge_decisions"
    ]
)

deferred_cleanup_final = (
    human_review_final[
        "deferred_text_cleanup"
    ]
)

speaker_overlays_final = (
    speaker_registry_final_disk[
        "human_attribution_overlays"
    ]
)


print()
print("SPEAKER / QUALITY / HUMAN REVIEW REVALIDATION")
print("=" * 110)

print(
    "Speaker registry programs:",
    speaker_registry_final_disk[
        "program_count"
    ],
)
print(
    "Unknown speaker inference allowed:",
    speaker_registry_final_disk[
        "unknown_speaker_inference_allowed"
    ],
)
print(
    "Speaker registry overlays:",
    len(
        speaker_overlays_final
    ),
)
print(
    "Quality candidates:",
    len(
        quality_candidates_final
    ),
)
print(
    "Timestamp annotations:",
    len(
        timestamp_annotations_final
    ),
)
print(
    "Human review status:",
    human_review_result_final[
        "structural_human_review_status"
    ],
)
print(
    "Human review blockers:",
    human_review_result_final[
        "blocking_issue_count"
    ],
)
print(
    "Human-confirmed merges:",
    len(
        merge_decisions_final
    ),
)
print(
    "Deferred cleanup candidates:",
    len(
        deferred_cleanup_final
    ),
)


assert (
    speaker_registry_final_disk[
        "program_count"
    ]
    == 13
)

assert (
    speaker_registry_final_disk[
        "unknown_speaker_inference_allowed"
    ]
    is False
)

assert len(
    speaker_overlays_final
) == 2

assert len(
    quality_candidates_final
) == 133

assert len(
    timestamp_annotations_final
) == 16

assert (
    human_review_result_final[
        "structural_human_review_status"
    ]
    == "PASS"
)

assert (
    human_review_result_final[
        "blocking_issue_count"
    ]
    == 0
)

assert len(
    merge_decisions_final
) == 3

assert all(
    record[
        "decision"
    ]
    == "MERGE_CONFIRMED"
    for record
    in merge_decisions_final
)

assert len(
    deferred_cleanup_final
) == 2

assert all(
    record[
        "apply_in_nb06"
    ]
    is False
    for record
    in deferred_cleanup_final
)


report_speaker_state = (
    final_report_disk[
        "speaker_state"
    ]
)

report_timestamp_state = (
    final_report_disk[
        "timestamp_state"
    ]
)

report_quality_state = (
    final_report_disk[
        "quality_state"
    ]
)

report_human_review = (
    final_report_disk[
        "human_review"
    ]
)


assert (
    report_speaker_state[
        "human_attribution_overlay_count"
    ]
    == final_human_overlay_count
    == 2
)

assert (
    report_speaker_state[
        "unknown_speaker_inference_allowed"
    ]
    is False
)

assert (
    report_timestamp_state[
        "source_timestamp_annotation_count"
    ]
    == final_timestamp_annotation_count
    == 16
)

assert (
    report_timestamp_state[
        "whole_segment_timestamp_assignment_count"
    ]
    == 0
)

assert (
    report_quality_state[
        "candidate_count"
    ]
    == len(
        quality_candidates_final
    )
    == 133
)

assert (
    report_human_review[
        "status"
    ]
    == "PASS"
)

assert (
    report_human_review[
        "blocking_issue_count"
    ]
    == 0
)

assert (
    report_human_review[
        "human_confirmed_merge_count"
    ]
    == 3
)

assert (
    report_human_review[
        "deferred_text_cleanup_count"
    ]
    == 2
)


report_processing_state = (
    final_report_disk[
        "processing_state"
    ]
)


assert (
    report_processing_state[
        "transcript_ingestion"
    ]
    == "COMPLETED"
)

assert (
    report_processing_state[
        "speaker_parsing"
    ]
    == "COMPLETED"
)

assert (
    report_processing_state[
        "lineage_validation"
    ]
    == "COMPLETED"
)

assert (
    report_processing_state[
        "source_conservation"
    ]
    == "COMPLETED"
)

assert (
    report_processing_state[
        "timestamp_preservation"
    ]
    == "COMPLETED"
)

assert (
    report_processing_state[
        "human_review"
    ]
    == "PASS"
)

assert (
    report_processing_state[
        "persian_normalization"
    ]
    == "NOT_PERFORMED"
)

assert (
    report_processing_state[
        "text_cleanup"
    ]
    == "DEFERRED"
)

assert (
    report_processing_state[
        "deduplication"
    ]
    == "NOT_PERFORMED"
)

assert (
    report_processing_state[
        "unified_schema"
    ]
    == "NOT_PERFORMED"
)

assert (
    report_processing_state[
        "chunking"
    ]
    == "NOT_PERFORMED"
)


stored_validation_checks = (
    final_report_disk[
        "validation_checks"
    ]
)

false_stored_checks = [
    key
    for key, value
    in stored_validation_checks.items()
    if value is not True
]


assert not (
    false_stored_checks
), (
    "Stored validation checks contain FAIL: "
    f"{false_stored_checks}"
)


independent_validation_contract = (
    final_report_disk[
        "independent_artifact_validation"
    ]
)

assert (
    independent_validation_contract[
        "status"
    ]
    == "PASS"
)

assert (
    independent_validation_contract[
        "source_file_count_validated"
    ]
    == 183
)

assert (
    independent_validation_contract[
        "segment_count_validated"
    ]
    == 16226
)

assert (
    independent_validation_contract[
        "source_hash_mismatch_count"
    ]
    == 0
)

assert (
    independent_validation_contract[
        "manifest_lineage_mismatch_count"
    ]
    == 0
)

assert (
    independent_validation_contract[
        "source_text_mismatch_count"
    ]
    == 0
)

assert (
    independent_validation_contract[
        "rights_mismatch_count"
    ]
    == 0
)

assert (
    independent_validation_contract[
        "timestamp_semantic_failure_count"
    ]
    == 0
)

assert (
    independent_validation_contract[
        "unknown_quality_utterance_reference_count"
    ]
    == 0
)


assert (
    final_report_disk[
        "acceptance_decision"
    ]
    == "PASS"
)


print()
print("FINAL ACCEPTANCE REPORT VALIDATION SUMMARY")
print("=" * 110)

print(
    "Stored validation checks:",
    sum(
        stored_validation_checks.values()
    ),
    "/",
    len(
        stored_validation_checks
    ),
)
print(
    "Accepted artifacts with matching SHA-256:",
    len(
        artifact_contract_actual_hashes
    ),
    "/",
    len(
        artifact_contract_paths
    ),
)
print(
    "Physical Source files revalidated:",
    len(
        accepted_sources_final
    ),
)
print(
    "Final segments revalidated:",
    final_segment_count,
)
print(
    "Programs revalidated:",
    len(
        final_program_counts
    ),
)
print(
    "Acceptance decision:",
    final_report_disk[
        "acceptance_decision"
    ],
)

print()
print(
    "NB06 FINAL ACCEPTANCE REPORT INDEPENDENT VALIDATION: PASS"
)
print(
    "FINAL NOTEBOOK HEALTH CHECK: PENDING"
)

FINAL ACCEPTANCE REPORT CONTRACT
Report: reports/phase2/transcript_ingestion_acceptance_v1.json
Report SHA-256: b9d4a5a29f52e9cc3031f21925d56f36f52d04d51e976a66843c491dc15a16ad
Report type: phase2_transcript_ingestion_acceptance
Report version: v1
Schema version: transcript_ingestion_acceptance_v1
Checkpoint role: FINAL_NOTEBOOK_CHECKPOINT
Acceptance decision: PASS

ACCEPTED ARTIFACT HASH VALIDATION

Artifact: source_manifest
Path: data/phase2/manifests/transcript_manifest_v1.json
Expected SHA-256: b328f512c67971b5e0e1f82cd5e9fe2ea148a89db3fbde31aebb474e78c6a9eb
Actual SHA-256: b328f512c67971b5e0e1f82cd5e9fe2ea148a89db3fbde31aebb474e78c6a9eb

Artifact: transcript_segments
Path: data/phase2/interim/transcripts_extracted_v1.jsonl
Expected SHA-256: 1e1d2ec379e9e8b5108efd1a5417e2a7d7394f78395fbf7164eae93ae0887848
Actual SHA-256: 1e1d2ec379e9e8b5108efd1a5417e2a7d7394f78395fbf7164eae93ae0887848

Artifact: speaker_registry
Path: data/phase2/interim/speaker_aliases_v1.json
Expected SHA-256: 9c

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Final Notebook Health Check — ممیزی نهایی Notebook 06</h3>

<p>
تمام Artifactهای Transcript Ingestion و Final Acceptance Report به‌صورت مستقل از روی Disk اعتبارسنجی شده‌اند.
</p>

<p>
این Cell آخرین Gate پیش از اجرای:
</p>

<p>
<code>Restart Kernel → Run All</code>
</p>

<p>
است.
</p>

<h4>بخش اول — Artifact Health</h4>

<ul>
<li>Final Acceptance Report وجود داشته باشد و وضعیت آن <code>PASS</code> باشد.</li>
<li>چهار Artifact ثبت‌شده در Report وجود داشته باشند.</li>
<li>SHA-256 واقعی هر Artifact با مقدار ثبت‌شده در Report برابر باشد.</li>
<li>۱۳ Program، ۱۸۳ Source و ۱۶۲۲۶ Segment حفظ شده باشند.</li>
<li>Human Review برابر PASS باشد.</li>
<li>۱۳۳ Quality Candidate، ۱۶ Timestamp Annotation، ۲ Human Attribution و ۲ Deferred Cleanup Candidate حفظ شده باشند.</li>
</ul>

<h4>بخش دوم — Notebook Structure</h4>

<p>
خود فایل Notebook مستقیماً از Disk خوانده می‌شود و موارد زیر بررسی می‌شوند:
</p>

<ul>
<li>تمام Code Cellها دارای شماره مشخص باشند.</li>
<li>شماره Code Cell تکراری وجود نداشته باشد.</li>
<li>شماره‌ها از ۰۱ تا آخرین Cell پیوسته باشند.</li>
<li>ترتیب شماره‌ها با ترتیب واقعی Cellها یکسان باشد.</li>
<li>تمام Code Cellها از نظر Python Syntax معتبر باشند.</li>
<li>هیچ Stored Error در Outputهای Notebook وجود نداشته باشد.</li>
<li>تمام Markdown Cellها دارای Wrapper راست‌به‌چپ و Closing Tag معتبر باشند.</li>
</ul>

<h4>نکته مهم</h4>

<p>
این Cell هیچ Repair خودکاری انجام نمی‌دهد.
</p>

<p>
اگر مشکل ساختاری مانند شماره تکراری، Cell قدیمی باقی‌مانده یا Syntax Error پیدا شود، فقط محل دقیق آن گزارش می‌شود تا قبل از Run All کمترین Repair لازم انجام شود.
</p>

<h4>بخش سوم — Scope Guard</h4>

<ul>
<li>Persian Normalization هنوز انجام نشده باشد.</li>
<li>Text Cleanup هنوز Deferred باشد.</li>
<li>Deduplication انجام نشده باشد.</li>
<li>Unified Schema انجام نشده باشد.</li>
<li>Chunking انجام نشده باشد.</li>
<li>Unknown Speaker Inference همچنان ممنوع باشد.</li>
</ul>

<h4>معیار PASS نهایی</h4>

<p>
فقط اگر Artifact Health، Notebook Structure، Syntax، Stored Error Check و Scope Guard همگی سالم باشند، خروجی نهایی برابر خواهد بود با:
</p>

<p>
<code>NOTEBOOK 06 FINAL HEALTH CHECK: PASS</code>
</p>

<p>
پس از آن Notebook باید با <code>Restart Kernel → Run All</code> از ابتدا اجرا شود و همین Final Health Check دوباره PASS شود.
</p>

</div>

In [49]:
# Code Cell 49

import ast
import hashlib
import json
import re
from collections import Counter


notebook_06_path = (
    project_root
    / "notebooks"
    / "phase2"
    / "06_transcript_ingestion.ipynb"
)

final_acceptance_path = (
    project_root
    / "reports"
    / "phase2"
    / "transcript_ingestion_acceptance_v1.json"
)


print("NOTEBOOK 06 FINAL HEALTH CHECK")
print("=" * 110)


health_failures = []
health_warnings = []


# ------------------------------------------------------------------
# Final acceptance artifact health
# ------------------------------------------------------------------

if not final_acceptance_path.is_file():
    health_failures.append({
        "category": "ARTIFACT",
        "reason": "FINAL_ACCEPTANCE_REPORT_MISSING",
        "detail": (
            "reports/phase2/"
            "transcript_ingestion_acceptance_v1.json"
        ),
    })

    final_acceptance_disk = None

else:
    final_acceptance_disk = json.loads(
        final_acceptance_path.read_text(
            encoding="utf-8"
        )
    )


if final_acceptance_disk is not None:
    if (
        final_acceptance_disk.get(
            "acceptance_decision"
        )
        != "PASS"
    ):
        health_failures.append({
            "category": "ARTIFACT",
            "reason": (
                "FINAL_ACCEPTANCE_DECISION_NOT_PASS"
            ),
            "detail": (
                final_acceptance_disk.get(
                    "acceptance_decision"
                )
            ),
        })

    if (
        final_acceptance_disk.get(
            "independent_artifact_validation",
            {},
        ).get(
            "status"
        )
        != "PASS"
    ):
        health_failures.append({
            "category": "ARTIFACT",
            "reason": (
                "INDEPENDENT_VALIDATION_NOT_PASS"
            ),
            "detail": (
                final_acceptance_disk.get(
                    "independent_artifact_validation",
                    {},
                ).get(
                    "status"
                )
            ),
        })


artifact_hash_failures = []

if final_acceptance_disk is not None:
    artifact_contracts = {
        "source_manifest": {
            "relative_path": (
                final_acceptance_disk[
                    "source_manifest"
                ][
                    "relative_path"
                ]
            ),
            "sha256": (
                final_acceptance_disk[
                    "source_manifest"
                ][
                    "sha256"
                ]
            ),
        },
        "transcript_segments": (
            final_acceptance_disk[
                "accepted_artifacts"
            ][
                "transcript_segments"
            ]
        ),
        "speaker_registry": (
            final_acceptance_disk[
                "accepted_artifacts"
            ][
                "speaker_registry"
            ]
        ),
        "quality_report": (
            final_acceptance_disk[
                "accepted_artifacts"
            ][
                "quality_report"
            ]
        ),
    }

    for artifact_name, contract in (
        artifact_contracts.items()
    ):
        relative_path = (
            contract[
                "relative_path"
            ]
        )

        artifact_path = (
            project_root
            / relative_path
        )

        if not artifact_path.is_file():
            artifact_hash_failures.append({
                "artifact": (
                    artifact_name
                ),
                "reason": (
                    "MISSING_FILE"
                ),
                "relative_path": (
                    relative_path
                ),
            })
            continue

        actual_sha256 = (
            hashlib.sha256(
                artifact_path.read_bytes()
            ).hexdigest()
        )

        if (
            actual_sha256
            != contract[
                "sha256"
            ]
        ):
            artifact_hash_failures.append({
                "artifact": (
                    artifact_name
                ),
                "reason": (
                    "SHA256_MISMATCH"
                ),
                "relative_path": (
                    relative_path
                ),
                "expected": (
                    contract[
                        "sha256"
                    ]
                ),
                "actual": (
                    actual_sha256
                ),
            })


if artifact_hash_failures:
    health_failures.extend(
        {
            "category": "ARTIFACT",
            **item,
        }
        for item
        in artifact_hash_failures
    )


print()
print("ARTIFACT HEALTH")
print("=" * 110)

print(
    "Final Acceptance Report:",
    (
        "PASS"
        if (
            final_acceptance_disk
            is not None
            and final_acceptance_disk.get(
                "acceptance_decision"
            )
            == "PASS"
        )
        else "FAIL"
    ),
)

print(
    "Accepted Artifact hash failures:",
    len(
        artifact_hash_failures
    ),
)

if final_acceptance_disk is not None:
    print(
        "Programs:",
        final_acceptance_disk[
            "accepted_scope"
        ][
            "program_count"
        ],
    )
    print(
        "Physical files:",
        final_acceptance_disk[
            "accepted_scope"
        ][
            "physical_file_count"
        ],
    )
    print(
        "Segments:",
        final_acceptance_disk[
            "accepted_scope"
        ][
            "segment_count"
        ],
    )
    print(
        "Human review:",
        final_acceptance_disk[
            "human_review"
        ][
            "status"
        ],
    )
    print(
        "Quality candidates:",
        final_acceptance_disk[
            "quality_state"
        ][
            "candidate_count"
        ],
    )
    print(
        "Timestamp annotations:",
        final_acceptance_disk[
            "timestamp_state"
        ][
            "source_timestamp_annotation_count"
        ],
    )


# ------------------------------------------------------------------
# Notebook structure
# ------------------------------------------------------------------

if not notebook_06_path.is_file():
    health_failures.append({
        "category": "NOTEBOOK_STRUCTURE",
        "reason": (
            "NOTEBOOK_FILE_MISSING"
        ),
        "detail": (
            notebook_06_path.relative_to(
                project_root
            ).as_posix()
        ),
    })

    notebook_06_disk = None

else:
    notebook_06_disk = json.loads(
        notebook_06_path.read_text(
            encoding="utf-8"
        )
    )


code_number_pattern = re.compile(
    r"^\s*#\s*Code Cell\s+(\d+)\s*$",
    re.IGNORECASE,
)


code_cell_records = []
markdown_cell_records = []
syntax_failures = []
stored_error_records = []


if notebook_06_disk is not None:
    notebook_cells = (
        notebook_06_disk.get(
            "cells",
            []
        )
    )

    for cell_index, cell in enumerate(
        notebook_cells
    ):
        cell_type = (
            cell.get(
                "cell_type"
            )
        )

        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        if (
            cell_type
            == "markdown"
        ):
            markdown_cell_records.append({
                "cell_index": (
                    cell_index
                ),
                "source": (
                    source
                ),
            })

            continue

        if (
            cell_type
            != "code"
        ):
            continue

        source_lines = (
            source.splitlines()
        )

        first_nonempty_line = next(
            (
                line
                for line
                in source_lines
                if line.strip()
            ),
            "",
        )

        number_match = (
            code_number_pattern.fullmatch(
                first_nonempty_line
            )
        )

        code_number = (
            int(
                number_match.group(1)
            )
            if number_match
            else None
        )

        code_cell_records.append({
            "cell_index": (
                cell_index
            ),
            "code_number": (
                code_number
            ),
            "first_line": (
                first_nonempty_line
            ),
            "source": (
                source
            ),
        })

        try:
            ast.parse(
                source
            )

        except SyntaxError as exc:
            syntax_failures.append({
                "cell_index": (
                    cell_index
                ),
                "code_number": (
                    code_number
                ),
                "line": (
                    exc.lineno
                ),
                "message": (
                    exc.msg
                ),
            })

        for output_index, output in enumerate(
            cell.get(
                "outputs",
                []
            )
        ):
            if (
                output.get(
                    "output_type"
                )
                == "error"
            ):
                stored_error_records.append({
                    "cell_index": (
                        cell_index
                    ),
                    "code_number": (
                        code_number
                    ),
                    "output_index": (
                        output_index
                    ),
                    "ename": (
                        output.get(
                            "ename"
                        )
                    ),
                    "evalue": (
                        output.get(
                            "evalue"
                        )
                    ),
                })


code_numbers = [
    record[
        "code_number"
    ]
    for record
    in code_cell_records
]


unnumbered_code_cells = [
    record
    for record
    in code_cell_records
    if record[
        "code_number"
    ]
    is None
]


numbered_code_numbers = [
    number
    for number
    in code_numbers
    if number is not None
]


code_number_counts = Counter(
    numbered_code_numbers
)


duplicate_code_numbers = {
    number: count
    for number, count
    in sorted(
        code_number_counts.items()
    )
    if count > 1
}


expected_final_code_number = 49

expected_code_numbers = list(
    range(
        1,
        expected_final_code_number + 1,
    )
)


missing_code_numbers = [
    number
    for number
    in expected_code_numbers
    if number
    not in code_number_counts
]


unexpected_code_numbers = [
    number
    for number
    in sorted(
        code_number_counts
    )
    if number
    not in expected_code_numbers
]


code_order_matches = (
    numbered_code_numbers
    == expected_code_numbers
)


if unnumbered_code_cells:
    health_failures.append({
        "category": "NOTEBOOK_STRUCTURE",
        "reason": (
            "UNNUMBERED_CODE_CELLS"
        ),
        "detail": [
            record[
                "cell_index"
            ]
            for record
            in unnumbered_code_cells
        ],
    })


if duplicate_code_numbers:
    health_failures.append({
        "category": "NOTEBOOK_STRUCTURE",
        "reason": (
            "DUPLICATE_CODE_NUMBERS"
        ),
        "detail": (
            duplicate_code_numbers
        ),
    })


if missing_code_numbers:
    health_failures.append({
        "category": "NOTEBOOK_STRUCTURE",
        "reason": (
            "MISSING_CODE_NUMBERS"
        ),
        "detail": (
            missing_code_numbers
        ),
    })


if unexpected_code_numbers:
    health_failures.append({
        "category": "NOTEBOOK_STRUCTURE",
        "reason": (
            "UNEXPECTED_CODE_NUMBERS"
        ),
        "detail": (
            unexpected_code_numbers
        ),
    })


if not code_order_matches:
    health_failures.append({
        "category": "NOTEBOOK_STRUCTURE",
        "reason": (
            "CODE_NUMBER_ORDER_NOT_CONTIGUOUS"
        ),
        "detail": (
            numbered_code_numbers
        ),
    })


if syntax_failures:
    health_failures.append({
        "category": "SYNTAX",
        "reason": (
            "PYTHON_SYNTAX_FAILURES"
        ),
        "detail": (
            syntax_failures
        ),
    })


if stored_error_records:
    health_failures.append({
        "category": "EXECUTION",
        "reason": (
            "STORED_ERROR_OUTPUTS"
        ),
        "detail": (
            stored_error_records
        ),
    })


print()
print("NOTEBOOK STRUCTURE")
print("=" * 110)

if notebook_06_disk is not None:
    print(
        "Total cells:",
        len(
            notebook_06_disk[
                "cells"
            ]
        ),
    )

print(
    "Markdown cells:",
    len(
        markdown_cell_records
    ),
)
print(
    "Code cells:",
    len(
        code_cell_records
    ),
)
print(
    "Numbered Code cells:",
    len(
        numbered_code_numbers
    ),
)
print(
    "Unnumbered Code cells:",
    len(
        unnumbered_code_cells
    ),
)
print(
    "Duplicate Code numbers:",
    duplicate_code_numbers,
)
print(
    "Missing Code numbers:",
    missing_code_numbers,
)
print(
    "Unexpected Code numbers:",
    unexpected_code_numbers,
)
print(
    "Code order contiguous 01–49:",
    code_order_matches,
)
print(
    "Python syntax failures:",
    len(
        syntax_failures
    ),
)
print(
    "Stored Error outputs:",
    len(
        stored_error_records
    ),
)


if duplicate_code_numbers:
    print()
    print("DUPLICATE CODE CELL LOCATIONS")
    print("-" * 110)

    for duplicated_number in (
        duplicate_code_numbers
    ):
        for record in (
            code_cell_records
        ):
            if (
                record[
                    "code_number"
                ]
                == duplicated_number
            ):
                print(
                    f"- Code Cell "
                    f"{duplicated_number:02d}"
                    f" | notebook cell index "
                    f"{record['cell_index']}"
                    f" | first line: "
                    f"{record['first_line']}"
                )


if syntax_failures:
    print()
    print("SYNTAX FAILURE LOCATIONS")
    print("-" * 110)

    for failure in (
        syntax_failures
    ):
        print(
            "-",
            failure,
        )


if stored_error_records:
    print()
    print("STORED ERROR LOCATIONS")
    print("-" * 110)

    for failure in (
        stored_error_records
    ):
        print(
            "-",
            failure,
        )


# ------------------------------------------------------------------
# Markdown structural wrapper audit
# ------------------------------------------------------------------

markdown_wrapper_failures = []

expected_wrapper_prefix = (
    '<div style="direction: rtl; '
    'white-space: normal; '
    'line-height:'
)

for record in (
    markdown_cell_records
):
    stripped_source = (
        record[
            "source"
        ].strip()
    )

    if not (
        stripped_source.startswith(
            expected_wrapper_prefix
        )
        and stripped_source.endswith(
            "</div>"
        )
    ):
        markdown_wrapper_failures.append({
            "cell_index": (
                record[
                    "cell_index"
                ]
            ),
            "first_line": (
                stripped_source.splitlines()[0]
                if stripped_source
                else ""
            ),
        })


if markdown_wrapper_failures:
    health_failures.append({
        "category": "MARKDOWN_STRUCTURE",
        "reason": (
            "INVALID_RTL_WRAPPER"
        ),
        "detail": (
            markdown_wrapper_failures
        ),
    })


exact_17_wrapper_count = sum(
    1
    for record
    in markdown_cell_records
    if record[
        "source"
    ].strip().startswith(
        '<div style="direction: rtl; '
        'white-space: normal; '
        'line-height: 1.7;">'
    )
)


print()
print("MARKDOWN STRUCTURE")
print("=" * 110)

print(
    "Markdown cells checked:",
    len(
        markdown_cell_records
    ),
)
print(
    "Valid RTL wrappers:",
    (
        len(
            markdown_cell_records
        )
        - len(
            markdown_wrapper_failures
        )
    ),
)
print(
    "Invalid RTL wrappers:",
    len(
        markdown_wrapper_failures
    ),
)
print(
    "Exact line-height 1.7 wrappers:",
    exact_17_wrapper_count,
)


# ------------------------------------------------------------------
# Scope guard
# ------------------------------------------------------------------

scope_failures = []

if final_acceptance_disk is not None:
    processing_state = (
        final_acceptance_disk[
            "processing_state"
        ]
    )

    expected_processing_state = {
        "transcript_ingestion": (
            "COMPLETED"
        ),
        "speaker_parsing": (
            "COMPLETED"
        ),
        "lineage_validation": (
            "COMPLETED"
        ),
        "source_conservation": (
            "COMPLETED"
        ),
        "timestamp_preservation": (
            "COMPLETED"
        ),
        "human_review": (
            "PASS"
        ),
        "persian_normalization": (
            "NOT_PERFORMED"
        ),
        "text_cleanup": (
            "DEFERRED"
        ),
        "deduplication": (
            "NOT_PERFORMED"
        ),
        "unified_schema": (
            "NOT_PERFORMED"
        ),
        "chunking": (
            "NOT_PERFORMED"
        ),
    }

    for key, expected_value in (
        expected_processing_state.items()
    ):
        actual_value = (
            processing_state.get(
                key
            )
        )

        if (
            actual_value
            != expected_value
        ):
            scope_failures.append({
                "field": (
                    key
                ),
                "expected": (
                    expected_value
                ),
                "actual": (
                    actual_value
                ),
            })

    if (
        final_acceptance_disk[
            "speaker_state"
        ][
            "unknown_speaker_inference_allowed"
        ]
        is not False
    ):
        scope_failures.append({
            "field": (
                "unknown_speaker_inference_allowed"
            ),
            "expected": False,
            "actual": (
                final_acceptance_disk[
                    "speaker_state"
                ][
                    "unknown_speaker_inference_allowed"
                ]
            ),
        })


if scope_failures:
    health_failures.append({
        "category": "SCOPE",
        "reason": (
            "PROCESSING_SCOPE_VIOLATION"
        ),
        "detail": (
            scope_failures
        ),
    })


print()
print("SCOPE GUARD")
print("=" * 110)

print(
    "Scope failures:",
    len(
        scope_failures
    ),
)

if final_acceptance_disk is not None:
    print(
        "Persian normalization:",
        final_acceptance_disk[
            "processing_state"
        ][
            "persian_normalization"
        ],
    )
    print(
        "Text cleanup:",
        final_acceptance_disk[
            "processing_state"
        ][
            "text_cleanup"
        ],
    )
    print(
        "Deduplication:",
        final_acceptance_disk[
            "processing_state"
        ][
            "deduplication"
        ],
    )
    print(
        "Unified schema:",
        final_acceptance_disk[
            "processing_state"
        ][
            "unified_schema"
        ],
    )
    print(
        "Chunking:",
        final_acceptance_disk[
            "processing_state"
        ][
            "chunking"
        ],
    )


# ------------------------------------------------------------------
# Final decision
# ------------------------------------------------------------------

print()
print("FINAL HEALTH SUMMARY")
print("=" * 110)

print(
    "Blocking failures:",
    len(
        health_failures
    ),
)
print(
    "Warnings:",
    len(
        health_warnings
    ),
)


if health_failures:
    print()
    print("BLOCKING FAILURE DETAILS")
    print("-" * 110)

    for failure_index, failure in enumerate(
        health_failures,
        start=1,
    ):
        print()
        print(
            "Failure:",
            failure_index,
        )
        print(
            "Category:",
            failure[
                "category"
            ],
        )
        print(
            "Reason:",
            failure[
                "reason"
            ],
        )
        print(
            "Detail:",
            failure.get(
                "detail"
            ),
        )


final_health_pass = (
    not health_failures
)


print()
print(
    "NOTEBOOK 06 FINAL HEALTH CHECK:",
    (
        "PASS"
        if final_health_pass
        else "FAIL"
    ),
)

if final_health_pass:
    print(
        "NEXT REQUIRED ACTION: "
        "Restart Kernel → Run All"
    )
else:
    print(
        "NEXT REQUIRED ACTION: "
        "Repair only the reported blocking "
        "Notebook issue(s), then rerun this Cell."
    )

NOTEBOOK 06 FINAL HEALTH CHECK

ARTIFACT HEALTH
Final Acceptance Report: PASS
Accepted Artifact hash failures: 0
Programs: 13
Physical files: 183
Segments: 16226
Human review: PASS
Quality candidates: 133
Timestamp annotations: 16

NOTEBOOK STRUCTURE
Total cells: 99
Markdown cells: 50
Code cells: 49
Numbered Code cells: 49
Unnumbered Code cells: 0
Duplicate Code numbers: {}
Missing Code numbers: []
Unexpected Code numbers: []
Code order contiguous 01–49: True
Python syntax failures: 0
Stored Error outputs: 0

MARKDOWN STRUCTURE
Markdown cells checked: 50
Valid RTL wrappers: 50
Invalid RTL wrappers: 0
Exact line-height 1.7 wrappers: 46

SCOPE GUARD
Scope failures: 0
Persian normalization: NOT_PERFORMED
Text cleanup: DEFERRED
Deduplication: NOT_PERFORMED
Unified schema: NOT_PERFORMED
Chunking: NOT_PERFORMED

FINAL HEALTH SUMMARY
Blocking failures: 0
Warnings: 0

NOTEBOOK 06 FINAL HEALTH CHECK: PASS
NEXT REQUIRED ACTION: Restart Kernel → Run All
